# Install dependencies

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.7 MB/s eta 0:00:00


# **datasets preparation**





In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    print(" Google Drive is mounted successfully.")
except Exception as e:
    print(f" Error mounting Google Drive: {e}")

Mounted at /content/drive
✅ Google Drive is mounted successfully.


**Combine all independent datasets**

In [ ]:
import pandas as pd
import os
from glob import glob
from google.colab import drive
import logging

#
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    # Exit or handle gracefully if drive mounting fails

# --- 2. CONFIGURATION ---
# Define the paths. You should customize these directories based on where your files are stored.
# ASSUME your source files (.csv,.xlsx) are in a folder named 'Input_Datasets' inside your Drive.
BASE_DIR = '/content/drive/MyDrive/'
INPUT_FOLDER = os.path.join(BASE_DIR, 'ML_Project_Data/Input_Datasets/')
OUTPUT_FILE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/consolidated_110k_data.csv')

# Configure logging for better feedback
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


# --- 3. CORE FUNCTION: DATA INGESTION AND CONSOLIDATION ---
def combine_datasets(input_dir: str) -> pd.DataFrame:
    """
    Finds all.csv and.xlsx files in the specified directory, reads them,
    and concatenates them into a single DataFrame.

    Handles files with ambiguous extensions (e.g.,.csv.xlsx) by checking only the last extension.
    """
    logging.info(f"Scanning directory for files: {input_dir}")

    # Use glob to find all files ending in.csv or.xlsx (case insensitive)
    search_pattern_csv = os.path.join(input_dir, '**', '*.csv')
    search_pattern_xlsx = os.path.join(input_dir, '**', '*.xlsx')

    all_files = glob(search_pattern_csv, recursive=True) + glob(search_pattern_xlsx, recursive=True)

    if not all_files:
        logging.warning("No.csv or.xlsx files found in the specified input directory.")
        return pd.DataFrame()

    df_list = []

    for filepath in all_files:
        # Check only the final extension to handle files like 'data.csv.xlsx'
        _, ext = os.path.splitext(filepath)
        ext = ext.lower()
        filename = os.path.basename(filepath)

        try:
            if ext == '.csv':
                df = pd.read_csv(filepath, encoding='utf-8')
            elif ext == '.xlsx':
                # Use openpyxl engine for modern Excel files
                df = pd.read_excel(filepath, engine='openpyxl')
            else:
                # Should be caught by glob, but included for safety
                logging.warning(f"Skipping file with unrecognized extension: {filename}")
                continue

            # Add source information for traceability (important for multi-event analysis)
            df['source_file'] = filename
            df_list.append(df)
            logging.info(f"Successfully loaded {filename} ({len(df)} rows) as {ext.upper()}.")

        except Exception as e:
            logging.error(f"Failed to read file {filename}: {e}")

    if not df_list:
        logging.error("No DataFrames were successfully loaded for concatenation.")
        return pd.DataFrame()

    # Concatenate all DataFrames into a single, large DataFrame
    logging.info(f"Concatenating {len(df_list)} datasets...")
    final_df = pd.concat(df_list, ignore_index=True)
    logging.info(f"Consolidation complete. Total rows: {len(final_df)}.")

    return final_df


# --- 4. EXECUTION AND EXPORT ---

if __name__ == '__main__':
    # Ensure the output directory exists before saving
    output_dir = os.path.dirname(OUTPUT_FILE_PATH)
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load and combine the data
    consolidated_data = combine_datasets(INPUT_FOLDER)

    if not consolidated_data.empty:
        logging.info(f"Starting export to {OUTPUT_FILE_PATH}...")

        # 2. Save the combined DataFrame as a CSV file in the specified Google Drive location
        try:
            # Use index=False to prevent writing the DataFrame index as a column
            consolidated_data.to_csv(OUTPUT_FILE_PATH, index=False, encoding='utf-8')
            logging.info(f"Successfully saved consolidated dataset to Google Drive.")

        except Exception as e:
            logging.error(f"FATAL ERROR: Failed to write output file to Drive: {e}")
            logging.warning("Please verify the output path and drive permissions.")
    else:
        print("Final DataFrame is empty. No file was saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


**Remove all old posts (currently posts before 16th August)**

In [ ]:
import pandas as pd
import os
import logging
from datetime import datetime, timezone

# --- 1. CONFIGURATION ---
BASE_DIR = '/content/drive/MyDrive/'
INPUT_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/consolidated_110k_data.csv')
OUTPUT_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/consolidated_new_records.csv')
CUTOFF_DATE_STR = '2025-08-16'

# Configure logging for clear feedback
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')

def filter_records_by_date(input_path: str, output_path: str, cutoff_date_str: str):
    """
    Loads a dataset and keeps only the records on or after a specified cutoff date.

    Args:
        input_path: The path to the source CSV file.
        output_path: The path to save the new, filtered CSV file.
        cutoff_date_str: The earliest date to keep, in 'YYYY-MM-DD' format.
    """
    try:
        logging.info(f"Loading dataset from: {input_path}")
        df = pd.read_csv(input_path, low_memory=False)
        original_count = len(df)
        logging.info(f"Successfully loaded {original_count:,} records.")

        # Ensure the 'timestamp' column is in a proper datetime format for comparison.
        # errors='coerce' will turn any unreadable dates into NaT (Not a Time).
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)

        # Remove rows where the timestamp could not be parsed
        df.dropna(subset=['timestamp'], inplace=True)
        if len(df) < original_count:
            logging.warning(f"Removed {original_count - len(df)} rows with invalid timestamp formats.")

        # Define the cutoff date. We make it timezone-aware to match the data.
        cutoff_date = pd.to_datetime(cutoff_date_str, utc=True)

        logging.info(f"Filtering dataset to keep records on or after {cutoff_date_str}...")

        # Perform the filtering operation
        filtered_df = df[df['timestamp'] >= cutoff_date].copy()

        new_count = len(filtered_df)
        removed_count = len(df) - new_count
        logging.info(f"Filtering complete. Kept {new_count:,} records and removed {removed_count:,}.")

        if new_count > 0:
            # Save the filtered DataFrame to the new CSV file
            filtered_df.to_csv(output_path, index=False, encoding='utf-8')
            logging.info(f"Successfully saved the {new_count:,} recent records to: {output_path}")
        else:
            logging.warning("No records found on or after the cutoff date. No output file was created.")

    except FileNotFoundError:
        logging.error(f"FATAL ERROR: The input file was not found at {input_path}")
    except KeyError:
        logging.error("FATAL ERROR: The input CSV does not contain a 'timestamp' column.")
    except Exception as e:
        logging.error(f"An unexpected error occurred: {e}", exc_info=True)

if __name__ == '__main__':
    filter_records_by_date(INPUT_PATH, OUTPUT_PATH, CUTOFF_DATE_STR)


**Check**

In [ ]:
import pandas as pd
import os
import logging

# --- 1. CONFIGURATION ---
BASE_DIR = '/content/drive/MyDrive/'
# The input is the final, fully-featured dataset we created in the last step.
INPUT_FILE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/consolidated_new_records.csv')
# The output file for the filtered, older posts.
OUTPUT_FILE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/old_posts.csv')

# Configure logging for clear feedback
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')

def filter_posts_by_date(input_path: str, output_path: str, cutoff_date_str: str):
    """
    Loads a dataset, filters out records with timestamps before a specified cutoff date,
    prints a complete summary of ticker/source file counts (including zeros),
    and saves the filtered result to a new CSV file.

    Args:
        input_path: Path to the input CSV file.
        output_path: Path to save the filtered CSV file.
        cutoff_date_str: The cutoff date in 'YYYY-MM-DD' format.
    """
    try:
        logging.info(f"Loading dataset from: {input_path}")
        df = pd.read_csv(input_path, low_memory=False)
        logging.info(f"Successfully loaded {len(df)} records.")

        # Ensure the timestamp column is in a datetime format for comparison.
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
        df.dropna(subset=['timestamp'], inplace=True)

        # --- NEW: Get all unique source_file/ticker combinations from the original dataframe ---
        # This will serve as our master list for the final report.
        if 'source_file' in df.columns and 'ticker' in df.columns:
            all_combinations = df[['source_file', 'ticker']].drop_duplicates()
        else:
            logging.warning("Missing 'ticker' or 'source_file' column. Cannot create detailed summary.")
            all_combinations = pd.DataFrame() # Create empty frame to avoid errors

        # Define the cutoff date and make it timezone-aware
        cutoff_date = pd.to_datetime(cutoff_date_str, utc=True)

        logging.info(f"Filtering for posts before {cutoff_date_str}...")

        # Perform the filtering
        old_posts_df = df[df['timestamp'] < cutoff_date].copy()

        total_old_posts = len(old_posts_df)
        logging.info(f"Found {total_old_posts} posts before the cutoff date.")

        # --- MODIFIED: Generate and display a complete summary ---
        if not all_combinations.empty:
            logging.info("Calculating record counts for all source files...")

            # Get counts only from the filtered (old) posts
            source_ticker_counts = old_posts_df.groupby(['source_file', 'ticker']).size().reset_index(name='count')

            # Merge with the master list of all combinations to include those with zero counts
            summary_df = pd.merge(all_combinations, source_ticker_counts, on=['source_file', 'ticker'], how='left')

            # Fill NaN counts with 0 (for files that had no old posts) and convert to integer
            summary_df['count'] = summary_df['count'].fillna(0).astype(int)

            # Set a multi-index for clear, grouped printing
            summary_df = summary_df.set_index(['source_file', 'ticker'])

            print("\n" + "="*60)
            print("--- Ticker Counts by Source File (All Sources Included) ---")
            print(summary_df.to_string())
            print("-" * 60)
            print(f"Total Old Posts Found: {total_old_posts}")
            print("="*60 + "\n")

        # Save the filtered DataFrame to the new CSV file only if it contains data
        if not old_posts_df.empty:
            old_posts_df.to_csv(output_path, index=False, encoding='utf-8')
            logging.info(f"Successfully saved the {total_old_posts} filtered posts to: {output_path}")
        else:
            logging.warning("No posts found before the cutoff date. No output file was saved.")

    except FileNotFoundError:
        logging.error(f"FATAL ERROR: The input file was not found at {input_path}")
    except Exception as e:
        logging.error(f"An unexpected error occurred: {e}", exc_info=True)

if __name__ == '__main__':
    # Set the cutoff date for filtering
    CUTOFF_DATE = '2025-08-27'
    filter_posts_by_date(INPUT_FILE_PATH, OUTPUT_FILE_PATH, CUTOFF_DATE)




--- Ticker Counts by Source File (All Sources Included) ---
                                                         count
source_file                                    ticker         
trump-usd-gpro_event_data_processed.csv        GPRO       1857
rblx_event_data_processed.csv                  RBLX        890
ukgdp.csv                                      EWU         127
eq_event_data_processed.csv                    EQX           0
poet_event_data_processed.csv                  POET          0
wif_event_data_processed.csv                   WIF-USD       0
iwto_event_data_processed.csv                  NaN           0
ovid_event_data_processed.csv                  OVID          0
gpro_event_data_processed.csv                  GPRO          0
myx_event_data_processed.csv                   MYX           0
lilpepe_event_data_processed.csv               PEPE-USD      0
snap_event_data_processed.csv                  SNAP          0
doge_event_data_processed.csv                  DOGE-USD  

# **Data Cleaning and Merging**

**Phase 1 : Remove old posts (Aug-27)**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

BASE_DIR = '/content/drive/MyDrive/ML_Project_Data/'

# ===== LOAD DATASETS =====
consolidated = pd.read_csv(f'{BASE_DIR}Combined/consolidated_final_features.csv')
train_seed = pd.read_csv(f'{BASE_DIR}Combined/Labeling_Sets/train_seed.csv')

print(f" Loaded consolidated: {len(consolidated)} records")
print(f" Loaded train_seed: {len(train_seed)} records")

# ===== PARSE TIMESTAMPS =====
consolidated['timestamp_utc'] = pd.to_datetime(consolidated['timestamp'], utc=True, errors='coerce')
CUTOFF_DATE = pd.to_datetime('2025-08-27', utc=True)

# ===== FILTER: KEEP ONLY >= AUGUST 27 =====
keep_mask = consolidated['timestamp_utc'] >= CUTOFF_DATE
old_records = consolidated[~keep_mask].copy()
filtered_data = consolidated[keep_mask].copy()

print(f"\n Records to remove: {len(old_records)}")
print(f" Records to keep: {len(filtered_data)}")

# ===== REMOVE OLD post_ids FROM train_seed =====
if 'post_id' in train_seed.columns:
    old_post_ids = set(old_records['post_id'].dropna().astype(int))
    train_seed_cleaned = train_seed[~train_seed['post_id'].isin(old_post_ids)].copy()
    removed_count = len(train_seed) - len(train_seed_cleaned)

    print(f"\n Removed {removed_count} old records from train_seed")
    print(f" train_seed after cleaning: {len(train_seed_cleaned)} records")
    train_seed = train_seed_cleaned

# ===== ARCHIVE OLD RECORDS =====
old_records.to_csv(f'{BASE_DIR}Combined/archived_pre_aug27.csv', index=False)
print(f"\n Archived {len(old_records)} old records")

# ===== SAVE CLEANED train_seed =====
train_seed.to_csv(f'{BASE_DIR}Combined/Labeling_Sets/train_seed_cleaned.csv', index=False)
print(f" Saved cleaned train_seed")

# ===== SAVE FILTERED CONSOLIDATED (NEW!) =====
filtered_data.to_csv(f'{BASE_DIR}Combined/consolidated_filtered_temp.csv', index=False)
print(f" Saved filtered consolidated (post_id NOT reassigned yet)")
print(f"\n post_id reassignment will happen in Phase 3 after merging with new data")


📊 Loaded consolidated: 104297 records
📊 Loaded train_seed: 5610 records

🗑️ Records to remove: 7962
✅ Records to keep: 96335

🧹 Removed 548 old records from train_seed
📊 train_seed after cleaning: 5062 records

📦 Archived 7962 old records
✅ Saved cleaned train_seed
✅ Saved filtered consolidated (post_id NOT reassigned yet)

⚠️ post_id reassignment will happen in Phase 3 after merging with new data


**Phase 1.5 remove recrods with headline_similarit < 0.65 in new datasets**

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

BASE_DIR = '/content/drive/MyDrive/ML_Project_Data/'
NEW_DATA_DIR = f'{BASE_DIR}Input_Datasets_new/'

print("=" * 60)
print("PHASE 1.5: FILTER BY headline_similarity_score ≥ 0.65")
print("=" * 60)

# ===== LOAD ALL FILES FROM Input_Datasets_new =====
new_files = glob.glob(f'{NEW_DATA_DIR}**/*.csv', recursive=True) + \
            glob.glob(f'{NEW_DATA_DIR}**/*.xlsx', recursive=True)

if not new_files:
    print(" No files found in Input_Datasets_new/")
else:
    print(f"\n Found {len(new_files)} files to filter")

    total_before = 0
    total_after = 0
    total_removed = 0

    for file_path in new_files:
        filename = os.path.basename(file_path)
        print(f"\n{'='*50}")
        print(f" Processing: {filename}")

        # ===== LOAD FILE =====
        try:
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path, low_memory=False)
            else:
                df = pd.read_excel(file_path)

            before_count = len(df)
            total_before += before_count
            print(f"   Loaded: {before_count} records")
        except Exception as e:
            print(f"   Error loading file: {e}")
            continue

        # ===== CHECK FOR headline_similarity_score COLUMN =====
        if 'headline_similarity_score' not in df.columns:
            print(f"   No 'headline_similarity_score' column found - skipping filter")
            total_after += before_count
            continue

        # ===== FILTER: KEEP ONLY headline_similarity_score >= 0.65 =====
        df_filtered = df[df['headline_similarity_score'] >= 0.65].copy()

        after_count = len(df_filtered)
        removed_count = before_count - after_count
        total_after += after_count
        total_removed += removed_count

        print(f"   headline_similarity_score distribution:")
        print(f"     Min: {df['headline_similarity_score'].min():.3f}")
        print(f"     Max: {df['headline_similarity_score'].max():.3f}")
        print(f"     Mean: {df['headline_similarity_score'].mean():.3f}")
        print(f"     Median: {df['headline_similarity_score'].median():.3f}")

        print(f"   Filter results:")
        print(f"     Before: {before_count} records")
        print(f"     After:  {after_count} records")
        print(f"     Removed: {removed_count} records ({removed_count/before_count*100:.1f}%)")

        # ===== SAVE FILTERED FILE (OVERWRITE ORIGINAL) =====
        try:
            if file_path.endswith('.csv'):
                df_filtered.to_csv(file_path, index=False)
            else:
                df_filtered.to_excel(file_path, index=False)

            print(f"   Saved filtered data back to {filename}")
        except Exception as e:
            print(f"   Error saving file: {e}")

    # ===== SUMMARY =====
    print(f"\n{'='*60}")
    print(" PHASE 1.5 SUMMARY")
    print(f"{'='*60}")
    print(f"Total files processed: {len(new_files)}")
    print(f"Total records before:  {total_before}")
    print(f"Total records after:   {total_after}")
    print(f"Total records removed: {total_removed}")

    if total_before > 0:
        print(f"Removal rate: {total_removed/total_before*100:.1f}%")

    print(f"\n Phase 1.5 complete! Proceed to Phase 2.")


PHASE 1.5: FILTER BY headline_similarity_score ≥ 0.65

📂 Found 8 files to filter

📄 Processing: tsla_event_data_processed.csv
  📊 Loaded: 410 records
  🔍 headline_similarity_score distribution:
     Min: 0.498
     Max: 1.000
     Mean: 0.655
     Median: 0.656
  ✂️ Filter results:
     Before: 410 records
     After:  362 records
     Removed: 48 records (11.7%)
  ✅ Saved filtered data back to tsla_event_data_processed.csv

📄 Processing: meta_event_data_processed.csv
  📊 Loaded: 72 records
  🔍 headline_similarity_score distribution:
     Min: 0.483
     Max: 0.711
     Mean: 0.589
     Median: 0.588
  ✂️ Filter results:
     Before: 72 records
     After:  11 records
     Removed: 61 records (84.7%)
  ✅ Saved filtered data back to meta_event_data_processed.csv

📄 Processing: aapl_event_data_processed.csv
  📊 Loaded: 1569 records
  🔍 headline_similarity_score distribution:
     Min: 0.429
     Max: 1.000
     Mean: 0.652
     Median: 0.610
  ✂️ Filter results:
     Before: 1569 records

**Phase 2: Merge new and old datasets**

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

BASE_DIR = '/content/drive/MyDrive/ML_Project_Data/'
NEW_DATA_DIR = f'{BASE_DIR}Input_Datasets_new/'

print("=" * 60)
print("PHASE 2: LOAD NEW DATASETS")
print("=" * 60)

# ===== LOAD NEW DATASETS =====
print("\n Loading new datasets from Input_Datasets_new/")
new_files = glob.glob(f'{NEW_DATA_DIR}**/*.csv', recursive=True) + \
            glob.glob(f'{NEW_DATA_DIR}**/*.xlsx', recursive=True)

print(f"Found {len(new_files)} files:")
for f in new_files:
    print(f"  - {os.path.basename(f)}")

if not new_files:
    print("\n No CSV or XLSX files found in Input_Datasets_new/")
    print("Proceeding with existing data only (no new data to merge)")
    new_data_combined = pd.DataFrame()
else:
    new_data_frames = []
    CUTOFF_DATE = pd.to_datetime('2025-08-27', utc=True)

    for file_path in new_files:
        filename = os.path.basename(file_path)
        print(f"\n{'='*50}")
        print(f" Processing: {filename}")

        # ===== LOAD FILE =====
        try:
            if file_path.endswith('.csv'):
                df_new = pd.read_csv(file_path, low_memory=False)
            else:
                df_new = pd.read_excel(file_path)

            print(f"   Loaded {len(df_new)} rows")
        except Exception as e:
            print(f"   Error loading file: {e}")
            continue

        # ===== ADD SOURCE TRACKING =====
        df_new['source_file'] = filename

        # ===== STANDARDIZE TIMESTAMPS =====
        if 'timestamp' in df_new.columns:
            df_new['timestamp_utc'] = pd.to_datetime(df_new['timestamp'], utc=True, errors='coerce')

            # Filter to keep only >= Aug 27
            before_filter = len(df_new)
            df_new = df_new[df_new['timestamp_utc'] >= CUTOFF_DATE].copy()
            after_filter = len(df_new)

            print(f"   Date filter: {before_filter} → {after_filter} records (kept ≥ Aug 27)")
        else:
            print(f"   No 'timestamp' column found - keeping all records")

        # ===== CHECK FOR REQUIRED COLUMNS =====
        # Show available columns for verification
        print(f"  📋 Columns: {list(df_new.columns[:5])}... ({len(df_new.columns)} total)")

        # Append to list
        new_data_frames.append(df_new)
        print(f"  📊 {len(df_new)} records added to queue")

    # ===== CONCATENATE ALL NEW DATA =====
    if new_data_frames:
        new_data_combined = pd.concat(new_data_frames, ignore_index=True)
        print(f"\n{'='*60}")
        print(f" Successfully combined {len(new_data_frames)} new datasets")
        print(f" Total new records: {len(new_data_combined)}")

        # Show source distribution
        print(f"\n Records by source file:")
        print(new_data_combined['source_file'].value_counts())

        # Show date range
        if 'timestamp_utc' in new_data_combined.columns:
            print(f"\n Date range:")
            print(f"   Earliest: {new_data_combined['timestamp_utc'].min()}")
            print(f"   Latest:   {new_data_combined['timestamp_utc'].max()}")
    else:
        print("\n No valid data loaded from new files")
        new_data_combined = pd.DataFrame()

# ===== SAVE NEW DATA (TEMPORARY) =====
if not new_data_combined.empty:
    new_data_combined.to_csv(f'{BASE_DIR}Combined/new_data_temp.csv', index=False)
    print(f"\n Saved new data to Combined/new_data_temp.csv")
    print(f"\n Phase 2 Summary:")
    print(f"   Old data (from Phase 1): 96,335 records")
    print(f"   New data (from Phase 2): {len(new_data_combined)} records")
    print(f"   Total to merge in Phase 3: {96335 + len(new_data_combined)} records")
else:
    print(f"\n Phase 2 Summary:")
    print(f"   Old data (from Phase 1): 96,335 records")
    print(f"   New data (from Phase 2): 0 records")
    print(f"    No new data to merge - will proceed with existing data only")

print(f"\n{'='*60}")
print(" PHASE 2 COMPLETE")
print("{'='*60}")
print("\n Ready for Phase 3: Merge + Reassign post_id")


PHASE 2: LOAD NEW DATASETS

📂 Loading new datasets from Input_Datasets_new/
Found 8 files:
  - tsla_event_data_processed.csv
  - meta_event_data_processed.csv
  - aapl_event_data_processed.csv
  - spy_cpi_event_data_processed.csv
  - spy_jobmarket_event_data_processed.csv
  - spy_fed_event_data_processed.csv
  - spy_rareMetal_event_data_processed.csv
  - spy_event_data_processed.csv

📄 Processing: tsla_event_data_processed.csv
  ✅ Loaded 362 rows
  🗓️ Date filter: 362 → 362 records (kept ≥ Aug 27)
  📋 Columns: ['timestamp', 'sentiment_score', 'emotional_valence', 'news_keyword_count', 'recency_keyword_flag']... (33 total)
  📊 362 records added to queue

📄 Processing: meta_event_data_processed.csv
  ✅ Loaded 11 rows
  🗓️ Date filter: 11 → 11 records (kept ≥ Aug 27)
  📋 Columns: ['timestamp', 'sentiment_score', 'emotional_valence', 'news_keyword_count', 'recency_keyword_flag']... (33 total)
  📊 11 records added to queue

📄 Processing: aapl_event_data_processed.csv
  ✅ Loaded 659 rows
  🗓

**Phase 3: Assign Post id to new dataset and map post id between old dataset and train set**

In [ ]:
import pandas as pd
import numpy as np

BASE_DIR = '/content/drive/MyDrive/ML_Project_Data/'

print("=" * 60)
print("PHASE 3: MERGE DATA & REASSIGN post_id")
print("=" * 60)

# ===== LOAD OLD DATA (FROM PHASE 1) =====
filtered_data = pd.read_csv(f'{BASE_DIR}Combined/consolidated_filtered_temp.csv')
print(f"\n Loaded old data: {len(filtered_data)} records")
print(f"   Has post_id: {'post_id' in filtered_data.columns}")
if 'post_id' in filtered_data.columns:
    print(f"   post_id range: {filtered_data['post_id'].min()} to {filtered_data['post_id'].max()}")

# ===== LOAD NEW DATA (FROM PHASE 2) =====
try:
    new_data_combined = pd.read_csv(f'{BASE_DIR}Combined/new_data_temp.csv')
    print(f"\n Loaded new data: {len(new_data_combined)} records")
    print(f"   Has post_id: {'post_id' in new_data_combined.columns}")
except FileNotFoundError:
    print(f"\n No new_data_temp.csv found - proceeding with old data only")
    new_data_combined = pd.DataFrame()

# ===== CHECK COLUMN COMPATIBILITY =====
if not new_data_combined.empty:
    old_cols = set(filtered_data.columns)
    new_cols = set(new_data_combined.columns)

    missing_in_new = old_cols - new_cols
    extra_in_new = new_cols - old_cols

    if missing_in_new:
        print(f"\n Columns in old data but NOT in new data ({len(missing_in_new)}):")
        print(f"   {list(missing_in_new)[:5]}...")  # Show first 5
        print(f"   → These will be filled with NaN in new data")

    if extra_in_new:
        print(f"\n Columns in new data but NOT in old data ({len(extra_in_new)}):")
        print(f"   {list(extra_in_new)[:5]}...")  # Show first 5
        print(f"   → These will be filled with NaN in old data")

# ===== MERGE OLD + NEW DATA =====
if not new_data_combined.empty:
    merged_raw = pd.concat([filtered_data, new_data_combined], ignore_index=True, sort=False)
    print(f"\n Merged old + new data: {len(merged_raw)} total records")
else:
    merged_raw = filtered_data.copy()
    print(f"\n Using old data only: {len(merged_raw)} total records")

# ===== STORE OLD post_id FOR MAPPING =====
if 'post_id' in merged_raw.columns:
    merged_raw['old_post_id'] = merged_raw['post_id'].copy()
    print(f"\n Stored old post_id for mapping")
    print(f"   Records with old post_id: {merged_raw['old_post_id'].notna().sum()}")
    print(f"   Records without old post_id (new data): {merged_raw['old_post_id'].isna().sum()}")
else:
    merged_raw['old_post_id'] = np.nan
    print(f"\n No post_id column found - all records will get new post_id")

# ===== SORT BY TIMESTAMP (CHRONOLOGICAL ORDER) =====
if 'timestamp_utc' in merged_raw.columns:
    merged_raw.sort_values('timestamp_utc', inplace=True)
    merged_raw.reset_index(drop=True, inplace=True)
    print(f"\n Sorted by timestamp_utc")
    print(f"   Earliest: {merged_raw['timestamp_utc'].min()}")
    print(f"   Latest:   {merged_raw['timestamp_utc'].max()}")
else:
    print(f"\n No timestamp_utc column - skipping sort")

# ===== REASSIGN post_id SEQUENTIALLY =====
merged_raw['post_id'] = range(1, len(merged_raw) + 1)
print(f"\n Reassigned post_id: 1 to {len(merged_raw)}")

# ===== CREATE MAPPING TABLE (old_post_id → new_post_id) =====
id_mapping = merged_raw[['old_post_id', 'post_id']].copy()
id_mapping.columns = ['old_post_id', 'new_post_id']

# Remove rows where old_post_id is NaN (new data doesn't need mapping)
id_mapping = id_mapping.dropna(subset=['old_post_id'])
id_mapping['old_post_id'] = id_mapping['old_post_id'].astype(int)
id_mapping['new_post_id'] = id_mapping['new_post_id'].astype(int)

print(f"\n Created post_id mapping:")
print(f"   Mapped records: {len(id_mapping)}")
print(f"   Old post_id range: {id_mapping['old_post_id'].min()} to {id_mapping['old_post_id'].max()}")
print(f"   New post_id range: {id_mapping['new_post_id'].min()} to {id_mapping['new_post_id'].max()}")

# ===== SAVE MAPPING =====
id_mapping.to_csv(f'{BASE_DIR}Combined/post_id_mapping.csv', index=False)
print(f"\n Saved mapping to Combined/post_id_mapping.csv")

# ===== DROP TEMPORARY old_post_id COLUMN =====
merged_raw.drop('old_post_id', axis=1, inplace=True)

# ===== SAVE MERGED RAW DATA =====
merged_raw.to_csv(f'{BASE_DIR}Combined/merged_raw_posts.csv', index=False)
print(f"\n Saved merged data to Combined/merged_raw_posts.csv")

# ===== VERIFICATION =====
print(f"\n{'='*60}")
print(" PHASE 3 SUMMARY")
print(f"{'='*60}")
print(f"Total records: {len(merged_raw)}")
print(f"post_id range: {merged_raw['post_id'].min()} to {merged_raw['post_id'].max()}")
print(f"Unique post_ids: {merged_raw['post_id'].nunique()}")
print(f"Duplicate post_ids: {merged_raw['post_id'].duplicated().sum()} (should be 0)")

if 'timestamp_utc' in merged_raw.columns:
    print(f"\n Date range:")
    print(f"   Earliest: {merged_raw['timestamp_utc'].min()}")
    print(f"   Latest:   {merged_raw['timestamp_utc'].max()}")

if 'source_file' in merged_raw.columns:
    print(f"\n Records by source:")
    print(merged_raw['source_file'].value_counts().head(10))

print(f"\n{'='*60}")
print(" PHASE 3 COMPLETE")
print(f"{'='*60}")
print("\n Next: Phase 4 (Recalculate market data + features)")
print("   Input: merged_raw_posts.csv")
print("   Output: consolidated_final_features_new.csv")


PHASE 3: MERGE DATA & REASSIGN post_id

📊 Loaded old data: 96335 records
   Has post_id: True
   post_id range: 128 to 104297

📊 Loaded new data: 1754 records
   Has post_id: False

⚠️ Columns in old data but NOT in new data (408):
   ['emo_emb_352', 'emo_emb_362', 'emo_emb_63', 'emo_emb_132', 'emo_emb_303']...
   → These will be filled with NaN in new data

✅ Merged old + new data: 98089 total records

📋 Stored old post_id for mapping
   Records with old post_id: 96335
   Records without old post_id (new data): 1754

🗓️ Sorted by timestamp_utc
   Earliest: 2025-08-27 00:31:30+00:00
   Latest:   2025-10-27 15:31:29+00:00

🔄 Reassigned post_id: 1 to 98089

📊 Created post_id mapping:
   Mapped records: 96335
   Old post_id range: 128 to 104297
   New post_id range: 1 to 97199

✅ Saved mapping to Combined/post_id_mapping.csv

✅ Saved merged data to Combined/merged_raw_posts.csv

📊 PHASE 3 SUMMARY
Total records: 98089
post_id range: 1 to 98089
Unique post_ids: 98089
Duplicate post_ids: 0 (

# **Feature engineering - adding some new features (ignore the emo_emd-0 to 383)**

# Repurposed Phase 4 and Phase 5

**Assigning ticker**

In [ ]:
import pandas as pd
import os
import logging
from google.colab import drive

# --- 1. CONFIGURE LOGGING FIRST (BEFORE ANY LOGGING CALLS) ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s: %(message)s',
    force=True  # Force reconfiguration in Colab
)

# --- 2. SETUP: GOOGLE DRIVE MOUNTING ---
try:
    drive.mount('/content/drive', force_remount=True)
    logging.info(" Google Drive mounted successfully.")
except Exception as e:
    logging.error(f" Error mounting Google Drive: {e}")

# --- 3. CONFIGURATION ---
BASE_DIR = '/content/drive/MyDrive/'
INPUT_FILE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv')
OUTPUT_FILE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv')

# --- 4. THE STRATEGIC MAPPING DICTIONARY ---
FILENAME_TO_TICKER_MAP = {
    # --- NEW EVENT DATASETS ---
    'meta_event_data_processed': 'META',
    'spy_fed_event_data_processed': 'SPY',
    'spy_event_data_processed': 'SPY',
    'spy_raremetal_event_data_processed': 'SPY',
    'spy_jobmarket_event_data_processed': 'SPY',
    'spy_cpi_event_data_processed': 'SPY',
    'aapl_event_data_processed': 'AAPL',
    'tsla_event_data_processed': 'TSLA',

    # --- CORPORATE STOCKS & CRYPTO ---
    'rblx': 'RBLX',
    'gpro': 'GPRO',
    'trump-usd-gpro': 'GPRO',
    'snap': 'SNAP',
    'myx': 'MYX',
    'poet': 'POET',
    'ovid': 'OVID',
    'wif': 'WIF-USD',
    'doge': 'DOGE-USD',
    'lilpepe': 'PEPE-USD',
    'inda': 'INDA',
    'inda_event_data_processed_modigsthaul': 'INDA',
    'saudiaramco': '2222.SR',
    'eq': 'EQX',

    # --- MACROECONOMIC / COUNTRY ETFS ---
    'australiareservebank': 'EWA',
    'australiaretail': 'EWA',
    'brazilgdp': 'EWZ',
    'brazilcentralbank': 'EWZ',
    'canadabank': 'EWC',
    'china_national_policy': 'MCHI',
    'european_centralbank': 'FEZ',
    'eurozone': 'FEZ',
    'euunemployment': 'FEZ',
    'frenchinflation': 'EWQ',
    'francepresedential': 'EWQ',
    'germancpi': 'EWG',
    'germanifo': 'EWG',
    'germanyzeweconomics': 'EWG',
    'indiarbi': 'INDA',
    'indiainflation': 'INDA',
    'indiaforeign': 'INDA',
    'indiamonsoon': 'INDA',
    'japancentralbank': 'EWJ',
    'japansurvey': 'EWJ',
    'korea': 'EWY',
    'russiasanctions': 'ERUS',
    'bankofengland': 'EWU',
    'ukboe': 'EWU',
    'uk_consumer_price_index': 'EWU',
    'uk_producer_price_inflation': 'EWU',
    'federalreserveinterest': 'SPY',
    'us_sandp': 'SPY',
    'usdurablegoods': 'SPY',
    'usdebt': 'SPY',
    'ussentiment': 'SPY',
    'usnonfarmpayroll': 'SPY',
    'usfederalreserve': 'SPY',
    'usism': 'SPY',
    'ustech_quarterly_earning': 'QQQ',
    'opec': 'USO',
    'tariff_us_india': 'SPY',
    'imfworldbankannualmeetings': 'SPY',
    'japancorecpi': 'EWJ',
    'sibosglobalfintech': 'QQQ',
    'chinaq3': 'FXI',
    'usq3': 'XLF',
    'ukgdp': 'EWU',

    # --- NO DIRECT TRADABLE ASSET ---
    'iwto': None,
    'nigeriapresedential': None,
}

# --- 5. CORE FUNCTION: MAPPING LOGIC ---
def map_filename_to_ticker(filename):
    """
    Looks for keywords in a filename and returns the corresponding ticker
    from the mapping dictionary. Case-insensitive.
    """
    if not isinstance(filename, str):
        return None

    lower_filename = filename.lower()

    for keyword, ticker in FILENAME_TO_TICKER_MAP.items():
        if keyword in lower_filename:
            return ticker

    return None  # Return None if no match is found

# --- 6. EXECUTION ---
print("=" * 60)
print("TICKER MAPPING SCRIPT - STARTING")
print("=" * 60)

try:
    logging.info(f"Loading merged dataset from: {INPUT_FILE_PATH}")
    df = pd.read_csv(INPUT_FILE_PATH, low_memory=False)
    logging.info(f" Successfully loaded {len(df)} rows.")

    if 'source_file' not in df.columns:
        logging.error(" FATAL: 'source_file' column not found in the dataset.")
        print("\n ERROR: 'source_file' column missing!")
    else:
        # --- CHECK EXISTING TICKERS ---
        if 'ticker' in df.columns:
            existing_tickers = df['ticker'].notna().sum()
            missing_tickers = df['ticker'].isna().sum()
            logging.info(f" Existing ticker status:")
            logging.info(f"   Records with ticker: {existing_tickers}")
            logging.info(f"   Records without ticker: {missing_tickers}")
            print(f"\n Existing tickers: {existing_tickers}")
            print(f" Missing tickers: {missing_tickers}")
        else:
            logging.info(" No 'ticker' column found - creating new column")
            print("\n Creating new 'ticker' column")
            df['ticker'] = None
            missing_tickers = len(df)

        # --- APPLY MAPPING ONLY TO MISSING TICKERS ---
        if missing_tickers > 0:
            logging.info(f"\n Applying mapping to {missing_tickers} records without tickers...")
            print(f"\n Mapping {missing_tickers} records...")

            # Create mask for rows without ticker
            missing_mask = df['ticker'].isna()

            # Apply mapping only to missing rows
            df.loc[missing_mask, 'ticker'] = df.loc[missing_mask, 'source_file'].apply(map_filename_to_ticker)

            # Report results
            newly_mapped = df.loc[missing_mask, 'ticker'].notna().sum()
            still_missing = df.loc[missing_mask, 'ticker'].isna().sum()

            logging.info(f"\n Mapping results:")
            logging.info(f"   Newly mapped: {newly_mapped}")
            logging.info(f"   Still missing: {still_missing}")
            print(f"\n Newly mapped: {newly_mapped}")
            print(f" Still missing: {still_missing}")
        else:
            logging.info("\n All records already have tickers - no mapping needed")
            print("\n All records already have tickers")

        # --- FINAL REPORT ---
        total_with_ticker = df['ticker'].notna().sum()
        total_without_ticker = df['ticker'].isna().sum()
        mapping_pct = (total_with_ticker / len(df)) * 100

        logging.info(f"\n Final ticker status:")
        logging.info(f"   Total records: {len(df)}")
        logging.info(f"   Records with ticker: {total_with_ticker} ({mapping_pct:.2f}%)")
        logging.info(f"   Records without ticker: {total_without_ticker}")

        print(f"\n" + "=" * 60)
        print(f" FINAL REPORT")
        print(f"=" * 60)
        print(f"Total records: {len(df)}")
        print(f"With ticker: {total_with_ticker} ({mapping_pct:.2f}%)")
        print(f"Without ticker: {total_without_ticker}")

        # Show ticker distribution
        logging.info(f"\n Ticker distribution:")
        print(f"\n Ticker distribution:")
        ticker_counts = df['ticker'].value_counts().head(10)  # Show top 10
        for ticker, count in ticker_counts.items():
            logging.info(f"   {ticker}: {count} records")
            print(f"   {ticker}: {count}")

        # Save the updated dataframe
        logging.info(f"\n Saving updated file to: {OUTPUT_FILE_PATH}")
        print(f"\n Saving to: merged_raw_posts.csv")
        df.to_csv(OUTPUT_FILE_PATH, index=False, encoding='utf-8')
        logging.info(" Script finished successfully!")
        print("\n SCRIPT FINISHED SUCCESSFULLY!")

except FileNotFoundError:
    logging.error(f" FATAL ERROR: Input file not found at {INPUT_FILE_PATH}.")
    print(f"\n ERROR: File not found at {INPUT_FILE_PATH}")
except Exception as e:
    logging.error(f" An unexpected error occurred: {e}", exc_info=True)
    print(f"\n ERROR: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("SCRIPT EXECUTION COMPLETE")
print("=" * 60)


2025-10-27 17:49:09,934 - INFO: ✅ Google Drive mounted successfully.
2025-10-27 17:49:09,939 - INFO: Loading merged dataset from: /content/drive/MyDrive/ML_Project_Data/Combined/merged_raw_posts.csv


Mounted at /content/drive
TICKER MAPPING SCRIPT - STARTING


2025-10-27 17:49:29,100 - INFO: ✅ Successfully loaded 98089 rows.
2025-10-27 17:49:29,114 - INFO: 📊 Existing ticker status:
2025-10-27 17:49:29,115 - INFO:    Records with ticker: 98089
2025-10-27 17:49:29,116 - INFO:    Records without ticker: 0
2025-10-27 17:49:29,117 - INFO: 
✅ All records already have tickers - no mapping needed
2025-10-27 17:49:29,130 - INFO: 
📊 Final ticker status:
2025-10-27 17:49:29,131 - INFO:    Total records: 98089
2025-10-27 17:49:29,132 - INFO:    Records with ticker: 98089 (100.00%)
2025-10-27 17:49:29,133 - INFO:    Records without ticker: 0
2025-10-27 17:49:29,136 - INFO: 
📊 Ticker distribution:
2025-10-27 17:49:29,145 - INFO:    GPRO: 29278 records
2025-10-27 17:49:29,146 - INFO:    RBLX: 20998 records
2025-10-27 17:49:29,148 - INFO:    SPY: 5493 records
2025-10-27 17:49:29,150 - INFO:    DOGE-USD: 4538 records
2025-10-27 17:49:29,152 - INFO:    SNAP: 4052 records
2025-10-27 17:49:29,154 - INFO:    PEPE-USD: 3851 records
2025-10-27 17:49:29,160 - INFO:


📊 Existing tickers: 98089
📊 Missing tickers: 0

✅ All records already have tickers

📊 FINAL REPORT
Total records: 98089
With ticker: 98089 (100.00%)
Without ticker: 0

📊 Ticker distribution:
   GPRO: 29278
   RBLX: 20998
   SPY: 5493
   DOGE-USD: 4538
   SNAP: 4052
   PEPE-USD: 3851
   EQX: 3813
   WIF-USD: 3090
   INDA: 2522
   EWU: 2413

💾 Saving to: merged_raw_posts.csv


2025-10-27 17:50:31,800 - INFO: ✅ Script finished successfully!



✅ SCRIPT FINISHED SUCCESSFULLY!

SCRIPT EXECUTION COMPLETE


**Phase 4.2 - Get finance data**

In [ ]:
# robust_hybrid_financial_fetch_fixed.py
# Enhanced version with expanded lookbacks and session-aware metadata

!pip install pycoingecko -q

import os
import time
import shutil
import logging
from datetime import datetime, timedelta, timezone

import pandas as pd
import numpy as np
import yfinance as yf

# Optional coin fallback
try:
    from pycoingecko import CoinGeckoAPI
    cg = CoinGeckoAPI()
    COINGECKO_AVAILABLE = True
    print(" CoinGecko API loaded successfully")
except Exception as e:
    COINGECKO_AVAILABLE = False
    print(f" CoinGecko not available: {e}")

# ---------------- CONFIG ----------------
BASE_DIR = '/content/drive/MyDrive/'

# Input/Output paths
INPUT_MAIN_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv')
OUTPUT_INTRADAY_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/financial_intraday_data_new.csv')
OUTPUT_PARQUET_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/financial_intraday_data_new.parquet')
LOG_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/financial_fetch_log_new.csv')

NOW_UTC = datetime.now(timezone.utc)
INTRADAY_LIMIT_DAYS = 58
INTRADAY_CUTOFF = NOW_UTC.date() - timedelta(days=INTRADAY_LIMIT_DAYS)

# Expanded lookbacks
LOOKBACK_1D_DAYS = 60
INTRADAY_BUFFER_DAYS = 2

TICKER_PAUSE = 0.6
YFINANCE_RETRIES = 3
YFINANCE_RETRY_DELAY = 2.0

COINGECKO_FALLBACK_MAP = {
    'MYX': 'myx-finance',
    'PEPE-USD': 'pepe',
    'DOGE-USD': 'dogecoin',
    'WIF-USD': 'dogwifhat',
}

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s: %(message)s',
    force=True
)

print("\n" + "=" * 60)
print("PHASE 4.2: MARKET DATA FETCH - CONFIGURATION")
print("=" * 60)
print(f" Current UTC: {NOW_UTC}")
print(f" Intraday cutoff: {INTRADAY_CUTOFF}")
print(f" Lookback windows:")
print(f"   - 1d historical: {LOOKBACK_1D_DAYS} days")
print(f"   - 5m buffer: {INTRADAY_BUFFER_DAYS} days")
print(f" Ticker pause: {TICKER_PAUSE}s")
print(f" yfinance retries: {YFINANCE_RETRIES}")

# ---------------- Helpers ----------------

def clear_yf_cache():
    try:
        cache_dir = os.path.expanduser('~/.yfinance')
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir)
            logging.info(" Cleared yfinance cache directory")
            print(" Cleared yfinance cache")
        else:
            logging.info(" No yfinance cache to clear")
    except Exception as e:
        logging.debug(f"Could not clear yfinance cache: {e}")
        print(f" Could not clear cache: {e}")

def yf_download_retry(ticker, start=None, end=None, interval="1d", prepost=False, retries=YFINANCE_RETRIES):
    logging.info(f" Attempting {ticker} fetch: {interval} from {start} to {end}")

    for attempt in range(1, retries + 1):
        try:
            logging.info(f"   Attempt {attempt}/{retries}...")
            df = yf.download(
                tickers=ticker,
                start=start,
                end=end,
                interval=interval,
                prepost=prepost,
                progress=False,
                threads=False
            )
            if isinstance(df, pd.DataFrame) and not df.dropna(how='all').empty:
                logging.info(f"    Success: fetched {len(df)} bars")
                return df.copy()
            else:
                logging.warning(f"    Empty result on attempt {attempt}")
        except Exception as e:
            logging.warning(f"    yf.download error on attempt {attempt}: {e}")

        if attempt < retries:
            logging.info(f"    Waiting {YFINANCE_RETRY_DELAY}s before retry...")
            time.sleep(YFINANCE_RETRY_DELAY)

    logging.error(f" Failed to fetch {ticker} {interval} after {retries} attempts")
    return None

def coingecko_fetch_price_volume(coin_id, start_dt, end_dt):
    if not COINGECKO_AVAILABLE:
        logging.warning(" CoinGecko not available")
        return None

    logging.info(f" Attempting CoinGecko fetch for {coin_id}")
    try:
        res = cg.get_coin_market_chart_range_by_id(
            coin_id, 'usd',
            int(start_dt.timestamp()),
            int(end_dt.timestamp())
        )
        prices = res.get('prices', [])
        vols = res.get('total_volumes', [])

        if not prices:
            logging.warning(f" No price data from CoinGecko for {coin_id}")
            return None

        logging.info(f"   Processing {len(prices)} CoinGecko price points")
        rows = []
        for i, (ts_ms, price) in enumerate(prices):
            ts = pd.to_datetime(int(ts_ms), unit='ms', utc=True)
            vol = vols[i][1] if i < len(vols) else np.nan
            rows.append({
                'market_timestamp': ts,
                'Open': price,
                'High': price,
                'Low': price,
                'Close': price,
                'Volume': vol
            })

        df = pd.DataFrame(rows)
        if df.empty:
            logging.warning(f" Empty DataFrame from CoinGecko for {coin_id}")
            return None

        df['data_granularity'] = 'coingecko_points'
        logging.info(f"    CoinGecko fetch successful: {len(df)} bars")
        return df

    except Exception as e:
        logging.error(f" CoinGecko error for {coin_id}: {e}")
        return None

# ---------------- ENHANCED standardize function ----------------

def standardize_yf_df(raw_df, granularity_label):
    """Robust standardizer for yfinance outputs"""
    if raw_df is None:
        logging.debug("    standardize_yf_df received None")
        return None

    logging.debug(f"    Standardizing {len(raw_df)} rows ({granularity_label})")
    df = raw_df.copy()

    if getattr(df, "empty", True):
        logging.warning("    DataFrame is empty")
        return None

    # Flatten MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        logging.debug("    Flattening MultiIndex columns")
        new_cols = []
        for col in df.columns:
            joined = "_".join([str(c) for c in col if str(c) and str(c) != ""])
            new_cols.append(joined)
        df.columns = new_cols

    df = df.reset_index()

    # Find timestamp column
    ts_candidates = ['Datetime', 'Date', 'market_timestamp', 'index']
    ts_col = None
    for c in ts_candidates:
        if c in df.columns:
            ts_col = c
            logging.debug(f"    Found timestamp column: {c}")
            break

    if ts_col is None:
        for c in df.columns:
            if np.issubdtype(df[c].dtype, np.datetime64):
                ts_col = c
                logging.debug(f"    Found datetime column: {c}")
                break

    if ts_col:
        df.rename(columns={ts_col: 'market_timestamp'}, inplace=True)
    else:
        logging.warning("    No timestamp column found, using first column")
        first = df.columns[0]
        try:
            df['market_timestamp'] = pd.to_datetime(df[first], errors='coerce', utc=True)
        except Exception:
            df['market_timestamp'] = pd.NaT

    # Ensure UTC
    try:
        df['market_timestamp'] = pd.to_datetime(df['market_timestamp'], utc=True)
    except Exception:
        try:
            df['market_timestamp'] = pd.to_datetime(df['market_timestamp'], errors='coerce')
            df['market_timestamp'] = df['market_timestamp'].dt.tz_localize(
                'America/New_York', ambiguous='infer'
            ).dt.tz_convert('UTC')
            logging.debug("    Localized to NY, converted to UTC")
        except Exception:
            df['market_timestamp'] = pd.to_datetime(df['market_timestamp'], utc=True, errors='coerce')

    # Helper to find column
    def find_col(target):
        t = target.lower()
        for c in df.columns:
            if str(c).lower() == t:
                return c
        for c in df.columns:
            name = str(c).lower()
            if name.endswith(t) or t in name:
                return c
        return None

    # Locate OHLCV
    ohlcv = {}
    for short in ['Open', 'High', 'Low', 'Close', 'Volume']:
        col_name = find_col(short)
        if col_name is not None:
            series = df[col_name]
            if isinstance(series, pd.DataFrame):
                series = series.iloc[:,0]
            ohlcv[short] = pd.to_numeric(series, errors='coerce')
        else:
            logging.debug(f"    Column {short} not found, filling with NaN")
            ohlcv[short] = pd.Series([np.nan] * len(df))

    out = pd.DataFrame({
        'market_timestamp': df['market_timestamp'],
        'Open': ohlcv['Open'],
        'High': ohlcv['High'],
        'Low': ohlcv['Low'],
        'Close': ohlcv['Close'],
        'Volume': ohlcv['Volume'],
    })
    out['data_granularity'] = granularity_label
    out = out[~out['market_timestamp'].isna()].reset_index(drop=True)

    logging.debug(f"    Standardized to {len(out)} rows")
    return out

# ---------------- Per-ticker fetch ----------------

def fetch_for_ticker(ticker, event_dates):
    logging.info(f"\n{'='*50}")
    logging.info(f" FETCHING: {ticker}")
    logging.info(f"{'='*50}")

    ticker = str(ticker).strip()
    if ticker == '' or pd.isna(ticker):
        logging.warning(" Invalid ticker (empty or NaN)")
        return None

    today = NOW_UTC.date()
    valid_dates = [d for d in event_dates if d <= today]

    if not valid_dates:
        logging.warning(f" No valid dates for {ticker}")
        return None

    min_date, max_date = min(valid_dates), max(valid_dates)
    logging.info(f" Date range: {min_date} to {max_date}")

    collected = []

    # Historical 1d fetch
    if min_date < INTRADAY_CUTOFF:
        start_1d = min_date - timedelta(days=LOOKBACK_1D_DAYS)
        end_1d = INTRADAY_CUTOFF
        logging.info(f" Fetching 1d bars: {start_1d} to {end_1d}")

        raw1 = yf_download_retry(ticker, start=start_1d, end=end_1d, interval="1d")
        df1 = standardize_yf_df(raw1, 'yfinance_1d')

        if df1 is not None:
            collected.append(df1)
            logging.info(f" 1d fetch successful: {len(df1)} bars")
            print(f"    1d: {len(df1)} bars")
        else:
            logging.warning(" 1d fetch failed")
            print(f"    1d fetch failed")

    # Recent 5m fetch
    if max_date >= INTRADAY_CUTOFF:
        start_5m = max(min_date - timedelta(days=INTRADAY_BUFFER_DAYS), INTRADAY_CUTOFF)
        logging.info(f" Fetching 5m bars: {start_5m} onwards (prepost=True)")

        raw5 = yf_download_retry(ticker, start=start_5m, end=None, interval="5m", prepost=True)
        df5 = standardize_yf_df(raw5, 'yfinance_5m')

        if df5 is not None:
            collected.append(df5)
            logging.info(f" 5m fetch successful: {len(df5)} bars")
            print(f"    5m: {len(df5)} bars")
        else:
            logging.warning(" 5m fetch failed, trying 1d fallback")
            print(f"    5m failed, trying 1d fallback")

            start_fb = start_5m - timedelta(days=1)
            end_fb = today + timedelta(days=1)
            raw_fb = yf_download_retry(ticker, start=start_fb, end=end_fb, interval="1d")
            df_fb = standardize_yf_df(raw_fb, 'yfinance_1d_fallback')

            if df_fb is not None:
                collected.append(df_fb)
                logging.info(f" 1d fallback successful: {len(df_fb)} bars")
                print(f"    1d fallback: {len(df_fb)} bars")
            else:
                logging.warning(" 1d fallback also failed")
                print(f"    1d fallback failed")

    # CoinGecko fallback
    if not collected and ticker in COINGECKO_FALLBACK_MAP and COINGECKO_AVAILABLE:
        coin_id = COINGECKO_FALLBACK_MAP[ticker]
        logging.info(f" Trying CoinGecko fallback for {ticker} → {coin_id}")
        print(f"    Trying CoinGecko: {coin_id}")

        start_dt = datetime.combine(
            min_date - timedelta(days=LOOKBACK_1D_DAYS),
            datetime.min.time()
        ).replace(tzinfo=timezone.utc)
        end_dt = datetime.combine(max_date, datetime.max.time()).replace(tzinfo=timezone.utc)

        cg_df = coingecko_fetch_price_volume(coin_id, start_dt, end_dt)

        if cg_df is not None:
            collected.append(cg_df)
            logging.info(f" CoinGecko successful: {len(cg_df)} bars")
            print(f"    CoinGecko: {len(cg_df)} bars")
        else:
            logging.warning(" CoinGecko fallback failed")
            print(f"    CoinGecko failed")

    if not collected:
        logging.error(f" No data collected for {ticker}")
        print(f"    No data for {ticker}")
        return None

    # Combine and process
    logging.info(f" Combining {len(collected)} data sources")
    combined = pd.concat(collected, ignore_index=True)

    combined['market_timestamp_utc'] = pd.to_datetime(
        combined['market_timestamp'], utc=True, errors='coerce'
    )
    combined['ticker'] = ticker

    for c in ['Open','High','Low','Close','Volume']:
        combined[c] = pd.to_numeric(combined[c], errors='coerce')

    before_dedup = len(combined)
    combined.drop_duplicates(subset=['ticker','market_timestamp_utc'], inplace=True)
    after_dedup = len(combined)

    if before_dedup > after_dedup:
        logging.info(f" Removed {before_dedup - after_dedup} duplicate bars")

    combined.sort_values(['ticker','market_timestamp_utc'], inplace=True)

    # Add session metadata
    logging.info(" Adding session-aware metadata (NY timezone)")
    combined['ts_ny'] = combined['market_timestamp_utc'].dt.tz_convert('America/New_York')
    combined['session_date_ny'] = combined['ts_ny'].dt.date
    hhmm = combined['ts_ny'].dt.strftime('%H:%M')
    combined['is_rth_ny'] = (hhmm >= '09:30') & (hhmm <= '16:00')
    combined['session_part_ny'] = np.where(
        hhmm < '09:30', 'premarket',
        np.where(hhmm <= '16:00', 'rth', 'afterhours')
    )

    final_cols = [
        'ticker','market_timestamp_utc','Open','High','Low','Close','Volume',
        'data_granularity','session_date_ny','is_rth_ny','session_part_ny'
    ]

    result = combined[final_cols]
    logging.info(f" {ticker} complete: {len(result)} bars")
    return result

# ---------------- Runner ----------------

def run_fetch_pipeline():
    print("\n" + "=" * 60)
    print("PHASE 4.2: MARKET DATA FETCH - STARTING")
    print("=" * 60)

    logging.info(" Loading input data")
    try:
        df_main = pd.read_csv(INPUT_MAIN_PATH, low_memory=False)
        logging.info(f" Loaded {len(df_main)} records from merged_raw_posts.csv")
        print(f"\n Loaded: {len(df_main)} records")
    except Exception as e:
        logging.error(f" Failed to read input file: {e}")
        print(f" Error: {e}")
        return

    if 'timestamp' not in df_main.columns or 'ticker' not in df_main.columns:
        logging.error(" Missing required columns: 'timestamp' and/or 'ticker'")
        print(" Missing required columns")
        return

    logging.info(" Processing timestamps")
    df_main['timestamp'] = pd.to_datetime(df_main['timestamp'], errors='coerce', utc=True)
    df_main.dropna(subset=['timestamp'], inplace=True)
    df_main['event_date'] = df_main['timestamp'].dt.date

    tickers = df_main['ticker'].dropna().unique().tolist()
    logging.info(f" Found {len(tickers)} unique tickers")
    logging.info(f" Date range: {df_main['event_date'].min()} to {df_main['event_date'].max()}")

    print(f"\n Fetching data for {len(tickers)} tickers")
    print(f" Date range: {df_main['event_date'].min()} to {df_main['event_date'].max()}")

    all_frames = []
    report = []

    start_time = time.time()

    for i, t in enumerate(tickers, 1):
        dates = df_main.loc[df_main['ticker'] == t, 'event_date'].unique().tolist()
        print(f"\n[{i}/{len(tickers)}] {t}")
        print(f"    {min(dates)} to {max(dates)}")

        df_t = fetch_for_ticker(t, dates)

        if df_t is not None and not df_t.empty:
            all_frames.append(df_t)
            report.append({
                'ticker': t,
                'status': 'Success',
                'rows': len(df_t),
                'latest_ts': df_t['market_timestamp_utc'].max()
            })
            print(f"    Total: {len(df_t)} bars")
        else:
            report.append({
                'ticker': t,
                'status': 'Failed',
                'rows': 0,
                'latest_ts': pd.NaT
            })
            print(f"    Failed")

        logging.info(f" Pausing {TICKER_PAUSE}s before next ticker")
        time.sleep(TICKER_PAUSE)

    elapsed_time = time.time() - start_time
    logging.info(f" Total fetch time: {elapsed_time/60:.1f} minutes")

    if not all_frames:
        logging.error(" No market data fetched")
        print("\n No data fetched")
        pd.DataFrame(report).to_csv(LOG_PATH, index=False)
        return

    logging.info(" Combining all ticker data")
    final = pd.concat(all_frames, ignore_index=True)

    before_dedup = len(final)
    final.drop_duplicates(subset=['ticker','market_timestamp_utc'], inplace=True)
    after_dedup = len(final)

    if before_dedup > after_dedup:
        logging.info(f" Removed {before_dedup - after_dedup} global duplicates")

    final.sort_values(['ticker','market_timestamp_utc'], inplace=True)

    # Save outputs
    logging.info(f" Saving {len(final)} rows to CSV")
    final.to_csv(OUTPUT_INTRADAY_PATH, index=False)

    logging.info(f" Saving {len(final)} rows to Parquet")
    final.to_parquet(OUTPUT_PARQUET_PATH, index=False)

    logging.info(f" Saving fetch report")
    pd.DataFrame(report).to_csv(LOG_PATH, index=False)

    print(f"\n" + "=" * 60)
    print(f" PHASE 4.2 COMPLETE")
    print(f"=" * 60)
    print(f" Total time: {elapsed_time/60:.1f} minutes")
    print(f" Total bars: {len(final):,}")
    print(f" Saved to:")
    print(f"   - financial_intraday_data_new.csv")
    print(f"   - financial_intraday_data_new.parquet")
    print(f"   - financial_fetch_log_new.csv")

    print("\n=== Top 20 Tickers by Bar Count ===")
    report_df = pd.DataFrame(report).sort_values('rows', ascending=False)
    print(report_df.head(20))

    print("\n=== Failed Tickers ===")
    failed = report_df[report_df['status'] == 'Failed']
    if len(failed) > 0:
        print(failed[['ticker', 'status']])
    else:
        print(" No failed tickers")

    return final, report_df

# ---------------- Main execution ----------------

if __name__ == "__main__":
    print("\n Starting Phase 4.2")
    print(f" {datetime.now()}")

    clear_yf_cache()

    # Preflight
    print("\n Preflight check...")
    logging.info(" Running preflight with SPY")

    try:
        test = yf_download_retry('SPY', start=NOW_UTC.date()-timedelta(days=7), interval="1d")

        if test is None:
            logging.error(" SPY preflight failed")
            print(" Preflight failed - yfinance may be down")
        else:
            logging.info(" Preflight successful")
            print(" Preflight successful\n")
            run_fetch_pipeline()

    except Exception as e:
        logging.exception(f" Fatal preflight error: {e}")
        print(f" Preflight error: {e}")


2025-10-27 18:01:17,418 - INFO: ℹ️ No yfinance cache to clear
2025-10-27 18:01:17,419 - INFO: 🔍 Running preflight with SPY
2025-10-27 18:01:17,420 - INFO: 🔍 Attempting SPY fetch: 1d from 2025-10-20 to None
2025-10-27 18:01:17,420 - INFO:    Attempt 1/3...


✅ CoinGecko API loaded successfully

PHASE 4.2: MARKET DATA FETCH - CONFIGURATION
📅 Current UTC: 2025-10-27 18:01:17.407479+00:00
📅 Intraday cutoff: 2025-08-30
📊 Lookback windows:
   - 1d historical: 60 days
   - 5m buffer: 2 days
⏱️ Ticker pause: 0.6s
🔄 yfinance retries: 3

🚀 Starting Phase 4.2
📅 2025-10-27 18:01:17.418234

🔍 Preflight check...


/tmp/ipython-input-2794396843.py:93: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
2025-10-27 18:01:18,109 - INFO:    ✅ Success: fetched 6 bars
2025-10-27 18:01:18,111 - INFO: ✅ Preflight successful
2025-10-27 18:01:18,112 - INFO: 📂 Loading input data


✅ Preflight successful


PHASE 4.2: MARKET DATA FETCH - STARTING


2025-10-27 18:01:38,047 - INFO: ✅ Loaded 98089 records from merged_raw_posts.csv
2025-10-27 18:01:38,048 - INFO: 🔧 Processing timestamps


...


[31/31] TSLA
   📅 2025-10-15 to 2025-10-25


/tmp/ipython-input-2794396843.py:93: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
2025-10-27 18:02:41,754 - INFO:    ✅ Success: fetched 2041 bars
2025-10-27 18:02:41,767 - INFO: ✅ 5m fetch successful: 2041 bars
2025-10-27 18:02:41,768 - INFO: 🔧 Combining 1 data sources
2025-10-27 18:02:41,785 - INFO: 🔧 Adding session-aware metadata (NY timezone)
2025-10-27 18:02:41,831 - INFO: ✅ TSLA complete: 2041 bars
2025-10-27 18:02:41,833 - INFO: ⏳ Pausing 0.6s before next ticker


   ✅ 5m: 2041 bars
   ✅ Total: 2041 bars


2025-10-27 18:02:42,434 - INFO: ⏱️ Total fetch time: 1.1 minutes
2025-10-27 18:02:42,436 - INFO: 🔧 Combining all ticker data
2025-10-27 18:02:42,596 - INFO: 💾 Saving 166987 rows to CSV
2025-10-27 18:02:45,771 - INFO: 💾 Saving 166987 rows to Parquet
2025-10-27 18:02:46,194 - INFO: 📝 Saving fetch report



✅ PHASE 4.2 COMPLETE
⏱️ Total time: 1.1 minutes
📊 Total bars: 166,987
📂 Saved to:
   - financial_intraday_data_new.csv
   - financial_intraday_data_new.parquet
   - financial_fetch_log_new.csv

=== Top 20 Tickers by Bar Count ===
      ticker   status   rows                 latest_ts
21  DOGE-USD  Success  14253 2025-10-27 17:59:00+00:00
24   WIF-USD  Success  13867 2025-10-27 17:57:00+00:00
1        QQQ  Success   7532 2025-10-27 18:00:00+00:00
11       SPY  Success   7491 2025-10-27 18:00:00+00:00
0       RBLX  Success   7469 2025-10-27 18:00:00+00:00
16       USO  Success   7170 2025-10-27 18:00:00+00:00
15       FXI  Success   7074 2025-10-27 18:00:00+00:00
12       XLF  Success   6997 2025-10-27 18:00:00+00:00
20      SNAP  Success   6628 2025-10-27 18:00:00+00:00
14       EWY  Success   6098 2025-10-27 18:00:00+00:00
19      GPRO  Success   6089 2025-10-27 18:00:00+00:00
13      INDA  Success   5819 2025-10-27 18:00:00+00:00
2        EWJ  Success   5773 2025-10-27 18:00:00+00:00

**Phase 4.3 - Join Market data with records**

In [ ]:
# ============================================================
# PHASE 4.3: MARKET DATA ALIGNMENT WITH PRE-TIMESTAMP FEATURES
# ULTRA-FIXED VERSION: Handles duplicates, sorting issues, mixed dtypes
# ============================================================

import pandas as pd
import numpy as np
from datetime import timedelta
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s: %(message)s',
    force=True
)

print("=" * 60)
print("PHASE 4.3: MARKET DATA ALIGNMENT")
print("=" * 60)

def compute_pre_timestamp_features(posts_df, intraday_df, daily_df=None,
                                    rolling_windows=[5, 10, 20],
                                    min_bars_for_rolling=3):
    """Joins posts to market data with strict time-cut (no lookahead)"""

    logging.info(" Starting market data alignment")
    print("\n Preparing data for alignment...")

    posts = posts_df.copy()
    intraday = intraday_df.copy()

    # ==================== STEP 1: CONVERT TIMESTAMPS ====================

    logging.info(" Converting timestamps to datetime")

    if 'timestamp' in posts.columns:
        posts['timestamp_utc'] = pd.to_datetime(posts['timestamp'], utc=True, errors='coerce')
    elif 'timestamp_utc' in posts.columns:
        posts['timestamp_utc'] = pd.to_datetime(posts['timestamp_utc'], utc=True, errors='coerce')
    else:
        raise ValueError("Posts dataframe must have 'timestamp' or 'timestamp_utc' column")

    intraday['market_timestamp_utc'] = pd.to_datetime(intraday['market_timestamp_utc'], utc=True, errors='coerce')

    # ==================== STEP 2: DROP INVALID RECORDS ====================

    logging.info(" Cleaning invalid records")

    posts = posts.dropna(subset=['timestamp_utc', 'ticker']).copy()
    posts['ticker'] = posts['ticker'].astype(str)

    intraday = intraday.dropna(subset=['market_timestamp_utc', 'ticker']).copy()
    intraday['ticker'] = intraday['ticker'].astype(str)

    # ==================== STEP 3: ENSURE NUMERIC OHLCV ====================

    logging.info(" Converting OHLCV to numeric")

    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if col in intraday.columns:
            intraday[col] = pd.to_numeric(intraday[col], errors='coerce')

    # ==================== STEP 4: SORT AND DEDUPLICATE ====================

    logging.info(" Sorting and deduplicating data")
    print("\n Sorting and removing duplicates...")

    posts = posts.sort_values(['ticker', 'timestamp_utc']).reset_index(drop=True)
    before_dedup = len(posts)
    posts = posts.drop_duplicates(subset=['ticker', 'timestamp_utc'], keep='last').reset_index(drop=True)
    if len(posts) < before_dedup:
        print(f" Removed {before_dedup - len(posts)} duplicate posts")

    intraday = intraday.sort_values(['ticker', 'market_timestamp_utc']).reset_index(drop=True)
    before_dedup = len(intraday)
    intraday = intraday.drop_duplicates(subset=['ticker', 'market_timestamp_utc'], keep='last').reset_index(drop=True)
    if len(intraday) < before_dedup:
        print(f" Removed {before_dedup - len(intraday)} duplicate bars")

    # ==================== STEP 5: ENSURE PROPER DTYPES ====================

    posts['timestamp_utc'] = posts['timestamp_utc'].astype('datetime64[ns, UTC]')
    intraday['market_timestamp_utc'] = intraday['market_timestamp_utc'].astype('datetime64[ns, UTC]')

    print(f"\n Posts ready: {len(posts):,}")
    print(f" Market bars ready: {len(intraday):,}")

    # ==================== STEP 6: PERFORM MERGE_ASOF ====================

    logging.info(" Performing merge_asof (backward direction)")
    print("\n Matching posts to market bars...")

    # Convert to categorical for consistent sorting
    ticker_categories = sorted(set(posts['ticker'].unique()) | set(intraday['ticker'].unique()))
    posts['ticker'] = pd.Categorical(posts['ticker'], categories=ticker_categories, ordered=True)
    intraday['ticker'] = pd.Categorical(intraday['ticker'], categories=ticker_categories, ordered=True)

    posts = posts.sort_values(['ticker', 'timestamp_utc']).reset_index(drop=True)
    intraday = intraday.sort_values(['ticker', 'market_timestamp_utc']).reset_index(drop=True)

    logging.info(" Converted tickers to categorical and re-sorted")

    try:
        posts_with_bar = pd.merge_asof(
            posts,
            intraday.rename(columns={'market_timestamp_utc': 'bar_timestamp'}),
            left_on='timestamp_utc',
            right_on='bar_timestamp',
            by='ticker',
            direction='backward',
            tolerance=pd.Timedelta(days=365),
            allow_exact_matches=True,
            suffixes=('', '_bar')
        )
        logging.info(" merge_asof successful")
        posts_with_bar['ticker'] = posts_with_bar['ticker'].astype(str)

    except Exception as e:
        logging.error(f" merge_asof failed: {e}")
        print(f"\n ERROR: {e}")
        print("\n Falling back to manual loop merge...")

        posts_with_bar = manual_loop_merge(posts, intraday)
        if posts_with_bar is None:
            raise RuntimeError("Both merge_asof and loop merge failed")

        logging.info(" Manual loop merge successful")

    # ==================== STEP 7: COMPUTE DAILY AGGREGATES ====================

    logging.info(" Computing daily aggregates")
    print("\n Computing prior-day features...")

    # Convert categorical back to string
    if isinstance(intraday['ticker'].dtype, pd.CategoricalDtype):
        intraday['ticker'] = intraday['ticker'].astype(str)
    if isinstance(posts_with_bar['ticker'].dtype, pd.CategoricalDtype):
        posts_with_bar['ticker'] = posts_with_bar['ticker'].astype(str)

    # Flag matches
    posts_with_bar['has_market_data'] = ~posts_with_bar['bar_timestamp'].isna()
    matched = posts_with_bar['has_market_data'].sum()
    print(f" Matched: {matched:,} / {len(posts_with_bar):,} ({matched/len(posts_with_bar)*100:.1f}%)")

    # Daily aggregation
    if daily_df is None:
        daily_agg = intraday.copy()
        daily_agg['date'] = daily_agg['market_timestamp_utc'].dt.date

        daily_close = daily_agg.groupby(['ticker', 'date'], as_index=False, observed=True).agg({
            'Close': 'last',
            'Volume': 'sum',
            'High': 'max',
            'Low': 'min',
            'Open': 'first'
        })
        daily_df = daily_close.copy()
        logging.info(f"   Created {len(daily_df)} daily bars")
    else:
        daily_df = daily_df.copy()
        if 'date' not in daily_df.columns:
            daily_df['date'] = pd.to_datetime(daily_df['date'], errors='coerce').dt.date

    daily_df = daily_df.sort_values(['ticker', 'date']).reset_index(drop=True)
    daily_df['daily_return'] = daily_df.groupby('ticker')['Close'].pct_change()

    # ==================== STEP 8: COMPUTE ROLLING FEATURES ====================

    logging.info(f" Computing rolling features: {rolling_windows}")

    for window in rolling_windows:
        daily_df[f'rolling_{window}d_return'] = (
            daily_df.groupby('ticker')['daily_return']
            .transform(lambda x: x.rolling(window, min_periods=min_bars_for_rolling).mean())
        )
        daily_df[f'rolling_{window}d_volatility'] = (
            daily_df.groupby('ticker')['daily_return']
            .transform(lambda x: x.rolling(window, min_periods=min_bars_for_rolling).std())
        )
        daily_df[f'rolling_{window}d_volume'] = (
            daily_df.groupby('ticker')['Volume']
            .transform(lambda x: x.rolling(window, min_periods=min_bars_for_rolling).mean())
        )

    daily_df['prior_close'] = daily_df.groupby('ticker')['Close'].shift(1)
    daily_df['prior_volume'] = daily_df.groupby('ticker')['Volume'].shift(1)

    # ==================== STEP 9: JOIN PRIOR-DAY FEATURES ====================

    logging.info(" Joining prior-day features")
    print("\n Joining prior-day features...")

    posts_with_bar['post_date'] = posts_with_bar['timestamp_utc'].dt.date
    posts_with_bar['prior_date'] = posts_with_bar['post_date'].apply(
        lambda x: x - timedelta(days=1) if pd.notna(x) else None
    )

    posts_enriched = posts_with_bar.merge(
        daily_df.add_suffix('_daily'),
        left_on=['ticker', 'prior_date'],
        right_on=['ticker_daily', 'date_daily'],
        how='left'
    )

    # ==================== STEP 10: RENAME AND DERIVE FEATURES ====================

    logging.info(" Renaming features")

    feature_cols = {
        'Close': 'last_available_close',
        'Open': 'last_available_open',
        'High': 'last_available_high',
        'Low': 'last_available_low',
        'Volume': 'last_bar_volume',
        'bar_timestamp': 'last_bar_timestamp',
        'Close_daily': 'prior_day_close',
        'Volume_daily': 'prior_day_volume',
    }

    for window in rolling_windows:
        feature_cols[f'rolling_{window}d_return'] = f'prior_{window}d_avg_return'
        feature_cols[f'rolling_{window}d_volatility'] = f'prior_{window}d_volatility'
        feature_cols[f'rolling_{window}d_volume'] = f'prior_{window}d_avg_volume'

    posts_enriched = posts_enriched.rename(columns=feature_cols)

    # Derived features
    posts_enriched['price_change_from_prior_close'] = (
        posts_enriched['last_available_close'] - posts_enriched['prior_day_close']
    ) / (posts_enriched['prior_day_close'] + 1e-8)

    posts_enriched['volume_vs_prior_avg'] = (
        posts_enriched['last_bar_volume'] / (posts_enriched['prior_day_volume'] + 1)
    )

    posts_enriched['has_live_intraday'] = False  # Simplified for now

    logging.info(" Feature computation complete")

    print(f"\n Alignment complete!")
    print(f"   Posts with market data: {posts_enriched['has_market_data'].sum():,}")

    return posts_enriched

def manual_loop_merge(posts, intraday):
    """Manual per-ticker merge fallback"""
    print("   Processing ticker by ticker...")

    all_results = []
    tickers = posts['ticker'].cat.categories if isinstance(posts['ticker'].dtype, pd.CategoricalDtype) else posts['ticker'].unique()

    for i, ticker in enumerate(tickers, 1):
        print(f"   [{i}/{len(tickers)}] {ticker}", end='\r')

        ticker_posts = posts[posts['ticker'] == ticker].copy()
        ticker_bars = intraday[intraday['ticker'] == ticker].copy()

        if ticker_bars.empty:
            ticker_posts['bar_timestamp'] = pd.NaT
            all_results.append(ticker_posts)
            continue

        bar_indices = []
        for post_time in ticker_posts['timestamp_utc']:
            valid_bars = ticker_bars[ticker_bars['market_timestamp_utc'] <= post_time]
            if not valid_bars.empty:
                bar_indices.append(valid_bars.index[-1])
            else:
                bar_indices.append(None)

        ticker_posts['_bar_idx'] = bar_indices
        ticker_posts = ticker_posts.merge(
            ticker_bars.rename(columns={'market_timestamp_utc': 'bar_timestamp'}),
            left_on='_bar_idx',
            right_index=True,
            how='left',
            suffixes=('', '_bar')
        )
        ticker_posts.drop('_bar_idx', axis=1, errors='ignore', inplace=True)
        all_results.append(ticker_posts)

    print("\n    All tickers processed")
    return pd.concat(all_results, ignore_index=True) if all_results else None

# ==================== MAIN EXECUTION ====================

if __name__ == "__main__":
    BASE_DIR = '/content/drive/MyDrive/'

    print("\n Loading input data...")

    posts = pd.read_csv(
        os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv'),
        low_memory=False
    )

    intraday = pd.read_parquet(
        os.path.join(BASE_DIR, 'ML_Project_Data/Combined/financial_intraday_data_new.parquet')
    )

    print(f" Loaded {len(posts):,} posts")
    print(f" Loaded {len(intraday):,} market bars")

    posts_with_features = compute_pre_timestamp_features(
        posts_df=posts,
        intraday_df=intraday,
        rolling_windows=[5, 10, 20, 30],
        min_bars_for_rolling=3
    )

    output_path = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_safe_market_features_new.csv')
    posts_with_features.to_csv(output_path, index=False)

    print(f"\n Saved {len(posts_with_features):,} posts with features")
    print(f" Output: posts_with_safe_market_features_new.csv")

    print("\n" + "=" * 60)
    print(" PHASE 4.3 COMPLETE")
    print("=" * 60)


PHASE 4.3: MARKET DATA ALIGNMENT

📂 Loading input data...


2025-10-27 18:28:46,306 - INFO: 🔧 Starting market data alignment
2025-10-27 18:28:46,420 - INFO: 📅 Converting timestamps to datetime


✅ Loaded 98,089 posts
✅ Loaded 166,987 market bars

🔧 Preparing data for alignment...


2025-10-27 18:28:46,554 - INFO: 🧹 Cleaning invalid records
2025-10-27 18:28:47,045 - INFO: 🔢 Converting OHLCV to numeric
2025-10-27 18:28:47,050 - INFO: 🔄 Sorting and deduplicating data



🔄 Sorting and removing duplicates...


...


📊 Computing prior-day features...
✅ Matched: 80,002 / 80,002 (100.0%)


2025-10-27 18:30:02,782 - INFO:    Created 2174 daily bars
2025-10-27 18:30:02,799 - INFO: 📊 Computing rolling features: [5, 10, 20, 30]
2025-10-27 18:30:02,980 - INFO: 🔗 Joining prior-day features



🔗 Joining prior-day features...


2025-10-27 18:30:03,890 - INFO: 🏷️ Renaming features
2025-10-27 18:30:04,346 - INFO: ✅ Feature computation complete



✅ Alignment complete!
   Posts with market data: 80,002

💾 Saved 80,002 posts with features
📂 Output: posts_with_safe_market_features_new.csv

✅ PHASE 4.3 COMPLETE


*Check for disorted Timestamps*

In [ ]:
import pandas as pd

BASE_DIR = '/content/drive/MyDrive/'
posts = pd.read_csv(f'{BASE_DIR}ML_Project_Data/Combined/merged_raw_posts.csv')

posts['timestamp'] = pd.to_datetime(posts['timestamp'], utc=True, errors='coerce')

print("=== Timestamp Issues ===")
print(f"Total posts: {len(posts)}")
print(f"Posts with NaT timestamps: {posts['timestamp'].isna().sum()}")
print(f"Posts without ticker: {posts['ticker'].isna().sum()}")

print("\n=== Sample problematic rows ===")
print(posts[posts['timestamp'].isna()][['post_id', 'timestamp', 'ticker', 'source_file']].head())


/tmp/ipython-input-745882305.py:4: DtypeWarning: Columns (37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  posts = pd.read_csv(f'{BASE_DIR}ML_Project_Data/Combined/merged_raw_posts.csv')


=== Timestamp Issues ===
Total posts: 98089
Posts with NaT timestamps: 0
Posts without ticker: 0

=== Sample problematic rows ===
Empty DataFrame
Columns: [post_id, timestamp, ticker, source_file]
Index: []


**Check data matching**

**Phase 4.4 - Calculating new features**

*Clean prev runs*


In [ ]:
import pandas as pd
import os

BASE_DIR = '/content/drive/MyDrive/'
INPUT_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_safe_market_features_new.csv')

print(" Cleaning input file...")

# Load data
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f" Loaded: {len(df):,} rows, {len(df.columns)} columns")

# Identify emotion feature columns to drop
cols_to_drop = [c for c in df.columns if c.startswith('emo_') or c in [
    'emotional_news_delta', 'A_Emotional_News_Delta',
    'H_Social_Momentum_Ratio', 'H_Jargon_Sentiment_Homogeneity',
    'A_Signal_Novelty', 'H_Retail_Volume_Float_Rotation',
    'H_Network_Influence_Clustering', 'emo_val_code',
    'inferred_float_rotation'
]]

print(f"\n Dropping {len(cols_to_drop)} emotion/behavioral feature columns")
print(f"   Sample: {cols_to_drop[:10]}")

# Drop the columns
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

print(f"\n Cleaned: {len(df_clean.columns)} columns remaining")

# Save cleaned file
df_clean.to_csv(INPUT_PATH, index=False)
print(f" Saved cleaned file: posts_with_safe_market_features_new.csv")

print("\n" + "=" * 60)
print(" Input file cleaned - ready for Phase 4.4")
print("=" * 60)


🧹 Cleaning input file...
✅ Loaded: 80,002 rows, 480 columns

🗑️ Dropping 404 emotion/behavioral feature columns
   Sample: ['emo_val_code', 'emo_anger', 'emo_anticipation', 'emo_disgust', 'emo_fear', 'emo_joy', 'emo_negative', 'emo_neutral', 'emo_positive', 'emo_sadness']

✅ Cleaned: 76 columns remaining
✅ Saved cleaned file: posts_with_safe_market_features_new.csv

✅ Input file cleaned - ready for Phase 4.4


*Calculate features*

In [ ]:
# ============================================================
# PHASE 4.4: BEHAVIORAL FEATURE ENGINEERING
# (Market features already computed in Phase 4.3)
# ============================================================

# Install required packages
!pip install -q sentence-transformers

import os
import logging
import numpy as np
import pandas as pd

# Try to import sentence-transformers
TRANSFORMERS_AVAILABLE = True
try:
    from sentence_transformers import SentenceTransformer
    import torch
except Exception:
    TRANSFORMERS_AVAILABLE = False

# ---------------- CONFIG ----------------
BASE_DIR = '/content/drive/MyDrive/'
INPUT_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_safe_market_features_new.csv')
OUTPUT_PATH_CSV = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_new.csv')
OUTPUT_PATH_PARQUET = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_new.parquet')
MODEL_CACHE_DIR = os.path.join(BASE_DIR, 'models', 'sentence_emotion_model')
SENTENCE_MODEL_NAME = "all-MiniLM-L6-v2"

# Canonical emotions list
KNOWN_EMOTIONS = [
    'anger', 'anticipation', 'disgust', 'fear', 'joy',
    'negative', 'neutral', 'positive', 'sadness', 'surprise', 'trust'
]

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s: %(message)s',
    force=True
)

print("=" * 60)
print("PHASE 4.4: BEHAVIORAL FEATURE ENGINEERING")
print("=" * 60)

# ---------------- Model Utils ----------------

def load_or_download_sentence_model(cache_dir=MODEL_CACHE_DIR, model_name=SENTENCE_MODEL_NAME):
    """Load or download SentenceTransformer model"""
    if not TRANSFORMERS_AVAILABLE:
        logging.warning("sentence-transformers not available. Embeddings will be zeros.")
        return None, None

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Try load cached
    if os.path.isdir(cache_dir):
        try:
            logging.info(f"Loading cached model from: {cache_dir}")
            model = SentenceTransformer(cache_dir, device=device)
            return model, device
        except Exception as e:
            logging.warning(f"Failed to load cached model: {e}")

    # Download and cache
    try:
        logging.info(f"Downloading model '{model_name}'...")
        model = SentenceTransformer(model_name, device=device)
        os.makedirs(cache_dir, exist_ok=True)
        model.save(cache_dir)
        logging.info(f"Model saved to: {cache_dir}")
        return model, device
    except Exception as e:
        logging.warning(f"Could not download model: {e}")
        return None, None

def build_sentence_embeddings_map(unique_labels, model=None, batch_size=16):
    """Compute sentence embeddings for emotion labels"""
    unique_labels = list(sorted(unique_labels))

    if model is None:
        logging.warning("No model available: using zero embeddings")
        emb_dim = 1
        return {lab: np.zeros(emb_dim, dtype=float) for lab in unique_labels}, emb_dim

    try:
        logging.info(f"Computing embeddings for {len(unique_labels)} emotions")
        emb_arr = model.encode(unique_labels, batch_size=batch_size, convert_to_numpy=True)
        emb_dim = emb_arr.shape[1]
        emb_map = {lab: emb_arr[i].astype(float) for i, lab in enumerate(unique_labels)}
        logging.info(f"Built embeddings map (dim={emb_dim})")
        return emb_map, emb_dim
    except Exception as e:
        logging.warning(f"Failed to compute embeddings: {e}")
        emb_dim = 1
        return {lab: np.zeros(emb_dim, dtype=float) for lab in unique_labels}, emb_dim

# ---------------- Helpers ----------------

def normalize_emotion_label_series(s: pd.Series) -> pd.Series:
    """Normalize emotion labels: lowercase, strip, blank->'neutral'"""
    s = s.fillna('').astype(str).str.strip().str.lower()
    s = s.replace('', 'neutral')
    s = s.where(s.isin(KNOWN_EMOTIONS), 'neutral')
    return s

# ---------------- Behavioral Feature Engineering ----------------

def engineer_behavioral_features(df_main, emb_map, emb_dim):
    """
    Create behavioral features:
    - Emotion embeddings
    - Emotional news delta
    - Herd behavior indicators
    - Availability heuristic features
    """
    logging.info("Starting behavioral feature engineering...")
    print("\n🔧 Engineering behavioral features...")

    df = df_main.copy()

    # Ensure timestamp
    df['timestamp'] = pd.to_datetime(df.get('timestamp_utc', df.get('timestamp')), utc=True, errors='coerce')
    df = df[~df['timestamp'].isna()].reset_index(drop=True)

    # Normalize emotion labels
    if 'emotional_valence' in df.columns:
        df['emotional_valence'] = normalize_emotion_label_series(df['emotional_valence'])
    else:
        df['emotional_valence'] = 'neutral'

    # Force categorical with canonical categories
    df['emotional_valence'] = pd.Categorical(df['emotional_valence'], categories=KNOWN_EMOTIONS)

    # Emotion code
    df['emo_val_code'] = df['emotional_valence'].cat.codes.astype(int)

    # One-hot encoding (deterministic columns)
    one_hot = pd.get_dummies(df['emotional_valence'], prefix='emo')
    one_hot = one_hot.reindex(columns=sorted(one_hot.columns), fill_value=0)

    # Build embedding matrix
    labels = df['emotional_valence'].astype(str).tolist()
    N = len(labels)

    if emb_dim is None or emb_dim <= 0:
        emb_dim = 1
        emb_matrix = np.zeros((N, emb_dim), dtype=np.float32)
    else:
        emb_matrix = np.empty((N, emb_dim), dtype=np.float32)
        for i, lab in enumerate(labels):
            vec = emb_map.get(lab)
            if vec is None:
                emb_matrix[i, :] = 0.0
            else:
                arr = np.asarray(vec, dtype=np.float32)
                if arr.shape[0] != emb_dim:
                    if arr.shape[0] < emb_dim:
                        tmp = np.zeros(emb_dim, dtype=np.float32)
                        tmp[:arr.shape[0]] = arr
                        emb_matrix[i, :] = tmp
                    else:
                        emb_matrix[i, :] = arr[:emb_dim]
                else:
                    emb_matrix[i, :] = arr

    emb_cols = [f'emo_emb_{i}' for i in range(emb_dim)]
    emb_df = pd.DataFrame(emb_matrix, columns=emb_cols, index=df.index, dtype=np.float32)

    # Concat all new columns
    df = pd.concat([df, one_hot, emb_df], axis=1)

    # Compute emotional news delta (embedding L2 distance)
    df.sort_values(['ticker', 'timestamp'], inplace=True)
    prev_emb = df.groupby('ticker')[emb_cols].shift(1).fillna(0.0)
    emb_vals = df[emb_cols].values.astype(np.float32)
    prev_vals = prev_emb.values.astype(np.float32)
    delta = np.linalg.norm(emb_vals - prev_vals, axis=1)
    df['emotional_news_delta'] = delta.astype(float)
    df['A_Emotional_News_Delta'] = df['emotional_news_delta'].astype(float)

    # Numeric conversions (safe)
    num_cols = [
        'news_sentiment_score', 'social_proof_velocity', 'initial_engagement_velocity',
        'jargon_prevalence', 'sentiment_homogeneity', 'headline_similarity_score',
        'comment_reply_count', 'upvote_like_ratio', 'user_network_influence',
        'temporal_clustering_flag'
    ]

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0.0)

    # Behavioral features (vectorized)
    eps = 1e-6

    # H_Social_Momentum_Ratio
    if 'social_proof_velocity' in df.columns and 'initial_engagement_velocity' in df.columns:
        df['H_Social_Momentum_Ratio'] = df['social_proof_velocity'] / (df['initial_engagement_velocity'] + eps)
        df['H_Social_Momentum_Ratio'] = np.tanh(df['H_Social_Momentum_Ratio'])
    else:
        df['H_Social_Momentum_Ratio'] = 0.0

    # H_Jargon_Sentiment_Homogeneity
    if 'jargon_prevalence' in df.columns and 'sentiment_homogeneity' in df.columns:
        df['H_Jargon_Sentiment_Homogeneity'] = df['jargon_prevalence'] * df['sentiment_homogeneity']
    else:
        df['H_Jargon_Sentiment_Homogeneity'] = 0.0

    # A_Signal_Novelty
    if 'headline_similarity_score' in df.columns:
        df['A_Signal_Novelty'] = 1.0 / (df['headline_similarity_score'] + eps)
    else:
        df['A_Signal_Novelty'] = 1.0

    # H_Retail_Volume_Float_Rotation
    if 'comment_reply_count' in df.columns and 'upvote_like_ratio' in df.columns:
        if 'inferred_float_rotation' not in df.columns:
            df['inferred_float_rotation'] = 1.0
        df['H_Retail_Volume_Float_Rotation'] = (
            df['comment_reply_count'] / (df['upvote_like_ratio'] + eps)
        ) * df['inferred_float_rotation']
    else:
        df['H_Retail_Volume_Float_Rotation'] = 0.0

    # H_Network_Influence_Clustering
    if 'user_network_influence' in df.columns and 'temporal_clustering_flag' in df.columns:
        df['H_Network_Influence_Clustering'] = df['user_network_influence'] * df['temporal_clustering_flag']
    else:
        df['H_Network_Influence_Clustering'] = 0.0

    # Smoothing and normalization
    behavioral_cols = [
        'H_Social_Momentum_Ratio', 'H_Jargon_Sentiment_Homogeneity',
        'A_Signal_Novelty', 'H_Retail_Volume_Float_Rotation',
        'A_Emotional_News_Delta', 'H_Network_Influence_Clustering'
    ]

    for col in behavioral_cols:
        if col in df.columns:
            # Winsorize
            lb = df[col].quantile(0.01)
            ub = df[col].quantile(0.99)
            df[col] = df[col].clip(lb, ub)

            # Per-source normalization if available
            if 'source_file' in df.columns:
                df[col] = df.groupby('source_file')[col].transform(
                    lambda x: (x - x.mean()) / (x.std() + 1e-6)
                )
            else:
                # Global normalization
                df[col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-6)

    df = df.copy()  # Defragment

    logging.info(" Behavioral features complete")
    print(" Behavioral features engineered")

    return df

# ---------------- Main Execution ----------------

if __name__ == '__main__':
    try:
        print("\n Loading input data...")
        logging.info("Loading posts with market features...")

        df = pd.read_csv(INPUT_PATH, low_memory=False)

        print(f" Loaded {len(df):,} posts with market features")
        logging.info(f"Loaded {len(df)} rows")

        # Load or download model
        print("\n Loading sentence transformer model...")
        model, device = load_or_download_sentence_model()

        # Normalize emotions and build embedding map
        print("\n Preparing emotion embeddings...")
        if 'emotional_valence' in df.columns:
            df['emotional_valence'] = normalize_emotion_label_series(df['emotional_valence'])
        else:
            df['emotional_valence'] = 'neutral'

        uniq_emotions = sorted(set(df['emotional_valence'].unique()))
        emb_map, emb_dim = build_sentence_embeddings_map(uniq_emotions, model=model)

        # Engineer behavioral features
        final_df = engineer_behavioral_features(df, emb_map=emb_map, emb_dim=emb_dim)

        # Save outputs
        print("\n Saving final dataset...")
        logging.info("Saving to CSV...")
        final_df.to_csv(OUTPUT_PATH_CSV, index=False, float_format='%.6f')
        print(f" Saved CSV: posts_with_all_features_new.csv")

        logging.info("Saving to Parquet...")
        final_df.to_parquet(OUTPUT_PATH_PARQUET, index=False, engine='pyarrow', compression='snappy')
        print(f" Saved Parquet: posts_with_all_features_new.parquet")

        print("\n" + "=" * 60)
        print("FEATURE SUMMARY")
        print("=" * 60)
        print(f"Total records: {len(final_df):,}")
        print(f"Total columns: {len(final_df.columns)}")
        print(f"\nKey behavioral features:")
        for col in ['H_Social_Momentum_Ratio', 'A_Emotional_News_Delta', 'A_Signal_Novelty']:
            if col in final_df.columns:
                print(f"  {col}: {final_df[col].notna().sum():,} non-null")

        print("\n" + "=" * 60)
        print(" PHASE 4.4 COMPLETE")
        print("=" * 60)

    except FileNotFoundError as e:
        logging.error(f"Input file not found: {INPUT_PATH}")
        print(f"\n ERROR: Input file not found")
        print(f"   Expected: {INPUT_PATH}")
        print(f"   Make sure Phase 4.3 completed successfully")

    except Exception as e:
        logging.exception(f"Error: {e}")
        print(f"\n ERROR: {e}")


2025-10-27 18:48:50,062 - INFO: Loading posts with market features...


PHASE 4.4: BEHAVIORAL FEATURE ENGINEERING

📂 Loading input data...


2025-10-27 18:48:52,290 - INFO: Loaded 80002 rows
2025-10-27 18:48:52,292 - INFO: Loading cached model from: /content/drive/MyDrive/models/sentence_emotion_model
2025-10-27 18:48:52,299 - INFO: Load pretrained SentenceTransformer: /content/drive/MyDrive/models/sentence_emotion_model


✅ Loaded 80,002 posts with market features

🤖 Loading sentence transformer model...

🔧 Preparing emotion embeddings...


2025-10-27 18:48:54,050 - INFO: Computing embeddings for 11 emotions


...


🔧 Engineering behavioral features...


2025-10-27 18:49:00,787 - INFO: ✅ Behavioral features complete
2025-10-27 18:49:00,831 - INFO: Saving to CSV...


✅ Behavioral features engineered

💾 Saving final dataset...


2025-10-27 18:50:35,285 - INFO: Saving to Parquet...


✅ Saved CSV: posts_with_all_features_new.csv
✅ Saved Parquet: posts_with_all_features_new.parquet

FEATURE SUMMARY
Total records: 80,002
Total columns: 480

Key behavioral features:
  H_Social_Momentum_Ratio: 80,002 non-null
  A_Emotional_News_Delta: 80,002 non-null
  A_Signal_Novelty: 80,002 non-null

✅ PHASE 4.4 COMPLETE


*Verification*

In [ ]:
import pandas as pd

# Load output
df = pd.read_csv('/content/drive/MyDrive/ML_Project_Data/Combined/posts_with_all_features_new.csv')

print("=" * 60)
print("PHASE 4.4 VERIFICATION")
print("=" * 60)

# 1. Check row count
print(f"\n Total rows: {len(df):,}")
print(f"   Expected: ~80,002 (from Phase 4.3)")

# 2. Check embedding columns
emb_cols = [c for c in df.columns if c.startswith('emo_emb_')]
print(f"\n Embedding columns: {len(emb_cols)}")
print(f"   Expected: 384")
print(f"   Sample: {emb_cols[:5]}")

# 3. Check one-hot emotion columns
onehot_cols = [c for c in df.columns if c.startswith('emo_') and c != 'emo_val_code']
print(f"\n One-hot emotion columns: {len(onehot_cols)}")
print(f"   Expected: 11")
print(f"   Columns: {sorted(onehot_cols)}")

# 4. Check behavioral features
behavioral = [
    'A_Emotional_News_Delta',
    'H_Social_Momentum_Ratio',
    'H_Jargon_Sentiment_Homogeneity',
    'A_Signal_Novelty',
    'H_Retail_Volume_Float_Rotation',
    'H_Network_Influence_Clustering'
]
missing = [f for f in behavioral if f not in df.columns]
print(f"\n Behavioral features: {len(behavioral) - len(missing)}/{len(behavioral)}")
if missing:
    print(f"    Missing: {missing}")
else:
    print(f"    All present")

# 5. Check for NaN/zeros in embeddings
if emb_cols:
    emb_data = df[emb_cols]
    all_zero_rows = (emb_data == 0).all(axis=1).sum()
    nan_rows = emb_data.isna().any(axis=1).sum()

    print(f"\n Embedding quality:")
    print(f"   Rows with all-zero embeddings: {all_zero_rows} ({all_zero_rows/len(df)*100:.1f}%)")
    print(f"   Rows with NaN embeddings: {nan_rows}")
    print(f"   Sample embedding values (first row):")
    print(f"      {df[emb_cols].iloc[0][:5].to_dict()}")

# 6. Check behavioral feature distributions
print(f"\n Behavioral feature stats:")
for feat in behavioral:
    if feat in df.columns:
        non_zero = (df[feat] != 0).sum()
        print(f"   {feat}:")
        print(f"      Non-zero: {non_zero:,} ({non_zero/len(df)*100:.1f}%)")
        print(f"      Mean: {df[feat].mean():.4f}, Std: {df[feat].std():.4f}")

# 7. Check emotion distribution
print(f"\n Emotion distribution:")
if 'emotional_valence' in df.columns:
    print(df['emotional_valence'].value_counts().head(10))

print("\n" + "=" * 60)


/tmp/ipython-input-1241082690.py:4: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/ML_Project_Data/Combined/posts_with_all_features_new.csv')


PHASE 4.4 VERIFICATION

📊 Total rows: 80,002
   Expected: ~80,002 (from Phase 4.3)

🔢 Embedding columns: 384
   Expected: 384
   Sample: ['emo_emb_0', 'emo_emb_1', 'emo_emb_2', 'emo_emb_3', 'emo_emb_4']

🏷️ One-hot emotion columns: 395
   Expected: 11
   Columns: ['emo_anger', 'emo_anticipation', 'emo_disgust', 'emo_emb_0', 'emo_emb_1', 'emo_emb_10', 'emo_emb_100', 'emo_emb_101', 'emo_emb_102', 'emo_emb_103', 'emo_emb_104', 'emo_emb_105', 'emo_emb_106', 'emo_emb_107', 'emo_emb_108', 'emo_emb_109', 'emo_emb_11', 'emo_emb_110', 'emo_emb_111', 'emo_emb_112', 'emo_emb_113', 'emo_emb_114', 'emo_emb_115', 'emo_emb_116', 'emo_emb_117', 'emo_emb_118', 'emo_emb_119', 'emo_emb_12', 'emo_emb_120', 'emo_emb_121', 'emo_emb_122', 'emo_emb_123', 'emo_emb_124', 'emo_emb_125', 'emo_emb_126', 'emo_emb_127', 'emo_emb_128', 'emo_emb_129', 'emo_emb_13', 'emo_emb_130', 'emo_emb_131', 'emo_emb_132', 'emo_emb_133', 'emo_emb_134', 'emo_emb_135', 'emo_emb_136', 'emo_emb_137', 'emo_emb_138', 'emo_emb_139', 'emo_

**Phase 4.5 - Get real inferred Float rotation and reclaculate H_retail_volume_float_rotation**

In [ ]:
# ============================================================
# PHASE 4.5: FLOAT ROTATION REFINEMENT
# Fetches real shares float data and recalculates H_Retail feature
# ============================================================

!pip install yfinance pyarrow --quiet

import pandas as pd
import yfinance as yf
import os
import logging
import time
import numpy as np

# --- CONFIG ---
BASE_DIR = '/content/drive/MyDrive/'

#  FIXED: Use Phase 4.4 output files
INPUT_PATH_CSV = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_new.csv')
INPUT_PATH_PARQUET = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_new.parquet')

# Output: refined version
OUTPUT_PATH_CSV = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_refined_new.csv')
OUTPUT_PATH_PARQUET = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_refined_new.parquet')

#  FIXED: Use Phase 4.2 financial data
FINANCIAL_DATA_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/financial_intraday_data_new.parquet')

FLOAT_CACHE_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/float_cache.csv')
TICKER_PAUSE = 0.5
MAX_RETRIES = 3
RETRY_BACKOFF = 1.5
EPS = 1e-9

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s: %(message)s',
    force=True
)

print("=" * 60)
print("PHASE 4.5: FLOAT ROTATION REFINEMENT")
print("=" * 60)

# ---------------- Float Fetching ----------------

def fetch_shares_float_for_tickers(tickers, cache_path=FLOAT_CACHE_PATH):
    """Fetches shares float from yfinance with caching."""
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    # Load existing cache
    if os.path.exists(cache_path):
        try:
            cache_df = pd.read_csv(cache_path)
            logging.info(f"Loaded float cache: {len(cache_df)} rows")
        except Exception:
            cache_df = pd.DataFrame(columns=['ticker', 'shares_float'])
    else:
        cache_df = pd.DataFrame(columns=['ticker', 'shares_float'])

    cache_df['ticker'] = cache_df['ticker'].astype(str)

    # Find tickers to fetch
    tickers = sorted(list(set([t for t in tickers if isinstance(t, str) and t.strip() != ""])))
    to_fetch = [t for t in tickers if t not in cache_df['ticker'].values]

    logging.info(f"Total: {len(tickers)}, To fetch: {len(to_fetch)}")
    print(f"\n Fetching shares float data:")
    print(f"   Cached: {len(tickers) - len(to_fetch)}")
    print(f"   To fetch: {len(to_fetch)}")

    fetched = []
    for i, t in enumerate(to_fetch, 1):
        # Skip cryptos
        if '-USD' in t:
            fetched.append({'ticker': t, 'shares_float': np.nan})
            print(f"   [{i}/{len(to_fetch)}] {t}: Skipped (crypto)")
            continue

        retries = 0
        while retries < MAX_RETRIES:
            try:
                print(f"   [{i}/{len(to_fetch)}] {t}: Fetching...", end='')
                tk = yf.Ticker(t)
                info = tk.info
                shares_float = info.get('floatShares', info.get('sharesOutstanding', None))

                if shares_float in (None, '', 0):
                    shares_float = np.nan

                fetched.append({'ticker': t, 'shares_float': shares_float})
                print(f"  {shares_float:,.0f}" if pd.notna(shares_float) else " ⚠️ N/A")
                time.sleep(TICKER_PAUSE)
                break

            except Exception as e:
                retries += 1
                time.sleep((RETRY_BACKOFF ** retries) * TICKER_PAUSE)
        else:
            fetched.append({'ticker': t, 'shares_float': np.nan})
            print(f"  Failed")

    # Update cache
    if fetched:
        fetched_df = pd.DataFrame(fetched)
        combined = pd.concat([cache_df, fetched_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=['ticker'], keep='first').reset_index(drop=True)
        combined.to_csv(cache_path, index=False)
        print(f"\n Cache updated: {len(combined)} tickers")
        return combined
    else:
        return cache_df

# ---------------- Main Processing ----------------

def add_true_float_rotation(features_csv_path, features_parquet_path, financial_path,
                            output_csv, output_parquet):
    """Adds real float rotation data to features."""
    logging.info("Loading features and financial data...")
    print("\n Loading data...")

    # Load features (prefer parquet if exists)
    try:
        df_features = pd.read_parquet(features_csv_path.replace('.csv', '.parquet'))
        print(f" Loaded {len(df_features):,} rows (from parquet)")
    except:
        df_features = pd.read_csv(features_csv_path, low_memory=False)
        print(f" Loaded {len(df_features):,} rows (from CSV)")

    # Load financial data
    try:
        df_fin = pd.read_parquet(financial_path)
        print(f" Loaded {len(df_fin):,} market bars (parquet)")
    except:
        df_fin = pd.read_csv(financial_path, low_memory=False)
        print(f" Loaded {len(df_fin):,} market bars (CSV)")

    # Ensure timestamps
    if 'timestamp_utc' in df_features.columns:
        df_features['timestamp'] = pd.to_datetime(df_features['timestamp_utc'], utc=True, errors='coerce')
    elif 'timestamp' in df_features.columns:
        df_features['timestamp'] = pd.to_datetime(df_features['timestamp'], utc=True, errors='coerce')

    df_fin['market_timestamp_utc'] = pd.to_datetime(df_fin['market_timestamp_utc'], utc=True, errors='coerce')

    # Compute daily volume
    print("\n Computing daily volumes...")
    df_fin['date'] = df_fin['market_timestamp_utc'].dt.date
    df_features['date'] = df_features['timestamp'].dt.date

    df_fin['Volume'] = pd.to_numeric(df_fin['Volume'], errors='coerce').fillna(0.0)
    daily_volume = df_fin.groupby(['ticker', 'date'], as_index=False)['Volume'].sum().rename(
        columns={'Volume': 'daily_volume'}
    )
    print(f" {len(daily_volume):,} ticker-date pairs")

    # Fetch shares float
    unique_tickers = df_features['ticker'].dropna().astype(str).unique().tolist()
    float_df = fetch_shares_float_for_tickers(unique_tickers, cache_path=FLOAT_CACHE_PATH)

    # Merge
    print("\n Merging float and volume data...")
    df_merged = df_features.merge(float_df, on='ticker', how='left')
    df_merged = df_merged.merge(daily_volume, on=['ticker', 'date'], how='left')

    df_merged['shares_float'] = pd.to_numeric(df_merged['shares_float'], errors='coerce')
    df_merged['daily_volume'] = pd.to_numeric(df_merged['daily_volume'], errors='coerce')

    # Calculate float rotation
    df_merged['inferred_float_rotation'] = df_merged['daily_volume'] / (df_merged['shares_float'] + EPS)
    df_merged.replace([np.inf, -np.inf], np.nan, inplace=True)

    n_missing_float = df_merged['shares_float'].isna().sum()
    n_missing_vol = df_merged['daily_volume'].isna().sum()

    print(f"\n Data quality:")
    print(f"   Missing float: {n_missing_float:,} ({n_missing_float/len(df_merged)*100:.1f}%)")
    print(f"   Missing volume: {n_missing_vol:,} ({n_missing_vol/len(df_merged)*100:.1f}%)")

    df_merged['inferred_float_rotation'] = df_merged['inferred_float_rotation'].fillna(0.0)

    # Recalculate H_Retail
    print("\n Recalculating H_Retail_Volume_Float_Rotation...")
    df_merged['comment_reply_count'] = pd.to_numeric(df_merged.get('comment_reply_count', 0), errors='coerce').fillna(0.0)
    df_merged['upvote_like_ratio'] = pd.to_numeric(df_merged.get('upvote_like_ratio', 0), errors='coerce').fillna(0.0)

    df_merged['H_Retail_Volume_Float_Rotation'] = (
        df_merged['comment_reply_count'] / (df_merged['upvote_like_ratio'] + EPS)
    ) * df_merged['inferred_float_rotation']

    # Drop helper columns
    df_out = df_merged.drop(columns=['shares_float', 'daily_volume', 'date'], errors='ignore')

    # Save
    print("\n Saving refined features...")
    df_out.to_csv(output_csv, index=False)
    print(f" Saved CSV: {os.path.basename(output_csv)}")

    try:
        df_out.to_parquet(output_parquet, index=False, engine='pyarrow', compression='snappy')
        print(f" Saved Parquet: {os.path.basename(output_parquet)}")
    except Exception as e:
        logging.warning(f"Could not save Parquet: {e}")

    print("\n=== Sample Results ===")
    sample_cols = ['ticker', 'inferred_float_rotation', 'H_Retail_Volume_Float_Rotation']
    print(df_out[sample_cols].head(10).to_string(index=False))

    print("\n" + "=" * 60)
    print(" PHASE 4.5 COMPLETE")
    print("=" * 60)

    return df_out

# ---------------- Execute ----------------

if __name__ == '__main__':
    try:
        add_true_float_rotation(
            features_csv_path=INPUT_PATH_CSV,
            features_parquet_path=INPUT_PATH_PARQUET,
            financial_path=FINANCIAL_DATA_PATH,
            output_csv=OUTPUT_PATH_CSV,
            output_parquet=OUTPUT_PATH_PARQUET
        )
    except FileNotFoundError as e:
        print(f"\n ERROR: File not found")
        print(f"   Make sure Phase 4.4 completed successfully")
        print(f"   Expected: {INPUT_PATH_CSV}")
    except Exception as e:
        logging.exception(f"Error: {e}")


2025-10-27 18:55:34,191 - INFO: Loading features and financial data...


PHASE 4.5: FLOAT ROTATION REFINEMENT

📂 Loading data...
✅ Loaded 80,002 rows (from parquet)
✅ Loaded 166,987 market bars (parquet)

📊 Computing daily volumes...
✅ 2,174 ticker-date pairs


2025-10-27 18:55:35,405 - INFO: Loaded float cache: 28 rows
2025-10-27 18:55:35,409 - INFO: Total: 31, To fetch: 3



📊 Fetching shares float data:
   Cached: 28
   To fetch: 3
   [1/3] AAPL: Fetching... ✅ 14,814,270,914
   [2/3] META: Fetching... ✅ 2,164,461,095
   [3/3] TSLA: Fetching... ✅ 2,809,419,225

✅ Cache updated: 31 tickers

🔗 Merging float and volume data...

📊 Data quality:
   Missing float: 14,048 (17.6%)
   Missing volume: 15,709 (19.6%)

🔧 Recalculating H_Retail_Volume_Float_Rotation...

💾 Saving refined features...
✅ Saved CSV: posts_with_all_features_refined_new.csv
✅ Saved Parquet: posts_with_all_features_refined_new.parquet

=== Sample Results ===
 ticker  inferred_float_rotation  H_Retail_Volume_Float_Rotation
2222.SR                 0.001296                        0.005508
2222.SR                 0.001296                        0.216274
2222.SR                 0.001296                        0.010918
2222.SR                 0.001296                        0.124750
2222.SR                 0.001296                        0.015507
2222.SR                 0.001296                    

*Verification*

In [ ]:
import pandas as pd

df = pd.read_parquet('/content/drive/MyDrive/ML_Project_Data/Combined/posts_with_all_features_refined_new.parquet')

print("=" * 60)
print("FINAL DATASET VERIFICATION")
print("=" * 60)

print(f"\n Dataset: {len(df):,} rows × {len(df.columns)} columns")

print(f"\n Key features:")
print(f"   Emotion embeddings: {len([c for c in df.columns if c.startswith('emo_emb_')])}")
print(f"   Market features: {len([c for c in df.columns if c in ['last_available_close', 'prior_day_close', 'prior_5d_avg_return']])}")
print(f"   Behavioral features: {len([c for c in df.columns if c.startswith('H_') or c.startswith('A_')])}")

print(f"\n Float rotation statistics:")
print(f"   Posts with non-zero float rotation: {(df['inferred_float_rotation'] > 0).sum():,} ({(df['inferred_float_rotation'] > 0).mean()*100:.1f}%)")
print(f"   Mean float rotation: {df['inferred_float_rotation'].mean():.6f}")
print(f"   Median float rotation: {df['inferred_float_rotation'].median():.6f}")

print(f"\n H_Retail_Volume_Float_Rotation:")
print(f"   Non-zero values: {(df['H_Retail_Volume_Float_Rotation'] != 0).sum():,}")
print(f"   Mean: {df['H_Retail_Volume_Float_Rotation'].mean():.6f}")
print(f"   Std: {df['H_Retail_Volume_Float_Rotation'].std():.6f}")

print("\n" + "=" * 60)
print(" ALL FEATURE ENGINEERING COMPLETE")
print("=" * 60)


FINAL DATASET VERIFICATION

📊 Dataset: 80,002 rows × 480 columns

🔢 Key features:
   Emotion embeddings: 384
   Market features: 2
   Behavioral features: 6

💰 Float rotation statistics:
   Posts with non-zero float rotation: 50,014 (62.5%)
   Mean float rotation: 0.041061
   Median float rotation: 0.010924

🧮 H_Retail_Volume_Float_Rotation:
   Non-zero values: 27,609
   Mean: 65555274.788372
   Std: 658097428.183173

✅ ALL FEATURE ENGINEERING COMPLETE


**Phase 4.6 - Normalization**

In [ ]:
# ============================================================
# PHASE 4.6: FEATURE NORMALIZATION (Z-SCORE)
# Creates _z versions of all numeric features
# ============================================================

import pandas as pd
import numpy as np
import os

BASE_DIR = '/content/drive/MyDrive/'
INPUT_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_refined_new.parquet')
OUTPUT_PATH_PARQUET = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_normalized_new.parquet')
OUTPUT_PATH_CSV = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_normalized_new.csv')

print("=" * 60)
print("PHASE 4.6: FEATURE NORMALIZATION")
print("=" * 60)

# Load data
print("\n Loading refined dataset...")
df = pd.read_parquet(INPUT_PATH)
print(f" Loaded {len(df):,} rows × {len(df.columns)} columns")

# Define features to normalize (exclude categorical, text, timestamp, etc.)
features_to_normalize = [
    'post_id',
    'sentiment_score',
    'news_keyword_count',
    'ner_mentions_count',
    'time_offset_from_event',
    'initial_engagement_velocity',
    'post_length_words',
    'post_length_chars',
    'user_posting_frequency',
    'reply_chain_length',
    'jargon_prevalence',
    'sentiment_homogeneity',
    'crowd_keyword_count',
    'text_similarity_top_posts',
    'ticker_repetition_window',
    'social_proof_velocity',
    'upvote_like_ratio',
    'comment_reply_count',
    'user_network_influence',
    'headline_similarity_score',
    'news_sentiment_score',
    'social_news_sentiment_delta',
    'fed',
    'intraday_return_30min',
    'intraday_return_24hr',
    'intraday_volatility',
    'volume_spike_ratio',
    'A_Emotional_News_Delta',
    'H_Social_Momentum_Ratio',
    'H_Jargon_Sentiment_Homogeneity',
    'A_Signal_Novelty',
    'inferred_float_rotation',
    'H_Retail_Volume_Float_Rotation',
    'H_Network_Influence_Clustering'
]

# Filter to only features that exist in the dataset
existing_features = [f for f in features_to_normalize if f in df.columns]
missing_features = [f for f in features_to_normalize if f not in df.columns]

print(f"\n Features to normalize:")
print(f"   Found: {len(existing_features)}")
print(f"   Missing: {len(missing_features)}")

if missing_features:
    print(f"\n Missing features (will skip):")
    for feat in missing_features:
        print(f"      - {feat}")

# Z-score normalization function
def zscore_normalize(series):
    """
    Z-score normalization: (x - mean) / std
    Handles NaN and infinite values
    """
    mean = series.mean()
    std = series.std()

    # Avoid division by zero
    if std == 0 or pd.isna(std):
        return series * 0  # Return zeros if no variance

    normalized = (series - mean) / std

    # Replace inf/-inf with NaN, then fill with 0
    normalized = normalized.replace([np.inf, -np.inf], np.nan).fillna(0)

    return normalized

# Normalize each feature
print(f"\n Normalizing {len(existing_features)} features...")

normalized_count = 0
skipped_count = 0

for feature in existing_features:
    try:
        # Convert to numeric (handles any string/mixed types)
        df[feature] = pd.to_numeric(df[feature], errors='coerce')

        # Apply z-score normalization
        df[f'{feature}_z'] = zscore_normalize(df[feature])

        normalized_count += 1

        # Show progress every 10 features
        if normalized_count % 10 == 0:
            print(f"   Processed {normalized_count}/{len(existing_features)} features...")

    except Exception as e:
        print(f"    Skipped {feature}: {e}")
        skipped_count += 1

print(f"\n Normalization complete:")
print(f"   Normalized: {normalized_count}")
print(f"   Skipped: {skipped_count}")

# Verify normalization
print(f"\n Verification (checking 5 random normalized features):")
sample_features = np.random.choice(existing_features, min(5, len(existing_features)), replace=False)

for feat in sample_features:
    z_col = f'{feat}_z'
    if z_col in df.columns:
        mean = df[z_col].mean()
        std = df[z_col].std()
        print(f"   {feat}_z:")
        print(f"      Mean: {mean:.6f} (should be ~0)")
        print(f"      Std:  {std:.6f} (should be ~1)")

# Show total columns
print(f"\n Final dataset:")
print(f"   Total rows: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Original features: {len(existing_features)}")
print(f"   Normalized features (_z): {normalized_count}")

# Save normalized dataset
print(f"\n Saving normalized dataset...")

df.to_parquet(OUTPUT_PATH_PARQUET, index=False, engine='pyarrow', compression='snappy')
print(f" Saved: posts_with_all_features_normalized_new.parquet")

df.to_csv(OUTPUT_PATH_CSV, index=False)
print(f" Saved: posts_with_all_features_normalized_new.csv")

# Summary statistics
print(f"\n" + "=" * 60)
print("NORMALIZATION SUMMARY")
print("=" * 60)

print(f"\n Sample normalized features:")
z_cols = [c for c in df.columns if c.endswith('_z')]
print(f"   Total _z columns: {len(z_cols)}")

# Show first 5 rows of a few normalized features
sample_display = z_cols[:5]
print(f"\n Sample data (first 3 rows, first 5 normalized features):")
print(df[sample_display].head(3).to_string(index=False))

print("\n" + "=" * 60)
print(" PHASE 4.6 COMPLETE - DATASET NORMALIZED")
print("=" * 60)


PHASE 4.6: FEATURE NORMALIZATION

📂 Loading refined dataset...
✅ Loaded 80,002 rows × 480 columns

🔢 Features to normalize:
   Found: 34
   Missing: 0

🔧 Normalizing 34 features...
   Processed 10/34 features...
   Processed 20/34 features...
   Processed 30/34 features...

✅ Normalization complete:
   Normalized: 34
   Skipped: 0

📊 Verification (checking 5 random normalized features):
   H_Retail_Volume_Float_Rotation_z:
      Mean: -0.000000 (should be ~0)
      Std:  1.000000 (should be ~1)
   post_length_chars_z:
      Mean: -0.000000 (should be ~0)
      Std:  1.000000 (should be ~1)
   headline_similarity_score_z:
      Mean: 0.000000 (should be ~0)
      Std:  1.000000 (should be ~1)
   intraday_volatility_z:
      Mean: 0.000000 (should be ~0)
      Std:  1.000000 (should be ~1)
   jargon_prevalence_z:
      Mean: 0.000000 (should be ~0)
      Std:  1.000000 (should be ~1)

📊 Final dataset:
   Total rows: 80,002
   Total columns: 514
   Original features: 34
   Normalized feat

**Phase 5 - mapping train set with new dataset**

In [ ]:
# ============================================================
# PHASE 5: UPDATE TRAIN SEED WITH NEW ENRICHED FEATURES
# Preserves ALL unique train_seed features + adds enriched features
# ============================================================

import pandas as pd
import numpy as np
import os

BASE_DIR = '/content/drive/MyDrive/'

# Input files
MAPPING_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/post_id_mapping.csv')
TRAIN_SEED_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets/train_seed.csv')  #  FIXED PATH
NEW_DATASET_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_normalized_new.parquet')

# Output files
OUTPUT_DIR = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets')
OUTPUT_TRAIN_SEED_PATH = os.path.join(OUTPUT_DIR, 'train_seed_updated_new.csv')
OUTPUT_TRAIN_SEED_PARQUET = os.path.join(OUTPUT_DIR, 'train_seed_updated_new.parquet')
OUTPUT_UNMAPPED_PATH = os.path.join(OUTPUT_DIR, 'train_seed_unmapped_posts.csv')

print("=" * 60)
print("PHASE 5: TRAIN SEED UPDATE")
print("=" * 60)

# ==================== STEP 1: LOAD DATA ====================

print("\n Loading data...")

# Load mapping
mapping_df = pd.read_csv(MAPPING_PATH)
print(f" Loaded mapping: {len(mapping_df):,} rows")

# Load train seed
train_seed = pd.read_csv(TRAIN_SEED_PATH, low_memory=False)
print(f" Loaded train seed: {len(train_seed):,} rows × {len(train_seed.columns)} columns")

# Load new enriched dataset
new_dataset = pd.read_parquet(NEW_DATASET_PATH)
print(f" Loaded new dataset: {len(new_dataset):,} rows × {len(new_dataset.columns)} columns")

# ==================== STEP 2: IDENTIFY UNIQUE TRAIN SEED FEATURES ====================

print("\n Identifying unique train_seed features...")

# Find columns that exist ONLY in train_seed (not in new_dataset)
train_seed_cols = set(train_seed.columns)
new_dataset_cols = set(new_dataset.columns)

unique_to_train = train_seed_cols - new_dataset_cols
unique_to_new = new_dataset_cols - train_seed_cols
common_cols = train_seed_cols & new_dataset_cols

print(f"\n Column analysis:")
print(f"   Unique to train_seed: {len(unique_to_train)} columns")
print(f"   Unique to new_dataset: {len(unique_to_new)} columns")
print(f"   Common columns: {len(common_cols)} columns")

if unique_to_train:
    print(f"\n Unique train_seed features (will be preserved):")
    for col in sorted(unique_to_train):
        print(f"      - {col}")

# ==================== STEP 3: PREPARE MAPPING ====================

print("\n Preparing mapping...")

# Ensure mapping has required columns
if 'old_post_id' not in mapping_df.columns or 'new_post_id' not in mapping_df.columns:
    print(" ERROR: Mapping file must have 'old_post_id' and 'new_post_id' columns")
    exit(1)

# Create lookup dictionary
old_to_new = dict(zip(mapping_df['old_post_id'], mapping_df['new_post_id']))
print(f" Created mapping dictionary: {len(old_to_new):,} mappings")

# ==================== STEP 4: MAP POST IDs ====================

print("\n Mapping old post IDs to new post IDs...")

# Map post_id in train_seed to new_post_id
train_seed['new_post_id'] = train_seed['post_id'].map(old_to_new)

# Count mappings
mapped_count = train_seed['new_post_id'].notna().sum()
unmapped_count = train_seed['new_post_id'].isna().sum()

print(f" Mapping results:")
print(f"   Mapped: {mapped_count:,} ({mapped_count/len(train_seed)*100:.1f}%)")
print(f"   Unmapped: {unmapped_count:,} ({unmapped_count/len(train_seed)*100:.1f}%)")

# ==================== STEP 5: PRESERVE UNIQUE TRAIN SEED FEATURES ====================

print("\n Preserving unique train_seed features...")

# Keep only mapped rows
train_seed_mapped = train_seed[train_seed['new_post_id'].notna()].copy()

# Extract unique train_seed columns to preserve
unique_features_df = train_seed_mapped[['new_post_id'] + list(unique_to_train)]
print(f" Extracted {len(unique_to_train)} unique features")

# ==================== STEP 6: MERGE WITH NEW DATASET ====================

print("\n Merging with enriched features...")

# Start with new_dataset features
train_seed_updated = new_dataset[new_dataset['post_id'].isin(train_seed_mapped['new_post_id'])].copy()
print(f" Found {len(train_seed_updated):,} matching posts in new dataset")

# Merge unique train_seed features back
train_seed_updated = train_seed_updated.merge(
    unique_features_df,
    left_on='post_id',
    right_on='new_post_id',
    how='left',
    suffixes=('', '_preserve')
)

# Drop the extra new_post_id column created by merge
if 'new_post_id' in train_seed_updated.columns and 'post_id' in train_seed_updated.columns:
    train_seed_updated = train_seed_updated.drop(columns=['new_post_id'])

print(f" Merge complete: {len(train_seed_updated):,} rows × {len(train_seed_updated.columns)} columns")

# ==================== STEP 7: HANDLE COMMON COLUMNS (KEEP NEW VALUES) ====================

print("\n Handling overlapping columns...")

# For common columns, prefer new_dataset values (enriched/normalized)
# except for specific columns we want to preserve from train_seed
preserve_from_train = [
    'label', 'target', 'human_label', 'herd_behavior', 'availability_heuristic',
    'predicted_label', 'confidence', 'annotator', 'annotation_date',
    'notes', 'review_status', 'quality_flag'
]

# Check if any preserved columns exist in common_cols
preserved_cols = []
for col in preserve_from_train:
    if col in common_cols:
        preserved_cols.append(col)
        # If the column was overwritten by new_dataset, restore from train_seed
        if f"{col}_preserve" in train_seed_updated.columns:
            train_seed_updated[col] = train_seed_updated[f"{col}_preserve"]
            train_seed_updated = train_seed_updated.drop(columns=[f"{col}_preserve"])

if preserved_cols:
    print(f"   Preserved {len(preserved_cols)} label/annotation columns:")
    for col in preserved_cols:
        non_null = train_seed_updated[col].notna().sum()
        print(f"      - {col}: {non_null:,} non-null")

# ==================== STEP 8: ADD METADATA ====================

print("\n Adding metadata...")

# Map back to original post_id for reference
new_to_old = {v: k for k, v in old_to_new.items()}
train_seed_updated['old_post_id'] = train_seed_updated['post_id'].map(new_to_old)

# Add update metadata
train_seed_updated['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
train_seed_updated['feature_version'] = 'v2_enriched_normalized'

# ==================== STEP 9: SAVE UNMAPPED POSTS ====================

if unmapped_count > 0:
    print(f"\n Saving {unmapped_count:,} unmapped posts...")

    unmapped_posts = train_seed[train_seed['new_post_id'].isna()].copy()
    unmapped_posts.to_csv(OUTPUT_UNMAPPED_PATH, index=False)

    print(f" Saved: train_seed_unmapped_posts.csv")
    print(f"   These posts need manual review or re-mapping")

# ==================== STEP 10: SAVE UPDATED TRAIN SEED ====================

print(f"\n Saving updated train seed...")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save as CSV
train_seed_updated.to_csv(OUTPUT_TRAIN_SEED_PATH, index=False)
print(f" Saved CSV: train_seed_updated_new.csv")

# Save as Parquet
train_seed_updated.to_parquet(OUTPUT_TRAIN_SEED_PARQUET, index=False, engine='pyarrow', compression='snappy')
print(f" Saved Parquet: train_seed_updated_new.parquet")

# ==================== STEP 11: FINAL SUMMARY ====================

print("\n" + "=" * 60)
print("UPDATE SUMMARY")
print("=" * 60)

print(f"\n Dataset sizes:")
print(f"   Original train_seed: {len(train_seed):,} rows")
print(f"   Mapped successfully: {mapped_count:,} rows ({mapped_count/len(train_seed)*100:.1f}%)")
print(f"   Unmapped (review needed): {unmapped_count:,} rows")
print(f"   Updated train_seed: {len(train_seed_updated):,} rows")

print(f"\n Feature counts:")
print(f"   Original train_seed columns: {len(train_seed.columns)}")
print(f"   New dataset columns: {len(new_dataset.columns)}")
print(f"   Updated train_seed columns: {len(train_seed_updated.columns)}")

print(f"\n Feature breakdown:")
print(f"   Unique to train_seed (preserved): {len(unique_to_train)}")
print(f"   New enriched features added: {len(unique_to_new)}")
print(f"   Updated common features: {len(common_cols)}")

# List the unique preserved features
if unique_to_train:
    print(f"\n Preserved unique train_seed features:")
    for col in sorted(unique_to_train):
        if col in train_seed_updated.columns:
            non_null = train_seed_updated[col].notna().sum()
            print(f"      - {col}: {non_null:,} non-null values")

print("\n" + "=" * 60)
print(" PHASE 5 COMPLETE - TRAIN SEED UPDATED")
print("=" * 60)

# ==================== VERIFICATION ====================

print("\n Sample of updated train_seed (first 3 rows):")
sample_cols = ['post_id', 'old_post_id', 'ticker', 'timestamp_utc', 'sentiment_score_z']

# Add unique train_seed columns to sample
unique_sample = list(unique_to_train)[:2]
sample_cols.extend(unique_sample)

available_cols = [c for c in sample_cols if c in train_seed_updated.columns]
print(train_seed_updated[available_cols].head(3).to_string(index=False))

print("\n All unique train_seed features have been preserved!")


PHASE 5: TRAIN SEED UPDATE

📂 Loading data...
✅ Loaded mapping: 96,335 rows
✅ Loaded train seed: 5,610 rows × 95 columns
✅ Loaded new dataset: 80,002 rows × 514 columns

🔍 Identifying unique train_seed features...

📊 Column analysis:
   Unique to train_seed: 5 columns
   Unique to new_dataset: 424 columns
   Common columns: 90 columns

✨ Unique train_seed features (will be preserved):
      - SA_SH_sum
      - SA_score
      - SH_score
      - pool
      - predicted_label

🔧 Preparing mapping...
✅ Created mapping dictionary: 96,335 mappings

🔗 Mapping old post IDs to new post IDs...
✅ Mapping results:
   Mapped: 5,062 (90.2%)
   Unmapped: 548 (9.8%)

💾 Preserving unique train_seed features...
✅ Extracted 5 unique features

🔗 Merging with enriched features...
📊 Found 4,180 matching posts in new dataset
✅ Merge complete: 4,180 rows × 519 columns

🔧 Handling overlapping columns...

📝 Adding metadata...

💾 Saving 548 unmapped posts...
✅ Saved: train_seed_unmapped_posts.csv
   These posts n

*Check*

In [ ]:
import pandas as pd
import os

BASE_DIR = '/content/drive/MyDrive/'

# Load files
mapping_df = pd.read_csv(os.path.join(BASE_DIR, 'ML_Project_Data/Combined/post_id_mapping.csv'))
train_seed = pd.read_csv(os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets/train_seed.csv'))
new_dataset = pd.read_parquet(os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_normalized_new.parquet'))
merged_raw = pd.read_csv(os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv'))

print("=" * 60)
print("MISSING POSTS ANALYSIS")
print("=" * 60)

# Get mapped IDs
old_to_new = dict(zip(mapping_df['old_post_id'], mapping_df['new_post_id']))
train_seed['new_post_id'] = train_seed['post_id'].map(old_to_new)
mapped = train_seed[train_seed['new_post_id'].notna()]

print(f"\n Train seed mapping:")
print(f"   Total train_seed posts: {len(train_seed):,}")
print(f"   Successfully mapped: {len(mapped):,}")
print(f"   Found in enriched dataset: {len(new_dataset[new_dataset['post_id'].isin(mapped['new_post_id'])]):,}")

# Find missing posts
mapped_ids = set(mapped['new_post_id'])
enriched_ids = set(new_dataset['post_id'])
missing_ids = mapped_ids - enriched_ids

print(f"\n Missing posts breakdown:")
print(f"   Mapped but not in enriched dataset: {len(missing_ids):,}")

if len(missing_ids) > 0:
    # Check if they exist in merged_raw
    merged_raw_ids = set(merged_raw['post_id'])
    in_merged_raw = missing_ids & merged_raw_ids

    print(f"\n Where did they go?")
    print(f"   Still in merged_raw_posts.csv: {len(in_merged_raw):,}")
    print(f"   Removed during deduplication (Phase 4.3): {len(missing_ids) - len(in_merged_raw):,}")

    # Sample of missing posts
    if len(in_merged_raw) > 0:
        sample_missing = list(in_merged_raw)[:5]
        missing_posts = merged_raw[merged_raw['post_id'].isin(sample_missing)]

        print(f"\n Sample of missing posts still in merged_raw:")
        display_cols = ['post_id', 'ticker', 'timestamp', 'text']
        available_cols = [c for c in display_cols if c in missing_posts.columns]
        print(missing_posts[available_cols].head(3).to_string(index=False))

        # Check for duplicates
        for pid in sample_missing[:3]:
            post_data = merged_raw[merged_raw['post_id'] == pid]
            duplicates = merged_raw[
                (merged_raw['ticker'] == post_data.iloc[0]['ticker']) &
                (merged_raw['timestamp'] == post_data.iloc[0]['timestamp'])
            ]
            if len(duplicates) > 1:
                print(f"\n   Post {pid}: Found {len(duplicates)} posts with same ticker+timestamp (dedup victim)")

print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)

loss_rate = len(missing_ids) / len(mapped) * 100

if loss_rate < 5:
    print(f" Loss rate {loss_rate:.1f}% is acceptable (< 5%)")
    print("   Current train_seed_updated_new.csv is ready to use")
elif loss_rate < 15:
    print(f" Loss rate {loss_rate:.1f}% is moderate (5-15%)")
    print("   Options:")
    print("   1. Use current dataset (acceptable for most use cases)")
    print("   2. Review unmapped posts and try manual remapping")
else:
    print(f" Loss rate {loss_rate:.1f}% is high (> 15%)")
    print("   Recommended actions:")
    print("   1. Review train_seed_unmapped_posts.csv (548 posts)")
    print("   2. Check if missing posts are duplicates")
    print("   3. Consider relaxing deduplication rules in Phase 4.3")


/tmp/ipython-input-1701024807.py:10: DtypeWarning: Columns (37,38,39,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_raw = pd.read_csv(os.path.join(BASE_DIR, 'ML_Project_Data/Combined/merged_raw_posts.csv'))


MISSING POSTS ANALYSIS

📊 Train seed mapping:
   Total train_seed posts: 5,610
   Successfully mapped: 5,062
   Found in enriched dataset: 4,180

⚠️ Missing posts breakdown:
   Mapped but not in enriched dataset: 882

🔍 Where did they go?
   Still in merged_raw_posts.csv: 882
   Removed during deduplication (Phase 4.3): 0

📋 Sample of missing posts still in merged_raw:
 post_id ticker                 timestamp
    2049    EWU 2025-08-28 21:06:41+00:00
   12288    EWJ 2025-09-04 05:29:41+00:00
   75777   GPRO 2025-10-12 15:01:38+00:00

   Post 77824.0: Found 2 posts with same ticker+timestamp (dedup victim)

   Post 2049.0: Found 5 posts with same ticker+timestamp (dedup victim)

   Post 12288.0: Found 3 posts with same ticker+timestamp (dedup victim)

RECOMMENDATION
❌ Loss rate 17.4% is high (> 15%)
   Recommended actions:
   1. Review train_seed_unmapped_posts.csv (548 posts)
   2. Check if missing posts are duplicates
   3. Consider relaxing deduplication rules in Phase 4.3


# **Threshold and score calculation**

** 0,1 and 0,0**

In [ ]:
!pip install pyarrow  # ensure parquet support if needed

import pandas as pd
import numpy as np
import os
import logging
from scipy.stats import zscore

# --- 1. CONFIGURATION ---
BASE_DIR = '/content/drive/MyDrive/'
INPUT_ENGINEERED_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/consolidated_final_features.parquet')
OUTPUT_DIR = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets/')

LABELING_PLAN = {
    'pool_1_1_both': {
        'label': '[1,1]', 'sort_by': 'SA_SH_sum', 'ascending': False,
        'top_n': 700, 'bottom_n': 300
    },
    'pool_1_0_availability_only': {
        'label': '[1,0]', 'sort_by': 'SA_score', 'ascending': False,
        'top_n': 1000, 'bottom_n': 500
    },
    'pool_0_1_herd_only': {
        'label': '[0,1]', 'sort_by': 'SH_score', 'ascending': False,
        'top_n': 700, 'bottom_n': 300
    },
    'pool_0_0_control': {
        'label': '[0,0]', 'sort_by': 'SA_SH_sum', 'ascending': True,
        'top_n': 500, 'bottom_n': 1000
    }
}
GOLD_SET_FRACTION = 0.15
LABEL_BUFFER = 200
RANDOM_STATE = 42

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')


def create_labeling_and_unlabeled_sets(df: pd.DataFrame, plan: dict, output_dir: str):
    logging.info("Starting dataset partitioning process...")
    os.makedirs(output_dir, exist_ok=True)

    # --- 1) Compute z-scored numeric columns (create deterministic _z columns) ---
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if 'flag' not in col and 'emo' not in col and not col.endswith('_z')]
    logging.info(f"Z-scoring {len(numeric_cols)} numeric columns.")
    for col in numeric_cols:
        df[f'{col}_z'] = zscore(df[col].fillna(0))

    # --- 2) Build SA/SH/combined scores (ensure referenced z-cols exist) ---
    # availability_features and herd_features reference *_z names
    availability_features = ['recency_keyword_flag', 'headline_similarity_score_z', 'news_sentiment_score_z']
    # if recency_keyword_flag is a flag (0/1) it's fine; ensure time_offset_from_event_z is present
    if 'time_offset_from_event_z' not in df.columns and 'time_offset_from_event' in df.columns:
        df['time_offset_from_event_z'] = zscore(df['time_offset_from_event'].fillna(0))

    herd_features = ['sentiment_homogeneity_z', 'social_proof_velocity_z', 'reply_chain_length_z', 'comment_reply_count_z', 'crowd_keyword_count_z']

    # defensive: ensure missing z cols exist (fill with 0)
    for c in availability_features + herd_features:
        if c not in df.columns:
            logging.warning(f"Missing column {c}; creating zero column to keep pipeline running.")
            df[c] = 0.0

    df['SA_score'] = df[['recency_keyword_flag']].fillna(0)['recency_keyword_flag'] + df['headline_similarity_score_z'].fillna(0) + df['news_sentiment_score_z'].fillna(0) - df['time_offset_from_event_z'].fillna(0)
    df['SH_score'] = df[['sentiment_homogeneity_z', 'social_proof_velocity_z', 'reply_chain_length_z', 'comment_reply_count_z', 'crowd_keyword_count_z']].fillna(0).sum(axis=1)
    df['SA_SH_sum'] = df['SA_score'] + df['SH_score']

    SA_threshold = df['SA_score'].quantile(0.75)
    SH_threshold = df['SH_score'].quantile(0.75)
    logging.info(f"SA threshold: {SA_threshold:.4f}, SH threshold: {SH_threshold:.4f}")

    # --- 3) define pool conditions ---
    conditions = {
        'pool_1_1_both': (df['SA_score'] >= SA_threshold) & (df['SH_score'] >= SH_threshold),
        'pool_1_0_availability_only': (df['SA_score'] >= SA_threshold) & (df['SH_score'] < SH_threshold),
        'pool_0_1_herd_only': (df['SA_score'] < SA_threshold) & (df['SH_score'] >= SH_threshold),
        'pool_0_0_control': (df['SA_score'] < SA_threshold) & (df['SH_score'] < SH_threshold)
    }

    all_labeling_indices = []
    gold_set_dfs, train_seed_dfs = [], []

    for name, cfg in plan.items():
        logging.info(f"Processing pool: {name}")
        idx_mask = conditions[name]
        pool_df = df[idx_mask].copy()
        if pool_df.empty:
            logging.warning(f"Pool {name} is empty; skipping.")
            continue

        # sort by requested key; if key missing, default to SA_SH_sum
        sort_key = cfg.get('sort_by', 'SA_SH_sum')
        if sort_key not in pool_df.columns:
            logging.warning(f"Sort key {sort_key} not present in pool {name}; defaulting to SA_SH_sum.")
            sort_key = 'SA_SH_sum'
        pool_df = pool_df.sort_values(by=sort_key, ascending=cfg.get('ascending', False))

        # Top + bottom slices (with buffer); if pool smaller than requested, head/tail gracefully give smaller
        top_slice = pool_df.head(cfg['top_n'] + LABEL_BUFFER)
        bottom_slice = pool_df.tail(cfg['bottom_n'] + LABEL_BUFFER)

        labeling_candidates = pd.concat([top_slice, bottom_slice]).drop_duplicates()
        if labeling_candidates.empty:
            logging.warning(f"No labeling candidates for pool {name} after concatenation; skipping.")
            continue

        # Tag pool name so we can assign predicted_label reliably later
        labeling_candidates = labeling_candidates.copy()
        labeling_candidates['pool'] = name

        # Remember indices for exclusion from unlabeled pool
        all_labeling_indices.extend(labeling_candidates.index.tolist())

        # Split into gold and seed
        gold_sample = labeling_candidates.sample(frac=GOLD_SET_FRACTION, random_state=RANDOM_STATE)
        seed_sample = labeling_candidates.drop(gold_sample.index)

        gold_set_dfs.append(gold_sample)
        train_seed_dfs.append(seed_sample)

        logging.info(f"Pool {name}: candidates={len(labeling_candidates)}, gold={len(gold_sample)}, seed={len(seed_sample)}")

    # --- 4) assemble final sets ---
    if gold_set_dfs:
        gold_set_df = pd.concat(gold_set_dfs, axis=0).reset_index(drop=True)
    else:
        gold_set_df = pd.DataFrame(columns=df.columns.tolist() + ['pool'])

    if train_seed_dfs:
        train_seed_df = pd.concat(train_seed_dfs, axis=0).reset_index(drop=True)
    else:
        train_seed_df = pd.DataFrame(columns=df.columns.tolist() + ['pool'])

    unlabeled_pool_df = df.loc[~df.index.isin(all_labeling_indices)].copy()
    logging.info(f"Final dataset sizes => gold_set: {len(gold_set_df)}, train_seed: {len(train_seed_df)}, unlabeled_pool: {len(unlabeled_pool_df)}")

    # --- 5) Add predicted_label via pool -> label mapping, drop heavy embedding cols for CSVs ---
    label_map = {name: cfg['label'] for name, cfg in plan.items()}
    for out_df in [gold_set_df, train_seed_df]:
        if out_df.empty:
            continue
        out_df['predicted_label'] = out_df['pool'].map(label_map).fillna('unclassified')

    # Drop embeddings from CSVs to minimize size and speed up human labeling
    emo_emb_cols = [c for c in df.columns if c.startswith('emo_emb_')]
    csv_drop_cols = emo_emb_cols  # can add more if needed

    gold_csv_path = os.path.join(output_dir, 'gold_set.csv')
    seed_csv_path = os.path.join(output_dir, 'train_seed.csv')
    unlabeled_parquet_path = os.path.join(output_dir, 'unlabeled_pool.parquet')

    # Save CSVs (if empty will create empty files with headers)
    gold_set_df.drop(columns=csv_drop_cols, errors='ignore').to_csv(gold_csv_path, index=False, encoding='utf-8')
    train_seed_df.drop(columns=csv_drop_cols, errors='ignore').to_csv(seed_csv_path, index=False, encoding='utf-8')

    # Save unlabeled full feature pool as parquet (keeps embeddings & types)
    unlabeled_pool_df.to_parquet(unlabeled_parquet_path, index=False, engine='pyarrow', compression='snappy')

    logging.info("Saved gold_set.csv, train_seed.csv, and unlabeled_pool.parquet.")
    logging.info(f"gold_set: {gold_csv_path}, train_seed: {seed_csv_path}, unlabeled_pool: {unlabeled_parquet_path}")


if __name__ == '__main__':
    try:
        logging.info(f"Loading engineered dataset from: {INPUT_ENGINEERED_PATH}")
        engineered_data = pd.read_parquet(INPUT_ENGINEERED_PATH, engine='pyarrow')
        logging.info(f"Loaded dataset: {len(engineered_data)} rows.")
        create_labeling_and_unlabeled_sets(engineered_data, LABELING_PLAN, OUTPUT_DIR)
    except FileNotFoundError:
        logging.error(f"Input not found: {INPUT_ENGINEERED_PATH}")
    except Exception as e:
        logging.exception("Unhandled error in partitioning script.")


**For 1,0 and 1,1**

In [ ]:
!pip install pyarrow --quiet

import pandas as pd
import numpy as np
import os
import logging
from scipy.stats import zscore

# --- CONFIGURATION ---
BASE_DIR = '/content/drive/MyDrive/'
INPUT_ENGINEERED_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/posts_with_all_features_normalized_new.parquet')
TRAIN_SEED_PATH = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets/train_seed_updated_new.csv')  #  Existing train_seed
OUTPUT_DIR = os.path.join(BASE_DIR, 'ML_Project_Data/Combined/Labeling_Sets/')

#  STRICT FILTERING
HEADLINE_SIMILARITY_MIN = 0.65  # Hard cutoff
EXCLUDE_TRAIN_SEED = True       # Prevent duplicates

# Aggressive for 1% classes
LABELING_PLAN = {
    'pool_1_1_both': {
        'label': '[1,1]',
        'sort_by': 'SA_SH_sum',
        'ascending': False,
        'top_n': 200,
        'bottom_n': 50
    },
    'pool_1_0_availability_only': {
        'label': '[1,0]',
        'sort_by': 'SA_score',
        'ascending': False,
        'top_n': 200,
        'bottom_n': 50
    }
}

LABEL_BUFFER = 25
RANDOM_STATE = 42

# Weights
HEADLINE_SIMILARITY_WEIGHT = 9.0
RECENCY_WEIGHT = 0.5
SENTIMENT_WEIGHT = 0.5

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s', force=True)

def create_extra_train_set(df: pd.DataFrame, plan: dict, output_dir: str,
                           train_seed_path: str = None, min_headline_sim: float = 0.65):
    """
    Create labeling set with strict filters:
    - headline_similarity_score > min_headline_sim
    - Exclude posts already in train_seed
    """
    logging.info("=" * 60)
    logging.info("STRICT LABELING SET CREATION")
    logging.info("=" * 60)

    os.makedirs(output_dir, exist_ok=True)

    # ==================== STEP 1: EXCLUDE TRAIN SEED ====================

    if train_seed_path and os.path.exists(train_seed_path):
        logging.info(f"\n Loading existing train_seed to exclude...")
        train_seed = pd.read_csv(train_seed_path, low_memory=False)

        # Get post IDs from train_seed
        train_seed_ids = set(train_seed['post_id'].dropna().astype(str))

        # Also check old_post_id if exists (mapped IDs)
        if 'old_post_id' in train_seed.columns:
            train_seed_ids.update(train_seed['old_post_id'].dropna().astype(str))

        logging.info(f"   Found {len(train_seed_ids):,} existing labeled post IDs")

        # Exclude from dataset
        before_count = len(df)
        df['post_id_str'] = df['post_id'].astype(str)
        df = df[~df['post_id_str'].isin(train_seed_ids)].copy()
        excluded_count = before_count - len(df)

        logging.info(f"    Excluded {excluded_count:,} posts already in train_seed")
        logging.info(f"   Remaining: {len(df):,} posts")
    else:
        logging.warning(f"    Train seed not found at {train_seed_path}")

    # ==================== STEP 2: APPLY HEADLINE SIMILARITY FILTER ====================

    logging.info(f"\n Applying headline_similarity_score > {min_headline_sim}")

    if 'headline_similarity_score' not in df.columns:
        logging.error(" headline_similarity_score column missing!")
        return

    before_count = len(df)
    df = df[df['headline_similarity_score'] > min_headline_sim].copy()
    filtered_count = before_count - len(df)

    logging.info(f"    Filtered out {filtered_count:,} posts with low similarity")
    logging.info(f"   Remaining: {len(df):,} posts")

    if df.empty:
        logging.error(" No posts remain after filtering! Consider lowering min_headline_sim")
        return

    # ==================== STEP 3: COMPUTE SCORES ====================

    logging.info(f"\n Computing SA/SH scores...")

    # Check if already normalized
    has_z_features = any(col.endswith('_z') for col in df.columns)

    if not has_z_features:
        logging.info("   Creating z-score features...")
        numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                        if 'flag' not in col and 'emo' not in col]
        for col in numeric_cols:
            df[f'{col}_z'] = zscore(df[col].fillna(0))

    # Time offset
    if 'time_offset_from_event' in df.columns:
        df['abs_time_offset'] = df['time_offset_from_event'].abs()
        if 'abs_time_offset_z' not in df.columns:
            df['abs_time_offset_z'] = zscore(df['abs_time_offset'].fillna(0))
    else:
        df['abs_time_offset_z'] = 0.0

    # Ensure features exist
    availability_features = ['recency_keyword_flag', 'headline_similarity_score_z', 'news_sentiment_score_z']
    herd_features = ['sentiment_homogeneity_z', 'social_proof_velocity_z', 'reply_chain_length_z',
                     'comment_reply_count_z', 'crowd_keyword_count_z']

    for c in availability_features + herd_features:
        if c not in df.columns:
            df[c] = 0.0

    # Compute SA/SH scores
    df['SA_score'] = (
        RECENCY_WEIGHT * df['recency_keyword_flag'].fillna(0) +
        HEADLINE_SIMILARITY_WEIGHT * df['headline_similarity_score_z'].fillna(0) +
        SENTIMENT_WEIGHT * df['news_sentiment_score_z'].fillna(0) -
        df['abs_time_offset_z'].fillna(0)
    )
    df['SH_score'] = df[herd_features].fillna(0).sum(axis=1)
    df['SA_SH_sum'] = df['SA_score'] + df['SH_score']

    # ==================== STEP 4: DEFINE POOLS ====================

    SA_threshold = df['SA_score'].quantile(0.75)
    SH_threshold = df['SH_score'].quantile(0.75)

    logging.info(f"\n Thresholds (75th percentile):")
    logging.info(f"   SA: {SA_threshold:.4f}")
    logging.info(f"   SH: {SH_threshold:.4f}")

    conditions = {
        'pool_1_1_both': (df['SA_score'] >= SA_threshold) & (df['SH_score'] >= SH_threshold),
        'pool_1_0_availability_only': (df['SA_score'] >= SA_threshold) & (df['SH_score'] < SH_threshold)
    }

    logging.info(f"\n Candidate pools (after all filters):")
    for name, cond in conditions.items():
        pool_size = cond.sum()
        pct = (pool_size / len(df)) * 100 if len(df) > 0 else 0
        logging.info(f"   {name}: {pool_size:,} ({pct:.2f}%)")

    # ==================== STEP 5: SELECT SAMPLES ====================

    train_extra_dfs = []

    for name, cfg in plan.items():
        logging.info(f"\n{'='*60}")
        logging.info(f"Processing: {name}")
        logging.info(f"{'='*60}")

        base_pool = df[conditions[name]].copy()

        if base_pool.empty:
            logging.warning(f"    Pool empty, skipping")
            continue

        sort_key = cfg.get('sort_by', 'SA_SH_sum')
        ascending = cfg.get('ascending', False)

        # TOP samples
        top_n = cfg.get('top_n', 200) + LABEL_BUFFER
        top_samples = base_pool.sort_values(by=sort_key, ascending=ascending).head(top_n)

        logging.info(f"   Selected {len(top_samples)} TOP samples")

        # BOTTOM samples (boundary cases)
        bottom_n = cfg.get('bottom_n', 50) + LABEL_BUFFER
        remaining_pool = base_pool.drop(index=top_samples.index, errors='ignore')
        bottom_samples = remaining_pool.sort_values(by=sort_key, ascending=ascending).tail(bottom_n)

        logging.info(f"   Selected {len(bottom_samples)} BOTTOM samples")

        # Combine
        combined = pd.concat([top_samples, bottom_samples]).drop_duplicates()
        combined['pool'] = name
        train_extra_dfs.append(combined)

        # Stats
        hs = combined['headline_similarity_score']
        logging.info(f"   Total: {len(combined)} | hs: min={hs.min():.4f}, mean={hs.mean():.4f}, max={hs.max():.4f}")

    # ==================== STEP 6: SAVE ====================

    if not train_extra_dfs:
        logging.error(" No samples selected from any pool!")
        return

    train_extra_df = pd.concat(train_extra_dfs, axis=0).reset_index(drop=True)

    # Add labels
    label_map = {name: cfg['label'] for name, cfg in plan.items()}
    train_extra_df['predicted_label'] = train_extra_df['pool'].map(label_map)

    # Drop embedding columns (save space)
    emo_emb_cols = [c for c in train_extra_df.columns if c.startswith('emo_emb_')]
    train_extra_df = train_extra_df.drop(columns=emo_emb_cols, errors='ignore')

    # Save
    out_path = os.path.join(output_dir, 'train_set_extra_strict.csv')
    train_extra_df.to_csv(out_path, index=False, encoding='utf-8')

    # ==================== STEP 7: FINAL SUMMARY ====================

    logging.info(f"\n{'='*60}")
    logging.info("FINAL SUMMARY")
    logging.info(f"{'='*60}")

    logging.info(f"\n Total selected: {len(train_extra_df):,}")
    logging.info(f"\n By pool:")
    for pool in train_extra_df['pool'].unique():
        count = (train_extra_df['pool'] == pool).sum()
        label = label_map.get(pool, 'unknown')
        logging.info(f"   {pool} ({label}): {count:,}")

    hs = train_extra_df['headline_similarity_score']
    logging.info(f"\n Headline similarity stats:")
    logging.info(f"   Min: {hs.min():.4f}")
    logging.info(f"   Mean: {hs.mean():.4f}")
    logging.info(f"   Median: {hs.median():.4f}")
    logging.info(f"   Max: {hs.max():.4f}")
    logging.info(f"   ALL > {min_headline_sim}: {'✅ YES' if hs.min() > min_headline_sim else '❌ NO'}")

    logging.info(f"\n Saved to: {out_path}")

    # Verify no train_seed overlap
    if train_seed_path and os.path.exists(train_seed_path):
        overlap = set(train_extra_df['post_id'].astype(str)) & train_seed_ids
        if overlap:
            logging.error(f"    WARNING: {len(overlap)} posts overlap with train_seed!")
        else:
            logging.info(f"    No overlap with existing train_seed")

if __name__ == '__main__':
    try:
        logging.info(f"Loading: {INPUT_ENGINEERED_PATH}")
        df_in = pd.read_parquet(INPUT_ENGINEERED_PATH, engine='pyarrow')
        logging.info(f"Loaded: {len(df_in):,} rows")

        create_extra_train_set(
            df_in,
            LABELING_PLAN,
            OUTPUT_DIR,
            train_seed_path=TRAIN_SEED_PATH,
            min_headline_sim=HEADLINE_SIMILARITY_MIN
        )
    except Exception as e:
        logging.exception("Error occurred")


2025-10-27 19:24:56,530 - INFO: Loading: /content/drive/MyDrive/ML_Project_Data/Combined/posts_with_all_features_normalized_new.parquet
2025-10-27 19:24:57,663 - INFO: Loaded: 80,002 rows
2025-10-27 19:24:57,666 - INFO: ============================================================
2025-10-27 19:24:57,667 - INFO: STRICT LABELING SET CREATION
2025-10-27 19:24:57,668 - INFO: ============================================================
2025-10-27 19:24:57,677 - INFO: 
📂 Loading existing train_seed to exclude...
2025-10-27 19:24:58,363 - INFO:    Found 8,188 existing labeled post IDs
2025-10-27 19:24:58,673 - INFO:    ✅ Excluded 7,294 posts already in train_seed
2025-10-27 19:24:58,674 - INFO:    Remaining: 72,708 posts
2025-10-27 19:24:58,677 - INFO: 
🔍 Applying headline_similarity_score > 0.65
2025-10-27 19:24:58,692 - INFO:    ✅ Filtered out 70,919 posts with low similarity
2025-10-27 19:24:58,693 - INFO:    Remaining: 1,789 posts
2025-10-27 19:24:58,695 - INFO: 
🔧 Computing SA/SH scores.

# Dimensionality Reduction

**Correct and merge the datasets**

In [ ]:
# ============================================================
# MANUAL LABELLED FILES → MAP TO NEW POST IDs → ATTACH ALL FEATURES (NO NaNs)
# - Normalizes IDs consistently
# - Strict INNER joins: mapped ids only + present in features parquet only
# - Preserves manual label fields; replaces overlapping raw fields with feature-store values
# - Saves unmatched IDs for audit per file
# - Special robust reader for rahul_final.csv with header/data gap handling
# ============================================================

import pandas as pd
import numpy as np
import re
import io
import os
import sys
import logging
import time
import gc

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s: %(message)s")

# ---------------- Paths ----------------
BASE_DIR    = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
COMBINED_DIR= os.path.join(BASE_DIR, "Combined")
MANUAL_DIR  = os.path.join(BASE_DIR, "Manual_labelled_data")

# Allow override via environment for portability
MAPPING_PATH  = os.environ.get("POST_ID_MAPPING_PATH", os.path.join(COMBINED_DIR, "post_id_mapping.csv"))
FEATURES_PATH = os.environ.get("FEATURES_PARQUET_PATH", os.path.join(COMBINED_DIR, "posts_with_all_features_normalized_new.parquet"))

# Files to process: (filename, filetype, idtype)
# idtype: 'old_id' -> needs mapping; 'new_id' -> already new
FILES = [
    ('pritika_final.xlsx',           'excel', 'old_id'),
    ('rahul_final.csv',              'csv',   'old_id'),  # special loader handles header/data gap
    ('varsha_final.xlsx',            'excel', 'old_id'),
    ('New_data/Harini_final.xlsx',   'excel', 'new_id'),
]

# Manual columns to preserve from the original files (if present)
PRESERVE_FROM_MANUAL = [
    'human_label','predicted_label','SA_score','SH_score','SA_SH_sum','pool',
    'annotator','annotation_date','notes','review_status','quality_flag'
]

# ---------------- Utils ----------------

def norm_id(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = re.sub(r'\.0$', '', s)  # strip Excel-int as float artifacts
    return s

def canon_colname(c):
    return str(c).strip()

def canon_lower(c):
    return str(c).strip().lower()

def is_delim_only_line(ln: str) -> bool:
    # Treat lines that have only separators/quotes/spaces/tabs as empty
    return bool(re.fullmatch(r"[,\t;|\s\"']*", ln))

def decode_bytes_with_fallback(raw: bytes) -> str:
    # Try UTF-8 with BOM handling first, then fall back
    encodings = ['utf-8-sig', 'utf-8', 'latin-1', 'utf-16']
    for enc in encodings:
        try:
            return raw.decode(enc, errors='replace')
        except Exception:
            continue
    # Last resort
    return raw.decode('utf-8', errors='replace')

def rebuild_text_without_gaps(text: str, header_regex: re.Pattern = None) -> str:
    """
    Remove empty/delimiter-only lines and if header_regex provided, ensure the header
    line is used from the first occurrence of that pattern.
    """
    lines = text.splitlines()
    # Drop empty/delimiter-only lines
    filtered = [ln for ln in lines if ln is not None and not is_delim_only_line(ln)]
    if not filtered:
        return ""

    if header_regex is None:
        # Use the first remaining non-empty line as header
        header_idx = 0
    else:
        header_idx = None
        for i, ln in enumerate(filtered):
            if header_regex.search(ln):
                header_idx = i
                break
        if header_idx is None:
            # Fallback to first line if header not found
            header_idx = 0

    # Build from header to end
    return "\n".join(filtered[header_idx:])

def try_parse_with_detectors(cleaned_text: str) -> pd.DataFrame:
    """
    Try pandas with sep inference, then iterate over common seps.
    """
    if not cleaned_text.strip():
        raise ValueError("No non-empty content after cleaning")

    # 1) Let pandas sniff
    try:
        df = pd.read_csv(io.StringIO(cleaned_text), sep=None, engine='python')
        if len(df.columns) > 1 or len(df) > 0:
            return df
    except Exception:
        pass

    # 2) Try common separators
    for sep in [',', '\t', ';', '|']:
        try:
            df = pd.read_csv(io.StringIO(cleaned_text), sep=sep)
            if len(df.columns) > 1 or len(df) > 0:
                return df
        except Exception:
            continue

    # 3) Last attempt with on_bad_lines
    return pd.read_csv(io.StringIO(cleaned_text), on_bad_lines='skip')

def read_csv_robust_generic(path: str) -> pd.DataFrame:
    """
    Generic robust CSV reader that:
      - Decodes with BOM-aware fallback
      - Drops blank and delimiter-only lines
      - Uses first non-empty line as header
    """
    with open(path, 'rb') as f:
        raw = f.read()
    text = decode_bytes_with_fallback(raw)

    cleaned_text = rebuild_text_without_gaps(text)
    if not cleaned_text.strip():
        raise ValueError(f"No parseable lines in {path}")

    df = try_parse_with_detectors(cleaned_text)
    return df

def read_csv_rahul_special(path: str) -> pd.DataFrame:
    """
    Special loader for rahul_final.csv:
      - Removes large gaps between header and data
      - Explicitly finds header row by searching for a 'post_id' token (case-insensitive)
      - Drops delimiter-only spacer lines throughout the file
    """
    with open(path, 'rb') as f:
        raw = f.read()
    text = decode_bytes_with_fallback(raw)

    # Header detection: any token that matches post[_ ]?id (case-insensitive)
    header_re = re.compile(r'(?i)\bpost[\s_]*id\b')
    cleaned_text = rebuild_text_without_gaps(text, header_regex=header_re)
    if not cleaned_text.strip():
        # Fallback to generic if special cleaning yields nothing
        logging.warning("rahul_final.csv special clean produced empty text; falling back to generic reader")
        return read_csv_robust_generic(path)

    df = try_parse_with_detectors(cleaned_text)

    # If 'post_id' not present but a variant exists, rename it
    lowered = {c.lower(): c for c in df.columns}
    if 'post_id' not in lowered:
        # Search for variants
        for lc, orig in lowered.items():
            if re.fullmatch(r'post[\s_]*id', lc):
                df = df.rename(columns={orig: 'post_id'})
                break
    return df

def read_excel_robust(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, engine='openpyxl')
    return df

def load_manual_file(fname: str, ftype: str, manual_dir: str) -> pd.DataFrame:
    fpath = os.path.join(manual_dir, fname)
    base = os.path.basename(fname).lower()
    if ftype == 'excel':
        return read_excel_robust(fpath)
    # CSV
    if base == 'rahul_final.csv':
        logging.info("Using special CSV reader for rahul_final.csv")
        return read_csv_rahul_special(fpath)
    return read_csv_robust_generic(fpath)

def ensure_post_id_column(df: pd.DataFrame) -> pd.DataFrame:
    cols = [canon_colname(c) for c in df.columns]
    df.columns = cols

    # Fast-path
    if 'post_id' in df.columns:
        return df

    # Case-insensitive/variant search
    lower_map = {c.lower(): c for c in df.columns}
    if 'post_id' in lower_map:
        df = df.rename(columns={lower_map['post_id']: 'post_id'})
        return df

    for c in df.columns:
        if re.fullmatch(r'(?i)post[\s_]*id', c):
            df = df.rename(columns={c: 'post_id'})
            return df

    raise KeyError("No 'post_id' column found (case-insensitive variants tried)")

# ---------------- Load shared data ----------------

print("="*70)
print("LOADING SHARED RESOURCES")
print("="*70)

# 1) Mapping (normalize keys)
print("\n Loading post_id_mapping.csv ...")
mapping_df = pd.read_csv(MAPPING_PATH)
mapping_df.columns = [canon_lower(c) for c in mapping_df.columns]
# Flexible renaming
rename_map = {}
if 'oldpostid' in mapping_df.columns: rename_map['oldpostid'] = 'old_post_id'
if 'newpostid' in mapping_df.columns: rename_map['newpostid'] = 'new_post_id'
if 'old_post_id' not in mapping_df.columns:
    # Try variants
    for c in mapping_df.columns:
        if re.fullmatch(r'old[\s_]*post[\s_]*id', c):
            rename_map[c] = 'old_post_id'
            break
if 'new_post_id' not in mapping_df.columns:
    for c in mapping_df.columns:
        if re.fullmatch(r'new[\s_]*post[\s_]*id', c):
            rename_map[c] = 'new_post_id'
            break
if rename_map:
    mapping_df = mapping_df.rename(columns=rename_map)

required_map_cols = {'old_post_id','new_post_id'}
missing = required_map_cols - set(mapping_df.columns)
if missing:
    raise KeyError(f"Mapping file missing columns: {missing}")

mapping_df['old_key'] = mapping_df['old_post_id'].apply(norm_id)
mapping_df['new_key'] = mapping_df['new_post_id'].apply(norm_id)
mapping_df = (mapping_df
              .dropna(subset=['old_key','new_key'])
              .drop_duplicates(subset=['old_key'], keep='first')
              .reset_index(drop=True))
print(f"   ✓ {len(mapping_df):,} normalized mappings")

# 2) Features parquet (normalize key, one row per key)
print(" Loading features parquet ...")
features_df = pd.read_parquet(FEATURES_PATH)

if 'post_id' not in features_df.columns:
    raise KeyError("Features parquet missing required column 'post_id'")

features_df['post_id'] = features_df['post_id'].astype(str)
features_df['key'] = features_df['post_id'].apply(norm_id)
if features_df['key'].isna().any():
    nnull = int(features_df['key'].isna().sum())
    raise ValueError(f"Null keys in features_df after normalization: {nnull}")

# keep one per key (canonical)
dups = features_df['key'].duplicated().sum()
if dups:
    logging.warning(f"Features duplicate keys detected: {dups} (keeping first per key)")
features_df = (features_df
               .dropna(subset=['key'])
               .drop_duplicates(subset=['key'], keep='first')
               .reset_index(drop=True))
feature_keys = set(features_df['key'])
print(f"   ✓ {len(features_df):,} feature rows, {len(features_df.columns)} columns")

# ---------------- Processing function ----------------

def process_manual_file(fname, ftype, idtype):
    stem = fname.replace('New_data/','').rsplit('.',1)[0]
    fpath = os.path.join(MANUAL_DIR, fname)

    print("\n" + "="*70)
    print(f"PROCESSING: {fname}  [{ftype} | {idtype}]")
    print("="*70)

    # Load manual file (special handler for rahul_final.csv)
    df = load_manual_file(fname, ftype, MANUAL_DIR)
    # Canonicalize columns early
    df.columns = [canon_colname(c) for c in df.columns]
    df = df.dropna(how='all')
    df = ensure_post_id_column(df)

    orig = len(df)
    df['post_id'] = df['post_id'].astype(str)
    df['post_id_norm'] = df['post_id'].apply(norm_id)
    print(f"   ✓ Loaded {orig:,} rows")

    # Map to new_post_id
    if idtype == 'old_id':
        # Inner join to keep only mapped rows
        df_map = df.merge(mapping_df[['old_key','new_key']], left_on='post_id_norm', right_on='old_key', how='inner')
        df_map['new_post_id'] = df_map['new_key']
        df_map['old_post_id'] = df_map['post_id']  # keep original for traceability
        kept = len(df_map)
        print(f"   ✓ Mapped old→new: kept {kept:,}, dropped {orig - kept:,} unmapped")
        df = df_map
    else:
        # Already new ids
        df['new_post_id'] = df['post_id_norm']
        df['old_post_id'] = np.nan
        print(f"   ✓ Treated as new_id: {len(df):,} rows")

    # Normalize join key
    df['key'] = df['new_post_id'].apply(norm_id)

    # Drop duplicate keys (keep first)
    before_dups = len(df)
    df = df.dropna(subset=['key']).drop_duplicates(subset=['key'], keep='first').reset_index(drop=True)
    if len(df) != before_dups:
        print(f"   ✓ Deduped manual keys: removed {before_dups - len(df):,} duplicates")

    # Anti-join: which mapped keys are NOT present in features?
    unmatched = df[~df['key'].isin(feature_keys)].copy()
    matched   = df[df['key'].isin(feature_keys)].copy()
    print(f"   ✓ Matched to features: {len(matched):,}, unmatched: {len(unmatched):,}")

    # Strict INNER merge to attach ALL features (no NaNs)
    # Use suffixes so features columns win (manual overlap → '_man')
    df_final = matched.merge(features_df, on='key', how='inner', suffixes=('_man',''))

    # Column reconciliation:
    # - Keep/restore manual preserved columns (if present as *_man), tolerant to case drift
    final_cols_lower = {c.lower(): c for c in df_final.columns}
    for col in PRESERVE_FROM_MANUAL:
        man_col_candidates = [c for c in df_final.columns if c.lower() == f"{col.lower()}_man"]
        if man_col_candidates:
            man_col = man_col_candidates[0]
            df_final[col] = df_final[man_col]

    # - Drop other *_man columns (non-preserved), retain preserved originals
    drop_man = [c for c in df_final.columns if c.endswith('_man') and c[:-4].lower() not in {x.lower() for x in PRESERVE_FROM_MANUAL}]
    if drop_man:
        df_final = df_final.drop(columns=drop_man)

    # Canonical identifiers in output
    # post_id (from features) is the canonical new id; keep old_post_id and new_post_id for trace
    # Ensure consistent ordering of important columns
    front_cols = [c for c in [
        'post_id','new_post_id','old_post_id','timestamp','ticker','human_label','predicted_label','pool'
    ] if c in df_final.columns]
    other_cols = [c for c in df_final.columns if c not in front_cols and c != 'key']
    df_final = df_final[front_cols + other_cols]

    # Safety: ensure there are no NaNs inside feature columns due to merge
    # We cannot assert absolutely no NaNs in entire frame because manual-only fields may have NaNs.
    # But ensure features columns from features_df are intact.
    # Example guard: check a few sentinel columns if known; optional comprehensive check:
    # If any NaNs exist, they should be in preserved manual columns only.
    # Uncomment for strict check across all columns:
    # if df_final.isna().any().any():
    #     missing_cols = df_final.columns[df_final.isna().any()].tolist()
    #     logging.warning(f"NaNs present after merge (likely in manual fields): {missing_cols}")

    # Write outputs
    out_clean = os.path.join(MANUAL_DIR, f"{stem}_cleaned.csv")
    df_final.to_csv(out_clean, index=False)
    print(f"    SAVED: {os.path.basename(out_clean)}  ({len(df_final):,} rows × {len(df_final.columns)} cols)")

    if not unmatched.empty:
        out_unmatched = os.path.join(MANUAL_DIR, f"{stem}_unmatched_ids.csv")
        unmatched[['post_id','old_post_id','new_post_id','key']].to_csv(out_unmatched, index=False)
        print(f"    Unmatched saved: {os.path.basename(out_unmatched)}  ({len(unmatched):,} rows)")

    # Cleanup
    del df_final, matched, unmatched, df
    gc.collect()
    time.sleep(0.25)

# ---------------- Run all ----------------

SUCCESS, FAILED = [], []

try:
    os.makedirs(MANUAL_DIR, exist_ok=True)
except Exception:
    pass

for fname, ftype, idtype in FILES:
    try:
        process_manual_file(fname, ftype, idtype)
        SUCCESS.append(fname)
    except Exception as e:
        logging.exception(f"FAILED: {fname} :: {e}")
        FAILED.append(fname)

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f" SUCCESS: {len(SUCCESS)}")
for s in SUCCESS: print(f"   • {s}")
if FAILED:
    print(f" FAILED: {len(FAILED)}")
    for f in FAILED: print(f"   • {f}")
else:
    print(" ALL FILES SAVED SUCCESSFULLY")
print("="*70)

# ---------------- Optional: consolidate all cleaned outputs ----------------
try:
    outs = []
    for fname, _, _ in FILES:
        stem = fname.replace('New_data/','').rsplit('.',1)[0]
        out_clean = os.path.join(MANUAL_DIR, f"{stem}_cleaned.csv")
        if os.path.exists(out_clean):
            outs.append(pd.read_csv(out_clean))
    if outs:
        combined = pd.concat(outs, ignore_index=True)
        combined_out = os.path.join(MANUAL_DIR, "manual_cleaned_all.csv")
        combined.to_csv(combined_out, index=False)
        print(f"    SAVED: manual_cleaned_all.csv ({len(combined):,} rows × {len(combined.columns)} cols)")
except Exception as e:
    logging.warning(f"Consolidation step skipped due to error: {e}")


LOADING SHARED RESOURCES

📂 Loading post_id_mapping.csv ...
   ✓ 96,335 normalized mappings
📂 Loading features parquet ...
   ✓ 80,002 feature rows, 515 columns

PROCESSING: pritika_final.xlsx  [excel | old_id]
   ✓ Loaded 1,871 rows
   ✓ Mapped old→new: kept 1,779, dropped 92 unmapped
   ✓ Matched to features: 1,423, unmatched: 356
   ✅ SAVED: pritika_final_cleaned.csv  (1,423 rows × 525 cols)
   ⚠️ Unmatched saved: pritika_final_unmatched_ids.csv  (356 rows)

PROCESSING: rahul_final.csv  [csv | old_id]
   ✓ Loaded 1,870 rows
   ✓ Mapped old→new: kept 1,733, dropped 137 unmapped
   ✓ Matched to features: 1,490, unmatched: 243
   ✅ SAVED: rahul_final_cleaned.csv  (1,490 rows × 525 cols)
   ⚠️ Unmatched saved: rahul_final_unmatched_ids.csv  (243 rows)

PROCESSING: varsha_final.xlsx  [excel | old_id]
   ✓ Loaded 1,870 rows
   ✓ Mapped old→new: kept 1,551, dropped 319 unmapped
   ✓ Matched to features: 1,268, unmatched: 283
   ✅ SAVED: varsha_final_cleaned.csv  (1,268 rows × 525 cols)
   

**Check schema**

In [ ]:
import os
import re
import io
import glob
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s: %(message)s")

# --------- Paths ---------
BASE_DIR     = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
MANUAL_DIR   = os.path.join(BASE_DIR, "Manual_labelled_data")

# Files of interest (manual inputs)
FILES = [
    ('pritika_final.xlsx',           'excel'),
    ('rahul_final.csv',              'csv'),    # special handling for header/data gap
    ('varsha_final.xlsx',            'excel'),
    ('New_data/Harini_final.xlsx',   'excel'),
]

# --------- Helpers: robust CSV header detection (esp. for rahul_final.csv) ---------
def is_delim_only_line(ln: str) -> bool:
    return bool(re.fullmatch(r"[,\t;|\s\"']*", ln))

def decode_bytes_with_fallback(raw: bytes) -> str:
    for enc in ['utf-8-sig', 'utf-8', 'latin-1', 'utf-16']:
        try:
            return raw.decode(enc, errors='replace')
        except Exception:
            continue
    return raw.decode('utf-8', errors='replace')

def rebuild_text_without_gaps(text: str, header_regex: re.Pattern | None) -> str:
    lines = text.splitlines()
    filtered = [ln for ln in lines if ln and not is_delim_only_line(ln)]
    if not filtered:
        return ""
    if header_regex is None:
        header_idx = 0
    else:
        header_idx = None
        for i, ln in enumerate(filtered):
            if header_regex.search(ln):
                header_idx = i
                break
        if header_idx is None:
            header_idx = 0
    return "\n".join(filtered[header_idx:])

def read_columns_from_csv(path: str, special_rahul: bool = False) -> list[str]:
    with open(path, 'rb') as f:
        raw = f.read()
    text = decode_bytes_with_fallback(raw)
    header_re = re.compile(r'(?i)\bpost[\s_]*id\b') if special_rahul else None
    cleaned_text = rebuild_text_without_gaps(text, header_re)
    if not cleaned_text.strip():
        raise ValueError(f"No parseable header for {path}")
    # Try pandas sniff first with nrows=0 for header only
    try:
        df0 = pd.read_csv(io.StringIO(cleaned_text), sep=None, engine='python', nrows=0)
        if len(df0.columns) > 0:
            return list(df0.columns)
    except Exception:
        pass
    # Try common seps
    for sep in [',', '\t', ';', '|']:
        try:
            df0 = pd.read_csv(io.StringIO(cleaned_text), sep=sep, nrows=0)
            if len(df0.columns) > 0:
                return list(df0.columns)
        except Exception:
            continue
    # Last-resort read then return cols
    df = pd.read_csv(io.StringIO(cleaned_text))
    return list(df.columns)

def read_columns_from_excel(path: str) -> list[str]:
    # nrows=0 reads only header
    df0 = pd.read_excel(path, engine='openpyxl', nrows=0)
    return list(df0.columns)

# --------- Reporting ---------
def print_columns(title: str, cols: list[str]):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    print(f"Total columns: {len(cols)}")
    for c in cols:
        print(c)

def main():
    # 1) Manual input files
    for rel, ftype in FILES:
        fpath = os.path.join(MANUAL_DIR, rel)
        if not os.path.exists(fpath):
            logging.warning(f"Missing: {fpath}")
            continue
        try:
            if ftype == 'excel':
                cols = read_columns_from_excel(fpath)
            else:
                # Special handling only for rahul_final.csv
                special = (os.path.basename(rel).lower() == 'rahul_final.csv')
                cols = read_columns_from_csv(fpath, special_rahul=special)
            print_columns(f"INPUT SCHEMA: {rel}", cols)
        except Exception as e:
            logging.exception(f"Failed reading columns for {rel}: {e}")

    # 2) Any cleaned outputs in MANUAL_DIR
    cleaned_paths = sorted(glob.glob(os.path.join(MANUAL_DIR, "*_cleaned.csv")))
    for cpath in cleaned_paths:
        try:
            # Read header only
            cols = read_columns_from_csv(cpath, special_rahul=False)
            print_columns(f"CLEANED SCHEMA: {os.path.basename(cpath)}", cols)
        except Exception as e:
            logging.exception(f"Failed reading columns for cleaned file {cpath}: {e}")

if __name__ == "__main__":
    main()



INPUT SCHEMA: pritika_final.xlsx
Total columns: 96
post_id
timestamp
sentiment_score
emotional_valence
news_keyword_count
recency_keyword_flag
ner_mentions_count
time_offset_from_event
initial_engagement_velocity
media_richness_flag
post_length_words
post_length_chars
user_posting_frequency
reply_chain_length
jargon_prevalence
sentiment_homogeneity
crowd_keyword_count
text_similarity_top_posts
ticker_repetition_window
social_proof_velocity
upvote_like_ratio
comment_reply_count
user_network_influence
temporal_clustering_flag
headline_similarity_score
event_linkage_flag
trading_volume_spike_flag
news_sentiment_score
social_news_sentiment_delta
source_file
fed
ticker
intraday_return_30min
intraday_return_24hr
intraday_volatility
volume_spike_ratio
emo_val_code
emo_anger
emo_anticipation
emo_disgust
emo_fear
emo_joy
emo_negative
emo_neutral
emo_positive
emo_sadness
emo_surprise
emo_trust
emotional_news_delta
A_Emotional_News_Delta
H_Social_Momentum_Ratio
H_Jargon_Sentiment_Homogeneity
A_S

Checking labels

In [ ]:
import os
import io
import re
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s: %(message)s")

BASE_DIR   = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")
INPUT_PATH = os.path.join(MANUAL_DIR, "manual_cleaned_all.csv")
OUT_PATH   = os.path.join(MANUAL_DIR, "human_label_pair_counts.csv")

def decode_bytes_with_fallback(raw: bytes) -> str:
    for enc in ['utf-8-sig', 'utf-8', 'latin-1', 'utf-16']:
        try:
            return raw.decode(enc, errors='replace')
        except Exception:
            continue
    return raw.decode('utf-8', errors='replace')

def load_csv_robust(path: str) -> pd.DataFrame:
    with open(path, 'rb') as f:
        raw = f.read()
    text = decode_bytes_with_fallback(raw)
    # Prefer pandas sniffing first; fall back to common separators
    try:
        return pd.read_csv(io.StringIO(text), sep=None, engine='python')
    except Exception:
        for sep in [',', '\t', ';', '|']:
            try:
                return pd.read_csv(io.StringIO(text), sep=sep)
            except Exception:
                continue
    # Last resort
    return pd.read_csv(io.StringIO(text), engine='python', on_bad_lines='skip')

def to01(token):
    if token is None:
        return None
    s = str(token).strip().lower()
    if s in {'1', 'true', 't', 'y', 'yes'}:
        return 1
    if s in {'0', 'false', 'f', 'n', 'no'}:
        return 0
    # Try numeric
    try:
        v = float(s)
        if v == 1.0:
            return 1
        if v == 0.0:
            return 0
    except Exception:
        return None
    return None

def parse_pair(val):
    # Already a list/tuple-like?
    if isinstance(val, (list, tuple)):
        if len(val) == 2:
            a, b = to01(val[0]), to01(val[1])
            return (a, b) if a in (0,1) and b in (0,1) else None
        return None
    # String-like patterns: "[0,1]", "0,1", "0|1", "0 1", "(0, 1)"
    s = str(val)
    # Remove surrounding brackets/parentheses
    s = re.sub(r'^[\[\(\s]*', '', s)
    s = re.sub(r'[\]\)\s]*$', '', s)
    # Normalize separators to comma
    s = re.sub(r'[|/;\s]+', ',', s)
    parts = [p for p in s.split(',') if p != '']
    if len(parts) != 2:
        return None
    a, b = to01(parts[0]), to01(parts[1])
    return (a, b) if a in (0,1) and b in (0,1) else None

def main():
    if not os.path.exists(INPUT_PATH):
        raise FileNotFoundError(f"File not found: {INPUT_PATH}")
    df = load_csv_robust(INPUT_PATH)

    # Resolve label column: prefer 'human_label', fallback to 'human_labeling'
    lower_map = {c.lower(): c for c in df.columns}
    label_col = None
    for cand in ('human_label', 'human_labeling'):
        if cand in lower_map:
            label_col = lower_map[cand]
            break
    if label_col is None:
        raise KeyError("Neither 'human_label' nor 'human_labeling' column found")

    # Parse into pairs
    pairs = df[label_col].apply(parse_pair)

    # Compute counts for the four canonical buckets
    buckets = [(0,0), (0,1), (1,0), (1,1)]
    counts = {str(b): 0 for b in buckets}
    for t in pairs.dropna():
        if t in buckets:
            counts[str(t)] += 1

    # Print nicely
    print("human_label pair counts (in order [0,0], [0,1], [1,0], [1,1]):")
    ordered_keys = [str(b) for b in buckets]
    for k in ordered_keys:
        print(f"{k}: {counts[k]}")

    # Optional: save to CSV
    out_df = pd.DataFrame({
        'pair': ordered_keys,
        'count': [counts[k] for k in ordered_keys]
    })
    out_df.to_csv(OUT_PATH, index=False)
    print(f"Saved counts to: {OUT_PATH}")

if __name__ == "__main__":
    main()


human_label pair counts (in order [0,0], [0,1], [1,0], [1,1]):
(0, 0): 1595
(0, 1): 1288
(1, 0): 202
(1, 1): 200
Saved counts to: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/human_label_pair_counts.csv


Sanity Check

In [ ]:
import os
import pandas as pd

# Paths (adjust if different)
BASE_DIR     = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
COMBINED_DIR = os.path.join(BASE_DIR, "Combined")
MANUAL_DIR   = os.path.join(BASE_DIR, "Manual_labelled_data")

FEATURES_PATH = os.path.join(COMBINED_DIR, "posts_with_all_features_normalized_new.parquet")
LABELED_PATH  = os.path.join(MANUAL_DIR, "manual_cleaned_all.csv")

# 1) Load data
feat_df = pd.read_parquet(FEATURES_PATH)              # Parquet -> DataFrame [features] [web:111]
lab_df  = pd.read_csv(LABELED_PATH, low_memory=False) # CSV -> DataFrame [manual_cleaned] [web:126]

# 2) Column lists
parquet_cols_all = feat_df.columns.tolist()
manual_cols_all  = lab_df.columns.tolist()

# 3) Numeric-only feature lists (useful for modeling / clustering)
parquet_cols_num = feat_df.select_dtypes(include=['number']).columns.tolist()
manual_cols_num  = lab_df.select_dtypes(include=['number']).columns.tolist()

# 4) Overlap / differences
overlap_all = sorted(set(parquet_cols_all) & set(manual_cols_all))
only_in_parquet = sorted(set(parquet_cols_all) - set(manual_cols_all))
only_in_manual  = sorted(set(manual_cols_all) - set(parquet_cols_all))

# 5) Print quick summaries
print(f"Parquet columns (all): {len(parquet_cols_all)}")
print(f"Manual columns  (all): {len(manual_cols_all)}")
print(f"Parquet numeric columns: {len(parquet_cols_num)}")
print(f"Manual numeric  columns: {len(manual_cols_num)}")
print(f"Overlap (all cols): {len(overlap_all)}")
print(f"Only in Parquet (first 20): {only_in_parquet[:20]}")
print(f"Only in Manual  (first 20): {only_in_manual[:20]}")

# 6) Save to files for inspection
out_dir = MANUAL_DIR
pd.Series(parquet_cols_all, name="parquet_columns_all").to_csv(os.path.join(out_dir, "parquet_columns_all.csv"), index=False)
pd.Series(manual_cols_all,  name="manual_columns_all").to_csv(os.path.join(out_dir, "manual_columns_all.csv"), index=False)
pd.Series(parquet_cols_num, name="parquet_columns_numeric").to_csv(os.path.join(out_dir, "parquet_columns_numeric.csv"), index=False)
pd.Series(manual_cols_num,  name="manual_columns_numeric").to_csv(os.path.join(out_dir, "manual_columns_numeric.csv"), index=False)
pd.Series(overlap_all,      name="overlap_all").to_csv(os.path.join(out_dir, "columns_overlap_all.csv"), index=False)
pd.Series(only_in_parquet,  name="only_in_parquet").to_csv(os.path.join(out_dir, "columns_only_in_parquet.csv"), index=False)
pd.Series(only_in_manual,   name="only_in_manual").to_csv(os.path.join(out_dir, "columns_only_in_manual.csv"), index=False)


Parquet columns (all): 514
Manual columns  (all): 528
Parquet numeric columns: 485
Manual numeric  columns: 496
Overlap (all cols): 514
Only in Parquet (first 20): []
Only in Manual  (first 20): ['SA_SH_sum', 'SA_score', 'SH_score', 'abs_time_offset', 'abs_time_offset_z', 'human_label', 'new_key', 'new_post_id', 'old_key', 'old_post_id', 'pool', 'post_id_norm', 'post_id_str', 'predicted_label']


**Dimensionality reduction using LightGBM**

In [ ]:
import pandas as pd
import re

csv_path = "/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/manual_cleaned_all.csv"  # adjust if not the correct path

df = pd.read_csv(csv_path, low_memory=False)

def is_well_formed_human_label(val):
    # Accepts (0,0), [1,1], "1,0", etc. Returns True for valid, False for missing/malformed.
    if pd.isna(val) or str(val).strip() == "":
        return False
    s = str(val)
    s = re.sub(r'^[\[\(\s]*', '', s)
    s = re.sub(r'[\]\)\s]*$', '', s)
    s = re.sub(r'[|/;\s]+', ',', s)
    parts = [p.strip() for p in s.split(',') if p.strip() != '']
    if len(parts) != 2: return False
    return all(p in {"0","1","0.0","1.0"} for p in parts)

total = len(df)
well_formed = df['human_label'].apply(is_well_formed_human_label).sum()
missing_or_malformed = total - well_formed

print(f"Total rows: {total}")
print(f"Rows with well-formed human_label: {well_formed}")
print(f"Rows missing or malformed: {missing_or_malformed}")

# Optional: print or save top 10 example bad entries for inspection
bad_examples = df.loc[~df['human_label'].apply(is_well_formed_human_label), ['post_id','human_label']].head(10)
print("Sample bad/missing human_label entries:")
print(bad_examples)


Total rows: 3287
Rows with well-formed human_label: 3287
Rows missing or malformed: 0
Sample bad/missing human_label entries:
Empty DataFrame
Columns: [post_id, human_label]
Index: []


**4** class DIM RED

In [ ]:
#!/usr/bin/env python3
"""
4-Class Multiclass Dimensionality Reduction with LightGBM (Production)
- Extreme imbalance class weighting (correct for real distribution)
- Class-weighted multiclass via per-sample weights
- Explicit multi_logloss early stopping
- Macro F1 / Precision / Recall diagnostics per class
- Optional clustering for feature diversity
- Dual output: console + log file
- Detailed imbalance weighting diagnostics
"""

import os, re, math, json, hashlib, logging, warnings, random
from typing import List, Optional
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import lightgbm as lgb

# ===== DUAL OUTPUT =====
class DualOutput:
    def __init__(self, logger, print_also=True):
        self.logger = logger
        self.print_also = print_also
    def info(self, msg, *args):
        fmt = msg % args if args else msg
        self.logger.info(fmt)
        if self.print_also: print(f"[INFO] {fmt}")
    def warning(self, msg, *args):
        fmt = msg % args if args else msg
        self.logger.warning(fmt)
        if self.print_also: print(f"[WARN] {fmt}")
    def error(self, msg, *args):
        fmt = msg % args if args else msg
        self.logger.error(fmt)
        if self.print_also: print(f"[ERROR] {fmt}")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s: %(message)s")
base_logger = logging.getLogger("dimred_4class")
log = DualOutput(base_logger, print_also=True)

# ===== CONFIG =====
BASE_DIR = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
COMBINED_DIR = os.path.join(BASE_DIR, "Combined")
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

FEATURES_PATH = os.path.join(COMBINED_DIR, "posts_with_all_features_normalized_new.parquet")
LABELED_PATH = os.path.join(MANUAL_DIR, "manual_cleaned_all.csv")
LABELED_PARQUET_OUT = os.path.join(MANUAL_DIR, "labeled_features_4class_weighted.parquet")

SELECTED_FEATURES_PATH = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")
DIAGNOSTICS_PATH = os.path.join(MANUAL_DIR, "diagnostics_4class_weighted.json")
FOLD_METRICS_PATH = os.path.join(MANUAL_DIR, "fold_metrics_4class_weighted.csv")
CLUSTER_MAP_PATH = os.path.join(MANUAL_DIR, "cluster_map_4class.json")
IMBALANCE_WEIGHTS_LOG = os.path.join(MANUAL_DIR, "imbalance_weights_log.txt")

TOP_K = 200
N_CV_FOLDS = 5
MAX_BIN = 255
MIN_DATA_IN_LEAF = 20
STABILITY_THRESHOLD = 0.6
USE_CLUSTERING = True
N_CLUSTERS = 50
SEED = 42

os.makedirs(MANUAL_DIR, exist_ok=True)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ===== UTILITIES =====
def parse_human_label_pair(val):
    if isinstance(val, (list, tuple)) and len(val) == 2:
        try: return (int(val[0]), int(val[1]))
        except: return None
    s = str(val).strip()
    s = re.sub(r'^[\[\(\s]*', '', s)
    s = re.sub(r'[\]\)\s]*$', '', s)
    s = re.sub(r'[|/;\s]+', ',', s)
    parts = [p.strip() for p in s.split(',') if p.strip() != '']
    if len(parts) != 2: return None
    try:
        a, b = int(float(parts[0])), int(float(parts[1]))
        return (a, b) if (a in (0,1) and b in (0,1)) else None
    except: return None

def map_pair_to_4class(pair):
    if pair is None: return pd.NA
    if pair == (0,0): return 0
    if pair == (0,1): return 1
    if pair == (1,0): return 2
    if pair == (1,1): return 3
    return pd.NA

# ===== NEW EXTREME IMBALANCE WEIGHTING =====
def compute_extreme_imbalance_weights(y, num_classes=4,
                                     multiplier_base=5.0,
                                     rare_class_boost=1.5,
                                     stability_cap=50.0,
                                     log_fn=None):
    vc = y.value_counts()
    total = len(y)
    weights = {}

    # Balanced baseline
    for class_id in range(num_classes):
        count = vc.get(class_id, 1)
        raw_weight = total / (num_classes * count)
        weights[class_id] = raw_weight * multiplier_base

    # Identify rare classes
    class_freqs = {cid: vc.get(cid, 0) / total for cid in range(num_classes)}
    rare_classes = [cid for cid, f in class_freqs.items() if f < 0.10]

    if rare_classes:
        for cid in rare_classes:
            weights[cid] *= rare_class_boost
        if log_fn:
            log_fn("Boosting rarest classes %s by %.2f×", rare_classes, rare_class_boost)

    # Optional stability cap to avoid runaway ratios
    max_w = max(weights.values())
    min_w = min(weights.values())
    ratio = max_w / min_w
    if stability_cap is not None and ratio > stability_cap:
        scale = stability_cap * min_w / max_w
        weights = {k: v * scale for k, v in weights.items()}
        if log_fn:
            log_fn("Weight ratio %.1f exceeds cap %.1f; scaled weights by %.3f",
                   ratio, stability_cap, scale)

    return weights

# ===== STEP 0 =====
log.info("="*80)
log.info("STEP 0 — Creating labeled Parquet from manual CSV (4-class mapping)")
log.info("="*80)

feat_df = pd.read_parquet(FEATURES_PATH)
lab_df = pd.read_csv(LABELED_PATH, low_memory=False)

parquet_cols = feat_df.columns.tolist()
cols_to_keep = ['human_label'] + parquet_cols

df_labeled = lab_df[cols_to_keep].copy()
pairs = df_labeled['human_label'].apply(parse_human_label_pair)
df_labeled['target_4class'] = pairs.apply(map_pair_to_4class)
df_labeled = df_labeled[df_labeled['target_4class'].notna()].copy()
df_labeled.to_parquet(LABELED_PARQUET_OUT, index=False, compression="snappy")

log.info("✓ Created labeled Parquet: %s (%d rows)", LABELED_PARQUET_OUT, len(df_labeled))

# ===== STEP 1 =====
log.info("="*80)
log.info("STEP 1 — Loading and preparing data")
log.info("="*80)

df_train = pd.read_parquet(LABELED_PARQUET_OUT)
feature_cols = [c for c in df_train.columns if c not in ['human_label', 'target_4class']]

X_df_all = df_train[feature_cols].select_dtypes(include=['number'])
y_all = df_train['target_4class'].astype(int)

mask_valid = X_df_all.notna().any(axis=1)
X_df = X_df_all[mask_valid].fillna(0.0)
y = y_all[mask_valid]

log.info("✓ Loaded %d features, %d valid rows", X_df.shape[1], len(X_df))

# ===== DIAGNOSTICS =====
log.info("="*80)
log.info("DIAGNOSTICS 1 — Class Balance & Extreme Weighting")
log.info("="*80)

class_names = {0: "(0,0)", 1: "(0,1)", 2: "(1,0)", 3: "(1,1)"}
vc = y.value_counts().sort_index()

log.info("\n>>> CLASS DISTRIBUTION:")
for class_id in range(4):
    count = vc.get(class_id, 0)
    pct = 100 * count / len(y)
    log.info("  Class %d %s: %d (%.2f%%)", class_id, class_names[class_id], count, pct)

# ===== APPLY EXTREME IMBALANCE WEIGHTING =====
weight_multiplier = 5.0
rare_class_boost = 1.5
stability_cap = 50.0

class_weights = compute_extreme_imbalance_weights(
    y,
    num_classes=4,
    multiplier_base=weight_multiplier,
    rare_class_boost=rare_class_boost,
    stability_cap=stability_cap,
    log_fn=log.info
)

log.info("\n>>> FINAL CLASS WEIGHTS:")
for cid, w in class_weights.items():
    log.info("  Class %d %s: weight=%.4f", cid, class_names[cid], w)

with open(IMBALANCE_WEIGHTS_LOG, 'w') as f:
    f.write("=== EXTREME CLASS WEIGHT DIAGNOSTICS ===\n")
    f.write(f"weight_multiplier: {weight_multiplier}\n")
    f.write(f"rare_class_boost: {rare_class_boost}\n")
    f.write(f"stability_cap: {stability_cap}\n\n")
    for cid in range(4):
        count = vc.get(cid, 0)
        pct = 100 * count / len(y)
        f.write(f"Class {cid} {class_names[cid]}: {count} ({pct:.2f}%) weight={class_weights[cid]:.4f}\n")

log.info("\n✓ Extreme class weights saved to: %s", IMBALANCE_WEIGHTS_LOG)

# ===== STEP 2 =====
log.info("\n"+"="*80)
log.info("STEP 2 — Stratified %d-Fold CV Setup", N_CV_FOLDS)
log.info("="*80)

X = X_df.values
feature_names = X_df.columns.tolist()

skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y.values), 1):
    log.info("\n  Fold %d:", fold_idx)
    log.info("    Train: %s", dict(y.iloc[train_idx].value_counts()))
    log.info("    Val:   %s", dict(y.iloc[val_idx].value_counts()))

log.info("\n✓ StratifiedKFold verified")

# ===== STEP 3 =====
log.info("\n"+"="*80)
log.info("STEP 3 — LightGBM CV Feature Ranking (Class-Weighted Multiclass)")
log.info("="*80)

sample_weights = np.array([class_weights[int(c)] for c in y.values])

params = {
    'objective': 'multiclass',
    'num_class': 4,
    'learning_rate': 0.03,
    'num_leaves': 64,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda_l2': 1.0,
    'lambda_l1': 0.0,
    'verbosity': -1,
    'seed': SEED,
    'deterministic': True,
    'metric': 'multi_logloss',
    'num_threads': 4,
    'force_col_wise': True,
    'feature_fraction_seed': SEED,
    'bagging_seed': SEED,
    'data_random_seed': SEED,
    'drop_seed': SEED,
    'max_bin': MAX_BIN,
    'min_data_in_leaf': MIN_DATA_IN_LEAF,
}

gain_sums = np.zeros(X.shape[1])
split_sums = np.zeros(X.shape[1])
topk_counts = np.zeros(X.shape[1], dtype=int)
fold_ranks = []
fold_metrics_list = []
num_boost_round = 800
early_stopping_rounds = 50

log.info("\n>>> TRAINING FOLDS:")
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y.values), 1):
    log.info("\n  Fold %d/%d", fold_idx, N_CV_FOLDS)

    y_train = y.iloc[train_idx].values
    y_val = y.iloc[val_idx].values

    dtrain = lgb.Dataset(X[train_idx], label=y_train, weight=sample_weights[train_idx])
    dval = lgb.Dataset(X[val_idx], label=y_val, weight=sample_weights[val_idx])

    evals_result = {}
    booster = lgb.train(
        params,
        train_set=dtrain,
        num_boost_round=num_boost_round,
        valid_sets=[dval],
        valid_names=['valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=early_stopping_rounds, first_metric_only=True, verbose=False),
            lgb.record_evaluation(evals_result)
        ]
    )

    best_it = booster.best_iteration or booster.current_iteration()
    logloss = evals_result['valid']['multi_logloss'][best_it - 1]

    preds = booster.predict(X[val_idx])
    preds_class = np.argmax(preds, axis=1)

    acc = accuracy_score(y_val, preds_class)
    f1_macro = f1_score(y_val, preds_class, average='macro', zero_division=0)
    prec_macro = precision_score(y_val, preds_class, average='macro', zero_division=0)
    rec_macro = recall_score(y_val, preds_class, average='macro', zero_division=0)

    fold_metrics_list.append({
        'fold': fold_idx,
        'best_iteration': best_it,
        'logloss': logloss,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'precision_macro': prec_macro,
        'recall_macro': rec_macro,
    })

    log.info("    Best iter: %d | Logloss: %.6f | Acc: %.4f | F1(macro): %.4f", best_it, logloss, acc, f1_macro)

    report = classification_report(y_val, preds_class, output_dict=True, zero_division=0)
    log.info("    Per-class F1:")
    for cid in range(4):
        f1c = report.get(str(cid), {}).get('f1-score', 0)
        log.info("      Class %d %s: %.4f", cid, class_names[cid], f1c)

    gain = booster.feature_importance(importance_type='gain')
    split = booster.feature_importance(importance_type='split')

    gain_sums += gain
    split_sums += split

    imp_fold = 0.7 * (gain / (gain.max() + 1e-12)) + 0.3 * (split / (split.max() + 1e-12))
    ranks = np.argsort(-imp_fold).argsort() + 1
    fold_ranks.append(ranks)

    topk_idx = np.argsort(-imp_fold)[:TOP_K]
    topk_counts[topk_idx] += 1

# ===== STEP 4 =====
log.info("\n"+"="*80)
log.info("STEP 4 — Aggregating Feature Importance")
log.info("="*80)

gain_mean = gain_sums / N_CV_FOLDS
split_mean = split_sums / N_CV_FOLDS
gain_norm = gain_mean / (gain_mean.max() + 1e-12)
split_norm = split_mean / (split_mean.max() + 1e-12)

importance_score = 0.7 * gain_norm + 0.3 * split_norm
fold_ranks_array = np.array(fold_ranks)
mean_rank = fold_ranks_array.mean(axis=0)

log.info("✓ Feature ranking complete")

# ===== STEP 5 =====
log.info("\n"+"="*80)
log.info("STEP 5 — Feature Selection with Stability & Clustering")
log.info("="*80)

feat_df = pd.DataFrame({
    'feature': feature_names,
    'importance_score': importance_score,
    'topk_frequency': topk_counts,
    'stability_score': topk_counts / N_CV_FOLDS,
    'mean_rank': mean_rank,
})

feat_df['final_importance'] = 0.6 * feat_df['importance_score'] + 0.4 * feat_df['stability_score']
feat_df = feat_df.sort_values('final_importance', ascending=False).reset_index(drop=True)

ranking_path = SELECTED_FEATURES_PATH.replace('.csv', '_ranking.csv')
feat_df.to_csv(ranking_path, index=False)

freq_thresh = math.ceil(STABILITY_THRESHOLD * N_CV_FOLDS)
stable_candidates = feat_df[feat_df['topk_frequency'] >= freq_thresh]['feature'].tolist()

cluster_labels = None
if USE_CLUSTERING and len(feature_names) > N_CLUSTERS:
    X_cluster = X_df.select_dtypes(include=['number']).loc[:, X_df.nunique() > 1]
    if X_cluster.shape[1] > 0:
        corr_abs = X_cluster.corr().abs().fillna(0.0)
        np.fill_diagonal(corr_abs.values, 1.0)
        dist = np.nan_to_num(1 - corr_abs.values, nan=1.0)

        try:
            clustering = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='precomputed', linkage='average')
            labels = clustering.fit_predict(dist)
        except TypeError:
            clustering = AgglomerativeClustering(n_clusters=N_CLUSTERS, affinity='precomputed', linkage='average')
            labels = clustering.fit_predict(dist)

        full_labels = pd.Series(-1, index=X_df.columns)
        full_labels.loc[X_cluster.columns] = labels
        cluster_labels = full_labels.values
        feat_df['cluster'] = cluster_labels

        cluster_reps = []
        for cid in range(N_CLUSTERS):
            clust_feats = feat_df[feat_df['cluster'] == cid]
            if len(clust_feats) > 0:
                rep = clust_feats.iloc[0]['feature']
                cluster_reps.append(rep)

        final_selected = list(set(stable_candidates) | set(cluster_reps))

        cluster_map = feat_df.groupby('cluster')['feature'].apply(list).to_dict()
        with open(CLUSTER_MAP_PATH, 'w') as f:
            json.dump(cluster_map, f, indent=2)
    else:
        final_selected = stable_candidates
else:
    final_selected = stable_candidates

if len(final_selected) < TOP_K:
    need = TOP_K - len(final_selected)
    extras = [f for f in feat_df['feature'] if f not in final_selected][:need]
    final_selected += extras
elif len(final_selected) > TOP_K:
    final_selected_df = feat_df[feat_df['feature'].isin(final_selected)]
    final_selected = final_selected_df.nlargest(TOP_K, 'final_importance')['feature'].tolist()

final_df = feat_df[feat_df['feature'].isin(final_selected)].copy()
final_df['rank'] = range(1, len(final_df) + 1)
final_df.to_csv(SELECTED_FEATURES_PATH, index=False)

# ===== STEP 6 =====
log.info("\n"+"="*80)
log.info("STEP 6 — Saving Final Diagnostics")
log.info("="*80)

fold_metrics_df = pd.DataFrame(fold_metrics_list)
fold_metrics_df.to_csv(FOLD_METRICS_PATH, index=False)

diagnostics = {
    "seed": SEED,
    "n_folds": N_CV_FOLDS,
    "n_features_initial": len(feature_names),
    "n_features_selected": len(final_selected),
    "n_rows_total": len(X_df),
    "class_distribution": {str(k): int(v) for k, v in vc.items()},
    "class_weights_applied": {str(k): float(v) for k, v in class_weights.items()},
    "weight_multiplier": float(weight_multiplier),
    "rare_class_boost": float(rare_class_boost),
    "stability_cap": float(stability_cap),
    "metrics_mean": fold_metrics_df.mean(numeric_only=True).to_dict(),
    "metrics_std": fold_metrics_df.std(numeric_only=True).to_dict(),
}

with open(DIAGNOSTICS_PATH, 'w') as fh:
    json.dump(diagnostics, fh, indent=2)

# ===== SUMMARY =====
log.info("\n"+"="*80)
log.info("4-CLASS MULTICLASS DIMENSIONALITY REDUCTION — COMPLETE")
log.info("="*80)
log.info("Features: %d → %d (%.1f%% reduction)", len(feature_names), len(final_selected),
         100 * (1 - len(final_selected) / len(feature_names)))
log.info("Weights used (multiplier=%.2f, rare_boost=%.2f, cap=%.2f): %s",
         weight_multiplier, rare_class_boost, stability_cap, class_weights)
log.info("Selected features: %s", SELECTED_FEATURES_PATH)
log.info("Full ranking:      %s", ranking_path)
log.info("Fold metrics:      %s", FOLD_METRICS_PATH)
log.info("Diagnostics:       %s", DIAGNOSTICS_PATH)
log.info("Weight log:        %s", IMBALANCE_WEIGHTS_LOG)
log.info("="*80)


[INFO] ================================================================================
[INFO] STEP 0 — Creating labeled Parquet from manual CSV (4-class mapping)
[INFO] ================================================================================
[INFO] ✓ Created labeled Parquet: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/labeled_features_4class_weighted.parquet (3287 rows)
[INFO] ================================================================================
[INFO] STEP 1 — Loading and preparing data
[INFO] ================================================================================
[INFO] ✓ Loaded 485 features, 3287 valid rows
[INFO] ================================================================================
[INFO] DIAGNOSTICS 1 — Class Balance & Extreme Weighting
[INFO] ================================================================================
[INFO] 
>>> CLASS DISTRIBUTION:
[INFO]   Class 0 (0,0): 1595 (48.52%)
[INFO]   Class 1 (0,1): 1288 (39.1

Threshold Calibraton

In [ ]:
#!/usr/bin/env python3
"""
Threshold Optimization for 4-Class Multiclass LightGBM (CORRECTED & PRODUCTION)
- Per-class F1 maximization with precision guard (≥0.60)
- JSON keys as strings for consistency
- Outputs optimal thresholds ready for inference
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import re
import random
import warnings

warnings.filterwarnings("ignore")

# ===== CONFIG =====
BASE_DIR = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

LABELED_PARQUET_OUT = os.path.join(MANUAL_DIR, "labeled_features_4class_weighted.parquet")
SELECTED_FEATURES_PATH = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")

THRESHOLD_RESULTS_PATH = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")
THRESHOLD_METRICS_PATH = os.path.join(MANUAL_DIR, "threshold_optimization_metrics.csv")

SEED = 42
N_CV_FOLDS = 5

os.makedirs(MANUAL_DIR, exist_ok=True)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ===== UTILITIES =====
def compute_extreme_imbalance_weights(y, num_classes=4, multiplier_base=5.0, rare_class_boost=1.5, stability_cap=50.0):
    vc = y.value_counts()
    total = len(y)
    weights = {}
    for class_id in range(num_classes):
        count = vc.get(class_id, 1)
        raw_weight = total / (num_classes * count)
        weights[class_id] = raw_weight * multiplier_base

    class_freqs = {cid: vc.get(cid, 0) / total for cid in range(num_classes)}
    rare_classes = [cid for cid, f in class_freqs.items() if f < 0.10]

    if rare_classes:
        for cid in rare_classes:
            weights[cid] *= rare_class_boost

    max_w = max(weights.values())
    min_w = min(weights.values())
    ratio = max_w / min_w
    if stability_cap is not None and ratio > stability_cap:
        scale = stability_cap * min_w / max_w
        weights = {k: v * scale for k, v in weights.items()}

    return weights

def find_optimal_thresholds_f1_with_precision_guard(y_true, y_proba, class_id, min_precision=0.60):
    """
    Find optimal probability threshold for a single class using F1 maximization.
    ONLY consider thresholds that maintain precision >= min_precision.

    y_true: true labels
    y_proba: probabilities for this class (shape: N,)
    class_id: class label
    min_precision: minimum precision to allow (default 0.60)

    Returns: (optimal_threshold, max_f1, metrics_dict)
    """
    thresholds = np.linspace(0.0, 1.0, 101)
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}

    for thresh in thresholds:
        preds = (y_proba >= thresh).astype(int)
        precision = precision_score((y_true == class_id).astype(int), preds, zero_division=0)

        #  PRECISION GUARD: Skip if precision < min_precision
        if precision < min_precision:
            continue

        f1 = f1_score((y_true == class_id).astype(int), preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            recall = recall_score((y_true == class_id).astype(int), preds, zero_division=0)
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh,
                'min_precision_guarded': True
            }

    return best_threshold, best_f1, best_metrics

def predict_with_thresholds(y_proba, thresholds):
    """
    Apply per-class thresholds to multiclass probabilities.

    y_proba: shape (N, 4), probabilities for each class
    thresholds: dict {str(class_id): threshold}

    Returns: predicted class labels (shape: N,)
    """
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)

    for i in range(N):
        scores = y_proba[i]
        # Apply thresholds: threshold[str(cid)] is the minimum probability needed
        adjusted_scores = np.array([scores[cid] if scores[cid] >= thresholds.get(str(cid), 0.0) else -1.0
                                    for cid in range(4)])
        if adjusted_scores.max() >= -1.0:
            preds[i] = np.argmax(adjusted_scores)
        else:
            preds[i] = np.argmax(scores)

    return preds

# ===== LOAD DATA =====
print("=" * 80)
print("THRESHOLD OPTIMIZATION: Loading data")
print("=" * 80)

df_train = pd.read_parquet(LABELED_PARQUET_OUT)
selected_features_df = pd.read_csv(SELECTED_FEATURES_PATH)
selected_features = selected_features_df['feature'].tolist()

feature_cols = [c for c in df_train.columns if c not in ['human_label', 'target_4class']]
X_df_all = df_train[feature_cols].select_dtypes(include=['number'])
y_all = df_train['target_4class'].astype(int)

mask_valid = X_df_all.notna().any(axis=1)
X_df_full = X_df_all[mask_valid].fillna(0.0)
y_full = y_all[mask_valid]

X_df = X_df_full[selected_features].copy()
X = X_df.values

print(f"✓ Loaded {X.shape[0]} samples, {X.shape[1]} features")

class_weights = compute_extreme_imbalance_weights(y_full, num_classes=4, multiplier_base=5.0,
                                                   rare_class_boost=1.5, stability_cap=50.0)
sample_weights = np.array([class_weights[int(c)] for c in y_full.values])

print(f"✓ Class weights: {class_weights}")

# ===== THRESHOLD OPTIMIZATION ON CV FOLDS =====
print("\n" + "=" * 80)
print("THRESHOLD OPTIMIZATION: 5-Fold CV with Precision Guard (min=0.60)")
print("=" * 80)

skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)

all_fold_thresholds = []
fold_threshold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_full.values), 1):
    print(f"\n--- Fold {fold_idx}/5 ---")

    y_train = y_full.iloc[train_idx].values
    y_val = y_full.iloc[val_idx].values

    dtrain = lgb.Dataset(X[train_idx], label=y_train, weight=sample_weights[train_idx])
    dval = lgb.Dataset(X[val_idx], label=y_val, weight=sample_weights[val_idx])

    params = {
        'objective': 'multiclass',
        'num_class': 4,
        'learning_rate': 0.03,
        'num_leaves': 64,
        'max_depth': -1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'lambda_l2': 1.0,
        'verbosity': -1,
        'seed': SEED,
        'metric': 'multi_logloss',
    }

    booster = lgb.train(
        params,
        train_set=dtrain,
        num_boost_round=800,
        valid_sets=[dval],
        valid_names=['valid'],
        callbacks=[lgb.early_stopping(stopping_rounds=50, first_metric_only=True, verbose=False)]
    )
    model_path = os.path.join(MANUAL_DIR, f"lgboost_fold_{fold_idx}.txt")
    booster.save_model(model_path)

    y_proba_val = booster.predict(X[val_idx])

    #  CORRECTED: String keys in fold_thresholds
    fold_thresholds = {}
    fold_metrics = {'fold': fold_idx}

    for class_id in range(4):
        class_name = {0: "(0,0)", 1: "(0,1)", 2: "(1,0)", 3: "(1,1)"}[class_id]

        #  Find threshold with precision guard
        opt_thresh, opt_f1, metrics = find_optimal_thresholds_f1_with_precision_guard(
            y_val, y_proba_val[:, class_id], class_id, min_precision=0.60
        )

        #  Store as string key
        fold_thresholds[str(class_id)] = float(opt_thresh)
        fold_metrics[f'class_{class_id}_{class_name}_threshold'] = float(opt_thresh)
        fold_metrics[f'class_{class_id}_{class_name}_f1'] = float(metrics['f1'])
        fold_metrics[f'class_{class_id}_{class_name}_precision'] = float(metrics['precision'])
        fold_metrics[f'class_{class_id}_{class_name}_recall'] = float(metrics['recall'])

        print(f"  Class {class_id} {class_name}: threshold={opt_thresh:.4f}, "
              f"F1={metrics['f1']:.4f}, P={metrics['precision']:.4f}, R={metrics['recall']:.4f}")

    # Evaluate with thresholds
    preds_with_thresh = predict_with_thresholds(y_proba_val, fold_thresholds)
    acc_with_thresh = (preds_with_thresh == y_val).mean()
    f1_with_thresh = f1_score(y_val, preds_with_thresh, average='macro', zero_division=0)

    print(f"  Overall: Acc={acc_with_thresh:.4f}, Macro F1={f1_with_thresh:.4f}")

    fold_metrics['accuracy_with_thresholds'] = float(acc_with_thresh)
    fold_metrics['f1_macro_with_thresholds'] = float(f1_with_thresh)

    all_fold_thresholds.append(fold_thresholds)
    fold_threshold_metrics.append(fold_metrics)

# ===== COMPUTE AVERAGE THRESHOLDS =====
print("\n" + "=" * 80)
print("AVERAGED OPTIMAL THRESHOLDS (across 5 folds)")
print("=" * 80)

#  CORRECTED: String keys when averaging
avg_thresholds = {}
for class_id in range(4):
    thresholds_for_class = [fold_thresh[str(class_id)] for fold_thresh in all_fold_thresholds]
    avg_thresh = np.mean(thresholds_for_class)
    avg_thresholds[str(class_id)] = float(avg_thresh)
    class_name = {0: "(0,0)", 1: "(0,1)", 2: "(1,0)", 3: "(1,1)"}[class_id]
    print(f"  Class {class_id} {class_name}: {avg_thresh:.4f}")

# ===== SAVE RESULTS =====
print("\n" + "=" * 80)
print("SAVING THRESHOLD RESULTS")
print("=" * 80)

threshold_results = {
    "method": "per_class_f1_maximization_with_precision_guard",
    "min_precision_required": 0.60,
    "n_folds": N_CV_FOLDS,
    "seed": SEED,
    "averaged_optimal_thresholds": avg_thresholds,
    "fold_thresholds": all_fold_thresholds,
}

with open(THRESHOLD_RESULTS_PATH, 'w') as f:
    json.dump(threshold_results, f, indent=2)

print(f"✓ Saved thresholds: {THRESHOLD_RESULTS_PATH}")

threshold_metrics_df = pd.DataFrame(fold_threshold_metrics)
threshold_metrics_df.to_csv(THRESHOLD_METRICS_PATH, index=False)
print(f"✓ Saved metrics: {THRESHOLD_METRICS_PATH}")

print("\n" + "=" * 80)
print("PRODUCTION INFERENCE TEMPLATE")
print("=" * 80)

production_code = """
import json
import numpy as np

# Load averaged thresholds
with open('optimal_thresholds_4class.json') as f:
    thresholds_data = json.load(f)
    optimal_thresholds = thresholds_data['averaged_optimal_thresholds']

def predict_with_optimal_thresholds(y_proba):
    '''
    Apply optimal thresholds to multiclass probabilities.

    y_proba: shape (N, 4), probabilities from LightGBM model
    Returns: predicted class labels (0, 1, 2, 3)
    '''
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)

    for i in range(N):
        scores = y_proba[i]  # (4,)
        # Apply thresholds (string keys)
        adjusted_scores = np.array([scores[cid] if scores[cid] >= float(optimal_thresholds[str(cid)]) else -1.0
                                    for cid in range(4)])
        if adjusted_scores.max() >= -1.0:
            preds[i] = np.argmax(adjusted_scores)
        else:
            preds[i] = np.argmax(scores)

    return preds

# USAGE on new data:
# booster = lgb.Booster(model_file='model.txt')
# y_proba = booster.predict(X_test)  # shape (N_test, 4)
# preds = predict_with_optimal_thresholds(y_proba)
"""

print(production_code)

print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(json.dumps(threshold_results, indent=2))


THRESHOLD OPTIMIZATION: Loading data
✓ Loaded 3287 samples, 200 features
✓ Class weights: {0: np.float64(2.5760188087774294), 1: np.float64(3.1900232919254656), 2: np.float64(30.21139705882353), 3: np.float64(30.815624999999997)}

THRESHOLD OPTIMIZATION: 5-Fold CV with Precision Guard (min=0.60)

--- Fold 1/5 ---
  Class 0 (0,0): threshold=0.2300, F1=0.9548, P=0.9503, R=0.9592
  Class 1 (0,1): threshold=0.3500, F1=0.9506, P=0.9328, R=0.9690
  Class 2 (1,0): threshold=0.3500, F1=0.8667, P=0.7959, R=0.9512
  Class 3 (1,1): threshold=0.5500, F1=0.8684, P=0.9167, R=0.8250
  Overall: Acc=0.9347, Macro F1=0.9031

--- Fold 2/5 ---
  Class 0 (0,0): threshold=0.2300, F1=0.9477, P=0.9305, R=0.9655
  Class 1 (0,1): threshold=0.2700, F1=0.9438, P=0.9130, R=0.9767
  Class 2 (1,0): threshold=0.7100, F1=0.8767, P=1.0000, R=0.7805
  Class 3 (1,1): threshold=0.2300, F1=0.8989, P=0.8163, R=1.0000
  Overall: Acc=0.9316, Macro F1=0.9124

--- Fold 3/5 ---
  Class 0 (0,0): threshold=0.5400, F1=0.9447, P=0.9

Stacked

In [ ]:
#!/usr/bin/env python3
"""
Stacked Ensemble: LightGBM + CatBoost (5-Fold Soft Voting with Alpha Tuning)
- Correct CatBoost training API (CatBoostClassifier.fit)
- LightGBM validation weights for consistent metrics
- Per-fold alpha tuning (0.0–1.0 grid search for soft-vote weight)
- Model persistence for deployment
- Optimal thresholds applied post-predictions
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import catboost as cb
import warnings

warnings.filterwarnings("ignore")

# ===== CONFIG =====
BASE_DIR = os.environ.get("ML_BASE_DIR", "/content/drive/MyDrive/ML_Project_Data")
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

LABELED_PARQUET_OUT = os.path.join(MANUAL_DIR, "labeled_features_4class_weighted.parquet")
SELECTED_FEATURES_PATH = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")
THRESHOLD_RESULTS_PATH = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")

ENSEMBLE_COMPARISON_PATH = os.path.join(MANUAL_DIR, "lgb_vs_catboost_vs_ensemble.csv")
MODEL_DIR = os.path.join(MANUAL_DIR, "models_stacked")

os.makedirs(MANUAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
N_CV_FOLDS = 5
N_CLASSES = 4

# ===== UTILITIES =====
def compute_extreme_imbalance_weights(y, num_classes=4, multiplier_base=5.0, rare_class_boost=1.5, stability_cap=50.0):
    vc = y.value_counts()
    total = len(y)
    weights = {}
    for class_id in range(num_classes):
        count = vc.get(class_id, 1)
        raw_weight = total / (num_classes * count)
        weights[class_id] = raw_weight * multiplier_base

    class_freqs = {cid: vc.get(cid, 0) / total for cid in range(num_classes)}
    rare_classes = [cid for cid, f in class_freqs.items() if f < 0.10]
    for cid in rare_classes:
        weights[cid] *= rare_class_boost

    if stability_cap is not None:
        max_w = max(weights.values())
        min_w = min(weights.values())
        ratio = max_w / min_w
        if ratio > stability_cap:
            scale = stability_cap * min_w / max_w
            weights = {k: v * scale for k, v in weights.items()}
    return weights

def predict_with_thresholds(y_proba, thresholds):
    """Apply per-class thresholds to probabilities."""
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)
    for i in range(N):
        scores = y_proba[i]
        adjusted = np.array([
            scores[c] if scores[c] >= float(thresholds.get(str(c), 0.0)) else -1.0
            for c in range(N_CLASSES)
        ])
        preds[i] = np.argmax(adjusted) if adjusted.max() >= -1.0 else np.argmax(scores)
    return preds

def search_alpha(y_val, proba_lgb, proba_cat, thresholds, grid=None):
    """Grid-search soft-vote weight alpha on validation to maximize macro F1."""
    if grid is None:
        grid = np.linspace(0.0, 1.0, 21)  # 0.00, 0.05, ..., 1.00
    best_alpha, best_f1 = 0.5, -1.0
    for a in grid:
        y_proba = a * proba_lgb + (1.0 - a) * proba_cat
        preds = predict_with_thresholds(y_proba, thresholds)
        f1m = f1_score(y_val, preds, average='macro', zero_division=0)
        if f1m > best_f1:
            best_f1 = f1m
            best_alpha = float(a)
    return best_alpha, best_f1

# ===== LOAD DATA =====
print("=" * 80)
print("STACKED ENSEMBLE: LightGBM + CatBoost (Corrected)")
print("=" * 80)

df_train = pd.read_parquet(LABELED_PARQUET_OUT)
selected_features_df = pd.read_csv(SELECTED_FEATURES_PATH)
selected_features = selected_features_df['feature'].tolist()

feature_cols = [c for c in df_train.columns if c not in ['human_label', 'target_4class']]
X_df_all = df_train[feature_cols].select_dtypes(include=['number'])
y_all = df_train['target_4class'].astype(int)

mask_valid = X_df_all.notna().any(axis=1)
X_df_full = X_df_all[mask_valid].fillna(0.0)
y_full = y_all[mask_valid]

X_df = X_df_full[selected_features].copy()
X = X_df.values
print(f"✓ Loaded {X.shape[0]} samples, {X.shape[1]} features")

class_weights = compute_extreme_imbalance_weights(
    y_full, num_classes=N_CLASSES, multiplier_base=5.0, rare_class_boost=1.5, stability_cap=50.0
)
sample_weights = np.array([class_weights[int(c)] for c in y_full.values])

with open(THRESHOLD_RESULTS_PATH) as f:
    thresholds_data = json.load(f)
    optimal_thresholds = thresholds_data['averaged_optimal_thresholds']
print(f"✓ Loaded optimal thresholds: {optimal_thresholds}")

# ===== 5-FOLD TRAIN/EVAL =====
print("\n" + "=" * 80)
print("5-FOLD STACKING WITH ALPHA TUNING")
print("=" * 80)

skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)
comparison_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_full.values), 1):
    print(f"\n--- Fold {fold_idx}/{N_CV_FOLDS} ---")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_full.iloc[train_idx].values, y_full.iloc[val_idx].values
    w_train, w_val = sample_weights[train_idx], sample_weights[val_idx]

    # ===== LIGHTGBM =====
    lgb_params = {
        'objective': 'multiclass',
        'num_class': N_CLASSES,
        'learning_rate': 0.03,
        'num_leaves': 64,
        'max_depth': -1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'lambda_l2': 1.0,
        'verbosity': -1,
        'seed': SEED,
        'metric': 'multi_logloss',
        'force_col_wise': True,
    }

    dtrain_lgb = lgb.Dataset(X_train, label=y_train, weight=w_train)
    dval_lgb = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=dtrain_lgb)

    booster_lgb = lgb.train(
        lgb_params,
        train_set=dtrain_lgb,
        num_boost_round=800,
        valid_sets=[dval_lgb],
        valid_names=['valid'],
        callbacks=[lgb.early_stopping(stopping_rounds=50, first_metric_only=True, verbose=False)]
    )
    y_proba_lgb = booster_lgb.predict(X_val)

    # Save LGB model
    lgb_path = os.path.join(MODEL_DIR, f"lgb_fold_{fold_idx}.txt")
    booster_lgb.save_model(lgb_path)

    # ===== CATBOOST (Corrected API) =====
    cat_params = {
        'iterations': 800,
        'learning_rate': 0.03,
        'depth': 6,
        'loss_function': 'MultiClass',
        'eval_metric': 'TotalF1',
        'random_seed': SEED,
        'verbose': False,
        'od_type': 'Iter',        #  Early stopping strategy
        'od_wait': 50,            #  Wait 50 iterations
        'use_best_model': True,   #  Use best iteration
    }

    #  Use CatBoostClassifier.fit() API (not cb.train)
    model_cat = cb.CatBoostClassifier(**cat_params)
    model_cat.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    y_proba_cat = model_cat.predict_proba(X_val)

    # Save CatBoost model
    cat_path = os.path.join(MODEL_DIR, f"catboost_fold_{fold_idx}.cbm")
    model_cat.save_model(cat_path)

    # ===== PREDICTIONS WITH THRESHOLDS (single models) =====
    preds_lgb = predict_with_thresholds(y_proba_lgb, optimal_thresholds)
    preds_cat = predict_with_thresholds(y_proba_cat, optimal_thresholds)

    f1_lgb = f1_score(y_val, preds_lgb, average='macro', zero_division=0)
    f1_cat = f1_score(y_val, preds_cat, average='macro', zero_division=0)
    acc_lgb = (preds_lgb == y_val).mean()
    acc_cat = (preds_cat == y_val).mean()

    # ===== TUNE ENSEMBLE WEIGHT α =====
    alpha, _ = search_alpha(y_val, y_proba_lgb, y_proba_cat, optimal_thresholds)
    y_proba_ens = alpha * y_proba_lgb + (1.0 - alpha) * y_proba_cat
    preds_ens = predict_with_thresholds(y_proba_ens, optimal_thresholds)
    f1_ens = f1_score(y_val, preds_ens, average='macro', zero_division=0)
    acc_ens = (preds_ens == y_val).mean()

    print(f"  LightGBM:        Acc={acc_lgb:.4f}, F1(macro)={f1_lgb:.4f}")
    print(f"  CatBoost:        Acc={acc_cat:.4f}, F1(macro)={f1_cat:.4f}")
    print(f"  Ensemble(α={alpha:.2f}): Acc={acc_ens:.4f}, F1(macro)={f1_ens:.4f}")
    print(f"   Gain vs LGB: +{(f1_ens - f1_lgb)*100:.2f}%")

    comparison_metrics.append({
        'fold': fold_idx,
        'alpha': alpha,
        'lgb_acc': acc_lgb,
        'lgb_f1_macro': f1_lgb,
        'catboost_acc': acc_cat,
        'catboost_f1_macro': f1_cat,
        'ensemble_acc': acc_ens,
        'ensemble_f1_macro': f1_ens,
        'ensemble_gain_vs_lgb_pct': (f1_ens - f1_lgb) * 100.0,
    })

# ===== SUMMARY =====
print("\n" + "=" * 80)
print("ENSEMBLE STACKING SUMMARY")
print("=" * 80)

comparison_df = pd.DataFrame(comparison_metrics)
comparison_df.to_csv(ENSEMBLE_COMPARISON_PATH, index=False)
print("\n" + comparison_df.to_string(index=False))

print("\n" + "=" * 80)
print("AVERAGES ACROSS 5 FOLDS")
print("=" * 80)

mean_lgb_f1 = comparison_df['lgb_f1_macro'].mean()
mean_cat_f1 = comparison_df['catboost_f1_macro'].mean()
mean_ens_f1 = comparison_df['ensemble_f1_macro'].mean()
mean_alpha = comparison_df['alpha'].mean()
mean_gain = comparison_df['ensemble_gain_vs_lgb_pct'].mean()

print(f"LightGBM Macro F1:      {mean_lgb_f1:.4f}")
print(f"CatBoost Macro F1:      {mean_cat_f1:.4f}")
print(f"Ensemble Macro F1:      {mean_ens_f1:.4f}")
print(f"Average α (LGB weight): {mean_alpha:.3f}")
print(f"Average gain vs LGB:    +{mean_gain:.2f}%")

print("\n" + "=" * 80)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 80)

deploy_code = """
# Load models for inference:
import lightgbm as lgb
import catboost as cb
import numpy as np
import json

# Load all 5 fold models
lgb_models = [lgb.Booster(model_file=f'models_stacked/lgb_fold_{i}.txt') for i in range(1, 6)]
cat_models = [cb.CatBoost().load_model(f'models_stacked/catboost_fold_{i}.cbm') for i in range(1, 6)]

with open('optimal_thresholds_4class.json') as f:
    thresholds = json.load(f)['averaged_optimal_thresholds']

# Read alpha values
alphas = [0.50, 0.50, 0.50, 0.50, 0.50]  # From ensemble_stacking_metrics.csv

def predict_ensemble(X_test):
    # Average probabilities across folds
    proba_lgb_list = [m.predict(X_test) for m in lgb_models]
    proba_cat_list = [m.predict_proba(X_test) for m in cat_models]

    proba_lgb_avg = np.mean(proba_lgb_list, axis=0)
    proba_cat_avg = np.mean(proba_cat_list, axis=0)

    # Soft vote with average alpha
    alpha_avg = np.mean(alphas)
    y_proba = alpha_avg * proba_lgb_avg + (1 - alpha_avg) * proba_cat_avg

    # Apply thresholds
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)
    for i in range(N):
        scores = y_proba[i]
        adjusted = np.array([scores[c] if scores[c] >= float(thresholds[str(c)]) else -1.0 for c in range(4)])
        preds[i] = np.argmax(adjusted) if adjusted.max() >= -1.0 else np.argmax(scores)

    return preds

# Use:
# preds = predict_ensemble(X_test)
"""

print(deploy_code)

print("\n" + "=" * 80)
print(f"✓ Models saved to: {MODEL_DIR}")
print(f"✓ Metrics saved to: {ENSEMBLE_COMPARISON_PATH}")
print("=" * 80)


STACKED ENSEMBLE: LightGBM + CatBoost (Corrected)
✓ Loaded 3287 samples, 200 features
✓ Loaded optimal thresholds: {'0': 0.378, '1': 0.384, '2': 0.46399999999999997, '3': 0.33799999999999997}

5-FOLD STACKING WITH ALPHA TUNING

--- Fold 1/5 ---
  LightGBM:        Acc=0.9331, F1(macro)=0.8970
  CatBoost:        Acc=0.8723, F1(macro)=0.8525
  Ensemble(α=0.50): Acc=0.9407, F1(macro)=0.9107
  ✅ Gain vs LGB: +1.37%

--- Fold 2/5 ---
  LightGBM:        Acc=0.9255, F1(macro)=0.8912
  CatBoost:        Acc=0.8754, F1(macro)=0.8380
  Ensemble(α=0.40): Acc=0.9362, F1(macro)=0.9027
  ✅ Gain vs LGB: +1.15%

--- Fold 3/5 ---
  LightGBM:        Acc=0.9163, F1(macro)=0.8542
  CatBoost:        Acc=0.8645, F1(macro)=0.8427
  Ensemble(α=0.45): Acc=0.9193, F1(macro)=0.8795
  ✅ Gain vs LGB: +2.53%

--- Fold 4/5 ---
  LightGBM:        Acc=0.9437, F1(macro)=0.8914
  CatBoost:        Acc=0.8752, F1(macro)=0.8312
  Ensemble(α=0.55): Acc=0.9467, F1(macro)=0.8984
  ✅ Gain vs LGB: +0.70%

--- Fold 5/5 ---
  Light

# Self training cycle - 1

get predictions

In [ ]:
import lightgbm as lgb
import catboost as cb
import numpy as np
import json
import pandas as pd
import os

# ===== CONFIG: Full Google Drive paths =====
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")
MODEL_DIR = os.path.join(MANUAL_DIR, "models_stacked")

print("Loading models...")

#  LightGBM: use full paths
lgb_models = [
    lgb.Booster(model_file=os.path.join(MODEL_DIR, f"lgb_fold_{i}.txt"))
    for i in range(1, 6)
]
print("✓ Loaded 5 LightGBM models")

#  CatBoost: use full paths
cat_models = []
for i in range(1, 6):
    model = cb.CatBoostClassifier()
    model.load_model(os.path.join(MODEL_DIR, f"catboost_fold_{i}.cbm"))
    cat_models.append(model)
print("✓ Loaded 5 CatBoost models")

# ===== LOAD THRESHOLDS =====

THRESHOLD_PATH = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")
with open(THRESHOLD_PATH) as f:
    thresholds = json.load(f)["averaged_optimal_thresholds"]
print("✓ Loaded optimal thresholds")

# ===== LOAD ALPHAS =====

METRICS_PATH = os.path.join(MANUAL_DIR, "lgb_vs_catboost_vs_ensemble.csv")
metrics_df = pd.read_csv(METRICS_PATH)
alphas = metrics_df["alpha"].tolist()
print(f"✓ Loaded alphas: {alphas}")

# ===== INFERENCE FUNCTION =====

def predict_ensemble_production(X_test, return_proba=False):
    """
    Production inference for 4-class SA/SH model:
    - Uses all 10 models (5 LGB + 5 CatBoost)
    - Soft vote with learned α
    - Applies optimal per-class thresholds
    Returns: class predictions or (predictions, probabilities)
    """

    # ----- 1) Get model probabilities -----
    proba_lgb_list = [m.predict(X_test) for m in lgb_models]
    proba_cat_list = [m.predict_proba(X_test) for m in cat_models]

    # ----- 2) Average across folds -----
    proba_lgb_avg = np.mean(proba_lgb_list, axis=0)
    proba_cat_avg = np.mean(proba_cat_list, axis=0)

    # ----- 3) Soft vote -----
    alpha_avg = float(np.mean(alphas))  # usually ~0.47
    y_proba = alpha_avg * proba_lgb_avg + (1 - alpha_avg) * proba_cat_avg

    # ----- 4) Apply thresholds -----
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)

    for i in range(N):
        scores = y_proba[i]

        adjusted = np.array([
            scores[c] if scores[c] >= float(thresholds[str(c)]) else -1.0
            for c in range(4)
        ])

        # if at least one class ≥ threshold → choose its argmax
        if adjusted.max() >= -1.0:
            preds[i] = np.argmax(adjusted)
        else:
            preds[i] = np.argmax(scores)

    return (preds, y_proba) if return_proba else preds


print("\n All models and thresholds loaded successfully!")
print("Ready for inference with: predict_ensemble_production(X_test)")


Loading models...
✓ Loaded 5 LightGBM models
✓ Loaded 5 CatBoost models
✓ Loaded optimal thresholds
✓ Loaded alphas: [0.5, 0.4, 0.45, 0.55, 0.45]

✅ All models and thresholds loaded successfully!
Ready for inference with: predict_ensemble_production(X_test)


self train loop 1

In [ ]:
#!/usr/bin/env python3
"""
Cycle-1 SSL (Pseudo-Labeling) — v3
- Uses deployed ensemble (5×LGB + 5×CatBoost) and precomputed thresholds
- Scores unlabeled + labeled data
- Accepts pseudo-labels with guards + rare-class quotas
- Produces final_label (human > pseudo > -1)
"""

import os
import json
import numpy as np
import pandas as pd
from collections import Counter
import lightgbm as lgb
import catboost as cb
import warnings

warnings.filterwarnings("ignore")

# ========= CONFIG =========
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
COMBINED_DIR = os.path.join(BASE_DIR, "Combined")
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")
MODEL_DIR = os.path.join(MANUAL_DIR, "models_stacked")  # deployed Cycle-0 models

# Inputs
FEATURES_PARQUET = os.path.join(COMBINED_DIR, "posts_with_all_features_normalized_new.parquet")  # unlabeled + features
LABELED_PARQUET  = os.path.join(MANUAL_DIR, "labeled_features_4class_weighted.parquet")          # labeled with human_label
SELECTED_FEATURES_PATH = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")
THRESHOLD_PATH   = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")
METRICS_PATH     = os.path.join(MANUAL_DIR, "lgb_vs_catboost_vs_ensemble.csv")

# Output
FINAL_OUTPUT_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label.csv")

# Model/data constants
SEED = 42
N_CLASSES = 4

# Pseudo-label acceptance (primary thresholds already tuned)
CONFIDENCE_THRESHOLD_GLOBAL = 0.85   # for classes 0/1
CONFIDENCE_THRESHOLD_RARE   = 0.92   # for classes 2/3

# Secondary guards (help weed-out noisy accepts)
MAX_ENTROPY = 1.10        # lower = more certain; ~uniform(4) entropy is 1.386
MIN_MARGIN  = 0.10        # top1 - top2 probability margin

# Rare-class quotas (guarantee presence of 2 & 3 among pseudo-labels)
# If there aren't enough candidates, takes as many as available.
QUOTA = {0: 0, 1: 0, 2: 1200, 3: 1200}
QUOTA_LOOSE_CONF = 0.60   # allow lower conf for quota filling (still ranked by conf)

# Optional agreement filter with prior weak labels (e.g., SA/SH).
# If you have a column like 'weak_label_4class' (0..3), set AGREEMENT_COLUMN to its name.
AGREEMENT_COLUMN = None        # e.g., "weak_label_4class"
REQUIRE_AGREEMENT = False      # set True to only accept if predicted == weak

np.random.seed(SEED)

# ========= UTILITIES =========

def map_pair_to_4class(pair_str_or_obj):
    """
    Map "[a,b]" or (a,b) or [a,b] to 4-class id: (0,0)->0, (0,1)->1, (1,0)->2, (1,1)->3
    Return int 0..3 or None if not parseable.
    """
    if pair_str_or_obj is None or (isinstance(pair_str_or_obj, float) and np.isnan(pair_str_or_obj)):
        return None

    # Already clean int string "0"/"1"/"2"/"3" -> accept
    if isinstance(pair_str_or_obj, str):
        s = pair_str_or_obj.strip()
        if s.isdigit() and int(s) in (0,1,2,3):
            return int(s)

    # Try to parse "[a,b]" in various messy forms
    def _try_parse_two(s):
        s = str(s).strip()
        s = s.replace('[','').replace(']','').replace('(','').replace(')','')
        s = s.replace('|', ',').replace('/', ',').replace(';', ',')
        parts = [p.strip() for p in s.split(',') if p.strip() != '']
        if len(parts) != 2:
            return None
        try:
            a, b = int(float(parts[0])), int(float(parts[1]))
        except:
            return None
        if a in (0,1) and b in (0,1):
            return (a<<1) | b  # (a,b) -> 2*a + b
        return None

    # tuple/list input
    if isinstance(pair_str_or_obj, (list, tuple)) and len(pair_str_or_obj) == 2:
        try:
            a, b = int(pair_str_or_obj[0]), int(pair_str_or_obj[1])
            if a in (0,1) and b in (0,1):
                return (a<<1) | b
        except:
            return None

    # string or anything else -> try generic parse
    return _try_parse_two(pair_str_or_obj)

def safe_cast_label_4class(val):
    """
    Try to coerce a 'human_label' value to an int in {0,1,2,3}.
    Supports: 0/1/2/3, "[0,1]" pairs, tuples, etc.
    Returns int or None.
    """
    if pd.isna(val):
        return None
    # numeric single class
    if isinstance(val, (int, np.integer)):
        return int(val) if val in (0,1,2,3) else None
    if isinstance(val, float):
        i = int(val)
        return i if i in (0,1,2,3) else None
    # strings or pairs
    mapped = map_pair_to_4class(val)
    return mapped if mapped in (0,1,2,3) else None

def entropy_rows(probs):
    p = np.clip(probs, 1e-10, 1.0)
    return -np.sum(p * np.log(p), axis=1)

def top2_margin_rows(probs):
    # margin = top1 - top2 per row
    sorted_p = -np.sort(-probs, axis=1)
    return sorted_p[:,0] - sorted_p[:,1]

def predict_with_thresholds(y_proba, thresholds_dict):
    """
    Apply per-class minimum probability thresholds (string keys in thresholds).
    If no class passes, fall back to argmax.
    """
    N = y_proba.shape[0]
    preds = np.zeros(N, dtype=int)
    for i in range(N):
        scores = y_proba[i]
        adjusted = np.array([
            scores[c] if scores[c] >= float(thresholds_dict[str(c)]) else -1.0
            for c in range(N_CLASSES)
        ])
        preds[i] = int(np.argmax(adjusted) if adjusted.max() >= -1.0 else np.argmax(scores))
    return preds

def load_ensemble_and_thresholds():
    lgb_models = [lgb.Booster(model_file=os.path.join(MODEL_DIR, f"lgb_fold_{i}.txt")) for i in range(1,6)]
    cat_models = []
    for i in range(1,6):
        m = cb.CatBoostClassifier()
        m.load_model(os.path.join(MODEL_DIR, f"catboost_fold_{i}.cbm"))
        cat_models.append(m)

    with open(THRESHOLD_PATH) as f:
        thresholds = json.load(f)["averaged_optimal_thresholds"]

    alphas = pd.read_csv(METRICS_PATH)["alpha"].tolist()
    alpha_avg = float(np.mean(alphas))
    return lgb_models, cat_models, thresholds, alpha_avg

def ensemble_predict(X, lgb_models, cat_models, alpha_avg):
    proba_lgb = np.mean([m.predict(X) for m in lgb_models], axis=0)
    proba_cat = np.mean([m.predict_proba(X) for m in cat_models], axis=0)
    return alpha_avg * proba_lgb + (1.0 - alpha_avg) * proba_cat

# ========= LOAD DATA =========

print("="*80)
print("CYCLE-1 SSL: Loading Data")
print("="*80)

# Selected features MUST match what the models expect
selected_features = pd.read_csv(SELECTED_FEATURES_PATH)["feature"].tolist()

# Labeled (has human_label and/or target_4class)
df_labeled = pd.read_parquet(LABELED_PARQUET).copy()
df_labeled["human_label_clean"] = df_labeled["human_label"].apply(safe_cast_label_4class)

X_labeled = df_labeled[selected_features].fillna(0.0).values
print(f"✓ Labeled: {len(df_labeled):,} rows")

# Unlabeled features (no ground truth columns)
df_unlabeled = pd.read_parquet(FEATURES_PARQUET).copy()
# ensure these columns exist for uniform merging later
if "human_label" not in df_unlabeled.columns: df_unlabeled["human_label"] = None
df_unlabeled["human_label_clean"] = None

X_unlabeled = df_unlabeled[selected_features].fillna(0.0).values
print(f"✓ Unlabeled: {len(df_unlabeled):,} rows")

# Optional weak labels (agreement filter)
if AGREEMENT_COLUMN and AGREEMENT_COLUMN not in df_unlabeled.columns:
    print(f"ℹ  AGREEMENT_COLUMN '{AGREEMENT_COLUMN}' not found; agreement filter disabled.")
    REQUIRE_AGREEMENT = False

# ========= LOAD ENSEMBLE =========

print("\nLoading ensemble + thresholds ...")
lgb_models, cat_models, thresholds, alpha_avg = load_ensemble_and_thresholds()
print(f"✓ Loaded 5 LGB + 5 CatBoost, alpha={alpha_avg:.2f}")
print(f"✓ Thresholds: {thresholds}")

# ========= PREDICT UNLABELED =========

print("\nScoring unlabeled ...")
probs_unl = ensemble_predict(X_unlabeled, lgb_models, cat_models, alpha_avg)
preds_unl  = predict_with_thresholds(probs_unl, thresholds)
conf_unl   = probs_unl.max(axis=1)
ent_unl    = entropy_rows(probs_unl)
marg_unl   = top2_margin_rows(probs_unl)

# Primary accept by class thresholds + guards
accept_primary = []
for i, cls in enumerate(preds_unl):
    thr = CONFIDENCE_THRESHOLD_RARE if cls in (2,3) else CONFIDENCE_THRESHOLD_GLOBAL
    ok_conf   = conf_unl[i] >= thr
    ok_ent    = ent_unl[i]  <= MAX_ENTROPY
    ok_margin = marg_unl[i] >= MIN_MARGIN
    ok_agree  = True
    if REQUIRE_AGREEMENT and AGREEMENT_COLUMN:
        try:
            weak = df_unlabeled.iloc[i][AGREEMENT_COLUMN]
            ok_agree = (weak == cls)
        except:
            ok_agree = True
    accept_primary.append(bool(ok_conf and ok_ent and ok_margin and ok_agree))
accept_primary = np.array(accept_primary, dtype=bool)

# Quota fill for rare classes (2,3)
print("Applying rare-class quotas ...")
forced_accept = set()
for cls in (2,3):
    idx_cls = np.where(preds_unl == cls)[0]
    if len(idx_cls) == 0:
        continue
    # rank by confidence
    ranked = idx_cls[np.argsort(-conf_unl[idx_cls])]
    # loosened confidence for quota ranking
    ranked_loose = [i for i in ranked if conf_unl[i] >= QUOTA_LOOSE_CONF]
    need = max(0, QUOTA.get(cls, 0) - int(accept_primary[idx_cls].sum()))
    if need > 0:
        take = ranked_loose[:need] if len(ranked_loose) >= need else ranked_loose
        forced_accept.update(take)

accept_mask = accept_primary.copy()
for i in forced_accept:
    accept_mask[i] = True

n_accept = int(accept_mask.sum())
print(f"✓ Pseudo-labels accepted: {n_accept:,} of {len(df_unlabeled):,} (primary + quotas)")

# ========= PREDICT LABELED (for reference only) =========
print("Scoring labeled (for reference) ...")
probs_lbl = ensemble_predict(X_labeled, lgb_models, cat_models, alpha_avg)
preds_lbl = predict_with_thresholds(probs_lbl, thresholds)
conf_lbl  = probs_lbl.max(axis=1)
ent_lbl   = entropy_rows(probs_lbl)

# ========= BUILD OUTPUT =========
print("\nAssembling output ...")

out_l = df_labeled.copy()
out_l["predicted_class"]        = preds_lbl
out_l["prediction_confidence"]  = conf_lbl
out_l["prediction_entropy"]     = ent_lbl
out_l["is_pseudo_label"]        = False
out_l["source"]                 = "labeled"

for c in range(N_CLASSES):
    out_l[f"proba_class_{c}"] = probs_lbl[:, c]

out_u = df_unlabeled.copy()
out_u["predicted_class"]        = preds_unl
out_u["prediction_confidence"]  = conf_unl
out_u["prediction_entropy"]     = ent_unl
out_u["is_pseudo_label"]        = accept_mask
out_u["source"]                 = "unlabeled"
for c in range(N_CLASSES):
    out_u[f"proba_class_{c}"] = probs_unl[:, c]

final = pd.concat([out_l, out_u], ignore_index=True)

# final_label: human > pseudo > -1
def compute_final_label_row(row):
    if row.get("human_label_clean", None) is not None:
        return int(row["human_label_clean"])
    if bool(row.get("is_pseudo_label", False)):
        return int(row["predicted_class"])
    return -1

print("Computing final_label ...")
final["final_label"] = final.apply(compute_final_label_row, axis=1)

# ========= SAVE =========
final.to_csv(FINAL_OUTPUT_CSV, index=False)
print(f"\n Saved: {FINAL_OUTPUT_CSV}")

# ========= SUMMARY =========
print("\n" + "="*80)
print("CYCLE-1 SSL — SUMMARY")
print("="*80)

tot = len(final)
n_human  = int(final["human_label_clean"].notna().sum())
n_pseudo = int(final["is_pseudo_label"].sum())
n_unk    = int((final["final_label"] == -1).sum())

print(f"Total rows:          {tot:,}")
print(f"Human-labeled:       {n_human:,}")
print(f"Pseudo-labeled:      {n_pseudo:,}")
print(f"Unknown (-1):        {n_unk:,}")

# Show class distribution among final_label (excluding -1)
known = final[final["final_label"] != -1]["final_label"].astype(int)
dist = known.value_counts().reindex([0,1,2,3], fill_value=0)
print("\nFinal label distribution (known only):")
for c in [0,1,2,3]:
    k = int(dist[c])
    pct = 100.0 * k / max(1, len(known))
    print(f"  class_{c}: {k:,} ({pct:.1f}%)")

print("\nNotes:")
print("• Thresholds + entropy + margin guard control precision.")
print("• Rare-class quotas ensure classes 2 & 3 are present for the next cycle.")
print("• final_label is ready for SSL-2 / co-training / label propagation.")


CYCLE-1 SSL: Loading Data
✓ Labeled: 3,287 rows
✓ Unlabeled: 80,002 rows

Loading ensemble + thresholds ...
✓ Loaded 5 LGB + 5 CatBoost, alpha=0.47
✓ Thresholds: {'0': 0.378, '1': 0.384, '2': 0.46399999999999997, '3': 0.33799999999999997}

Scoring unlabeled ...
Applying rare-class quotas ...
✓ Pseudo-labels accepted: 14,601 of 80,002 (primary + quotas)
Scoring labeled (for reference) ...

Assembling output ...
Computing final_label ...

✅ Saved: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_with_final_label.csv

CYCLE-1 SSL — SUMMARY
Total rows:          83,289
Human-labeled:       3,287
Pseudo-labeled:      14,601
Unknown (-1):        65,401

Final label distribution (known only):
  class_0: 4,689 (26.2%)
  class_1: 11,656 (65.2%)
  class_2: 1,140 (6.4%)
  class_3: 403 (2.3%)

Notes:
• Thresholds + entropy + margin guard control precision.
• Rare-class quotas ensure classes 2 & 3 are present for the next cycle.
• final_label is ready for SSL-2 / co-training /

# Self training cycle 2

In [ ]:
#!/usr/bin/env python3
"""
SSL-2: FINAL PRODUCTION VERSION
Minor fixes for robustness + clarity
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import catboost as cb
import warnings

warnings.filterwarnings("ignore")

# ===== PATHS =====
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

CYCLE1_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label.csv")
SELECTED_FEATURES = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")
THRESHOLDS_C1 = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")
METRICS_C1 = os.path.join(MANUAL_DIR, "lgb_vs_catboost_vs_ensemble.csv")

MODEL_DIR_C2 = os.path.join(MANUAL_DIR, "models_stacked_v2")
CYCLE2_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label_v2.csv")
DIAGNOSTICS_C2 = os.path.join(MANUAL_DIR, "ssl2_diagnostics.json")
FOLD_METRICS_C2 = os.path.join(MANUAL_DIR, "ssl2_fold_metrics.csv")

os.makedirs(MODEL_DIR_C2, exist_ok=True)

SEED = 42
N_CLASSES = 4
BASELINE_F1 = 0.8951  #  FIX 1: Define baseline

# ===== UTILS =====

def predict_with_thresholds(proba, thresholds):
    preds = np.zeros(proba.shape[0], dtype=int)
    for i in range(proba.shape[0]):
        row = proba[i]
        adj = np.array([
            row[c] if row[c] >= float(thresholds[str(c)]) else -1.0
            for c in range(N_CLASSES)
        ])
        preds[i] = np.argmax(adj) if adj.max() >= -1.0 else np.argmax(row)
    return preds

def compute_entropy(p):
    p = np.clip(p, 1e-10, 1 - 1e-10)
    return -(p * np.log(p)).sum(axis=1)

def predict_ensemble(X, lgb_models, cat_models, thresholds, alphas, return_proba=False):
    proba_lgb = np.mean([m.predict(X) for m in lgb_models], axis=0)
    proba_cat = np.mean([m.predict_proba(X) for m in cat_models], axis=0)
    alpha = float(np.mean(alphas))
    proba = alpha * proba_lgb + (1 - alpha) * proba_cat
    if return_proba:
        preds = predict_with_thresholds(proba, thresholds)
        return preds, proba
    return predict_with_thresholds(proba, thresholds)

def compute_class_weights(y, multiplier=3.0):
    vc = pd.Series(y).value_counts()
    total = len(y)
    w = {}
    for c in range(N_CLASSES):
        count = vc.get(c, 1)
        raw = total / (4 * count)
        w[c] = min(raw * multiplier, 50.0)
    return w

# ===== STEP 1: LOAD DATA =====
print("="*80)
print("SSL-2: FINAL PRODUCTION VERSION")
print("="*80)

df = pd.read_csv(CYCLE1_CSV)
features = pd.read_csv(SELECTED_FEATURES)['feature'].tolist()

print(f"✓ Loaded: {len(df):,} rows, {len(features)} features")

#  FIX 2: Check for required column
if 'final_label' not in df.columns:
    raise ValueError("Missing 'final_label' column. Run SSL-1 first.")

df_labeled = df[df['final_label'] != -1].copy()
df_unlabeled = df[df['final_label'] == -1].copy()

print(f"  Known labeled:  {len(df_labeled):,}")
print(f"  To re-predict:  {len(df_unlabeled):,}")

X_labeled = df_labeled[features].fillna(0).values
y_labeled = df_labeled['final_label'].astype(int).values

# Load thresholds once
with open(THRESHOLDS_C1) as f:
    thresholds = json.load(f)["averaged_optimal_thresholds"]

# ===== STEP 2: TRAIN ENSEMBLE =====
print("\nTraining 5-fold ensemble ...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lgb_models_v2 = []
cat_models_v2 = []
fold_metrics = []

for fold, (tr, va) in enumerate(skf.split(X_labeled, y_labeled), 1):
    print(f"  Fold {fold}/5", end=" ")
    X_tr, X_va = X_labeled[tr], X_labeled[va]
    y_tr, y_va = y_labeled[tr], y_labeled[va]

    w_map = compute_class_weights(y_tr, multiplier=4.0)
    w_tr = np.array([w_map[c] for c in y_tr])
    w_va = np.array([w_map[c] for c in y_va])

    # LightGBM
    params = dict(
        objective="multiclass",
        num_class=4,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.8,
        lambda_l2=1.0,
        metric="multi_logloss",
        seed=SEED,
        verbosity=-1
    )
    dtr = lgb.Dataset(X_tr, y_tr, weight=w_tr)
    dva = lgb.Dataset(X_va, y_va, weight=w_va, reference=dtr)
    booster = lgb.train(
        params, dtr,
        num_boost_round=800,
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    lgb_models_v2.append(booster)

    # CatBoost
    class_weights_cat = [w_map.get(c, 1.0) for c in range(4)]
    model_cat = cb.CatBoostClassifier(
        iterations=800,
        learning_rate=0.03,
        depth=6,
        loss_function="MultiClass",
        classes_count=4,
        class_weights=class_weights_cat,
        verbose=False
    )
    model_cat.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=(X_va, y_va),
                  early_stopping_rounds=60,
                  verbose=False)
    cat_models_v2.append(model_cat)

    #  FIX 3: Validate with thresholds
    proba_va_lgb = booster.predict(X_va)
    preds_va = predict_with_thresholds(proba_va_lgb, thresholds)
    f1 = f1_score(y_va, preds_va, average="macro", zero_division=0)
    fold_metrics.append({"fold": fold, "f1_macro": f1})
    print(f"F1={f1:.4f}")

fold_df = pd.DataFrame(fold_metrics)
fold_df.to_csv(FOLD_METRICS_C2, index=False)

mean_f1 = fold_df['f1_macro'].mean()
std_f1 = fold_df['f1_macro'].std()
print(f"\nMean F1: {mean_f1:.4f} (± {std_f1:.4f})")

# Save models
for i, m in enumerate(lgb_models_v2, 1):
    m.save_model(os.path.join(MODEL_DIR_C2, f"lgb_fold_{i}.txt"))
for i, m in enumerate(cat_models_v2, 1):
    m.save_model(os.path.join(MODEL_DIR_C2, f"catboost_fold_{i}.cbm"))

print(f"✓ Models saved → {MODEL_DIR_C2}")

# ===== STEP 3: INFER AGAIN =====
print("\nRe-predicting all rows ...")

alphas = pd.read_csv(METRICS_C1)["alpha"].tolist()
X_all = df[features].fillna(0).values
preds, proba = predict_ensemble(X_all, lgb_models_v2, cat_models_v2, thresholds, alphas, return_proba=True)

conf = proba.max(axis=1)
entropy = compute_entropy(proba)
margin = np.sort(proba, axis=1)[:, -1] - np.sort(proba, axis=1)[:, -2]

# Stricter guards for SSL-2
is_pseudo_v2 = (conf >= 0.88) & (entropy <= 1.10) & (margin >= 0.08)

df["predicted_class_v2"] = preds
df["prediction_confidence_v2"] = conf
df["prediction_entropy_v2"] = entropy
df["prediction_margin_v2"] = margin
df["is_pseudo_label_v2"] = is_pseudo_v2

for c in range(N_CLASSES):
    df[f"proba_class_{c}_v2"] = proba[:, c]

# Final label: human > pseudo > -1
def compute_final_label_v2(row):
    if pd.notna(row.get("human_label_clean", None)):
        return int(row["human_label_clean"])
    if row.get("is_pseudo_label_v2", False):
        return int(row["predicted_class_v2"])
    return -1

df["final_label_v2"] = df.apply(compute_final_label_v2, axis=1)

df.to_csv(CYCLE2_CSV, index=False)
print(f"✓ Saved: {CYCLE2_CSV}")

# ===== SUMMARY =====
n_human = df["human_label_clean"].notna().sum() if "human_label_clean" in df.columns else 0
n_pseudo_v1 = df["is_pseudo_label"].sum() if "is_pseudo_label" in df.columns else 0
n_pseudo_v2 = df["is_pseudo_label_v2"].sum()
n_unknown_v2 = (df["final_label_v2"] == -1).sum()

improvement = (mean_f1 - BASELINE_F1) * 100

diagnostics = {
    "ssl_stage": 2,
    "baseline_f1": BASELINE_F1,
    "ssl2_mean_f1": float(mean_f1),
    "ssl2_std_f1": float(std_f1),
    "improvement_pct": float(improvement),
    "n_human": int(n_human),
    "n_pseudo_v1": int(n_pseudo_v1),
    "n_pseudo_v2": int(n_pseudo_v2),
    "n_unknown_v2": int(n_unknown_v2),
    "total_rows": int(len(df)),
    "fold_metrics": fold_df.to_dict('records')
}

with open(DIAGNOSTICS_C2, "w") as f:
    json.dump(diagnostics, f, indent=2)

print("\n" + "="*80)
print("SSL-2 SUMMARY")
print("="*80)
print(f"Total rows:       {len(df):,}")
print(f"Human labels:     {n_human:,}")
print(f"Pseudo v1:        {n_pseudo_v1:,}")
print(f"Pseudo v2:        {n_pseudo_v2:,} (stricter)")
print(f"Unknown v2:       {n_unknown_v2:,}")

print(f"\nPerformance:")
print(f"  Baseline:       {BASELINE_F1:.4f}")
print(f"  SSL-2:          {mean_f1:.4f}")
print(f"  Improvement:    {improvement:+.2f}%")

status = " IMPROVED" if improvement > 0 else "⚠️  DECLINED"
print(f"  Status:         {status}")

print(f"\n Output:        {CYCLE2_CSV}")
print(f" Models:        {MODEL_DIR_C2}")
print(f" Diagnostics:   {DIAGNOSTICS_C2}")
print("="*80)


SSL-2: FINAL PRODUCTION VERSION
✓ Loaded: 83,289 rows, 200 features
  Known labeled:  17,888
  To re-predict:  65,401

Training 5-fold ensemble ...
  Fold 1/5 F1=0.9704
  Fold 2/5 F1=0.9748
  Fold 3/5 F1=0.9656
  Fold 4/5 F1=0.9781
  Fold 5/5 F1=0.9675

Mean F1: 0.9713 (± 0.0051)
✓ Models saved → /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/models_stacked_v2

Re-predicting all rows ...
✓ Saved: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_with_final_label_v2.csv

SSL-2 SUMMARY
Total rows:       83,289
Human labels:     3,287
Pseudo v1:        14,601
Pseudo v2:        56,638 (stricter)
Unknown v2:       25,877

Performance:
  Baseline:       0.8951
  SSL-2:          0.9713
  Improvement:    +7.62%
  Status:         ✅ IMPROVED

✅ Output:        /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_with_final_label_v2.csv
✅ Models:        /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/models_stacked_v2
✅ Diagnostics:

SSL2 Diagnostics

In [ ]:
#!/usr/bin/env python3
"""
SSL-2 DIAGNOSTIC REPORT (Production)
Comprehensive health check for SSL-2 results:
✓ Label volume growth (SSL-1 → SSL-2)
✓ Class distributions (known only)
✓ Confidence / entropy / margin stats (overall & per-class)
✓ Mode collapse / distribution shift (with symmetric KL)
✓ Rare-class preservation (2 & 3)
✓ Pseudo-label consistency (SSL-1 vs SSL-2)
✓ Gold-set evaluation on human labels (F1 + confusion matrix)
✓ Red-flag detection with actionable hints
Outputs CSV/JSON summaries to diagnostics_ssl2/
"""

import os
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import warnings

warnings.filterwarnings("ignore")

# ===== CONFIG =====
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

CYCLE1_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label.csv")
CYCLE2_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label_v2.csv")
DIAG_DIR   = os.path.join(MANUAL_DIR, "diagnostics_ssl2")

os.makedirs(DIAG_DIR, exist_ok=True)

def safe_series(x, fill=0):
    """Return numeric series with safe fill and dtype."""
    if x is None:
        return pd.Series(dtype=float)
    s = pd.to_numeric(x, errors="coerce")
    return s.fillna(fill)

def softmax_rows(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-12, 1)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def sym_kl(p, q):
    """Symmetric KL divergence between two discrete distributions."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = np.clip(p, 1e-12, 1)
    q = np.clip(q, 1e-12, 1)
    return float(np.sum(p*np.log(p/q) + q*np.log(q/p)))

print("="*80)
print("SSL-2 COMPREHENSIVE DIAGNOSTIC REPORT (Production)")
print("="*80)

# ===== LOAD =====
df1 = pd.read_csv(CYCLE1_CSV)
df2 = pd.read_csv(CYCLE2_CSV)

print(f"\n✓ SSL-1 rows: {len(df1):,}")
print(f"✓ SSL-2 rows: {len(df2):,}")

# Presence checks (tolerant)
need_c2 = [
    "final_label_v2","predicted_class_v2","prediction_confidence_v2",
    "prediction_entropy_v2","prediction_margin_v2","is_pseudo_label_v2"
]
missing = [c for c in need_c2 if c not in df2.columns]
if missing:
    raise ValueError(f"SSL-2 file missing required columns: {missing}")

# Normalize expected basics
for c in ["human_label_clean"]:
    if c not in df1.columns: df1[c] = np.nan
    if c not in df2.columns: df2[c] = np.nan

for c in ["is_pseudo_label"]:
    if c not in df1.columns: df1[c] = False
    if c not in df2.columns: df2[c] = False

# ===== 1) LABEL VOLUME GROWTH =====
print("\n" + "="*80)
print("1) LABEL GROWTH (SSL-1 → SSL-2)")
print("="*80)

human_c1   = df1["human_label_clean"].notna().sum()
pseudo_c1  = int(df1.get("is_pseudo_label", pd.Series([False]*len(df1))).sum())
unknown_c1 = int((df1["final_label"] == -1).sum()) if "final_label" in df1.columns else int(len(df1) - human_c1 - pseudo_c1)

human_c2   = df2["human_label_clean"].notna().sum()
pseudo_c2  = int(df2["is_pseudo_label_v2"].sum())
unknown_c2 = int((df2["final_label_v2"] == -1).sum())

growth_df = pd.DataFrame({
    "Metric": ["Human", "Pseudo v1", "Pseudo v2", "Total Known", "Unknown"],
    "SSL-1": [human_c1, pseudo_c1, 0, human_c1 + pseudo_c1, unknown_c1],
    "SSL-2": [human_c2, 0, pseudo_c2, human_c2 + pseudo_c2, unknown_c2]
})
growth_df["Growth"] = growth_df["SSL-2"] - growth_df["SSL-1"]
print(growth_df.to_string(index=False))
growth_df.to_csv(os.path.join(DIAG_DIR, "01_label_growth.csv"), index=False)

# ===== 2) CLASS DISTRIBUTION (KNOWN ONLY) =====
print("\n" + "="*80)
print("2) CLASS DISTRIBUTION (KNOWN LABELS ONLY)")
print("="*80)

known_c1 = df1[df1.get("final_label", -1) != -1].copy()
known_c2 = df2[df2["final_label_v2"] != -1].copy()

dist_df = pd.DataFrame({"Class": [0,1,2,3]})
dist_df["SSL-1 Count"] = [int((known_c1["final_label"]==i).sum()) for i in range(4)]
dist_df["SSL-2 Count"] = [int((known_c2["final_label_v2"]==i).sum()) for i in range(4)]
dist_df["SSL-1 %"] = 100*dist_df["SSL-1 Count"]/max(1, dist_df["SSL-1 Count"].sum())
dist_df["SSL-2 %"] = 100*dist_df["SSL-2 Count"]/max(1, dist_df["SSL-2 Count"].sum())
dist_df["Δ %"] = dist_df["SSL-2 %"] - dist_df["SSL-1 %"]
print(dist_df.to_string(index=False))
dist_df.to_csv(os.path.join(DIAG_DIR, "02_class_distribution_known.csv"), index=False)

# ===== 3) CONFIDENCE / ENTROPY / MARGIN (overall) + per class =====
print("\n" + "="*80)
print("3) CONFIDENCE / ENTROPY / MARGIN")
print("="*80)

conf2   = safe_series(df2["prediction_confidence_v2"])
ent2    = safe_series(df2["prediction_entropy_v2"])
margin2 = safe_series(df2["prediction_margin_v2"])

metrics_overall = pd.DataFrame([{
    "conf_mean": conf2.mean(), "conf_std": conf2.std(), "conf_min": conf2.min(), "conf_max": conf2.max(),
    "entropy_mean": ent2.mean(), "entropy_std": ent2.std(), "entropy_max": ent2.max(),
    "margin_mean": margin2.mean(), "margin_std": margin2.std(), "margin_min": margin2.min(),
}])
print(metrics_overall.to_string(index=False))
metrics_overall.to_csv(os.path.join(DIAG_DIR, "03_overall_conf_entropy_margin.csv"), index=False)

# Per-class on known labels
per_class_rows = []
for c in [0,1,2,3]:
    sub = df2[df2["final_label_v2"]==c]
    if len(sub)==0:
        per_class_rows.append({"class": c, "n":0, "conf_mean":np.nan, "entropy_mean":np.nan, "margin_mean":np.nan})
        continue
    per_class_rows.append({
        "class": c,
        "n": int(len(sub)),
        "conf_mean": float(safe_series(sub["prediction_confidence_v2"]).mean()),
        "entropy_mean": float(safe_series(sub["prediction_entropy_v2"]).mean()),
        "margin_mean": float(safe_series(sub["prediction_margin_v2"]).mean())
    })
per_class_df = pd.DataFrame(per_class_rows)
print("\nPer-class stats (known only):")
print(per_class_df.to_string(index=False))
per_class_df.to_csv(os.path.join(DIAG_DIR, "03b_per_class_conf_entropy_margin.csv"), index=False)

# ===== 4) MODE COLLAPSE + DISTRIBUTION SHIFT =====
print("\n" + "="*80)
print("4) PREDICTION DISTRIBUTION & SHIFT (SSL-1 vs SSL-2)")
print("="*80)

pred1 = df1.get("predicted_class", pd.Series([], dtype=float))
pred2 = df2["predicted_class_v2"]

pred_dist_df = pd.DataFrame({"Class":[0,1,2,3]})
pred_dist_df["SSL-1 Pred Count"] = [int((pred1==i).sum()) for i in range(4)]
pred_dist_df["SSL-2 Pred Count"] = [int((pred2==i).sum()) for i in range(4)]
pred_dist_df["SSL-1 %"] = 100*pred_dist_df["SSL-1 Pred Count"]/max(1, pred_dist_df["SSL-1 Pred Count"].sum())
pred_dist_df["SSL-2 %"] = 100*pred_dist_df["SSL-2 Pred Count"]/max(1, pred_dist_df["SSL-2 Pred Count"].sum())
print(pred_dist_df.to_string(index=False))
pred_dist_df.to_csv(os.path.join(DIAG_DIR, "04_prediction_distribution.csv"), index=False)

collapse_pct = pred_dist_df["SSL-2 %"].max()
collapse_class = int(pred_dist_df.loc[pred_dist_df["SSL-2 %"].idxmax(),"Class"])
if collapse_pct > 85.0:
    print(f"\n  POSSIBLE MODE COLLAPSE: class {collapse_class} at {collapse_pct:.1f}% of predictions")
else:
    print("\n No mode collapse detected")

# Symmetric KL on prediction distributions (robustness check)
p = pred_dist_df["SSL-1 Pred Count"].values.astype(float)
q = pred_dist_df["SSL-2 Pred Count"].values.astype(float)
if p.sum()==0 or q.sum()==0:
    skld = np.nan
else:
    p /= p.sum(); q /= q.sum()
    skld = sym_kl(p, q)
print(f"Prediction distribution shift (sym. KL): {skld:.4f}")
pd.DataFrame([{"sym_kl": skld}]).to_csv(os.path.join(DIAG_DIR, "04b_pred_distribution_shift_symKL.csv"), index=False)

# ===== 5) RARE CLASS PRESERVATION =====
print("\n" + "="*80)
print("5) RARE CLASS PRESERVATION (classes 2 & 3)")
print("="*80)

rare_c1 = int(known_c1["final_label"].isin([2,3]).sum()) if "final_label" in known_c1.columns else 0
rare_c2 = int(known_c2["final_label_v2"].isin([2,3]).sum())
pct1 = 100*rare_c1/max(1,len(known_c1))
pct2 = 100*rare_c2/max(1,len(known_c2))
print(f"  SSL-1 rare: {rare_c1:,} ({pct1:.2f}%)")
print(f"  SSL-2 rare: {rare_c2:,} ({pct2:.2f}%)")
pd.DataFrame([{"ssl1_rare":rare_c1,"ssl2_rare":rare_c2,"ssl1_pct":pct1,"ssl2_pct":pct2}]).to_csv(
    os.path.join(DIAG_DIR, "05_rare_class_preservation.csv"), index=False
)

# ===== 6) PSEUDO-LABEL CONSISTENCY (SSL-1 vs SSL-2) =====
print("\n" + "="*80)
print("6) PSEUDO-LABEL CONSISTENCY (SSL-1 vs SSL-2)")
print("="*80)

if "predicted_class" in df2.columns and "is_pseudo_label" in df2.columns:
    m = df2["is_pseudo_label"].fillna(False)
    n_c1_pseudo = int(m.sum())
    agree = int((df2.loc[m, "predicted_class"] == df2.loc[m, "predicted_class_v2"]).sum())
    agree_pct = 100*agree/max(1,n_c1_pseudo)
    print(f"  SSL-1 pseudo rows: {n_c1_pseudo:,}")
    print(f"  Agreement with SSL-2: {agree_pct:.2f}%")
    pd.DataFrame([{"ssl1_pseudo": n_c1_pseudo, "agreement_pct": agree_pct}]).to_csv(
        os.path.join(DIAG_DIR, "06_pseudo_consistency.csv"), index=False
    )
else:
    print("  (Skipped: missing SSL-1 pseudo columns in SSL-2 file)")

# ===== 7) GOLD-SET EVALUATION (HUMAN LABELS) =====
print("\n" + "="*80)
print("7) GOLD-SET EVALUATION (on human_label_clean)")
print("="*80)

gold = df2[df2["human_label_clean"].notna()].copy()
if len(gold) > 0:
    y_true = pd.to_numeric(gold["human_label_clean"], errors="coerce")
    y_pred = pd.to_numeric(gold["predicted_class_v2"], errors="coerce")
    mask = (~y_true.isna()) & (~y_pred.isna())
    y_true = y_true[mask].astype(int)
    y_pred = y_pred[mask].astype(int)

    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm  = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    print(f"  Gold-set size: {len(y_true):,}")
    print(f"  Macro F1 vs human: {f1m:.4f}")
    cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in range(4)], columns=[f"pred_{i}" for i in range(4)])
    cm_df.to_csv(os.path.join(DIAG_DIR, "07_gold_confusion_matrix.csv"))
    pd.DataFrame([{"gold_macro_f1": f1m, "gold_n": len(y_true)}]).to_csv(
        os.path.join(DIAG_DIR, "07_gold_f1.csv"), index=False
    )
else:
    print("  No human labels found in SSL-2 file (human_label_clean).")
    pd.DataFrame([{"gold_macro_f1": np.nan, "gold_n": 0}]).to_csv(
        os.path.join(DIAG_DIR, "07_gold_f1.csv"), index=False
    )

# ===== 8) RED-FLAG DETECTION =====
print("\n" + "="*80)
print("8) RED-FLAG DETECTION")
print("="*80)

red_flags = []

# Unknown ratio
unk_ratio = unknown_c2 / max(1, len(df2))
if unk_ratio > 0.40:
    red_flags.append(f"High unknown ratio: {unk_ratio:.2%} (>40%). Consider relaxing SSL-2 guards slightly.")

# Entropy / margin
if ent2.mean() > 1.20:
    red_flags.append(f"High mean entropy: {ent2.mean():.3f}. Check calibration / thresholds.")
if margin2.mean() < 0.06:
    red_flags.append(f"Low mean margin: {margin2.mean():.3f}. Consider boosting separation (regularization/tuning).")

# Severe class imbalance (known labels)
min_count = dist_df["SSL-2 Count"].replace(0, np.nan).min()
max_count = dist_df["SSL-2 Count"].max()
if pd.notna(min_count):
    imb_ratio = max_count / max(1, min_count)
    if imb_ratio > 20:
        red_flags.append(f"Severe class imbalance in known labels: ~{imb_ratio:.1f}:1. Use class weights / quotas in SSL-3.")
else:
    red_flags.append("Some classes have 0 known samples in SSL-2. Ensure quotas/guards allow them.")

# Mode collapse
if collapse_pct > 85.0:
    red_flags.append(f"Prediction distribution is {collapse_pct:.1f}% on class {collapse_class}. Possible mode collapse.")

# Rare classes drop
if rare_c1 > 0 and rare_c2 < 0.8*rare_c1:
    red_flags.append("Rare classes (2/3) declined by >20%. Consider class-specific confidence gates.")

if len(red_flags)==0:
    print(" No red flags detected.")
else:
    for rf in red_flags:
        print(" ", rf)

# ===== SAVE SUMMARY =====
summary = {
    "rows_ssl1": int(len(df1)),
    "rows_ssl2": int(len(df2)),
    "human_ssl1": int(human_c1),
    "pseudo_ssl1": int(pseudo_c1),
    "unknown_ssl1": int(unknown_c1),
    "human_ssl2": int(human_c2),
    "pseudo_ssl2": int(pseudo_c2),
    "unknown_ssl2": int(unknown_c2),
    "conf_mean_ssl2": float(conf2.mean()),
    "entropy_mean_ssl2": float(ent2.mean()),
    "margin_mean_ssl2": float(margin2.mean()),
    "pred_symKL_ssl1_ssl2": float(sym_kl(p, q)) if p.sum()>0 and q.sum()>0 else np.nan,
    "red_flags": red_flags
}
with open(os.path.join(DIAG_DIR, "00_summary_report.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print(" DIAGNOSTIC COMPLETE")
print("="*80)
print(f"Reports saved to: {DIAG_DIR}")
print("  • 00_summary_report.json")
print("  • 01_label_growth.csv")
print("  • 02_class_distribution_known.csv")
print("  • 03_overall_conf_entropy_margin.csv")
print("  • 03b_per_class_conf_entropy_margin.csv")
print("  • 04_prediction_distribution.csv")
print("  • 04b_pred_distribution_shift_symKL.csv")
print("  • 05_rare_class_preservation.csv")
print("  • 06_pseudo_consistency.csv")
print("  • 07_gold_f1.csv")
print("  • 07_gold_confusion_matrix.csv")


SSL-2 COMPREHENSIVE DIAGNOSTIC REPORT (Production)

✓ SSL-1 rows: 83,289
✓ SSL-2 rows: 83,289

1) LABEL GROWTH (SSL-1 → SSL-2)
     Metric  SSL-1  SSL-2  Growth
      Human   3287   3287       0
  Pseudo v1  14601      0  -14601
  Pseudo v2      0  56638   56638
Total Known  17888  59925   42037
    Unknown  65401  25877  -39524

2) CLASS DISTRIBUTION (KNOWN LABELS ONLY)
 Class  SSL-1 Count  SSL-2 Count   SSL-1 %   SSL-2 %        Δ %
     0         4689        41691 26.213104 72.617223  46.404119
     1        11656        13977 65.161002 24.345085 -40.815917
     2         1140         1326  6.372987  2.309622  -4.063366
     3          403          418  2.252907  0.728071  -1.524836

3) CONFIDENCE / ENTROPY / MARGIN
 conf_mean  conf_std  conf_min  conf_max  entropy_mean  entropy_std  entropy_max  margin_mean  margin_std  margin_min
  0.877324  0.144232   0.28453  0.998901        0.3316     0.270908     1.378768     0.769995    0.275011    0.000033

Per-class stats (known only):
 clas

# SSL Cycle - 3

In [ ]:
#!/usr/bin/env python3
"""
SSL-3 PRO MODE
--------------
Enhancements implemented:
1) Quantitative monitoring of new class-2/3 pseudo-labels:
   - Store margin, entropy, and k-NN proximity to human-labeled points.
   - Estimate "in-the-wild" precision from rows pseudo-labeled in earlier cycles
     that later received human labels.

2) Dynamic threshold tuning for rare classes (2 & 3):
   - On mined pool, tune acceptance (confidence/entropy/margin) to maximize a
     proxy precision (neighbor agreement) subject to minimum soft quotas.
   - Tighten thresholds if prediction distribution sym. KL drift exceeds a cap.

3) Automated report aggregation & plotting:
   - Collect deltas across SSL-1/2/3 into a bundle.
   - Produce trend plots: label growth, class balance, F1, rare precision, symKL.
   - Emit a top-level FAIL flag if any fail-safe/regression triggers.

Inputs expected (from prior cycles):
- predictions_with_final_label.csv           (SSL-1)
- predictions_with_final_label_v2.csv        (SSL-2)
- selected_features_topK_4class_weighted.csv
- optimal_thresholds_4class.json
- lgb_vs_catboost_vs_ensemble.csv            (for alpha)
Optional (if present):
- ssl2_fold_metrics.csv
- diagnostics_ssl2/00_summary_report.json

Outputs:
- models_stacked_v3/
- predictions_with_final_label_v3.csv
- diagnostics_ssl3/* (csv/json + png plots)
"""

import os
import json
import math
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, confusion_matrix
)
import lightgbm as lgb
import catboost as cb
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# =================== PATHS / CONFIG ===================
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

CYCLE1_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label.csv")
CYCLE2_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label_v2.csv")
SELECTED_FEATURES = os.path.join(MANUAL_DIR, "selected_features_topK_4class_weighted.csv")
THRESHOLDS_JSON = os.path.join(MANUAL_DIR, "optimal_thresholds_4class.json")
METRICS_C1 = os.path.join(MANUAL_DIR, "lgb_vs_catboost_vs_ensemble.csv")
FOLD_METRICS_SSL2 = os.path.join(MANUAL_DIR, "ssl2_fold_metrics.csv")

MODEL_DIR_V3 = os.path.join(MANUAL_DIR, "models_stacked_v3")
OUT_CSV_V3 = os.path.join(MANUAL_DIR, "predictions_with_final_label_v3.csv")

DIAG_DIR = os.path.join(MANUAL_DIR, "diagnostics_ssl3")
os.makedirs(MODEL_DIR_V3, exist_ok=True)
os.makedirs(DIAG_DIR, exist_ok=True)

SEED = 42
N_CLASSES = 4
BASELINE_F1 = 0.8951

# -------- Rare-class soft quotas & guards (tunable) --------
# Desired totals after SSL-3 (soft; data-aware)
QUOTA_DESIRED = {2: 1200, 3: 1200}

# Class-specific confidence floors (base and relaxed)
CONF_MIN_DEFAULT = 0.88
CONF_MIN_RARE_BASE = {2: 0.86, 3: 0.86}
CONF_MIN_RARE_RELAX = {2: 0.84, 3: 0.84}
# Looser gate if quota unmet (still with entropy/margin guards)
QUOTA_LOOSE_CONF = {2: 0.60, 3: 0.45}
ENTROPY_MAX_RARE = 1.10
MARGIN_MIN_RARE = 0.08

# KL drift cap (if exceeded, tighten thresholds by +0.01 and reselect)
SYM_KL_CAP = 0.06
MAX_TIGHTEN_STEPS = 3

# Oversampling for minorities (post-quota)
OVERSAMPLE_ENABLED = True
OVERSAMPLE_TARGET_RATIO = 0.20   # Rare vs majority target ratio
OVERSAMPLE_MAX_MULT = 3

# Fail-safe tolerance for class-3 precision drop (pp)
PRECISION_DROP_TOL_PPTS = 5.0

# =================== UTILS ===================

def sym_kl(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0).astype(float)
    q = np.clip(q, eps, 1.0).astype(float)
    p /= p.sum()
    q /= q.sum()
    return 0.5 * (np.sum(p * np.log(p / q)) + np.sum(q * np.log(q / p)))

def compute_entropy(p):
    p = np.clip(p, 1e-10, 1 - 1e-10)
    return -(p * np.log(p)).sum(axis=1)

def predict_with_thresholds(proba, thresholds):
    preds = np.zeros(proba.shape[0], dtype=int)
    for i in range(proba.shape[0]):
        row = proba[i]
        adjusted = np.array([
            row[c] if row[c] >= float(thresholds[str(c)]) else -1.0
            for c in range(N_CLASSES)
        ])
        preds[i] = np.argmax(adjusted) if adjusted.max() >= -1.0 else np.argmax(row)
    return preds

def predict_ensemble(X, lgb_models, cat_models, thresholds, alphas, return_proba=False):
    proba_lgb = np.mean([m.predict(X) for m in lgb_models], axis=0)
    proba_cat = np.mean([m.predict_proba(X) for m in cat_models], axis=0)
    alpha = float(np.mean(alphas))
    proba = alpha * proba_lgb + (1 - alpha) * proba_cat
    if return_proba:
        preds = predict_with_thresholds(proba, thresholds)
        return preds, proba
    return predict_with_thresholds(proba, thresholds)

def compute_class_weights(y, multiplier=4.0, cap=50.0):
    vc = pd.Series(y).value_counts()
    total = len(y)
    w = {}
    for c in range(N_CLASSES):
        cnt = vc.get(c, 1)
        raw = total / (N_CLASSES * cnt)
        w[c] = min(raw * multiplier, cap)
    return w

def random_oversample(X, y, w, target_ratio=OVERSAMPLE_TARGET_RATIO, max_mult=OVERSAMPLE_MAX_MULT, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    vc = Counter(y)
    maj = max(vc, key=vc.get)
    maj_n = vc[maj]
    X_aug, y_aug, w_aug = [X], [y], [w]
    for c in range(N_CLASSES):
        if c == maj:
            continue
        c_n = vc.get(c, 0)
        if c_n == 0:
            continue
        desired = int(min(maj_n * target_ratio, c_n * max_mult))
        if desired > c_n:
            idx = np.where(y == c)[0]
            take = rng.choice(idx, size=desired - c_n, replace=True)
            X_aug.append(X[take]); y_aug.append(y[take]); w_aug.append(w[take])
    return np.vstack(X_aug), np.concatenate(y_aug), np.concatenate(w_aug)

def knn_agreement_scores(X_ref, y_ref, X_cand, cand_class, k=5, metric='cosine'):
    """
    For each candidate, compute:
      - mean distance to k NN from gold set
      - neighbor agreement ratio = fraction of NN with y_ref == cand_class
    """
    if len(X_ref) == 0:
        return np.ones(len(X_cand))*np.nan, np.zeros(len(X_cand))
    nn = NearestNeighbors(n_neighbors=min(k, len(X_ref)), metric=metric)
    nn.fit(X_ref)
    dists, idxs = nn.kneighbors(X_cand, return_distance=True)
    neigh_labels = y_ref[idxs]
    agree = (neigh_labels == cand_class).mean(axis=1)
    mean_dist = dists.mean(axis=1)
    return mean_dist, agree

def load_models_for_alpha_and_thresholds(thresholds_json, metrics_csv):
    with open(thresholds_json) as f:
        thresholds = json.load(f)["averaged_optimal_thresholds"]
    alphas = pd.read_csv(metrics_csv)["alpha"].tolist()
    return thresholds, alphas

def load_fold_f1_ssl2(path):
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            if "f1_macro" in df.columns:
                return float(df["f1_macro"].mean())
        except Exception:
            pass
    return None

# =================== LOAD DATA ===================
print("="*80)
print("SSL-3 PRO MODE")
print("="*80)

df1 = pd.read_csv(CYCLE1_CSV)  # For historical diagnostics
df2 = pd.read_csv(CYCLE2_CSV)
features = pd.read_csv(SELECTED_FEATURES)['feature'].tolist()
thresholds, alphas = load_models_for_alpha_and_thresholds(THRESHOLDS_JSON, METRICS_C1)

print(f"✓ Loaded SSL-1: {len(df1):,} rows")
print(f"✓ Loaded SSL-2: {len(df2):,} rows | {len(features)} features")

# Identify known and unknown from SSL-2
known = df2[df2["final_label_v2"] != -1].copy()
unk   = df2[df2["final_label_v2"] == -1].copy()

print(f"  Known (train base): {len(known):,}")
print(f"  Unknown pool:       {len(unk):,}")

# Gold set (human-labeled rows) for proximity and auditing
gold = df2[~df2.get("human_label_clean", pd.Series([np.nan]*len(df2))).isna()].copy()
X_gold = gold[features].fillna(0.0).values if len(gold)>0 else np.zeros((0, len(features)))
y_gold = gold["human_label_clean"].astype(int).values if len(gold)>0 else np.zeros((0,), dtype=int)

# =================== DYNAMIC RARE MINING (with tuning) ===================
# Mine candidates for classes 2 & 3 from UNK predicted by SSL-2, with guards.
def mine_candidates(class_id, base_conf, relaxed_conf, quota_target,
                    entropy_max=ENTROPY_MAX_RARE, margin_min=MARGIN_MIN_RARE,
                    tighten_steps=0):
    pool = unk[unk["predicted_class_v2"] == class_id].copy()
    if pool.empty or quota_target <= 0:
        return pool.iloc[0:0], {"accepted":0, "tighten_steps":tighten_steps, "used_relaxed":False}

    # Run two-stage selection: strict -> relaxed (if needed)
    strict = pool[
        (pool["prediction_confidence_v2"] >= base_conf) &
        (pool["prediction_entropy_v2"]    <= entropy_max) &
        (pool["prediction_margin_v2"]     >= margin_min)
    ].copy()
    strict = strict.sort_values("prediction_confidence_v2", ascending=False)

    out = strict.head(quota_target)
    used_relaxed = False
    if len(out) < quota_target:
        remaining = quota_target - len(out)
        relaxed = pool[
            (pool["prediction_confidence_v2"] >= relaxed_conf) &
            (pool["prediction_entropy_v2"]    <= entropy_max) &
            (pool["prediction_margin_v2"]     >= margin_min)
        ].copy().sort_values("prediction_confidence_v2", ascending=False)
        if not out.empty:
            relaxed = relaxed.loc[~relaxed.index.isin(out.index)]
        extra = relaxed.head(remaining)
        if len(extra) > 0:
            used_relaxed = True
            out = pd.concat([out, extra], axis=0)

    # k-NN proximity stats to gold set (proxy precision)
    X_cand = out[features].fillna(0.0).values
    mean_dist, neigh_agree = knn_agreement_scores(X_gold[y_gold==class_id],
                                                  y_gold[y_gold==class_id] if len(y_gold)>0 else y_gold,
                                                  X_cand, class_id, k=5, metric='cosine')
    out[f"ssl3_knn_mean_dist_c{class_id}"] = mean_dist
    out[f"ssl3_knn_agree_c{class_id}"] = neigh_agree

    return out, {"accepted": len(out), "tighten_steps": tighten_steps, "used_relaxed": used_relaxed}

# Decide how many to add (soft) based on already known counts
have = known["final_label_v2"].value_counts()
already_have = {2: int(have.get(2,0)), 3: int(have.get(3,0))}
add_quota = {c: max(0, QUOTA_DESIRED[c] - already_have.get(c,0)) for c in (2,3)}

print("\nSoft-quota mining (pre-tuning):")
print(f"  Have: class2={already_have[2]}, class3={already_have[3]}")
print(f"  Want: class2={add_quota[2]}, class3={add_quota[3]}")

# Initial selection (before KL tightening)
cand2, meta2 = mine_candidates(2, CONF_MIN_RARE_BASE[2], CONF_MIN_RARE_RELAX[2], add_quota[2])
cand3, meta3 = mine_candidates(3, CONF_MIN_RARE_BASE[3], CONF_MIN_RARE_RELAX[3], add_quota[3])

print(f"  Selected: c2={len(cand2)} (relaxed={meta2['used_relaxed']}), c3={len(cand3)} (relaxed={meta3['used_relaxed']})")

# KL drift check: compare SSL-2 pred distribution vs distribution after adding these labels
def current_pred_dist(df_pred_cols):
    return df_pred_cols.value_counts().reindex([0,1,2,3], fill_value=0).values.astype(float)

pred_v2 = current_pred_dist(df2["predicted_class_v2"])
# simulate effect (rough): counts of accepted will bias future training; still we cap via KL
added_counts = np.array([
    0,
    0,
    len(cand2),
    len(cand3)
], dtype=float)
pred_sim = pred_v2 + added_counts
kl = sym_kl(pred_v2, pred_sim)

tighten = 0
base_conf2, base_conf3 = CONF_MIN_RARE_BASE[2], CONF_MIN_RARE_BASE[3]
while kl > SYM_KL_CAP and tighten < MAX_TIGHTEN_STEPS:
    tighten += 1
    base_conf2 += 0.01; base_conf3 += 0.01
    cand2, meta2 = mine_candidates(2, base_conf2, CONF_MIN_RARE_RELAX[2], add_quota[2], tighten_steps=tighten)
    cand3, meta3 = mine_candidates(3, base_conf3, CONF_MIN_RARE_RELAX[3], add_quota[3], tighten_steps=tighten)
    added_counts = np.array([0,0,len(cand2),len(cand3)], dtype=float)
    pred_sim = pred_v2 + added_counts
    kl = sym_kl(pred_v2, pred_sim)

print(f"  KL tighten steps: {tighten}, final symKL(sim)={kl:.4f}")
print(f"  Final selected: c2={len(cand2)} | c3={len(cand3)}")

# Assemble training set for SSL-3
train = known.copy()
train = train.assign(ssl3_extra=False, ssl3_extra_weight=1.0)
extra = []
if len(cand2)>0:
    cand2 = cand2.assign(final_label_ssl3=2, ssl3_extra=True, ssl3_extra_weight=0.3)
    extra.append(cand2)
if len(cand3)>0:
    cand3 = cand3.assign(final_label_ssl3=3, ssl3_extra=True, ssl3_extra_weight=0.3)
    extra.append(cand3)
if len(extra)>0:
    train = pd.concat([train] + extra, axis=0, ignore_index=True)

# Build training arrays
y_train = np.where(train.get("final_label_ssl3").notna(), train["final_label_ssl3"], train["final_label_v2"]).astype(int)
X_train = train[features].fillna(0.0).values
w_base = train["ssl3_extra_weight"].values
class_w_map = compute_class_weights(y_train, multiplier=4.0, cap=50.0)
w_class = np.array([class_w_map[c] for c in y_train])
w_train = w_base * w_class

print("\nTraining set summary:")
print(f"  Size: {len(train):,}")
print(f"  Class counts: {dict(Counter(y_train))}")
print(f"  Extra mined rows: {int(train['ssl3_extra'].sum())}")

# Optional oversampling
if OVERSAMPLE_ENABLED:
    X_train, y_train, w_train = random_oversample(X_train, y_train, w_train,
                                                  target_ratio=OVERSAMPLE_TARGET_RATIO,
                                                  max_mult=OVERSAMPLE_MAX_MULT)
    print(f"  Oversampled size: {len(y_train):,}")

# =================== TRAIN SSL-3 ENSEMBLE ===================
print("\nTraining SSL-3 ensemble (5-fold, thresholded F1)...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lgb_models_v3, cat_models_v3 = [], []
fold_metrics = []

for fold, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_va = X_train[tr], X_train[va]
    y_tr, y_va = y_train[tr], y_train[va]
    w_tr, w_va = w_train[tr], w_train[va]

    # LightGBM
    params = dict(
        objective="multiclass",
        num_class=N_CLASSES,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.8,
        lambda_l2=1.0,
        metric="multi_logloss",
        seed=SEED,
        verbosity=-1
    )
    dtr = lgb.Dataset(X_tr, y_tr, weight=w_tr)
    dva = lgb.Dataset(X_va, y_va, weight=w_va, reference=dtr)
    booster = lgb.train(
        params, dtr,
        num_boost_round=800,
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    lgb_models_v3.append(booster)

    # CatBoost
    cw = compute_class_weights(y_tr, multiplier=4.0, cap=50.0)
    class_weights_cat = [cw.get(c,1.0) for c in range(N_CLASSES)]
    model_cat = cb.CatBoostClassifier(
        iterations=800,
        learning_rate=0.03,
        depth=6,
        loss_function="MultiClass",
        classes_count=N_CLASSES,
        class_weights=class_weights_cat,
        verbose=False
    )
    model_cat.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=(X_va, y_va), early_stopping_rounds=60, verbose=False)
    cat_models_v3.append(model_cat)

    # Thresholded F1
    proba_va = booster.predict(X_va)
    preds_va = predict_with_thresholds(proba_va, thresholds)
    f1 = f1_score(y_va, preds_va, average="macro", zero_division=0)
    fold_metrics.append({"fold": fold, "f1_macro": f1})
    print(f"  Fold {fold}/5  F1={f1:.4f}")

fold_df = pd.DataFrame(fold_metrics)
fold_df.to_csv(os.path.join(DIAG_DIR, "ssl3_fold_metrics.csv"), index=False)
mean_f1 = fold_df["f1_macro"].mean()
std_f1  = fold_df["f1_macro"].std()
print(f"\nMean F1 (thr.): {mean_f1:.4f} (± {std_f1:.4f})")

# Save models
for i, m in enumerate(lgb_models_v3, 1):
    m.save_model(os.path.join(MODEL_DIR_V3, f"lgb_fold_{i}.txt"))
for i, m in enumerate(cat_models_v3, 1):
    m.save_model(os.path.join(MODEL_DIR_V3, f"catboost_fold_{i}.cbm"))
print(f"✓ Models saved → {MODEL_DIR_V3}")

# =================== INFER SSL-3 ON ALL ===================
print("\nRe-predicting all rows with SSL-3 models...")
alphas_list = pd.read_csv(METRICS_C1)["alpha"].tolist()
X_all = df2[features].fillna(0.0).values
preds_v3, proba_v3 = predict_ensemble(X_all, lgb_models_v3, cat_models_v3, thresholds, alphas_list, return_proba=True)
conf_v3 = proba_v3.max(axis=1)
ent_v3  = compute_entropy(proba_v3)
marg_v3 = np.sort(proba_v3, axis=1)[:, -1] - np.sort(proba_v3, axis=1)[:, -2]

df2["predicted_class_v3"] = preds_v3
df2["prediction_confidence_v3"] = conf_v3
df2["prediction_entropy_v3"] = ent_v3
df2["prediction_margin_v3"] = marg_v3
for c in range(N_CLASSES):
    df2[f"proba_class_{c}_v3"] = proba_v3[:, c]

# Acceptance rule for SSL-3 pseudo
is_pseudo_v3 = (conf_v3 >= CONF_MIN_DEFAULT) & (ent_v3 <= 1.10) & (marg_v3 >= 0.08)
df2["is_pseudo_label_v3"] = is_pseudo_v3

def compute_final_label_v3(row):
    if not pd.isna(row.get("human_label_clean", np.nan)):
        return int(row["human_label_clean"])
    if row.get("is_pseudo_label_v3", False):
        return int(row["predicted_class_v3"])
    return -1

df2["final_label_v3"] = df2.apply(compute_final_label_v3, axis=1)

# =================== “IN THE WILD” PRECISION ESTIMATION ===================
# Use rows that were pseudo in earlier cycles but now have human labels.
def estimate_precision_from_newly_labeled(df_prev, prev_pred_col, prev_pseudo_flag_col, df_now):
    if ("human_label_clean" not in df_now.columns) or (prev_pred_col not in df_prev.columns):
        return {}
    # Join on an implicit row index (assuming consistent ordering); if an ID column exists, use it instead.
    idx_prev = df_prev.reset_index().rename(columns={"index":"_row_id"})
    idx_now  = df_now.reset_index().rename(columns={"index":"_row_id"})
    merged = pd.merge(idx_prev[["_row_id", prev_pred_col, prev_pseudo_flag_col]],
                      idx_now[["_row_id","human_label_clean"]],
                      on="_row_id", how="inner")
    merged = merged[merged[prev_pseudo_flag_col] == True]
    merged = merged[~merged["human_label_clean"].isna()]
    if merged.empty:
        return {}
    y_true = merged["human_label_clean"].astype(int).values
    y_pred = merged[prev_pred_col].astype(int).values
    pr, rc, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0,1,2,3], zero_division=0)
    out = {int(k): {"precision": float(pr[i]), "recall": float(rc[i]), "f1": float(f1[i]), "support": int(sup[i])}
           for i,k in enumerate([0,1,2,3])}
    out["macro_f1"] = float(np.mean(f1))
    return out

est_ssl1 = estimate_precision_from_newly_labeled(df1, "predicted_class", "is_pseudo_label", df2)
est_ssl2 = estimate_precision_from_newly_labeled(df2, "predicted_class_v2", "is_pseudo_label_v2", df2)

# Trend warning if precision for 2/3 drops > 5pp across cycles
trend_warn = []
def get_prec(d, cls):
    return d.get(cls,{}).get("precision") if d else None
for cls in [2,3]:
    p1 = get_prec(est_ssl1, cls)
    p2 = get_prec(est_ssl2, cls)
    if p1 is not None and p2 is not None and (p2 < p1 - 0.05):
        trend_warn.append(f"Estimated precision for class {cls} dropped: SSL-1={p1:.3f} → SSL-2={p2:.3f}")

# =================== FAIL-SAFE ON GOLD (class-3 precision) ===================
gold_now = df2[~df2["human_label_clean"].isna()].copy() if "human_label_clean" in df2.columns else pd.DataFrame()
fail_safe_triggered = False
if not gold_now.empty and "predicted_class_v2" in gold_now.columns:
    y_true = gold_now["human_label_clean"].astype(int).values
    y_pred_v2 = gold_now["predicted_class_v2"].astype(int).values
    pr2, _, _, _ = precision_recall_fscore_support(y_true, y_pred_v2, labels=[0,1,2,3], zero_division=0)
    prec_v2_c3 = pr2[3]

    y_pred_v3 = gold_now["predicted_class_v3"].astype(int).values
    pr3, _, _, _ = precision_recall_fscore_support(y_true, y_pred_v3, labels=[0,1,2,3], zero_division=0)
    prec_v3_c3 = pr3[3]

    drop_pp = (prec_v2_c3 - prec_v3_c3) * 100.0
    if prec_v2_c3 > 0 and drop_pp > PRECISION_DROP_TOL_PPTS:
        fail_safe_triggered = True
        # Roll back class-3 v3 predictions (no human label) to v2
        mask_nohuman = df2["human_label_clean"].isna()
        mask_c3_v3 = (df2["predicted_class_v3"] == 3) & mask_nohuman
        df2.loc[mask_c3_v3, "predicted_class_v3"] = df2.loc[mask_c3_v3, "predicted_class_v2"]
        df2["final_label_v3"] = df2.apply(compute_final_label_v3, axis=1)

# =================== KL SHIFT VS SSL-2 ===================
pred_dist_v2 = df2["predicted_class_v2"].value_counts().reindex([0,1,2,3], fill_value=0).values.astype(float)
pred_dist_v3 = df2["predicted_class_v3"].value_counts().reindex([0,1,2,3], fill_value=0).values.astype(float)
symkl = sym_kl(pred_dist_v2, pred_dist_v3)
pd.DataFrame({
    "class":[0,1,2,3],
    "count_v2": pred_dist_v2.astype(int),
    "count_v3": pred_dist_v3.astype(int)
}).to_csv(os.path.join(DIAG_DIR, "ssl3_prediction_distribution.csv"), index=False)
pd.DataFrame({"symKL":[symkl]}).to_csv(os.path.join(DIAG_DIR, "ssl3_prediction_distribution_shift_symKL.csv"), index=False)

# =================== SAVE OUTPUT ===================
df2.to_csv(OUT_CSV_V3, index=False)

# =================== REPORT AGGREGATION & PLOTS ===================
ssl2_f1 = load_fold_f1_ssl2(FOLD_METRICS_SSL2)
bundle = {
    "ssl1": {
        "rows": len(df1),
        "pseudo": int(df1["is_pseudo_label"].sum()) if "is_pseudo_label" in df1.columns else None,
        "unknown": int((df1["final_label"]==-1).sum()) if "final_label" in df1.columns else None
    },
    "ssl2": {
        "rows": len(df2),
        "pseudo": int(df2["is_pseudo_label_v2"].sum()) if "is_pseudo_label_v2" in df2.columns else None,
        "unknown": int((df2["final_label_v2"]==-1).sum()) if "final_label_v2" in df2.columns else None,
        "fold_f1_mean": ssl2_f1
    },
    "ssl3": {
        "rows": len(df2),
        "pseudo": int(df2["is_pseudo_label_v3"].sum()),
        "unknown": int((df2["final_label_v3"]==-1).sum()),
        "fold_f1_mean": float(mean_f1),
        "fold_f1_std": float(std_f1),
        "symKL_vs_ssl2": float(symkl),
        "fail_safe_triggered": bool(fail_safe_triggered),
        "trend_warnings": trend_warn
    },
    "rare_quota": {
        "already_have": already_have,
        "added_c2": int(len(cand2)),
        "added_c3": int(len(cand3)),
        "base_conf_used": {"2": base_conf2, "3": base_conf3}
    },
    "in_the_wild_precision": {
        "ssl1_pseudo_then_labeled": est_ssl1,
        "ssl2_pseudo_then_labeled": est_ssl2
    }
}
with open(os.path.join(DIAG_DIR, "ssl3_bundle.json"), "w") as f:
    json.dump(bundle, f, indent=2)

# ---- Plots ----
def save_plot(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)

# Label growth plot
def count_known(df, col):
    return int((df[col]!=-1).sum()) if col in df.columns else None

known1 = count_known(df1, "final_label")
known2 = count_known(df2, "final_label_v2")
known3 = count_known(df2, "final_label_v3")
x = ["SSL-1","SSL-2","SSL-3"]
y = [known1, known2, known3]
fig = plt.figure()
plt.plot(x, y, marker='o')
plt.title("Known Labels Growth")
plt.ylabel("# Known")
save_plot(fig, os.path.join(DIAG_DIR, "plot_known_growth.png"))

# Class balance bars (SSL-2 vs SSL-3)
k2 = df2[df2["final_label_v2"]!=-1]["final_label_v2"].value_counts().reindex([0,1,2,3], fill_value=0)
k3 = df2[df2["final_label_v3"]!=-1]["final_label_v3"].value_counts().reindex([0,1,2,3], fill_value=0)
fig = plt.figure()
width = 0.35
idx = np.arange(4)
plt.bar(idx - width/2, k2.values, width, label="SSL-2")
plt.bar(idx + width/2, k3.values, width, label="SSL-3")
plt.xticks(idx, [0,1,2,3])
plt.title("Known Class Distribution")
plt.legend()
save_plot(fig, os.path.join(DIAG_DIR, "plot_class_balance_ssl2_vs_ssl3.png"))

# F1 trend
f1_ssl2 = ssl2_f1 if ssl2_f1 is not None else np.nan
fig = plt.figure()
plt.plot(["Baseline","SSL-2","SSL-3"], [BASELINE_F1, f1_ssl2, mean_f1], marker='o')
plt.title("Macro F1 Trend (thresholded)")
plt.ylabel("F1")
save_plot(fig, os.path.join(DIAG_DIR, "plot_f1_trend.png"))

# Rare precision trend (estimated) if available
def get_prec_for(d, cls):
    return d.get(cls,{}).get("precision") if d else None
p2_ssl1 = get_prec_for(est_ssl1, 2)
p3_ssl1 = get_prec_for(est_ssl1, 3)
p2_ssl2 = get_prec_for(est_ssl2, 2)
p3_ssl2 = get_prec_for(est_ssl2, 3)
if any(v is not None for v in [p2_ssl1,p2_ssl2,p3_ssl1,p3_ssl2]):
    fig = plt.figure()
    xs = [1,2]
    if (p2_ssl1 is not None) and (p2_ssl2 is not None):
        plt.plot(xs, [p2_ssl1, p2_ssl2], marker='o', label="Class 2 est. precision")
    if (p3_ssl1 is not None) and (p3_ssl2 is not None):
        plt.plot(xs, [p3_ssl1, p3_ssl2], marker='o', label="Class 3 est. precision")
    plt.xticks(xs, ["SSL-1","SSL-2"])
    plt.ylim(0,1)
    plt.legend()
    plt.title("Estimated Rare-Class Precision (in the wild)")
    save_plot(fig, os.path.join(DIAG_DIR, "plot_estimated_precision_trend.png"))

# =================== FINAL SUMMARY ===================
print("\n" + "="*80)
print("SSL-3 PRO MODE — SUMMARY")
print("="*80)
print(f"Saved dataset: {OUT_CSV_V3}")
print(f"Models:       {MODEL_DIR_V3}")
print(f"Diagnostics:  {DIAG_DIR}")
print(f"Fold F1:      {mean_f1:.4f} (± {std_f1:.4f})")
print(f"symKL vs SSL-2 predictions: {symkl:.4f}")
if fail_safe_triggered:
    print("Fail-safe:     triggered for class-3 precision (rollback applied)")
if trend_warn:
    for w in trend_warn:
        print(f"Trend:         {w}")
print("="*80)


SSL-3 PRO MODE


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_with_final_label.csv'

diagnostics

In [ ]:
#!/usr/bin/env python3
"""
SSL-3 Diagnostic Report
------------------------
Comprehensive health check for predictions_with_final_label_v3.csv:

✓ Label growth from SSL-2 → SSL-3
✓ Known class distribution
✓ Confidence / entropy / margin health
✓ Mode collapse detection
✓ sym-KL prediction shift vs SSL-2
✓ Rare-class preservation check
✓ Gold-set precision & F1 vs human labels
✓ Red-flag detection
"""

import os
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import entropy as KL
import warnings

warnings.filterwarnings("ignore")

# ===== PATHS =====
BASE_DIR = "/content/drive/MyDrive/ML_Project_Data"
MANUAL_DIR = os.path.join(BASE_DIR, "Manual_labelled_data")

SSL2_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label_v2.csv")
SSL3_CSV = os.path.join(MANUAL_DIR, "predictions_with_final_label_v3.csv")

DIAG_DIR = os.path.join(MANUAL_DIR, "diagnostics_ssl3")
os.makedirs(DIAG_DIR, exist_ok=True)


print("="*80)
print("SSL-3 DIAGNOSTICS")
print("="*80)

# ===== LOAD DATA =====
df2 = pd.read_csv(SSL2_CSV)
df3 = pd.read_csv(SSL3_CSV)

print(f"\n✓ SSL-2 rows: {len(df2):,}")
print(f"✓ SSL-3 rows: {len(df3):,}")

# ===== 1. LABEL VOLUME =====
print("\n" + "="*80)
print("1) LABEL VOLUME (SSL-2 → SSL-3)")
print("="*80)

human = df3["human_label_clean"].notna().sum() if "human_label_clean" in df3.columns else 0
pseudo2 = df2["is_pseudo_label_v2"].sum()
pseudo3 = df3["is_pseudo_label_v3"].sum() if "is_pseudo_label_v3" in df3.columns else 0
unknown2 = (df2["final_label_v2"] == -1).sum()
unknown3 = (df3["final_label_v3"] == -1).sum()

vol_df = pd.DataFrame({
    "Metric": ["Human", "Pseudo-2", "Pseudo-3", "Total Known", "Unknown"],
    "SSL-2": [human, pseudo2, 0, human + pseudo2, unknown2],
    "SSL-3": [human, 0, pseudo3, human + pseudo3, unknown3],
})
print(vol_df.to_string(index=False))
vol_df.to_csv(os.path.join(DIAG_DIR, "01_label_volume.csv"), index=False)

# ===== 2. CLASS DISTRIBUTION =====
print("\n" + "="*80)
print("2) CLASS DISTRIBUTION (KNOWN ONLY)")
print("="*80)

known3 = df3[df3["final_label_v3"] != -1]

class_counts = known3["final_label_v3"].value_counts().sort_index()
total_known = len(known3)

dist_df = pd.DataFrame({
    "Class": [0, 1, 2, 3],
    "Count": [class_counts.get(i, 0) for i in range(4)],
    "%": [100 * class_counts.get(i, 0) / total_known for i in range(4)]
})
print(dist_df.to_string(index=False))
dist_df.to_csv(os.path.join(DIAG_DIR, "02_class_distribution.csv"), index=False)

# ===== 3. CONFIDENCE / ENTROPY / MARGIN =====
print("\n" + "="*80)
print("3) CONFIDENCE / ENTROPY / MARGIN")
print("="*80)

conf = df3["prediction_confidence_v3"].values
ent = df3["prediction_entropy_v3"].values
margin = df3["prediction_margin_v3"].values

stats = {
    "Metric": ["conf_mean", "conf_std", "conf_min", "conf_max",
               "entropy_mean", "entropy_std", "entropy_max",
               "margin_mean", "margin_std", "margin_min"],
    "SSL-3": [
        conf.mean(), conf.std(), conf.min(), conf.max(),
        ent.mean(), ent.std(), ent.max(),
        margin.mean(), margin.std(), margin.min()
    ]
}
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))
stats_df.to_csv(os.path.join(DIAG_DIR, "03_conf_entropy_margin.csv"), index=False)

# ===== 4. MODE COLLAPSE CHECK =====
print("\n" + "="*80)
print("4) PREDICTION DISTRIBUTION (MODE COLLAPSE?)")
print("="*80)

pred_dist2 = df2["predicted_class_v2"].value_counts().sort_index()
pred_dist3 = df3["predicted_class_v3"].value_counts().sort_index()

mode_df = pd.DataFrame({
    "Class": [0, 1, 2, 3],
    "SSL-2 Count": [pred_dist2.get(i, 0) for i in range(4)],
    "SSL-3 Count": [pred_dist3.get(i, 0) for i in range(4)],
})
mode_df["SSL-2 %"] = 100 * mode_df["SSL-2 Count"] / mode_df["SSL-2 Count"].sum()
mode_df["SSL-3 %"] = 100 * mode_df["SSL-3 Count"] / mode_df["SSL-3 Count"].sum()
print(mode_df.to_string(index=False))
mode_df.to_csv(os.path.join(DIAG_DIR, "04_prediction_distribution.csv"), index=False)

if mode_df["SSL-3 %"].max() > 85:
    print("  WARNING: Mode collapse possible")
else:
    print(" No mode collapse detected")

# ===== 5. sym-KL SHIFT =====
print("\n" + "="*80)
print("5) PREDICTION SHIFT (sym-KL between SSL-2 and SSL-3)")
print("="*80)

# Convert to normalized distribution
p2 = mode_df["SSL-2 %"].values / 100
p3 = mode_df["SSL-3 %"].values / 100

# sym-KL
symkl = 0.5 * (KL(p2, p3) + KL(p3, p2))
print(f"sym-KL (SSL-2 ↔ SSL-3): {symkl:.6f}")
pd.DataFrame({"sym_kl": [symkl]}).to_csv(os.path.join(DIAG_DIR, "05_symkl.csv"), index=False)

# ===== 6. RARE CLASS PRESERVATION =====
print("\n" + "="*80)
print("6) RARE CLASS PRESERVATION (classes 2,3)")
print("="*80)

rare_total = class_counts.get(2, 0) + class_counts.get(3, 0)
print(f"  Rare total: {rare_total:,} ({100*rare_total/total_known:.2f}%)")

if rare_total < 300:
    print("  Very few rare-class samples (check SSL-4 / active learning)")
else:
    print(" Rare-class count acceptable")

# ===== 7. GOLD SET EVALUATION (human label ground truth) =====
print("\n" + "="*80)
print("7) GOLD SET PERFORMANCE vs human_label_clean")
print("="*80)

if "human_label_clean" in df3.columns:
    gold = df3[df3["human_label_clean"].notna()]
    if len(gold) > 0:
        y_true = gold["human_label_clean"].astype(int)
        y_pred = gold["predicted_class_v3"]
        f1 = (classification_report(y_true, y_pred, output_dict=True)
              ["macro avg"]["f1-score"])

        print(f"  Gold-set size: {len(gold):,}")
        print(f"  Macro F1: {f1:.4f}")
        pd.DataFrame(confusion_matrix(y_true, y_pred)).to_csv(
            os.path.join(DIAG_DIR, "06_gold_confusion_matrix.csv"), index=False
        )
        print(" Saved confusion matrix")
    else:
        print("No gold labels found.")
else:
    print("No human_label_clean column — skipping.")

# ===== 8. RED FLAG DETECTION =====
print("\n" + "="*80)
print("8) RED FLAG SCAN")
print("="*80)

red_flags = []

if symkl > 0.10:
    red_flags.append(f"High prediction shift (sym-KL={symkl:.4f})")

if conf.mean() < 0.80:
    red_flags.append("Low mean confidence")

if ent.mean() > 1.00:
    red_flags.append("High entropy → uncertain predictions")

if rare_total < 200:
    red_flags.append("Rare-class count critically low")

if len(red_flags) == 0:
    print(" No red flags detected")
else:
    for f in red_flags:
        print(" ", f)

summary = {
    "ssl_stage": 3,
    "total_rows": len(df3),
    "human_labels": int(human),
    "pseudo_v3": int(pseudo3),
    "unknown": int(unknown3),
    "mean_conf": float(conf.mean()),
    "mean_entropy": float(ent.mean()),
    "mean_margin": float(margin.mean()),
    "sym_kl": float(symkl),
    "rare_total": int(rare_total),
    "rare_pct": float(100*rare_total/total_known),
    "red_flags": red_flags,
}

with open(os.path.join(DIAG_DIR, "00_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print(" SSL-3 DIAGNOSTIC COMPLETE")
print("="*80)
print(f"Reports saved → {DIAG_DIR}")


SSL-3 DIAGNOSTICS

✓ SSL-2 rows: 83,289
✓ SSL-3 rows: 83,289

1) LABEL VOLUME (SSL-2 → SSL-3)
     Metric  SSL-2  SSL-3
      Human   3287   3287
   Pseudo-2  56638      0
   Pseudo-3      0  59940
Total Known  59925  63227
    Unknown  25877  22415

2) CLASS DISTRIBUTION (KNOWN ONLY)
 Class  Count         %
     0  43228 71.012255
     1  15337 25.194664
     2   1880  3.088346
     3    429  0.704734

3) CONFIDENCE / ENTROPY / MARGIN
      Metric    SSL-3
   conf_mean 0.886288
    conf_std 0.161391
    conf_min 0.327204
    conf_max 0.999782
entropy_mean 0.257243
 entropy_std 0.263519
 entropy_max 1.352151
 margin_mean 0.778112
  margin_std 0.318853
  margin_min 0.000008

4) PREDICTION DISTRIBUTION (MODE COLLAPSE?)
 Class  SSL-2 Count  SSL-3 Count   SSL-2 %   SSL-3 %
     0        60136        58595 72.201611 70.351427
     1        20441        21571 24.542256 25.898978
     2         2136         2653  2.564564  3.185295
     3          576          470  0.691568  0.564300
✅ No mod

# Co training

In [ ]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.6 MB/s eta 0:00:00


**Correct Final Co training**

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import lightgbm as lgb
import catboost as cb
import warnings
from collections import Counter
warnings.filterwarnings("ignore")


MODEL_SAVE_DIR = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/models_stacked_v2'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


# Load features and data
selected_features = pd.read_csv('/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/selected_features_topK_4class_weighted.csv')['feature'].tolist()
mid_point = len(selected_features) // 2
viewA_features = selected_features[:mid_point]
viewB_features = selected_features[mid_point:]

df = pd.read_csv('/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_with_final_label_v3.csv')

# Split labeled into train/validation
df_labeled = df[df['final_label_v3'] != -1].copy()
df_train, df_val = train_test_split(df_labeled, test_size=0.1, stratify=df_labeled['final_label_v3'], random_state=42)

X_train_A = df_train[viewA_features].fillna(0).values
X_train_B = df_train[viewB_features].fillna(0).values
y_train = df_train['final_label_v3'].values

# Initialize separate label arrays for each view (CRITICAL FIX)
y_train_A = y_train.copy()
y_train_B = y_train.copy()

X_val_A = df_val[viewA_features].fillna(0).values
X_val_B = df_val[viewB_features].fillna(0).values
y_val = df_val['final_label_v3'].values

df_unlabeled = df[df['final_label_v3'] == -1].copy()
X_unlabeled_A = df_unlabeled[viewA_features].fillna(0).values
X_unlabeled_B = df_unlabeled[viewB_features].fillna(0).values

confidence_thresh = 0.88

print("="*80)
print("CO-TRAINING INITIALIZATION")
print("="*80)
print(f"View A features: {len(viewA_features)}")
print(f"View B features: {len(viewB_features)}")
print(f"Training samples: {len(df_train):,}")
print(f"Validation samples: {len(df_val):,}")
print(f"Unlabeled samples: {len(df_unlabeled):,}")
print(f"Initial class distribution: {dict(Counter(y_train))}")
print("="*80 + "\n")


def train_view_models(X, y, X_val, y_val, seed=42):
    # LightGBM (unchanged; it's already fast on CPU)
    params_lgb = {
        'objective': 'multiclass',
        'num_class': 4,
        'learning_rate': 0.05,
        'num_leaves': 64,
        'metric': 'multi_logloss',
        'seed': seed,
        'verbosity': -1,
        'force_col_wise': True
    }
    dtrain = lgb.Dataset(X, label=y)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    lgb_model = lgb.train(
        params_lgb, dtrain, num_boost_round=300,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    # CatBoost with safe GPU→CPU fallback
    def fit_cat(task_type):
        model = cb.CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function='MultiClass',
            task_type=task_type,
            verbose=False,
            random_seed=seed,
            early_stopping_rounds=30
        )
        model.fit(X, y, eval_set=(X_val, y_val))
        return model

    try:
        cat_model = fit_cat('GPU')   # try GPU first
    except Exception as e:
        print(f"⚠️ CatBoost GPU unavailable ({e}). Falling back to CPU.")
        cat_model = fit_cat('CPU')   # CPU fallback

    return lgb_model, cat_model



# Initialize models with labeled data
lgb_A, cat_A = train_models(X_train_A, X_train_B, y_train_A, y_train_B, seed=42)
lgb_B, cat_B = train_models(X_train_B, X_train_A, y_train_B, y_train_A, seed=43)

unlabeled_pool_indices = df_unlabeled.index.to_list()

# Track statistics across iterations
iteration_stats = []

for iteration in range(5):
    print("="*80)
    print(f"CO-TRAINING ITERATION {iteration+1}/5")
    print("="*80)

    if len(unlabeled_pool_indices) == 0:
        print("No more unlabeled samples. Early stop.")
        break

    print(f"Unlabeled pool size: {len(unlabeled_pool_indices):,}")

    # Predict on unlabeled samples with both models
    X_uA = df.loc[unlabeled_pool_indices, viewA_features].fillna(0).values
    X_uB = df.loc[unlabeled_pool_indices, viewB_features].fillna(0).values

    preds_A, confs_A, _ = predict(X_uA, lgb_A, cat_A)
    preds_B, confs_B, _ = predict(X_uB, lgb_B, cat_B)

    # Agreement filter: both models agree AND both confident
    agreed_mask = (preds_A == preds_B) & (confs_A >= confidence_thresh) & (confs_B >= confidence_thresh)

    agreed_indices = np.array(unlabeled_pool_indices)[agreed_mask]
    agreed_labels = preds_A[agreed_mask]

    # Remove agreed samples from unlabeled pool
    unlabeled_pool_indices = list(set(unlabeled_pool_indices) - set(agreed_indices))

    print(f"Samples pseudo-labeled: {len(agreed_indices):,}")

    if len(agreed_labels) > 0:
        class_counts = Counter(agreed_labels)
        print(f"Class distribution: {dict(class_counts)}")

    if len(agreed_indices) == 0:
        print("No confident agreed pseudo-labels. Stopping early.")
        break

    # Add new pseudo-labeled samples to training sets
    X_new_for_A = df.loc[agreed_indices, viewA_features].fillna(0).values
    y_new_for_A = agreed_labels

    X_new_for_B = df.loc[agreed_indices, viewB_features].fillna(0).values
    y_new_for_B = agreed_labels

    # CORRECTED: Append to view-specific arrays
    X_train_A = np.vstack([X_train_A, X_new_for_B])
    y_train_A = np.concatenate([y_train_A, y_new_for_B])  # ✅ Append to y_train_A

    X_train_B = np.vstack([X_train_B, X_new_for_A])
    y_train_B = np.concatenate([y_train_B, y_new_for_A])  # ✅ Append to y_train_B

    print(f"Training set sizes — Model A: {len(y_train_A):,}, Model B: {len(y_train_B):,}")

    # Validation monitoring
    val_preds_A, _, _ = predict(X_val_A, lgb_A, cat_A)
    val_f1_A = f1_score(y_val, val_preds_A, average='macro', zero_division=0)

    val_preds_B, _, _ = predict(X_val_B, lgb_B, cat_B)
    val_f1_B = f1_score(y_val, val_preds_B, average='macro', zero_division=0)

    print(f"Validation Macro F1 — Model A: {val_f1_A:.4f}, Model B: {val_f1_B:.4f}")

    # Track iteration statistics
    iteration_stats.append({
        'iteration': iteration + 1,
        'pseudo_labeled': len(agreed_indices),
        'unlabeled_remaining': len(unlabeled_pool_indices),
        'val_f1_A': val_f1_A,
        'val_f1_B': val_f1_B,
        'class_dist': dict(class_counts) if len(agreed_labels) > 0 else {}
    })

    # Retrain models with early stopping
    dtrain_A = lgb.Dataset(X_train_A, label=y_train_A)
    dval_A = lgb.Dataset(X_val_A, label=y_val, reference=dtrain_A)
    params_lgb = {
        'objective': 'multiclass',
        'num_class': 4,
        'learning_rate': 0.03,
        'num_leaves': 64,
        'metric': 'multi_logloss',
        'seed': 42 + iteration,
        'verbosity': -1
    }
    lgb_A = lgb.train(
        params_lgb, dtrain_A, num_boost_round=500,
        valid_sets=[dval_A],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=-1)]
    )

    dtrain_B = lgb.Dataset(X_train_B, label=y_train_B)
    dval_B = lgb.Dataset(X_val_B, label=y_val, reference=dtrain_B)
    params_lgb['seed'] = 43 + iteration
    lgb_B = lgb.train(
        params_lgb, dtrain_B, num_boost_round=500,
        valid_sets=[dval_B],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=-1)]
    )

    cat_A = cb.CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=6,
        loss_function='MultiClass', random_seed=42 + iteration,
        verbose=False, early_stopping_rounds=50
    )
    cat_A.fit(X_train_A, y_train_A, eval_set=(X_val_A, y_val))

    cat_B = cb.CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=6,
        loss_function='MultiClass', random_seed=43 + iteration,
        verbose=False, early_stopping_rounds=50
    )
    cat_B.fit(X_train_B, y_train_B, eval_set=(X_val_B, y_val))

    # Save models each iteration
    lgb_A.save_model(os.path.join(MODEL_SAVE_DIR, f"lgb_model_A_iter_{iteration+1}.txt"))
    lgb_B.save_model(os.path.join(MODEL_SAVE_DIR, f"lgb_model_B_iter_{iteration+1}.txt"))
    cat_A.save_model(os.path.join(MODEL_SAVE_DIR, f"catboost_model_A_iter_{iteration+1}.cbm"))
    cat_B.save_model(os.path.join(MODEL_SAVE_DIR, f"catboost_model_B_iter_{iteration+1}.cbm"))

    print(f"Models saved for iteration {iteration+1}.")
    print(f"Unlabeled remaining: {len(unlabeled_pool_indices):,}\n")


print("\n" + "="*80)
print("CO-TRAINING COMPLETED")
print("="*80)

# Print iteration summary
print("\nIteration Summary:")
print("-" * 80)
for stat in iteration_stats:
    print(f"Iter {stat['iteration']}: Pseudo-labeled={stat['pseudo_labeled']:,}, "
          f"Remaining={stat['unlabeled_remaining']:,}, "
          f"F1_A={stat['val_f1_A']:.4f}, F1_B={stat['val_f1_B']:.4f}")
print("-" * 80)

# ====================================================================
# FINAL PREDICTION (SELECTIVE LABELING)
# ====================================================================

print("\n" + "="*80)
print("FINAL PREDICTION: SELECTIVE LABELING")
print("="*80)

# Predict on full dataset with both models
X_full_A = df[viewA_features].fillna(0).values
X_full_B = df[viewB_features].fillna(0).values
preds_A, conf_A, proba_A = predict(X_full_A, lgb_A, cat_A)
preds_B, conf_B, proba_B = predict(X_full_B, lgb_B, cat_B)

# Agreement + confidence filter
agreed_mask = (preds_A == preds_B) & (conf_A >= confidence_thresh) & (conf_B >= confidence_thresh)

# Averaged predictions
proba_avg = (proba_A + proba_B) / 2
final_preds = np.argmax(proba_avg, axis=1)

# Initialize with -1 (unlabeled)
df['co_train_final_pred'] = -1
df['co_train_confidence'] = 0.0

# Assign predictions only where both models agreed and were confident
df.loc[agreed_mask, 'co_train_final_pred'] = final_preds[agreed_mask]
df.loc[agreed_mask, 'co_train_confidence'] = proba_avg.max(axis=1)[agreed_mask]

# Preserve existing labels from SSL-3
mask_already_labeled = df['final_label_v3'] != -1
df.loc[mask_already_labeled, 'co_train_final_pred'] = df.loc[mask_already_labeled, 'final_label_v3']

# Summary statistics
n_already_labeled = mask_already_labeled.sum()
n_new_pseudo = (agreed_mask & ~mask_already_labeled).sum()
n_still_unlabeled = (df['co_train_final_pred'] == -1).sum()

print(f"\nFinal Summary:")
print(f"  Already labeled (from SSL-3): {n_already_labeled:,}")
print(f"  New pseudo-labels (co-training): {n_new_pseudo:,}")
print(f"  Still unlabeled (for LP): {n_still_unlabeled:,}")
print(f"  Total labeled: {(df['co_train_final_pred'] != -1).sum():,}")

# Class distribution of new co-training pseudo-labels
cotrain_pseudo = df.loc[(agreed_mask & ~mask_already_labeled), 'co_train_final_pred']
if len(cotrain_pseudo) > 0:
    print(f"\nClass distribution of new co-training pseudo-labels:")
    print(dict(Counter(cotrain_pseudo)))

# Save output
output_path = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/predictions_co_training_final.csv'
df.to_csv(output_path, index=False)
print(f"\nOutput saved to: {output_path}")
print("Ready for label propagation.")
print("="*80)


**FINAL CORRECT CO TRAINING**

In [ ]:

#!/usr/bin/env python3
"""
Co-Training with Semantic View Split - CORRECTED & OPTIMIZED
============================================================
Fixes:
- Proper view separation (each model trained/predicts on one view)
- Saves co_train_proba_0..3 for ROC/Brier metrics
- Saves original_index for LP alignment
- GPU acceleration for CatBoost
- Parallel prediction batching
- Early stopping optimizations

Output: Replaces predictions_co_training_final.csv and models in models_stacked_v2/
"""

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import lightgbm as lgb
import catboost as cb
import warnings
from collections import Counter
import gc

warnings.filterwarnings("ignore")

# ========================================================================
# CONFIGURATION
# ========================================================================
MODEL_SAVE_DIR = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/models_stacked_v2'
DATA_DIR = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

CONFIDENCE_THRESH_INITIAL = 0.88
CONFIDENCE_DECAY = 0.03
NUM_ITERATIONS = 5
RANDOM_SEED = 42
BATCH_SIZE = 10000  # For large dataset predictions

# GPU settings (CatBoost will auto-detect)
USE_GPU = True

np.random.seed(RANDOM_SEED)

# ========================================================================
# SEMANTIC VIEW SPLIT
# ========================================================================

print("="*80)
print("CO-TRAINING WITH OPTIMIZED SEMANTIC VIEW SPLIT")
print("="*80)

# VIEW A: Social/Linguistic/Text/Engagement features (~64 features)
viewA_features = [
    'sentiment_score', 'sentiment_score_z', 'news_sentiment_score', 'news_sentiment_score_z',
    'sentiment_homogeneity', 'sentiment_homogeneity_z', 'social_news_sentiment_delta',
    'social_news_sentiment_delta_z', 'emotional_news_delta', 'A_Emotional_News_Delta',
    'A_Emotional_News_Delta_z',
    'headline_similarity_score', 'headline_similarity_score_z', 'text_similarity_top_posts',
    'text_similarity_top_posts_z', 'post_length_words', 'post_length_words_z',
    'post_length_chars', 'post_length_chars_z',
    'crowd_keyword_count', 'crowd_keyword_count_z', 'news_keyword_count', 'news_keyword_count_z',
    'ner_mentions_count', 'ner_mentions_count_z', 'ticker_repetition_window',
    'ticker_repetition_window_z', 'jargon_prevalence',
    'user_network_influence', 'user_network_influence_z', 'reply_chain_length',
    'reply_chain_length_z', 'comment_reply_count', 'comment_reply_count_z',
    'user_posting_frequency', 'user_posting_frequency_z', 'upvote_like_ratio',
    'upvote_like_ratio_z',
    'social_proof_velocity', 'social_proof_velocity_z', 'initial_engagement_velocity',
    'initial_engagement_velocity_z', 'H_Social_Momentum_Ratio', 'H_Social_Momentum_Ratio_z',
    'H_Network_Influence_Clustering', 'H_Network_Influence_Clustering_z',
    'H_Jargon_Sentiment_Homogeneity', 'H_Jargon_Sentiment_Homogeneity_z',
    'A_Signal_Novelty', 'A_Signal_Novelty_z',
    'media_richness_flag', 'recency_keyword_flag',
    'emo_emb_1', 'emo_emb_3', 'emo_emb_7', 'emo_emb_31', 'emo_emb_34', 'emo_emb_43',
    'post_id', 'post_id_z'
]

# VIEW B: Market/Temporal/Financial/Volume features (~136 features)
viewB_features = [
    'intraday_return_24hr', 'intraday_return_24hr_z', 'intraday_return_30min',
    'intraday_return_30min_z', 'daily_return_daily', 'rolling_5d_return_daily',
    'rolling_10d_return_daily', 'rolling_20d_return_daily', 'rolling_30d_return_daily',
    'intraday_volatility', 'intraday_volatility_z', 'rolling_5d_volatility_daily',
    'rolling_10d_volatility_daily', 'rolling_20d_volatility_daily', 'rolling_30d_volatility_daily',
    'last_bar_volume', 'prior_volume_daily', 'prior_day_volume', 'volume_vs_prior_avg',
    'rolling_5d_volume_daily', 'rolling_10d_volume_daily', 'rolling_20d_volume_daily',
    'rolling_30d_volume_daily', 'volume_spike_ratio', 'volume_spike_ratio_z',
    'trading_volume_spike_flag',
    'H_Retail_Volume_Float_Rotation', 'H_Retail_Volume_Float_Rotation_z',
    'inferred_float_rotation', 'inferred_float_rotation_z',
    'last_available_open', 'last_available_high', 'last_available_low',
    'last_available_close', 'prior_day_close', 'prior_close_daily',
    'price_change_from_prior_close', 'High_daily', 'Low_daily', 'Open_daily',
    'time_offset_from_event', 'time_offset_from_event_z', 'temporal_clustering_flag',
    'fed_z',
    'emo_emb_0', 'emo_emb_2', 'emo_emb_4', 'emo_emb_5', 'emo_emb_8', 'emo_emb_11',
    'emo_emb_12', 'emo_emb_13', 'emo_emb_16', 'emo_emb_18', 'emo_emb_19', 'emo_emb_20',
    'emo_emb_22', 'emo_emb_28', 'emo_emb_29', 'emo_emb_30', 'emo_emb_37', 'emo_emb_38',
    'emo_emb_45', 'emo_emb_48', 'emo_emb_49', 'emo_emb_55', 'emo_emb_56', 'emo_emb_59',
    'emo_emb_60', 'emo_emb_61', 'emo_emb_62', 'emo_emb_63', 'emo_emb_64', 'emo_emb_65',
    'emo_emb_66', 'emo_emb_67', 'emo_emb_68', 'emo_emb_69', 'emo_emb_70', 'emo_emb_71',
    'emo_emb_72', 'emo_emb_73', 'emo_emb_74', 'emo_emb_75', 'emo_emb_76', 'emo_emb_77',
    'emo_emb_78', 'emo_emb_79', 'emo_emb_80', 'emo_emb_81', 'emo_emb_82', 'emo_emb_83',
    'emo_emb_84', 'emo_emb_85', 'emo_emb_86', 'emo_emb_87', 'emo_emb_88', 'emo_emb_89',
    'emo_emb_90', 'emo_emb_91', 'emo_emb_92', 'emo_emb_93', 'emo_emb_94', 'emo_emb_95',
    'emo_emb_96', 'emo_emb_97', 'emo_emb_98', 'emo_emb_99', 'emo_emb_100', 'emo_emb_101',
    'emo_emb_102', 'emo_emb_103', 'emo_emb_104', 'emo_emb_105', 'emo_emb_106', 'emo_emb_107',
    'emo_emb_108', 'emo_emb_109', 'emo_emb_110', 'emo_emb_111', 'emo_emb_112', 'emo_emb_113',
    'emo_emb_114', 'emo_emb_115', 'emo_emb_116', 'emo_emb_117', 'emo_emb_118', 'emo_emb_119',
    'emo_emb_120', 'emo_emb_121', 'emo_emb_129', 'emo_emb_130', 'emo_emb_131', 'emo_emb_132',
    'emo_emb_133', 'emo_emb_134', 'emo_emb_135', 'emo_emb_136', 'emo_emb_137',
    'emo_val_code'
]

print(f"\nView A (Social/Linguistic): {len(viewA_features)} features")
print(f"View B (Market/Temporal): {len(viewB_features)} features")

overlap = set(viewA_features) & set(viewB_features)
if overlap:
    raise ValueError(f"Views overlap: {overlap}")
print("✓ No feature overlap")

# ========================================================================
# LOAD DATA
# ========================================================================
df = pd.read_csv(f'{DATA_DIR}/predictions_with_final_label_v3.csv')
print(f"\nTotal rows: {len(df):,}")

# Save original_index for LP alignment
if 'original_index' not in df.columns:
    df['original_index'] = np.arange(len(df))
    print("✓ Saved original_index column")

# Check missing features
missing_A = [f for f in viewA_features if f not in df.columns]
missing_B = [f for f in viewB_features if f not in df.columns]
if missing_A or missing_B:
    print(f"\n⚠️  WARNING: Missing features!")
    if missing_A:
        print(f"   View A missing: {missing_A[:5]}...")
    if missing_B:
        print(f"   View B missing: {missing_B[:5]}...")
    viewA_features = [f for f in viewA_features if f in df.columns]
    viewB_features = [f for f in viewB_features if f in df.columns]
    print(f"   Adjusted: View A={len(viewA_features)}, View B={len(viewB_features)}")

# ========================================================================
# TRAIN/VAL SPLIT
# ========================================================================
df_labeled = df[df['final_label_v3'] != -1].copy()
df_train, df_val = train_test_split(
    df_labeled, test_size=0.1, stratify=df_labeled['final_label_v3'],
    random_state=RANDOM_SEED
)

print(f"\nLabeled: {len(df_labeled):,}")
print(f"  Training: {len(df_train):,}")
print(f"  Validation: {len(df_val):,}")
print(f"Unlabeled: {(df['final_label_v3'] == -1).sum():,}")
print(f"Class distribution: {dict(Counter(df_train['final_label_v3']))}")
print("="*80 + "\n")

# Prepare matrices
X_train_A = df_train[viewA_features].fillna(0).values.astype(np.float32)
X_train_B = df_train[viewB_features].fillna(0).values.astype(np.float32)
y_train_A = df_train['final_label_v3'].values.copy()
y_train_B = df_train['final_label_v3'].values.copy()

X_val_A = df_val[viewA_features].fillna(0).values.astype(np.float32)
X_val_B = df_val[viewB_features].fillna(0).values.astype(np.float32)
y_val = df_val['final_label_v3'].values

unlabeled_pool_indices = df[df['final_label_v3'] == -1].index.to_list()

# ========================================================================
# MODEL TRAINING FUNCTIONS (CORRECTED)
# ========================================================================

def train_view_models(X, y, X_val, y_val, seed=42):
    # LightGBM (unchanged; it's already fast on CPU)
    params_lgb = {
        'objective': 'multiclass',
        'num_class': 4,
        'learning_rate': 0.05,
        'num_leaves': 64,
        'metric': 'multi_logloss',
        'seed': seed,
        'verbosity': -1,
        'force_col_wise': True
    }
    dtrain = lgb.Dataset(X, label=y)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    lgb_model = lgb.train(
        params_lgb, dtrain, num_boost_round=300,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    # CatBoost with safe GPU→CPU fallback
    def fit_cat(task_type):
        model = cb.CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function='MultiClass',
            task_type=task_type,
            verbose=False,
            random_seed=seed,
            early_stopping_rounds=30
        )
        model.fit(X, y, eval_set=(X_val, y_val))
        return model

    try:
        cat_model = fit_cat('GPU')   # try GPU first
    except Exception as e:
        print(f"⚠️ CatBoost GPU unavailable ({e}). Falling back to CPU.")
        cat_model = fit_cat('CPU')   # CPU fallback

    return lgb_model, cat_model



def predict_view_batched(X, lgb_model, cat_model, batch_size=BATCH_SIZE):
    """Batched prediction for memory efficiency."""
    n_samples = X.shape[0]
    proba_final = np.zeros((n_samples, 4), dtype=np.float32)

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        Xb = X[start:end]

        proba_lgb = lgb_model.predict(Xb)
        proba_cat = cat_model.predict_proba(Xb)
        proba_final[start:end] = (proba_lgb + proba_cat) / 2.0

    preds = np.argmax(proba_final, axis=1)
    confs = proba_final.max(axis=1)
    return preds, confs, proba_final


# ========================================================================
# INITIAL TRAINING
# ========================================================================
print("Training initial models...")
lgb_A, cat_A = train_view_models(X_train_A, y_train_A, X_val_A, y_val, seed=RANDOM_SEED)
print("  Model A trained")
lgb_B, cat_B = train_view_models(X_train_B, y_train_B, X_val_B, y_val, seed=RANDOM_SEED + 1)
print("  Model B trained")

# Validation
val_preds_A, _, _ = predict_view_batched(X_val_A, lgb_A, cat_A)
val_f1_A = f1_score(y_val, val_preds_A, average='macro', zero_division=0)

val_preds_B, _, _ = predict_view_batched(X_val_B, lgb_B, cat_B)
val_f1_B = f1_score(y_val, val_preds_B, average='macro', zero_division=0)

print(f"\nInitial validation F1 — Model A: {val_f1_A:.4f}, Model B: {val_f1_B:.4f}")

f1_A_pc = f1_score(y_val, val_preds_A, average=None, zero_division=0)
f1_B_pc = f1_score(y_val, val_preds_B, average=None, zero_division=0)
print(f"  View A per-class: {[f'{x:.3f}' for x in f1_A_pc]}")
print(f"  View B per-class: {[f'{x:.3f}' for x in f1_B_pc]}")

if val_f1_B < 0.70:
    print("\n⚠️  WARNING: View B F1 < 0.70. Co-training may not help.\n")
elif val_f1_B < 0.80:
    print("\n⚠️  NOTICE: View B F1 < 0.80. Monitor closely.\n")
else:
    print("\n✅ Both views strong. Co-training should help.\n")

iteration_stats = []

# ========================================================================
# CO-TRAINING LOOP
# ========================================================================
for iteration in range(NUM_ITERATIONS):
    print("="*80)
    print(f"CO-TRAINING ITERATION {iteration+1}/{NUM_ITERATIONS}")
    print("="*80)

    if len(unlabeled_pool_indices) == 0:
        print("No more unlabeled samples.")
        break

    confidence_thresh = max(0.70, CONFIDENCE_THRESH_INITIAL - (iteration // 2) * CONFIDENCE_DECAY)
    print(f"Confidence threshold: {confidence_thresh:.2f}")
    print(f"Unlabeled pool: {len(unlabeled_pool_indices):,}")

    # Predict on unlabeled
    X_uA = df.loc[unlabeled_pool_indices, viewA_features].fillna(0).values.astype(np.float32)
    X_uB = df.loc[unlabeled_pool_indices, viewB_features].fillna(0).values.astype(np.float32)

    preds_A, confs_A, _ = predict_view_batched(X_uA, lgb_A, cat_A)
    preds_B, confs_B, _ = predict_view_batched(X_uB, lgb_B, cat_B)

    # Agreement filter
    both_confident = (confs_A >= confidence_thresh) & (confs_B >= confidence_thresh)
    agreed_mask = (preds_A == preds_B) & both_confident

    print(f"  Both confident: {both_confident.sum():,} ({100*both_confident.sum()/len(unlabeled_pool_indices):.1f}%)")
    print(f"  Agreed: {agreed_mask.sum():,} ({100*agreed_mask.sum()/len(unlabeled_pool_indices):.1f}%)")

    if agreed_mask.sum() == 0:
        print("  No new pseudo-labels. Stopping.")
        break

    agreed_indices = np.array(unlabeled_pool_indices)[agreed_mask]
    agreed_labels = preds_A[agreed_mask]
    unlabeled_pool_indices = list(set(unlabeled_pool_indices) - set(agreed_indices))

    print(f"  Pseudo-labeled: {len(agreed_indices):,}")
    class_counts = Counter(agreed_labels)
    print(f"  Class dist: {dict(class_counts)}")

    # Add new samples (each view gets samples from SAME view)
    X_new_A = df.loc[agreed_indices, viewA_features].fillna(0).values.astype(np.float32)
    X_new_B = df.loc[agreed_indices, viewB_features].fillna(0).values.astype(np.float32)

    X_train_A = np.vstack([X_train_A, X_new_A])
    y_train_A = np.concatenate([y_train_A, agreed_labels])

    X_train_B = np.vstack([X_train_B, X_new_B])
    y_train_B = np.concatenate([y_train_B, agreed_labels])

    print(f"  Training sizes: A={len(y_train_A):,}, B={len(y_train_B):,}")

    # Retrain
    lgb_A, cat_A = train_view_models(X_train_A, y_train_A, X_val_A, y_val, seed=RANDOM_SEED + iteration)
    lgb_B, cat_B = train_view_models(X_train_B, y_train_B, X_val_B, y_val, seed=RANDOM_SEED + 1 + iteration)

    # Validation
    val_preds_A, _, _ = predict_view_batched(X_val_A, lgb_A, cat_A)
    val_f1_A = f1_score(y_val, val_preds_A, average='macro', zero_division=0)

    val_preds_B, _, _ = predict_view_batched(X_val_B, lgb_B, cat_B)
    val_f1_B = f1_score(y_val, val_preds_B, average='macro', zero_division=0)

    print(f"  Val F1: A={val_f1_A:.4f}, B={val_f1_B:.4f}")

    iteration_stats.append({
        'iteration': iteration + 1,
        'pseudo_labeled': len(agreed_indices),
        'remaining': len(unlabeled_pool_indices),
        'f1_A': val_f1_A,
        'f1_B': val_f1_B,
        'thresh': confidence_thresh
    })

    # Save models
    lgb_A.save_model(f'{MODEL_SAVE_DIR}/lgb_model_A_iter_{iteration+1}.txt')
    lgb_B.save_model(f'{MODEL_SAVE_DIR}/lgb_model_B_iter_{iteration+1}.txt')
    cat_A.save_model(f'{MODEL_SAVE_DIR}/catboost_model_A_iter_{iteration+1}.cbm')
    cat_B.save_model(f'{MODEL_SAVE_DIR}/catboost_model_B_iter_{iteration+1}.cbm')

    gc.collect()
    print()

# ========================================================================
# SUMMARY & FINAL PREDICTIONS
# ========================================================================
print("="*80)
print("CO-TRAINING COMPLETED")
print("="*80)
print("\nIteration Summary:")
for s in iteration_stats:
    print(f"Iter {s['iteration']}: Labeled={s['pseudo_labeled']:,}, Remain={s['remaining']:,}, "
          f"F1_A={s['f1_A']:.4f}, F1_B={s['f1_B']:.4f}, T={s['thresh']:.2f}")

total_added = sum(s['pseudo_labeled'] for s in iteration_stats)
print(f"\nTotal pseudo-labels: {total_added:,}\n")

# Save stats
if iteration_stats:
    pd.DataFrame(iteration_stats).to_csv(f'{DATA_DIR}/cotraining_iteration_stats.csv', index=False)

# Final predictions on full dataset
print("="*80)
print("FINAL PREDICTIONS")
print("="*80)

X_full_A = df[viewA_features].fillna(0).values.astype(np.float32)
X_full_B = df[viewB_features].fillna(0).values.astype(np.float32)

print("Predicting with Model A...")
preds_A, conf_A, proba_A = predict_view_batched(X_full_A, lgb_A, cat_A)
print("Predicting with Model B...")
preds_B, conf_B, proba_B = predict_view_batched(X_full_B, lgb_B, cat_B)

# Agreement filter
final_thresh = iteration_stats[-1]['thresh'] if iteration_stats else CONFIDENCE_THRESH_INITIAL
agreed_mask = (preds_A == preds_B) & (conf_A >= final_thresh) & (conf_B >= final_thresh)

# Average probabilities
proba_avg = (proba_A + proba_B) / 2.0
final_preds = np.argmax(proba_avg, axis=1)

# Initialize output columns
df['co_train_final_pred'] = -1
df['co_train_confidence'] = 0.0

# Assign where agreed
df.loc[agreed_mask, 'co_train_final_pred'] = final_preds[agreed_mask].astype(int)
df.loc[agreed_mask, 'co_train_confidence'] = proba_avg.max(axis=1)[agreed_mask]

# Preserve SSL-3 labels
mask_ssl3 = df['final_label_v3'] != -1
df.loc[mask_ssl3, 'co_train_final_pred'] = df.loc[mask_ssl3, 'final_label_v3'].astype(int)

# Save probabilities for metrics
for k in range(4):
    df[f'co_train_proba_{k}'] = proba_avg[:, k].astype(np.float32)

n_ssl3 = int(mask_ssl3.sum())
n_new = int((agreed_mask & ~mask_ssl3).sum())
n_unlabeled = int((df['co_train_final_pred'] == -1).sum())

print(f"\nFinal Summary:")
print(f"  SSL-3 labels: {n_ssl3:,}")
print(f"  New co-train: {n_new:,}")
print(f"  Still unlabeled: {n_unlabeled:,}")
print(f"  Total labeled: {(df['co_train_final_pred'] != -1).sum():,}")

# Save
output_path = f'{DATA_DIR}/predictions_co_training_final.csv'
df.to_csv(output_path, index=False)
print(f"\n✓ Saved: {output_path}")
print("✓ Ready for Label Propagation")
print("="*80)

CO-TRAINING WITH OPTIMIZED SEMANTIC VIEW SPLIT

View A (Social/Linguistic): 60 features
View B (Market/Temporal): 140 features
✓ No feature overlap

Total rows: 83,289
✓ Saved original_index column

Labeled: 60,874
  Training: 54,786
  Validation: 6,088
Unlabeled: 22,415
Class distribution: {0: 38905, 1: 13803, 2: 1692, 3: 386}

Training initial models...
  Model A trained
  Model B trained

Initial validation F1 — Model A: 0.9815, Model B: 0.7796
  View A per-class: ['0.999', '0.998', '0.987', '0.943']
  View B per-class: ['0.894', '0.655', '0.834', '0.736']

⚠️  NOTICE: View B F1 < 0.80. Monitor closely.

CO-TRAINING ITERATION 1/5
Confidence threshold: 0.88
Unlabeled pool: 22,415
  Both confident: 5,727 (25.5%)
  Agreed: 4,023 (17.9%)
  Pseudo-labeled: 4,023
  Class dist: {np.int64(0): 3913, np.int64(1): 34, np.int64(2): 76}
  Training sizes: A=58,809, B=58,809
  Val F1: A=0.9815, B=0.7823

CO-TRAINING ITERATION 2/5
Confidence threshold: 0.88
Unlabeled pool: 18,392
  Both confident: 

**Label Check**

In [ ]:
import pandas as pd

# Load your list of selected features for the project
selected_features_path = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/selected_features_topK_4class_weighted.csv'
selected_features = pd.read_csv(selected_features_path)['feature'].tolist()

# Split the features into View A and View B
mid_point = len(selected_features) // 2
viewA_features = selected_features[:mid_point]
viewB_features = selected_features[mid_point:]

# Print feature lists for each view
print("==== Features used in Model/View A (usually text/social/linguistic) ====")
for i, f in enumerate(viewA_features, 1):
    print(f"{i:3d}. {f}")

print("\n==== Features used in Model/View B (usually market/temporal/financial) ====")
for i, f in enumerate(viewB_features, 1):
    print(f"{i:3d}. {f}")

# OPTIONAL: Save each list to a CSV file for reporting/inspection
pd.Series(viewA_features).to_csv('/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/features_viewA.csv', index=False)
pd.Series(viewB_features).to_csv('/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/features_viewB.csv', index=False)


==== Features used in Model/View A (usually text/social/linguistic) ====
  1. headline_similarity_score
  2. sentiment_homogeneity
  3. crowd_keyword_count
  4. headline_similarity_score_z
  5. news_keyword_count
  6. reply_chain_length
  7. social_proof_velocity
  8. sentiment_homogeneity_z
  9. post_id
 10. initial_engagement_velocity
 11. news_sentiment_score
 12. user_network_influence
 13. social_proof_velocity_z
 14. sentiment_score
 15. time_offset_from_event
 16. upvote_like_ratio
 17. post_length_words
 18. post_length_chars
 19. H_Retail_Volume_Float_Rotation
 20. intraday_return_24hr
 21. text_similarity_top_posts
 22. crowd_keyword_count_z
 23. ticker_repetition_window
 24. H_Social_Momentum_Ratio
 25. intraday_return_30min
 26. H_Network_Influence_Clustering
 27. intraday_volatility
 28. last_available_open
 29. volume_spike_ratio
 30. comment_reply_count
 31. price_change_from_prior_close
 32. A_Emotional_News_Delta
 33. post_length_chars_z
 34. H_Jargon_Sentiment_Homogen

# Label Propagation

In [ ]:
#!/usr/bin/env python3
"""
Label Propagation (Final SSL Stage) - CORRECTED VERSION
=========================================================
Integrates Human + SSL-3 + Co-Training labels in priority cascade.
Uses KNN kernel with auto-tuning, HELD-OUT VALIDATION, confidence filtering,
comprehensive diagnostics, SHAP analysis, and model saving.

Inputs:
 - predictions_co_training_final.csv  (output from co-training)
 - selected_features_topK_4class_weighted.csv

Outputs:
 - predictions_labelprop_final.csv
 - labelprop_model.joblib
 - final_lgb_model.joblib (optional)
 - labelprop_plots/ (all diagnostic plots)
"""

import os
import logging
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.semi_supervised import LabelPropagation
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
import joblib

# Optional libs
try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False
    print("LightGBM not available. Skipping final classifier training.")

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False
    print("SHAP not available. Skipping SHAP analysis.")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s: %(message)s")

# ====================================================================
# CONFIGURATION
# ====================================================================

DATA_DIR = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data'
FEATURE_FILE = os.path.join(DATA_DIR, 'selected_features_topK_4class_weighted.csv')
INPUT_FILE = os.path.join(DATA_DIR, 'predictions_co_training_final.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'predictions_labelprop_final.csv')
PLOT_DIR = os.path.join(DATA_DIR, 'labelprop_plots')
MODEL_DIR = os.path.join(DATA_DIR, 'models')
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Label columns
label_col_ssl3 = 'final_label_v3'
label_col_cotrain = 'co_train_final_pred'
UNLABELED_VALUE = -1

# LabelPropagation params
DEFAULT_N_NEIGHBORS = 15
AUTO_TUNE_NEIGHBORS = True
CANDIDATE_NEIGHBORS = [7, 15, 25, 35]
LP_MAX_ITER = 1000

# Confidence filtering
CONFIDENCE_THRESHOLD = 0.85

# Final supervised model
TRAIN_FINAL_CLASSIFIER = True
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, 'final_lgb_model.joblib')
LP_MODEL_PATH = os.path.join(MODEL_DIR, 'labelprop_model.joblib')

RANDOM_SEED = 42
NEIGHBOR_TUNE_FOLDS = 3

# Plot styles
sns.set(style='whitegrid')

# ====================================================================
# UTILITY FUNCTIONS
# ====================================================================

def safe_load_features(feature_csv_path):
    """Load feature list from CSV."""
    df = pd.read_csv(feature_csv_path)
    if 'feature' not in df.columns:
        raise KeyError(f"Feature CSV must contain column 'feature'. Found: {list(df.columns)}")
    return df['feature'].tolist()


def tune_n_neighbors(X, y_labeled, candidate_list, folds=3, random_state=RANDOM_SEED):
    """
    Tune n_neighbors for LabelPropagation via stratified CV.
    Safety checks for class imbalance.
    """
    logging.info("Tuning n_neighbors via Stratified CV on labeled subset...")

    class_counts = Counter(y_labeled)
    min_class_count = min(class_counts.values())
    max_folds = min(folds, min_class_count, len(y_labeled) // 2)

    if max_folds < 2:
        logging.warning(f"Not enough samples for CV (min class count: {min_class_count}). "
                       f"Using default n_neighbors={DEFAULT_N_NEIGHBORS}")
        return DEFAULT_N_NEIGHBORS, {}

    skf = StratifiedKFold(n_splits=max_folds, shuffle=True, random_state=random_state)
    results = {}

    for k in candidate_list:
        scores = []
        try:
            for train_idx, val_idx in skf.split(X, y_labeled):
                y_masked = np.array(y_labeled, copy=True)
                y_masked[val_idx] = UNLABELED_VALUE

                lp = LabelPropagation(kernel='knn', n_neighbors=k, max_iter=LP_MAX_ITER)
                lp.fit(X, y_masked)
                y_pred = lp.transduction_[val_idx]
                acc = accuracy_score(y_labeled[val_idx], y_pred)
                scores.append(acc)

            mean_score = float(np.mean(scores))
            results[k] = mean_score
            logging.info(f"  k={k}: mean acc={mean_score:.4f}")
        except Exception as e:
            logging.warning(f"  k={k}: failed ({e})")
            continue

    if not results:
        logging.warning(f"All tuning failed. Using default n_neighbors={DEFAULT_N_NEIGHBORS}")
        return DEFAULT_N_NEIGHBORS, {}

    best_k = sorted(results.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    logging.info(f"Selected n_neighbors={best_k} (used {max_folds}-fold CV)")
    return best_k, results


def compute_entropy_from_distributions(dist_arr):
    """Compute entropy of posterior distributions."""
    eps = 1e-12
    return -(dist_arr * np.log(dist_arr + eps)).sum(axis=1)


# ====================================================================
# MAIN PIPELINE
# ====================================================================

def main():
    np.random.seed(RANDOM_SEED)

    logging.info("="*80)
    logging.info("LABEL PROPAGATION - FINAL SSL STAGE")
    logging.info("="*80)

    # Load features
    logging.info("Loading selected features...")
    selected_features = safe_load_features(FEATURE_FILE)

    # Load co-training output
    logging.info("Loading co-training output...")
    df = pd.read_csv(INPUT_FILE)

    if label_col_ssl3 not in df.columns:
        raise KeyError(f"Column '{label_col_ssl3}' not found")
    if label_col_cotrain not in df.columns:
        logging.warning(f"Column '{label_col_cotrain}' not found. Using only SSL-3 labels.")

    # Prepare feature matrix
    X = df[selected_features].fillna(0.0).values

    # Priority cascade: Human + SSL-3 → Co-training → Unlabeled
    logging.info("Building label array (priority: Human+SSL-3 → Co-training → Unlabeled)...")
    y = df[label_col_ssl3].copy()
    if label_col_cotrain in df.columns:
        y = y.fillna(df[label_col_cotrain])
    y = y.fillna(UNLABELED_VALUE).astype(int).values

    n_samples, n_features = X.shape
    n_labeled = (y != UNLABELED_VALUE).sum()
    n_unlabeled = (y == UNLABELED_VALUE).sum()
    n_from_ssl3 = df[label_col_ssl3].notna().sum()
    n_from_cotrain = 0
    if label_col_cotrain in df.columns:
        n_from_cotrain = (df[label_col_ssl3].isna() & df[label_col_cotrain].notna()).sum()

    logging.info(f"Total samples: {n_samples:,}")
    logging.info(f"Features: {n_features}")
    logging.info(f"Labeled samples: {n_labeled:,}")
    logging.info(f"  - From human/SSL-3: {n_from_ssl3:,}")
    logging.info(f"  - Added by co-training: {n_from_cotrain:,}")
    logging.info(f"Unlabeled samples: {n_unlabeled:,}")

    class_counts = Counter(y[y != UNLABELED_VALUE])
    logging.info(f"Class distribution: {dict(class_counts)}")

    # ================================================================
    # CREATE HELD-OUT VALIDATION SET (CRITICAL FIX)
    # ================================================================

    logging.info("\n" + "="*80)
    logging.info("CREATING HELD-OUT VALIDATION SET")
    logging.info("="*80)

    labeled_mask = (y != UNLABELED_VALUE)
    labeled_indices = np.where(labeled_mask)[0]

    if len(labeled_indices) < 10:
        logging.warning("Not enough labeled samples for validation. Skipping.")
        y_for_lp = y.copy()
        val_idx = np.array([])
    else:
        train_idx, val_idx = train_test_split(
            labeled_indices,
            test_size=0.2,
            stratify=y[labeled_indices],
            random_state=RANDOM_SEED
        )

        # Mask validation samples as unlabeled for LP training
        y_for_lp = y.copy()
        y_for_lp[val_idx] = UNLABELED_VALUE

        logging.info(f"Train (visible to LP): {len(train_idx):,} samples")
        logging.info(f"Validation (held out): {len(val_idx):,} samples")

    # ================================================================
    # TUNE n_neighbors (OPTIONAL)
    # ================================================================

    n_neighbors = DEFAULT_N_NEIGHBORS
    if AUTO_TUNE_NEIGHBORS and len(labeled_indices) >= 10:
        y_labeled = y[labeled_mask]
        X_labeled = X[labeled_mask]

        try:
            best_k, tune_results = tune_n_neighbors(
                X_labeled, y_labeled, CANDIDATE_NEIGHBORS, folds=NEIGHBOR_TUNE_FOLDS
            )
            n_neighbors = best_k
        except Exception as e:
            logging.warning(f"Neighbor tuning failed ({e}), using default k={DEFAULT_N_NEIGHBORS}")
    else:
        logging.info(f"Using default n_neighbors={n_neighbors}")

    # ================================================================
    # FIT LABEL PROPAGATION
    # ================================================================

    logging.info(f"\nFitting LabelPropagation (kernel='knn', n_neighbors={n_neighbors})...")
    lp = LabelPropagation(kernel='knn', n_neighbors=n_neighbors, max_iter=LP_MAX_ITER)
    lp.fit(X, y_for_lp)  # ← Use masked labels with validation hidden

    y_lp = lp.transduction_.astype(int)
    df['labelprop_pred'] = y_lp

    # Extract confidence and entropy
    try:
        label_distributions = lp.label_distributions_
        confidence = label_distributions[np.arange(len(y_lp)), y_lp]
        df['labelprop_confidence'] = confidence
        df['labelprop_entropy'] = compute_entropy_from_distributions(label_distributions)
        logging.info("Extracted confidence and entropy.")
    except AttributeError:
        logging.warning("label_distributions_ not available. Using placeholder.")
        df['labelprop_confidence'] = 1.0
        df['labelprop_entropy'] = 0.0

    # ================================================================
    # CLASS DISTRIBUTION ANALYSIS
    # ================================================================

    logging.info("\n" + "="*80)
    logging.info("CLASS DISTRIBUTION ANALYSIS")
    logging.info("="*80)

    before_counts = Counter(y[y != UNLABELED_VALUE])
    after_counts = Counter(y_lp)

    logging.info(f"Before propagation: {dict(before_counts)}")
    logging.info(f"After propagation:  {dict(after_counts)}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    keys_before, vals_before = zip(*sorted(before_counts.items()))
    axes[0].bar(keys_before, vals_before, color='steelblue')
    axes[0].set_title("Labeled Only (Before LP)")
    axes[0].set_xlabel("Class")
    axes[0].set_ylabel("Count")

    keys_after, vals_after = zip(*sorted(after_counts.items()))
    axes[1].bar(keys_after, vals_after, color='orange')
    axes[1].set_title("After Label Propagation")
    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'class_distribution_comparison.png'), dpi=150)
    plt.close()
    logging.info("Saved: class_distribution_comparison.png")

    # ================================================================
    # EVALUATION ON HELD-OUT VALIDATION SET
    # ================================================================

    if len(val_idx) > 0:
        y_true_val = y[val_idx]
        y_pred_val = y_lp[val_idx]

        logging.info("\n" + "="*80)
        logging.info("EVALUATION ON HELD-OUT VALIDATION SET")
        logging.info("="*80)
        logging.info(f"Validation samples: {len(val_idx):,}")
        logging.info(f"Accuracy: {accuracy_score(y_true_val, y_pred_val):.4f}")
        logging.info(f"Macro F1: {f1_score(y_true_val, y_pred_val, average='macro', zero_division=0):.4f}")
        logging.info(f"Weighted F1: {f1_score(y_true_val, y_pred_val, average='weighted', zero_division=0):.4f}")

        print("\nClassification Report (Held-Out Validation):")
        print(classification_report(y_true_val, y_pred_val,
                                   target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3'],
                                   zero_division=0))

        cm_val = confusion_matrix(y_true_val, y_pred_val)
        disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val,
                                          display_labels=['Class 0', 'Class 1', 'Class 2', 'Class 3'])
        disp_val.plot(cmap='Blues', values_format='d')
        plt.title("Confusion Matrix: LP on Held-Out Validation")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix_labelprop_heldout.png'), dpi=150)
        plt.close()
        logging.info("Saved: confusion_matrix_labelprop_heldout.png")
    else:
        logging.warning("No held-out validation set available.")

    # ================================================================
    # CONFIDENCE & ENTROPY ANALYSIS
    # ================================================================

    logging.info("\n" + "="*80)
    logging.info("CONFIDENCE & ENTROPY ANALYSIS")
    logging.info("="*80)

    if 'labelprop_confidence' in df.columns:
        plt.figure(figsize=(10, 4))
        plt.hist(df['labelprop_confidence'], bins=50, color='teal', edgecolor='black', alpha=0.7)
        plt.xlabel("Confidence Score")
        plt.ylabel("Frequency")
        plt.title("Label Propagation Confidence Distribution")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'confidence_distribution.png'), dpi=150)
        plt.close()
        logging.info("Saved: confidence_distribution.png")

    if 'labelprop_entropy' in df.columns:
        plt.figure(figsize=(10, 4))
        plt.hist(df['labelprop_entropy'], bins=50, color='coral', edgecolor='black', alpha=0.7)
        plt.xlabel("Entropy")
        plt.ylabel("Frequency")
        plt.title("Label Propagation Posterior Entropy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'entropy_distribution.png'), dpi=150)
        plt.close()
        logging.info("Saved: entropy_distribution.png")

    # ================================================================
    # CONFIDENCE-BASED FILTERING
    # ================================================================

    pseudo_mask = (y == UNLABELED_VALUE)
    high_conf_mask = pseudo_mask & (df['labelprop_confidence'] >= CONFIDENCE_THRESHOLD)
    low_conf_mask = pseudo_mask & (df['labelprop_confidence'] < CONFIDENCE_THRESHOLD)

    df['labelprop_final'] = df['labelprop_pred'].copy()
    df.loc[low_conf_mask, 'labelprop_final'] = UNLABELED_VALUE

    logging.info(f"Originally unlabeled: {pseudo_mask.sum():,}")
    logging.info(f"High-confidence kept: {high_conf_mask.sum():,}")
    logging.info(f"Low-confidence removed: {low_conf_mask.sum():,}")

    # ================================================================
    # SAVE MODEL & OUTPUT
    # ================================================================

    try:
        joblib.dump(lp, LP_MODEL_PATH)
        logging.info(f"Saved LP model: {LP_MODEL_PATH}")
    except Exception as e:
        logging.warning(f"Failed to save LP model: {e}")

    df.to_csv(OUTPUT_FILE, index=False)
    logging.info(f"Saved predictions: {OUTPUT_FILE}")

    # ================================================================
    # OPTIONAL: TRAIN FINAL CLASSIFIER
    # ================================================================

    if TRAIN_FINAL_CLASSIFIER and LGB_AVAILABLE:
        logging.info("\n" + "="*80)
        logging.info("TRAINING FINAL SUPERVISED CLASSIFIER")
        logging.info("="*80)

        mask_gold = (y != UNLABELED_VALUE)
        mask_pseudo_keep = high_conf_mask
        train_mask = mask_gold | mask_pseudo_keep

        X_train = X[train_mask]
        y_train = df.loc[train_mask, 'labelprop_final'].astype(int).values

        logging.info(f"Training samples: {len(y_train):,}")
        logging.info(f"  - Gold: {mask_gold.sum():,}")
        logging.info(f"  - High-conf pseudo: {mask_pseudo_keep.sum():,}")

        unique_classes = np.unique(y_train[y_train != UNLABELED_VALUE])
        n_classes = len(unique_classes)

        if n_classes >= 2:
            clf = LGBMClassifier(
                objective='multiclass',
                num_class=n_classes,
                learning_rate=0.05,
                num_leaves=31,
                n_estimators=300,
                random_state=RANDOM_SEED,
                verbosity=-1
            )
            clf.fit(X_train, y_train)

            try:
                joblib.dump(clf, FINAL_MODEL_PATH)
                logging.info(f"Saved final model: {FINAL_MODEL_PATH}")
            except Exception as e:
                logging.warning(f"Failed to save final model: {e}")

            # Evaluate on held-out validation
            if len(val_idx) > 0:
                X_val_final = X[val_idx]
                y_val_final = y[val_idx]
                y_pred_final = clf.predict(X_val_final)

                logging.info("\nFinal Classifier on Held-Out Validation:")
                logging.info(f"Accuracy: {accuracy_score(y_val_final, y_pred_final):.4f}")
                logging.info(f"Macro F1: {f1_score(y_val_final, y_pred_final, average='macro', zero_division=0):.4f}")
                print(classification_report(y_val_final, y_pred_final, zero_division=0))

                cm2 = confusion_matrix(y_val_final, y_pred_final)
                disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2)
                disp2.plot(cmap='Blues', values_format='d')
                plt.title("Confusion Matrix: Final Classifier")
                plt.tight_layout()
                plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix_final_classifier.png'), dpi=150)
                plt.close()
                logging.info("Saved: confusion_matrix_final_classifier.png")

            # Feature importance
            feature_importance = clf.feature_importances_
            importance_df = pd.DataFrame({
                'feature': selected_features,
                'importance': feature_importance
            }).sort_values('importance', ascending=False).head(20)

            plt.figure(figsize=(10, 6))
            plt.barh(importance_df['feature'], importance_df['importance'], color='purple')
            plt.xlabel("Feature Importance")
            plt.title("Top 20 Features (Final LightGBM)")
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'feature_importance_final.png'), dpi=150)
            plt.close()
            logging.info("Saved: feature_importance_final.png")

            # SHAP
            if SHAP_AVAILABLE:
                try:
                    logging.info("Computing SHAP summary...")
                    explainer = shap.TreeExplainer(clf)
                    sample_idx = np.random.choice(np.arange(len(X_train)),
                                                 size=min(2000, len(X_train)),
                                                 replace=False)
                    shap_values = explainer.shap_values(X_train[sample_idx])

                    if isinstance(shap_values, list):
                        shap.summary_plot(shap_values, X_train[sample_idx],
                                        feature_names=selected_features, show=False)
                    else:
                        shap.summary_plot(shap_values, X_train[sample_idx],
                                        feature_names=selected_features, show=False)

                    plt.tight_layout()
                    plt.savefig(os.path.join(PLOT_DIR, 'shap_summary_final.png'),
                              dpi=150, bbox_inches='tight')
                    plt.close()
                    logging.info("Saved: shap_summary_final.png")
                except Exception as e:
                    logging.warning(f"SHAP failed: {e}")
        else:
            logging.warning("Not enough classes for final classifier.")

    # ================================================================
    # FINAL SUMMARY
    # ================================================================

    logging.info("\n" + "="*80)
    logging.info("LABEL PROPAGATION COMPLETE")
    logging.info("="*80)
    logging.info(f"Output: {OUTPUT_FILE}")
    logging.info(f"Models: {MODEL_DIR}")
    logging.info(f"Plots: {PLOT_DIR}")
    logging.info(f"Total propagated: {(df['labelprop_final'] != UNLABELED_VALUE).sum():,}")
    logging.info("="*80)


if __name__ == "__main__":
    main()



Classification Report (Held-Out Validation):
              precision    recall  f1-score   support

     Class 0       0.90      0.96      0.93      8646
     Class 1       0.86      0.72      0.79      3067
     Class 2       0.78      0.72      0.75       376
     Class 3       0.54      0.41      0.46        86

    accuracy                           0.89     12175
   macro avg       0.77      0.70      0.73     12175
weighted avg       0.89      0.89      0.88     12175

              precision    recall  f1-score   support

           0       0.97      0.98      0.98      8646
           1       0.95      0.93      0.94      3067
           2       0.78      0.73      0.75       376
           3       0.54      0.41      0.46        86

    accuracy                           0.96     12175
   macro avg       0.81      0.76      0.78     12175
weighted avg       0.96      0.96      0.96     12175



**FINAL CORRECT LABEL PROPAGATION**

In [ ]:
#!/usr/bin/env python3
"""
Label Propagation (Final SSL Stage) - CPU, full 200 features
============================================================
Integrates Human + SSL-3 + Co-Training labels in priority cascade.
Uses KNN kernel with optional auto-tuning, held-out validation, confidence filtering,
diagnostics, and model saving.

Inputs:
- predictions_co_training_final.csv (output from co-training; must include original_index)
- selected_features_topK_4class_weighted.csv

Outputs:
- predictions_labelprop_final.csv
- models/labelprop_model.joblib
- models/final_lgb_model.joblib (optional)
- labelprop_plots/ (diagnostic plots)
"""

import os
import logging
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.semi_supervised import LabelPropagation
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
import joblib

# Optional libs (kept as optional; runs fine without)
try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s: %(message)s")

# ====================================================================
# CONFIGURATION
# ====================================================================

DATA_DIR = '/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data'
FEATURE_FILE = os.path.join(DATA_DIR, 'selected_features_topK_4class_weighted.csv')
INPUT_FILE = os.path.join(DATA_DIR, 'predictions_co_training_final.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'predictions_labelprop_final.csv')
PLOT_DIR = os.path.join(DATA_DIR, 'labelprop_plots')
MODEL_DIR = os.path.join(DATA_DIR, 'models')
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Label columns and constants
label_col_ssl3 = 'final_label_v3'
label_col_cotrain = 'co_train_final_pred'
ORIGINAL_INDEX_COL = 'original_index'
UNLABELED_VALUE = -1

# LabelPropagation params
DEFAULT_N_NEIGHBORS = 15
AUTO_TUNE_NEIGHBORS = True
CANDIDATE_NEIGHBORS = [7, 15, 25, 35]
LP_MAX_ITER = 1000

# Confidence filtering
CONFIDENCE_THRESHOLD = 0.85

# Final supervised model (optional)
TRAIN_FINAL_CLASSIFIER = True
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, 'final_lgb_model.joblib')
LP_MODEL_PATH = os.path.join(MODEL_DIR, 'labelprop_model.joblib')

RANDOM_SEED = 42
NEIGHBOR_TUNE_FOLDS = 3

# Plot styles
sns.set(style='whitegrid')

# ====================================================================
# UTILITY FUNCTIONS
# ====================================================================

def safe_load_features(feature_csv_path):
    """Load feature list from CSV (requires 'feature' column)."""
    df = pd.read_csv(feature_csv_path)
    if 'feature' not in df.columns:
        raise KeyError(f"Feature CSV must contain column 'feature'. Found: {list(df.columns)}")
    return df['feature'].tolist()

def tune_n_neighbors(X, y_labeled, candidate_list, folds=3, random_state=RANDOM_SEED):
    """
    Tune n_neighbors for LabelPropagation via stratified CV on labeled subset.
    """
    logging.info("Tuning n_neighbors via Stratified CV on labeled subset...")
    class_counts = Counter(y_labeled)
    min_class_count = min(class_counts.values())
    max_folds = min(folds, min_class_count, len(y_labeled) // 2)

    if max_folds < 2:
        logging.warning(f"Not enough samples for CV (min class count: {min_class_count}). "
                        f"Using default n_neighbors={DEFAULT_N_NEIGHBORS}")
        return DEFAULT_N_NEIGHBORS, {}

    skf = StratifiedKFold(n_splits=max_folds, shuffle=True, random_state=random_state)
    results = {}

    for k in candidate_list:
        scores = []
        try:
            for train_idx, val_idx in skf.split(X, y_labeled):
                y_masked = np.array(y_labeled, dtype=np.int32, copy=True)
                y_masked[val_idx] = UNLABELED_VALUE

                lp = LabelPropagation(kernel='knn', n_neighbors=k, max_iter=LP_MAX_ITER)
                lp.fit(X, y_masked)
                y_pred = lp.transduction_[val_idx]
                acc = accuracy_score(y_labeled[val_idx], y_pred)
                scores.append(acc)

            mean_score = float(np.mean(scores))
            results[k] = mean_score
            logging.info(f"  k={k}: mean acc={mean_score:.4f}")
        except Exception as e:
            logging.warning(f"  k={k}: failed ({e})")
            continue

    if not results:
        logging.warning(f"All tuning failed. Using default n_neighbors={DEFAULT_N_NEIGHBORS}")
        return DEFAULT_N_NEIGHBORS, {}

    best_k = sorted(results.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    logging.info(f"Selected n_neighbors={best_k} (used {max_folds}-fold CV)")
    return best_k, results

def compute_entropy_from_distributions(dist_arr):
    """Compute entropy of posterior distributions row-wise."""
    eps = 1e-12
    return -(dist_arr * np.log(dist_arr + eps)).sum(axis=1)

# ====================================================================
# MAIN PIPELINE
# ====================================================================

def main():
    np.random.seed(RANDOM_SEED)

    logging.info("="*80)
    logging.info("LABEL PROPAGATION - FINAL SSL STAGE (CPU, FULL FEATURES)")
    logging.info("="*80)

    # Load features (require all to be present)
    logging.info("Loading selected features...")
    selected_features = safe_load_features(FEATURE_FILE)

    # Load co-training output
    logging.info("Loading co-training output...")
    df = pd.read_csv(INPUT_FILE)

    # Ensure original_index exists
    if ORIGINAL_INDEX_COL not in df.columns:
        raise KeyError(f"Required column '{ORIGINAL_INDEX_COL}' not present in co-training output. "
                       f"Make sure the co-training script saved it.")

    # Verify required columns exist
    if label_col_ssl3 not in df.columns:
        raise KeyError(f"Column '{label_col_ssl3}' not found in co-training output - needed for priority cascade.")
    if label_col_cotrain not in df.columns:
        logging.warning(f"Column '{label_col_cotrain}' not found. LP will use SSL-3 labels only where present.")

    # Enforce all features (no reduction); error if any are missing
    missing_features = [f for f in selected_features if f not in df.columns]
    if missing_features:
        raise KeyError(f"The following required features are missing from INPUT_FILE: {missing_features[:10]}"
                       f"{' ...' if len(missing_features) > 10 else ''}. "
                       f"This script requires all selected features to be present.")

    used_features = selected_features  # all features, as requested
    X = df[used_features].fillna(0.0).values  # keep default dtype

    # ================================================================
    # BUILD LABEL ARRAY: PRIORITY CASCADE (SSL-3 -> CO-TRAIN -> UNLABELED)
    # ================================================================
    logging.info("Building label array (priority: SSL-3 -> Co-training -> Unlabeled)...")

    ssl3 = df[label_col_ssl3].astype(int)
    cot = df[label_col_cotrain].astype(int) if label_col_cotrain in df.columns else None

    if cot is not None:
        y = np.where(ssl3 != -1, ssl3, cot)
    else:
        y = ssl3.copy()

    y = np.where(y != -1, y, UNLABELED_VALUE).astype(int)

    n_samples, n_features = X.shape
    n_labeled = int((y != UNLABELED_VALUE).sum())
    n_unlabeled = int((y == UNLABELED_VALUE).sum())
    n_from_ssl3 = int((df[label_col_ssl3] != -1).sum())
    n_from_cotrain = 0
    if cot is not None:
        n_from_cotrain = int(((df[label_col_ssl3] == -1) & (df[label_col_cotrain] != -1)).sum())

    logging.info(f"Total samples: {n_samples:,}")
    logging.info(f"Features used: {len(used_features)}")
    logging.info(f"Labeled samples: {n_labeled:,} (SSL-3: {n_from_ssl3:,}, Co-train added: {n_from_cotrain:,})")
    logging.info(f"Unlabeled samples: {n_unlabeled:,}")

    class_counts = Counter(y[y != UNLABELED_VALUE])
    logging.info(f"Class distribution (visible to LP): {dict(class_counts)}")

    # ================================================================
    # CREATE HELD-OUT VALIDATION SET
    # ================================================================
    logging.info("\n" + "="*80)
    logging.info("CREATING HELD-OUT VALIDATION SET")
    logging.info("="*80)

    labeled_mask = (y != UNLABELED_VALUE)
    labeled_indices = np.where(labeled_mask)[0]

    if len(labeled_indices) < 10:
        logging.warning("Not enough labeled samples for validation. Skipping held-out validation.")
        y_for_lp = y.copy()
        val_idx = np.array([], dtype=int)
    else:
        train_idx, val_idx = train_test_split(
            labeled_indices,
            test_size=0.2,
            stratify=y[labeled_indices],
            random_state=RANDOM_SEED
        )
        # Mask validation samples as unlabeled for LP training
        y_for_lp = y.copy()
        y_for_lp[val_idx] = UNLABELED_VALUE

    logging.info(f"Train (visible to LP): {int((y_for_lp != UNLABELED_VALUE).sum()):,} samples")
    logging.info(f"Validation (held out): {len(val_idx):,} samples")

    # ================================================================
    # TUNE n_neighbors (OPTIONAL)
    # ================================================================
    n_neighbors = DEFAULT_N_NEIGHBORS
    if AUTO_TUNE_NEIGHBORS and len(labeled_indices) >= 10:
        y_labeled = y[labeled_mask]
        X_labeled = X[labeled_mask]
        try:
            best_k, tune_results = tune_n_neighbors(X_labeled, y_labeled, CANDIDATE_NEIGHBORS, folds=NEIGHBOR_TUNE_FOLDS)
            n_neighbors = best_k
        except Exception as e:
            logging.warning(f"Neighbor tuning failed ({e}), using default k={DEFAULT_N_NEIGHBORS}")
    else:
        logging.info(f"Using default n_neighbors={n_neighbors}")

    # ================================================================
    # FIT LABEL PROPAGATION (CPU)
    # ================================================================
    logging.info(f"\nFitting LabelPropagation (kernel='knn', n_neighbors={n_neighbors})...")
    lp = LabelPropagation(kernel='knn', n_neighbors=n_neighbors, max_iter=LP_MAX_ITER)
    lp.fit(X, y_for_lp)  # use masked labels (held-out val hidden)

    y_lp = lp.transduction_.astype(int)
    df['labelprop_pred'] = y_lp

    # Extract confidences and entropies
    try:
        label_distributions = np.asarray(lp.label_distributions_)
        conf_idx = np.clip(y_lp, 0, label_distributions.shape[1] - 1)
        confidences = label_distributions[np.arange(label_distributions.shape[0]), conf_idx]
        df['labelprop_confidence'] = confidences
        df['labelprop_entropy'] = compute_entropy_from_distributions(label_distributions)
        logging.info(f"Extracted label distributions shape: {label_distributions.shape}")
    except Exception as e:
        logging.warning(f"label_distributions_ unavailable or extraction failed: {e}")
        df['labelprop_confidence'] = 1.0
        df['labelprop_entropy'] = 0.0

    # ================================================================
    # CLASS DISTRIBUTION ANALYSIS & PLOTS
    # ================================================================
    logging.info("\n" + "="*80)
    logging.info("CLASS DISTRIBUTION ANALYSIS")
    logging.info("="*80)

    before_counts = Counter(y[y != UNLABELED_VALUE])
    after_counts = Counter(y_lp)

    logging.info(f"Before propagation: {dict(before_counts)}")
    logging.info(f"After propagation: {dict(after_counts)}")

    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        keys_before, vals_before = zip(*sorted(before_counts.items()))
        axes[0].bar(keys_before, vals_before, color='steelblue')
        axes[0].set_title("Labeled Only (Before LP)")
        axes[0].set_xlabel("Class")
        axes[0].set_ylabel("Count")

        keys_after, vals_after = zip(*sorted(after_counts.items()))
        axes[1].bar(keys_after, vals_after, color='orange')
        axes[1].set_title("After Label Propagation")
        axes[1].set_xlabel("Class")
        axes[1].set_ylabel("Count")

        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'class_distribution_comparison.png'), dpi=150)
        plt.close()
        logging.info("Saved: class_distribution_comparison.png")
    except Exception as e:
        logging.warning(f"Failed to create class distribution plot: {e}")

    # ================================================================
    # EVALUATION ON HELD-OUT VALIDATION SET
    # ================================================================
    if len(val_idx) > 0:
        y_true_val = y[val_idx]
        y_pred_val = y_lp[val_idx]

        logging.info("\n" + "="*80)
        logging.info("EVALUATION ON HELD-OUT VALIDATION SET")
        logging.info("="*80)
        logging.info(f"Validation samples: {len(val_idx):,}")
        logging.info(f"Accuracy: {accuracy_score(y_true_val, y_pred_val):.4f}")
        logging.info(f"Macro F1: {f1_score(y_true_val, y_pred_val, average='macro', zero_division=0):.4f}")
        logging.info(f"Weighted F1: {f1_score(y_true_val, y_pred_val, average='weighted', zero_division=0):.4f}")

        print("\nClassification Report (Held-Out Validation):")
        print(classification_report(y_true_val, y_pred_val, zero_division=0))

        try:
            cm_val = confusion_matrix(y_true_val, y_pred_val)
            disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val,
                                              display_labels=['Class 0', 'Class 1', 'Class 2', 'Class 3'])
            disp_val.plot(cmap='Blues', values_format='d')
            plt.title("Confusion Matrix: LP on Held-Out Validation")
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix_labelprop_heldout.png'), dpi=150)
            plt.close()
            logging.info("Saved: confusion_matrix_labelprop_heldout.png")
        except Exception as e:
            logging.warning(f"Failed to create confusion matrix plot: {e}")
    else:
        logging.warning("No held-out validation set available.")

    # ================================================================
    # CONFIDENCE & ENTROPY ANALYSIS
    # ================================================================
    logging.info("\n" + "="*80)
    logging.info("CONFIDENCE & ENTROPY ANALYSIS")
    logging.info("="*80)

    if 'labelprop_confidence' in df.columns:
        try:
            plt.figure(figsize=(10, 4))
            plt.hist(df['labelprop_confidence'], bins=50, edgecolor='black', alpha=0.7)
            plt.xlabel("Confidence Score")
            plt.ylabel("Frequency")
            plt.title("Label Propagation Confidence Distribution")
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'confidence_distribution.png'), dpi=150)
            plt.close()
            logging.info("Saved: confidence_distribution.png")
        except Exception as e:
            logging.warning(f"Failed to save confidence distribution plot: {e}")

    if 'labelprop_entropy' in df.columns:
        try:
            plt.figure(figsize=(10, 4))
            plt.hist(df['labelprop_entropy'], bins=50, edgecolor='black', alpha=0.7)
            plt.xlabel("Entropy")
            plt.ylabel("Frequency")
            plt.title("Label Propagation Posterior Entropy")
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'entropy_distribution.png'), dpi=150)
            plt.close()
            logging.info("Saved: entropy_distribution.png")
        except Exception as e:
            logging.warning(f"Failed to save entropy plot: {e}")

    # ================================================================
    # CONFIDENCE-BASED FILTERING
    # ================================================================
    pseudo_mask = (y == UNLABELED_VALUE)
    # If confidence is missing (fallback case), keep all; otherwise filter by threshold
    if 'labelprop_confidence' in df.columns:
        high_conf_mask = pseudo_mask & (df['labelprop_confidence'] >= CONFIDENCE_THRESHOLD)
        low_conf_mask = pseudo_mask & (df['labelprop_confidence'] < CONFIDENCE_THRESHOLD)
    else:
        high_conf_mask = pseudo_mask
        low_conf_mask = np.zeros(len(df), dtype=bool)

    df['labelprop_final'] = df['labelprop_pred'].astype(int)
    df.loc[low_conf_mask, 'labelprop_final'] = UNLABELED_VALUE

    logging.info(f"Originally unlabeled: {int(pseudo_mask.sum()):,}")
    logging.info(f"High-confidence kept: {int(high_conf_mask.sum()):,}")
    logging.info(f"Low-confidence removed: {int(low_conf_mask.sum()):,}")

    # ================================================================
    # SAVE MODEL & OUTPUT
    # ================================================================
    try:
        joblib.dump(lp, LP_MODEL_PATH)
        logging.info(f"Saved LP model: {LP_MODEL_PATH}")
    except Exception as e:
        logging.warning(f"Failed to save LP model: {e}")

    # Ensure original_index and co-training probability columns are preserved
    df.to_csv(OUTPUT_FILE, index=False)
    logging.info(f"Saved predictions: {OUTPUT_FILE}")

    # ================================================================
    # OPTIONAL: TRAIN FINAL CLASSIFIER (LightGBM)
    # ================================================================
    if TRAIN_FINAL_CLASSIFIER and LGB_AVAILABLE:
        logging.info("\n" + "="*80)
        logging.info("TRAINING FINAL SUPERVISED CLASSIFIER (CPU)")
        logging.info("="*80)

        mask_gold = (y != UNLABELED_VALUE)
        mask_pseudo_keep = high_conf_mask
        train_mask = mask_gold | mask_pseudo_keep

        X_train = X[train_mask]
        y_train = df.loc[train_mask, 'labelprop_final'].astype(int).values

        logging.info(f"Training samples: {len(y_train):,}")
        logging.info(f" - Gold: {int(mask_gold.sum()):,}")
        logging.info(f" - High-conf pseudo: {int(mask_pseudo_keep.sum()):,}")

        # Ensure at least 2 classes are present
        unique_classes = np.unique(y_train[y_train != UNLABELED_VALUE])
        if len(unique_classes) >= 2:
            clf = LGBMClassifier(
                objective='multiclass',
                num_class=4,
                learning_rate=0.05,
                num_leaves=31,
                n_estimators=300,
                random_state=RANDOM_SEED,
                verbosity=-1
            )
            clf.fit(X_train, y_train)

            try:
                joblib.dump(clf, FINAL_MODEL_PATH)
                logging.info(f"Saved final model: {FINAL_MODEL_PATH}")
            except Exception as e:
                logging.warning(f"Failed to save final model: {e}")

            # Evaluate on held-out validation (if any)
            if len(val_idx) > 0:
                X_val_final = X[val_idx]
                y_val_final = y[val_idx]
                y_pred_final = clf.predict(X_val_final)

                logging.info("\nFinal Classifier on Held-Out Validation:")
                logging.info(f"Accuracy: {accuracy_score(y_val_final, y_pred_final):.4f}")
                logging.info(f"Macro F1: {f1_score(y_val_final, y_pred_final, average='macro', zero_division=0):.4f}")
                print(classification_report(y_val_final, y_pred_final, zero_division=0))

                try:
                    cm2 = confusion_matrix(y_val_final, y_pred_final)
                    disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2)
                    disp2.plot(cmap='Blues', values_format='d')
                    plt.title("Confusion Matrix: Final Classifier")
                    plt.tight_layout()
                    plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix_final_classifier.png'), dpi=150)
                    plt.close()
                    logging.info("Saved: confusion_matrix_final_classifier.png")
                except Exception as e:
                    logging.warning(f"Failed to save final classifier confusion matrix: {e}")

            # Feature importance plot
            try:
                feature_importance = clf.feature_importances_
                importance_df = pd.DataFrame({
                    'feature': used_features,
                    'importance': feature_importance
                }).sort_values('importance', ascending=False).head(20)

                plt.figure(figsize=(10, 6))
                plt.barh(importance_df['feature'], importance_df['importance'], color='purple')
                plt.xlabel("Feature Importance")
                plt.title("Top 20 Features (Final LightGBM)")
                plt.gca().invert_yaxis()
                plt.tight_layout()
                plt.savefig(os.path.join(PLOT_DIR, 'feature_importance_final.png'), dpi=150)
                plt.close()
                logging.info("Saved: feature_importance_final.png")
            except Exception as e:
                logging.warning(f"Failed to create feature importance plot: {e}")

            # SHAP analysis (optional)
            if SHAP_AVAILABLE:
                try:
                    logging.info("Computing SHAP summary...")
                    explainer = shap.TreeExplainer(clf)
                    # Subsample for plot readability (no speed optimization intent)
                    sample_idx = np.random.choice(np.arange(len(X_train)),
                                                  size=min(2000, len(X_train)),
                                                  replace=False)
                    shap_values = explainer.shap_values(X_train[sample_idx])

                    if isinstance(shap_values, list):
                        shap.summary_plot(shap_values, X_train[sample_idx],
                                          feature_names=used_features, show=False)
                    else:
                        shap.summary_plot(shap_values, X_train[sample_idx],
                                          feature_names=used_features, show=False)

                    plt.tight_layout()
                    plt.savefig(os.path.join(PLOT_DIR, 'shap_summary_final.png'),
                                dpi=150, bbox_inches='tight')
                    plt.close()
                    logging.info("Saved: shap_summary_final.png")
                except Exception as e:
                    logging.warning(f"SHAP failed: {e}")
        else:
            logging.warning("Not enough classes for final classifier.")

    # ================================================================
    # FINAL SUMMARY
    # ================================================================
    logging.info("\n" + "="*80)
    logging.info("LABEL PROPAGATION COMPLETE")
    logging.info("="*80)
    logging.info(f"Output: {OUTPUT_FILE}")
    logging.info(f"Models: {MODEL_DIR}")
    logging.info(f"Plots: {PLOT_DIR}")
    logging.info(f"Total propagated (labelprop_final != -1): "
                 f"{int((df['labelprop_final'] != UNLABELED_VALUE).sum()):,}")
    logging.info("="*80)

if __name__ == "__main__":
    main()



Classification Report (Held-Out Validation):
              precision    recall  f1-score   support

           0       0.91      0.96      0.93     10009
           1       0.84      0.73      0.78      3076
           2       0.78      0.74      0.76       396
           3       0.60      0.34      0.43        86

    accuracy                           0.89     13567
   macro avg       0.79      0.69      0.73     13567
weighted avg       0.89      0.89      0.89     13567

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     10009
           1       0.95      0.94      0.95      3076
           2       0.78      0.74      0.76       396
           3       0.60      0.34      0.43        86

    accuracy                           0.96     13567
   macro avg       0.83      0.75      0.78     13567
weighted avg       0.96      0.96      0.96     13567



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Evaluation metrics

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive SSL Metrics Evaluation - MERGED & STANDALONE
=========================================================
Computes all metrics (Accuracy, Precision, Recall, F1, ROC-AUC, MCC, Kappa, Brier)
for all classes across SSL-1, SSL-2, SSL-3, Co-Training, and Label Propagation.

This file merges the user's original evaluation flow with improved, robust helpers:
- safer CatBoost loading with GPU fallback
- robust probability/pred column finding
- memory-efficient batched ensemble prediction
- safer co-training & label-propagation probability alignment
- path validation and clear logging

Output: ssl_comprehensive_metrics.csv
"""

import os
import gc
import re
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
import logging
from collections import defaultdict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score
)
import lightgbm as lgb
import catboost as cb
import joblib

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data"

# File paths
FEATURES_FILE = os.path.join(BASE_DIR, "selected_features_topK_4class_weighted.csv")
SSL1_CSV = os.path.join(BASE_DIR, "predictions_with_final_label.csv")
SSL2_CSV = os.path.join(BASE_DIR, "predictions_with_final_label_v2.csv")
SSL3_CSV = os.path.join(BASE_DIR, "predictions_with_final_label_v3.csv")
COTRAIN_CSV = os.path.join(BASE_DIR, "predictions_co_training_final.csv")
LP_CSV = os.path.join(BASE_DIR, "predictions_labelprop_final.csv")

# Model paths
MODEL_DIR_SSL1 = os.path.join(BASE_DIR, "models_stacked")
MODEL_DIR_SSL2 = os.path.join(BASE_DIR, "models_stacked_v2")
MODEL_DIR_SSL3 = os.path.join(BASE_DIR, "models_stacked_v3")
MODEL_DIR_COTRAIN = MODEL_DIR_SSL2
LP_MODEL = os.path.join(BASE_DIR, "models/labelprop_model.joblib")

ALPHA_CSV = os.path.join(BASE_DIR, "lgb_vs_catboost_vs_ensemble.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "ssl_comprehensive_metrics.csv")

GOLD_LABEL_COL = "human_label_clean"
NUM_CLASSES = 4
ORDINAL_CLASSES = True  # Set True if classes are ordered

# Performance settings
USE_GPU = True  # Attempt CatBoost GPU acceleration
BATCH_SIZE = 10000  # Batch size for predictions


# =============================================================================
# VALIDATION
# =============================================================================

def validate_paths():
    """Check all required files exist before starting."""
    required = [FEATURES_FILE, SSL1_CSV, SSL2_CSV, SSL3_CSV, COTRAIN_CSV, LP_CSV]
    missing = [p for p in required if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Missing files: {missing}")
    logging.info("All input files validated.")


# =============================================================================
# IMPROVED UTILITY FUNCTIONS (safe loaders, finders, batched predictions)
# =============================================================================

def safe_catboost_load(path: str, use_gpu: bool = False):
    """
    Load CatBoost model from path. Attempt GPU load if requested; fall back to CPU.
    Returns loaded model or raises Exception.
    """
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    try:
        model = cb.CatBoost()
        model.load_model(path)
        return model
    except Exception as e:
        logging.warning(f"CatBoost load failed for {path}: {e}. Retrying direct load without wrapper.")
        model = cb.CatBoost()
        model.load_model(path)
        return model


def find_proba_columns(df, version='', n_classes=NUM_CLASSES):
    """
    More robust probability column finder.
    Returns list of column names or None.
    """
    df_cols = list(df.columns)
    variants = []
    v_raw = version or ''
    v_strip = v_raw.replace('_', '')
    for i in range(n_classes):
        candidates = [
            f"f_proba_class_{i}{v_raw}",
            f"f_proba_class_{i}{v_strip}",
            f"f_proba_class_{i}",
            f"proba_class_{i}{v_raw}",
            f"fprobaclass{i}{v_strip}",
            f"proba_{i}{v_strip}",
        ]
        found = None
        for cand in candidates:
            for col in df_cols:
                if col.lower().replace('_', '') == cand.lower().replace('_', ''):
                    found = col
                    break
            if found:
                break
        if not found:
            variants = None
            break
        variants.append(found)
    if variants:
        return variants

    # regex fallback
    candidates = [c for c in df_cols if 'proba' in c.lower()]
    class_map = {}
    for col in candidates:
        m = re.search(r'(\d+)', col)
        if m:
            idx = int(m.group(1))
            if 0 <= idx < n_classes and idx not in class_map:
                class_map[idx] = col
    if len(class_map) == n_classes:
        return [class_map[i] for i in range(n_classes)]

    # final fallback: first n_classes columns containing 'proba'
    proba_group = [c for c in df_cols if 'proba' in c.lower()]
    if len(proba_group) >= n_classes:
        return proba_group[:n_classes]

    return None


def load_ensemble_models(model_dir, num_folds=5, use_gpu=USE_GPU):
    """
    Load LightGBM and CatBoost ensemble models if available.
    Returns (lgb_models, cat_models) where each is a list (possibly empty).
    """
    model_dir = Path(model_dir)
    lgb_models = []
    cat_models = []

    for i in range(1, num_folds + 1):
        lgb_path = model_dir / f"lgb_fold_{i}.txt"
        cat_path = model_dir / f"catboost_fold_{i}.cbm"

        if lgb_path.exists():
            try:
                lgb_models.append(lgb.Booster(model_file=str(lgb_path)))
            except Exception as e:
                logging.warning(f"Failed to load LightGBM model {lgb_path}: {e}")

        if cat_path.exists():
            try:
                cat_models.append(safe_catboost_load(str(cat_path), use_gpu=use_gpu))
            except Exception as e:
                logging.warning(f"Failed to load CatBoost model {cat_path}: {e}")

    logging.info(f"Loaded ensemble models from {model_dir}: lgb={len(lgb_models)}, cat={len(cat_models)}")
    return lgb_models, cat_models


def predict_ensemble_batched(X, lgb_models, cat_models, alpha=0.47, batch_size=BATCH_SIZE):
    """
    Predict in batches and return probability matrix (n_samples, n_classes).
    If no models available, returns an array of NaNs (so metrics fail fast/obvious).
    """
    X = np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    n_samples = X.shape[0]
    n_classes = NUM_CLASSES
    final_proba = np.zeros((n_samples, n_classes), dtype=np.float32)

    has_lgb = bool(lgb_models)
    has_cat = bool(cat_models)

    if not (has_lgb or has_cat):
        logging.warning("No ensemble models found. Returning NaN probabilities.")
        return np.full((n_samples, n_classes), np.nan, dtype=np.float32)

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        Xb = X[start:end]

        # LGB predictions
        if has_lgb:
            lgb_preds = []
            for m in lgb_models:
                try:
                    p = m.predict(Xb)
                except TypeError:
                    p = m.predict(Xb)
                lgb_preds.append(np.asarray(p, dtype=np.float32))
            lgb_avg = np.mean(lgb_preds, axis=0)
        else:
            lgb_avg = None

        # CatBoost predictions
        if has_cat:
            cat_preds = []
            for m in cat_models:
                try:
                    p = m.predict_proba(Xb)
                except Exception:
                    p = m.predict(Xb)
                cat_preds.append(np.asarray(p, dtype=np.float32))
            cat_avg = np.mean(cat_preds, axis=0)
        else:
            cat_avg = None

        # Combine
        if lgb_avg is None:
            combined = cat_avg
        elif cat_avg is None:
            combined = lgb_avg
        else:
            # ensure both have shape (n, n_classes)
            if lgb_avg.ndim == 1:
                lgb_avg = np.vstack([1 - lgb_avg, lgb_avg]).T
            if cat_avg.ndim == 1:
                cat_avg = np.vstack([1 - cat_avg, cat_avg]).T
            combined = alpha * lgb_avg + (1.0 - alpha) * cat_avg

        final_proba[start:end] = combined.astype(np.float32)

    return final_proba


# =============================================================================
# MISSING/ORIGINAL UTILITY FUNCTIONS (load_features, alpha, pred finder, brier)
# =============================================================================

def load_features():
    """Load feature list from CSV."""
    df = pd.read_csv(FEATURES_FILE)
    if 'feature' not in df.columns:
        raise KeyError("Feature CSV must contain 'feature' column")
    return df['feature'].tolist()


def load_alpha(alpha_csv):
    """Load ensemble alpha weight."""
    try:
        df = pd.read_csv(alpha_csv)
        if 'alpha' in df.columns:
            return float(df['alpha'].mean())
    except Exception:
        pass
    return 0.47


def find_pred_column(df, version=''):
    """Flexibly find predicted class column."""
    patterns = [
        f'predicted_class{version}',
        f'predictedclass{version}',
        f'predicted{version}',
        f'pred_class{version}'
    ]
    for col in df.columns:
        col_lower = col.lower().replace('_', '')
        for pattern in patterns:
            if col_lower == pattern.replace('_', ''):
                return col
    return None


def multiclass_brier(y_true, y_proba, n_classes=NUM_CLASSES):
    """Compute overall and per-class Brier scores."""
    y_true = np.asarray(y_true, dtype=np.int32)
    y_proba = np.asarray(y_proba, dtype=np.float32)
    n = len(y_true)

    # One-hot encode
    Y = np.zeros((n, n_classes), dtype=np.float32)
    Y[np.arange(n), y_true] = 1.0

    # Brier score = MSE of probabilities
    per_sample = np.sum((y_proba - Y) ** 2, axis=1)
    overall = float(np.mean(per_sample))

    # Per-class Brier
    per_class = []
    for c in range(n_classes):
        y_c = (y_true == c).astype(np.float32)
        per_class.append(float(np.mean((y_proba[:, c] - y_c) ** 2)))

    return overall, per_class


def compute_metrics_for_stage(y_true, y_pred, y_proba=None, stage_name=""):
    """Compute all metrics for a single stage (full original behavior)."""
    results = []

    # Ensure arrays
    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    # Basic metrics
    acc = float(accuracy_score(y_true, y_pred))
    precision = precision_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    recall = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    f1 = f1_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)

    macro_precision = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    macro_recall = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))

    weighted_precision = float(precision_score(y_true, y_pred, average='weighted', zero_division=0))
    weighted_recall = float(recall_score(y_true, y_pred, average='weighted', zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))

    # ROC-AUC
    if y_proba is not None:
        try:
            roc_auc_ovr_macro = float(roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro'))
            roc_auc_ovr_weighted = float(roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted'))
            roc_auc_per_class = []
            for cls in range(NUM_CLASSES):
                y_bin = (y_true == cls).astype(int)
                if len(np.unique(y_bin)) > 1:
                    roc_auc_per_class.append(float(roc_auc_score(y_bin, y_proba[:, cls])))
                else:
                    roc_auc_per_class.append(np.nan)
        except Exception as e:
            logging.warning(f"ROC-AUC computation failed for {stage_name}: {e}")
            roc_auc_ovr_macro = np.nan
            roc_auc_ovr_weighted = np.nan
            roc_auc_per_class = [np.nan] * NUM_CLASSES
    else:
        roc_auc_ovr_macro = np.nan
        roc_auc_ovr_weighted = np.nan
        roc_auc_per_class = [np.nan] * NUM_CLASSES

    # MCC and Kappa
    try:
        mcc = float(matthews_corrcoef(y_true, y_pred))
    except Exception:
        mcc = np.nan

    try:
        kappa = float(cohen_kappa_score(y_true, y_pred))
        kappa_weighted = float(cohen_kappa_score(y_true, y_pred, weights='quadratic')) if ORDINAL_CLASSES else np.nan
    except Exception:
        kappa = np.nan
        kappa_weighted = np.nan

    # Brier score
    if y_proba is not None:
        try:
            brier_overall, brier_per_class = multiclass_brier(y_true, y_proba, NUM_CLASSES)
        except Exception as e:
            logging.warning(f"Brier computation failed for {stage_name}: {e}")
            brier_overall, brier_per_class = np.nan, [np.nan] * NUM_CLASSES
    else:
        brier_overall, brier_per_class = np.nan, [np.nan] * NUM_CLASSES

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    # Per-class rows
    for cls in range(NUM_CLASSES):
        results.append({
            'Stage': stage_name,
            'Class': int(cls),
            'Metric_Type': 'Per_Class',
            'Accuracy': acc,
            'Precision': float(precision[cls]) if len(precision) > cls else np.nan,
            'Recall': float(recall[cls]) if len(recall) > cls else np.nan,
            'F1_Score': float(f1[cls]) if len(f1) > cls else np.nan,
            'ROC_AUC': float(roc_auc_per_class[cls]) if roc_auc_per_class[cls] is not np.nan else np.nan,
            'Brier': float(brier_per_class[cls]) if brier_per_class[cls] is not np.nan else np.nan,
            'MCC': np.nan,
            'Kappa': np.nan,
            'Kappa_Weighted': np.nan,
            'Support': int(np.sum(y_true == cls)),
            'TP': int(cm[cls, cls]),
            'FP': int(np.sum(cm[:, cls]) - cm[cls, cls]),
            'FN': int(np.sum(cm[cls, :]) - cm[cls, cls]),
            'TN': int(np.sum(cm) - np.sum(cm[cls, :]) - np.sum(cm[:, cls]) + cm[cls, cls])
        })

    # Macro average row
    results.append({
        'Stage': stage_name,
        'Class': 'Macro_Avg',
        'Metric_Type': 'Macro',
        'Accuracy': acc,
        'Precision': macro_precision,
        'Recall': macro_recall,
        'F1_Score': macro_f1,
        'ROC_AUC': roc_auc_ovr_macro,
        'Brier': brier_overall,
        'MCC': mcc,
        'Kappa': kappa,
        'Kappa_Weighted': kappa_weighted,
        'Support': int(len(y_true)),
        'TP': np.nan, 'FP': np.nan, 'FN': np.nan, 'TN': np.nan
    })

    # Weighted average row
    results.append({
        'Stage': stage_name,
        'Class': 'Weighted_Avg',
        'Metric_Type': 'Weighted',
        'Accuracy': acc,
        'Precision': weighted_precision,
        'Recall': weighted_recall,
        'F1_Score': weighted_f1,
        'ROC_AUC': roc_auc_ovr_weighted,
        'Brier': brier_overall,
        'MCC': mcc,
        'Kappa': kappa,
        'Kappa_Weighted': kappa_weighted,
        'Support': int(len(y_true)),
        'TP': np.nan, 'FP': np.nan, 'FN': np.nan, 'TN': np.nan
    })

    return results


# =============================================================================
# STAGE EVALUATORS (SSL1, SSL2, SSL3, Co-training, Label Propagation)
# =============================================================================

def evaluate_ssl1():
    """Evaluate SSL-1."""
    print("\n" + "="*80)
    print("EVALUATING SSL-1")
    print("="*80)

    df = pd.read_csv(SSL1_CSV)
    print(f"Loaded: {len(df):,} rows")

    features = load_features()
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()

    if len(df_gold) == 0:
        print("No gold labels found. Skipping.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    # Find predicted class column
    pred_col = find_pred_column(df_gold, version='')
    if pred_col:
        y_pred = df_gold[pred_col].astype(int).values
        print(f"Using prediction column: {pred_col}")
    else:
        print("Predicted class column not found. Regenerating predictions...")
        X = df_gold[features].fillna(0).values
        lgb_models, cat_models = load_ensemble_models(MODEL_DIR_SSL1)
        alpha = load_alpha(ALPHA_CSV)
        y_proba_tmp = predict_ensemble_batched(X, lgb_models, cat_models, alpha)
        y_pred = np.argmax(y_proba_tmp, axis=1)
        # Clean up
        del lgb_models, cat_models
        gc.collect()

    # Find probability columns
    proba_cols = find_proba_columns(df_gold, version='')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32)
        print(f"Using probability columns: {proba_cols}")
    else:
        print("Probability columns not found. Regenerating...")
        X = df_gold[features].fillna(0).values
        lgb_models, cat_models = load_ensemble_models(MODEL_DIR_SSL1)
        alpha = load_alpha(ALPHA_CSV)
        y_proba = predict_ensemble_batched(X, lgb_models, cat_models, alpha)
        # Clean up
        del lgb_models, cat_models
        gc.collect()

    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-1")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results


def evaluate_ssl2():
    """Evaluate SSL-2."""
    print("\n" + "="*80)
    print("EVALUATING SSL-2")
    print("="*80)

    df = pd.read_csv(SSL2_CSV)
    print(f"Loaded: {len(df):,} rows")

    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0:
        print("No gold labels found. Skipping.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    pred_col = find_pred_column(df_gold, version='_v2')
    if not pred_col:
        raise KeyError("Predicted class column for SSL-2 not found")

    y_pred = df_gold[pred_col].astype(int).values
    print(f"Using prediction column: {pred_col}")

    proba_cols = find_proba_columns(df_gold, version='_v2')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32)
        print(f"Found {len(proba_cols)} probability columns")
    else:
        print("Warning: No probability columns found. ROC-AUC/Brier will be NaN.")
        y_proba = None

    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-2")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results


def evaluate_ssl3():
    """Evaluate SSL-3."""
    print("\n" + "="*80)
    print("EVALUATING SSL-3")
    print("="*80)

    df = pd.read_csv(SSL3_CSV)
    print(f"Loaded: {len(df):,} rows")

    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0:
        print("No gold labels found. Skipping.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    pred_col = find_pred_column(df_gold, version='_v3')
    if not pred_col:
        raise KeyError("Predicted class column for SSL-3 not found")

    y_pred = df_gold[pred_col].astype(int).values
    print(f"Using prediction column: {pred_col}")

    proba_cols = find_proba_columns(df_gold, version='_v3')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32)
        print(f"Found {len(proba_cols)} probability columns")
    else:
        print("Warning: No probability columns found. ROC-AUC/Brier will be NaN.")
        y_proba = None

    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-3")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results


def evaluate_cotraining():
    """Evaluate Co-training."""
    print("\n" + "="*80)
    print("EVALUATING CO-TRAINING")
    print("="*80)

    df = pd.read_csv(COTRAIN_CSV)
    print(f"Loaded: {len(df):,} rows")

    features = load_features()
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()

    if len(df_gold) == 0:
        print("No gold labels found. Skipping.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    if 'co_train_final_pred' not in df_gold.columns:
        print("co_train_final_pred column not found. Skipping.")
        return []

    y_pred = df_gold['co_train_final_pred'].astype(int).values

    # Regenerate probabilities from saved models
    print("Regenerating probabilities from co-training models...")
    mid = max(1, len(features) // 2)
    viewA_features = features[:mid]
    viewB_features = features[mid:]

    X_A = df_gold[viewA_features].fillna(0).values
    X_B = df_gold[viewB_features].fillna(0).values

    try:
        lgb_A_path = Path(MODEL_DIR_COTRAIN) / "lgb_model_A_iter_5.txt"
        cat_A_path = Path(MODEL_DIR_COTRAIN) / "catboost_model_A_iter_5.cbm"
        lgb_B_path = Path(MODEL_DIR_COTRAIN) / "lgb_model_B_iter_5.txt"
        cat_B_path = Path(MODEL_DIR_COTRAIN) / "catboost_model_B_iter_5.cbm"

        have_A = lgb_A_path.exists() or cat_A_path.exists()
        have_B = lgb_B_path.exists() or cat_B_path.exists()

        if not (have_A or have_B):
            logging.warning("No co-training view models found. Skipping probability regeneration.")
            y_proba = None
        else:
            proba_A = None
            proba_B = None

            if lgb_A_path.exists():
                try:
                    lgb_A = lgb.Booster(model_file=str(lgb_A_path))
                    proba_A = lgb_A.predict(X_A)
                except Exception as e:
                    logging.warning(f"Failed LGB A predict: {e}")

            if cat_A_path.exists():
                try:
                    cat_A = safe_catboost_load(str(cat_A_path), use_gpu=USE_GPU)
                    proba_catA = cat_A.predict_proba(X_A)
                    proba_A = proba_catA if proba_A is None else (np.asarray(proba_A) + np.asarray(proba_catA)) / 2.0
                except Exception as e:
                    logging.warning(f"Failed Cat A predict: {e}")

            if lgb_B_path.exists():
                try:
                    lgb_B = lgb.Booster(model_file=str(lgb_B_path))
                    proba_B = lgb_B.predict(X_B)
                except Exception as e:
                    logging.warning(f"Failed LGB B predict: {e}")

            if cat_B_path.exists():
                try:
                    cat_B = safe_catboost_load(str(cat_B_path), use_gpu=USE_GPU)
                    proba_catB = cat_B.predict_proba(X_B)
                    proba_B = proba_catB if proba_B is None else (np.asarray(proba_B) + np.asarray(proba_catB)) / 2.0
                except Exception as e:
                    logging.warning(f"Failed Cat B predict: {e}")

            if proba_A is not None and proba_B is not None:
                proba_A = np.asarray(proba_A, dtype=np.float32)
                proba_B = np.asarray(proba_B, dtype=np.float32)
                y_proba = ((proba_A + proba_B) / 2.0).astype(np.float32)
            elif proba_A is not None:
                y_proba = np.asarray(proba_A, dtype=np.float32)
            elif proba_B is not None:
                y_proba = np.asarray(proba_B, dtype=np.float32)
            else:
                y_proba = None

            logging.info(f"Co-training probabilities generated: {'yes' if y_proba is not None else 'no'}")
    except Exception as e:
        logging.warning(f"Failed to regenerate co-training probabilities: {e}")
        y_proba = None
    finally:
        gc.collect()

    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "Co-Training")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results


def evaluate_labelprop():
    """Evaluate Label Propagation."""
    print("\n" + "="*80)
    print("EVALUATING LABEL PROPAGATION")
    print("="*80)

    df = pd.read_csv(LP_CSV)
    print(f"Loaded: {len(df):,} rows")

    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0:
        print("No gold labels found. Skipping.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    # Find prediction column
    if 'labelprop_final' in df_gold.columns:
        y_pred = df_gold['labelprop_final'].astype(int).values
        print("Using: labelprop_final")
    elif 'labelprop_pred' in df_gold.columns:
        y_pred = df_gold['labelprop_pred'].astype(int).values
        print("Using: labelprop_pred")
    else:
        print("No label propagation predictions found. Skipping.")
        return []

    # Extract probabilities from saved LP model
    try:
        logging.info("Loading LP model for probability extraction...")
        lp = joblib.load(LP_MODEL)
        if not hasattr(lp, 'label_distributions_'):
            raise AttributeError("Loaded LP object has no 'label_distributions_' attribute")

        label_distributions = np.asarray(lp.label_distributions_, dtype=np.float32)

        # Use original_index if available, otherwise try lp_row_id or DataFrame index
        # Coerce to integer indices in case of dtype/object issues
        if 'original_index' in df_gold.columns:
          df_gold['original_index'] = pd.to_numeric(df_gold['original_index'], errors='coerce').astype('Int64')
          gold_indices = df_gold['original_index'].astype(int).values

            try:
                y_proba = label_distributions[gold_indices].astype(np.float32)
            except Exception as e:
                logging.warning(f"Alignment with original_index failed: {e}")
                y_proba = None
        elif 'lp_row_id' in df_gold.columns:
            gold_indices = df_gold['lp_row_id'].astype(int).values
            logging.info("Using lp_row_id column for alignment")
            try:
                y_proba = label_distributions[gold_indices].astype(np.float32)
            except Exception as e:
                logging.warning(f"Alignment with lp_row_id failed: {e}")
                y_proba = None
        else:
            # Last resort: try DataFrame index (risky)
            try:
                gold_indices = df_gold.index.values
                logging.warning("No explicit index; attempting to align using DataFrame index (risky).")
                y_proba = label_distributions[gold_indices].astype(np.float32)
            except Exception as e:
                logging.warning(f"Fallback alignment failed: {e}")
                y_proba = None

        if y_proba is not None:
            logging.info(f"✓ Extracted probabilities: shape {y_proba.shape}")
        else:
            logging.warning("LP probabilities could not be aligned; ROC/Brier will be NaN.")
    except Exception as e:
        logging.warning(f"Failed to extract LP probabilities: {e}")
        y_proba = None

    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "Label_Propagation")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*80)
    print("COMPREHENSIVE SSL METRICS EVALUATION")
    print("="*80)
    print(f"GPU Acceleration: {'Enabled' if USE_GPU else 'Disabled'}")
    print(f"Ordinal Classes: {ORDINAL_CLASSES}")
    print("="*80)

    # Validate presence of files (optional but recommended)
    try:
        validate_paths()
    except FileNotFoundError as e:
        logging.error(e)
        # Still continue — validation may be relaxed if some stages produce predictions internally
        logging.info("Continuing despite missing files (some stages may regenerate predictions).")

    all_rows = []

    # Evaluate each stage
    for fn in (evaluate_ssl1, evaluate_ssl2, evaluate_ssl3, evaluate_cotraining, evaluate_labelprop):
        try:
            rows = fn()
            all_rows.extend(rows)
            gc.collect()  # Free memory between stages
        except Exception as e:
            logging.error(f"Stage {fn.__name__} failed: {e}")
            import traceback
            traceback.print_exc()

    if not all_rows:
        print("\n✗ No metrics generated. Check file paths and columns.")
        return

    # Save results
    df_out = pd.DataFrame(all_rows)
    cols = ['Stage','Class','Metric_Type','Accuracy','Precision','Recall','F1_Score',
            'ROC_AUC','Brier','MCC','Kappa','Kappa_Weighted','Support','TP','FP','FN','TN']
    # Ensure all columns exist in DataFrame (fill missing)
    for c in cols:
        if c not in df_out.columns:
            df_out[c] = np.nan
    df_out = df_out[cols]
    df_out.to_csv(OUTPUT_CSV, index=False, float_format='%.5f')

    print("\n" + "="*80)
    print("EVALUATION COMPLETE")
    print("="*80)
    print(f"Output: {OUTPUT_CSV}")
    print(f"Total rows: {len(df_out)}")
    print("\nPer-Stage Summary (Macro F1):")
    try:
        print(df_out[df_out['Metric_Type'] == 'Macro'][['Stage', 'F1_Score', 'ROC_AUC', 'Brier', 'MCC', 'Kappa']])
    except Exception:
        pass
    print("="*80)


if __name__ == '__main__':
    main()


KeyboardInterrupt: 

**Final Eval Metrics**

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive SSL Metrics Evaluation - CORRECTED & FINAL
=========================================================
Computes Accuracy, per-class Precision/Recall/F1, Macro/Weighted F1,
ROC-AUC (macro/weighted and per-class), Brier (overall/per-class), MCC,
Cohen's Kappa, and Quadratic-Weighted Kappa for SSL-1, SSL-2, SSL-3,
Co-Training, and Label Propagation.

Fixes:
- Semantic View A/B split embedded (matches co-training)
- Auto-detect latest co-training iteration models
- CatBoost predict_proba fallback via prediction_type='Probability' or softmax(RawFormulaVal)
- Label Propagation probability alignment + normalization
- Single CSV output, same schema

Output: ssl_comprehensive_metrics.csv
"""

import os
import gc
import re
import glob
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
import logging
from collections import defaultdict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score
)
import lightgbm as lgb
import catboost as cb
import joblib

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data"

# File paths
FEATURES_FILE = os.path.join(BASE_DIR, "selected_features_topK_4class_weighted.csv")
SSL1_CSV = os.path.join(BASE_DIR, "predictions_with_final_label.csv")
SSL2_CSV = os.path.join(BASE_DIR, "predictions_with_final_label_v2.csv")
SSL3_CSV = os.path.join(BASE_DIR, "predictions_with_final_label_v3.csv")
COTRAIN_CSV = os.path.join(BASE_DIR, "predictions_co_training_final.csv")
LP_CSV = os.path.join(BASE_DIR, "predictions_labelprop_final.csv")

# Model paths
MODEL_DIR_SSL1 = os.path.join(BASE_DIR, "models_stacked")
MODEL_DIR_SSL2 = os.path.join(BASE_DIR, "models_stacked_v2")
MODEL_DIR_SSL3 = os.path.join(BASE_DIR, "models_stacked_v3")
MODEL_DIR_COTRAIN = MODEL_DIR_SSL2
LP_MODEL = os.path.join(BASE_DIR, "models/labelprop_model.joblib")

ALPHA_CSV = os.path.join(BASE_DIR, "lgb_vs_catboost_vs_ensemble.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "ssl_comprehensive_metrics.csv")

GOLD_LABEL_COL = "human_label_clean"
NUM_CLASSES = 4
ORDINAL_CLASSES = True

# Performance settings
USE_GPU = False  # Keep CPU for CatBoost to avoid CUDA mismatch
BATCH_SIZE = 10000

# =============================================================================
# HARDCODED SEMANTIC VIEW SPLIT (matches co-training script)
# =============================================================================

VIEW_A_FEATURES = [
    'sentiment_score', 'sentiment_score_z', 'news_sentiment_score', 'news_sentiment_score_z',
    'sentiment_homogeneity', 'sentiment_homogeneity_z', 'social_news_sentiment_delta',
    'social_news_sentiment_delta_z', 'emotional_news_delta', 'A_Emotional_News_Delta',
    'A_Emotional_News_Delta_z',
    'headline_similarity_score', 'headline_similarity_score_z', 'text_similarity_top_posts',
    'text_similarity_top_posts_z', 'post_length_words', 'post_length_words_z',
    'post_length_chars', 'post_length_chars_z',
    'crowd_keyword_count', 'crowd_keyword_count_z', 'news_keyword_count', 'news_keyword_count_z',
    'ner_mentions_count', 'ner_mentions_count_z', 'ticker_repetition_window',
    'ticker_repetition_window_z', 'jargon_prevalence',
    'user_network_influence', 'user_network_influence_z', 'reply_chain_length',
    'reply_chain_length_z', 'comment_reply_count', 'comment_reply_count_z',
    'user_posting_frequency', 'user_posting_frequency_z', 'upvote_like_ratio',
    'upvote_like_ratio_z',
    'social_proof_velocity', 'social_proof_velocity_z', 'initial_engagement_velocity',
    'initial_engagement_velocity_z', 'H_Social_Momentum_Ratio', 'H_Social_Momentum_Ratio_z',
    'H_Network_Influence_Clustering', 'H_Network_Influence_Clustering_z',
    'H_Jargon_Sentiment_Homogeneity', 'H_Jargon_Sentiment_Homogeneity_z',
    'A_Signal_Novelty', 'A_Signal_Novelty_z',
    'media_richness_flag', 'recency_keyword_flag',
    'emo_emb_1', 'emo_emb_3', 'emo_emb_7', 'emo_emb_31', 'emo_emb_34', 'emo_emb_43',
    'post_id', 'post_id_z'
]

VIEW_B_FEATURES = [
    'intraday_return_24hr', 'intraday_return_24hr_z', 'intraday_return_30min',
    'intraday_return_30min_z', 'daily_return_daily', 'rolling_5d_return_daily',
    'rolling_10d_return_daily', 'rolling_20d_return_daily', 'rolling_30d_return_daily',
    'intraday_volatility', 'intraday_volatility_z', 'rolling_5d_volatility_daily',
    'rolling_10d_volatility_daily', 'rolling_20d_volatility_daily', 'rolling_30d_volatility_daily',
    'last_bar_volume', 'prior_volume_daily', 'prior_day_volume', 'volume_vs_prior_avg',
    'rolling_5d_volume_daily', 'rolling_10d_volume_daily', 'rolling_20d_volume_daily',
    'rolling_30d_volume_daily', 'volume_spike_ratio', 'volume_spike_ratio_z',
    'trading_volume_spike_flag',
    'H_Retail_Volume_Float_Rotation', 'H_Retail_Volume_Float_Rotation_z',
    'inferred_float_rotation', 'inferred_float_rotation_z',
    'last_available_open', 'last_available_high', 'last_available_low',
    'last_available_close', 'prior_day_close', 'prior_close_daily',
    'price_change_from_prior_close', 'High_daily', 'Low_daily', 'Open_daily',
    'time_offset_from_event', 'time_offset_from_event_z', 'temporal_clustering_flag',
    'fed_z',
    'emo_emb_0', 'emo_emb_2', 'emo_emb_4', 'emo_emb_5', 'emo_emb_8', 'emo_emb_11',
    'emo_emb_12', 'emo_emb_13', 'emo_emb_16', 'emo_emb_18', 'emo_emb_19', 'emo_emb_20',
    'emo_emb_22', 'emo_emb_28', 'emo_emb_29', 'emo_emb_30', 'emo_emb_37', 'emo_emb_38',
    'emo_emb_45', 'emo_emb_48', 'emo_emb_49', 'emo_emb_55', 'emo_emb_56', 'emo_emb_59',
    'emo_emb_60', 'emo_emb_61', 'emo_emb_62', 'emo_emb_63', 'emo_emb_64', 'emo_emb_65',
    'emo_emb_66', 'emo_emb_67', 'emo_emb_68', 'emo_emb_69', 'emo_emb_70', 'emo_emb_71',
    'emo_emb_72', 'emo_emb_73', 'emo_emb_74', 'emo_emb_75', 'emo_emb_76', 'emo_emb_77',
    'emo_emb_78', 'emo_emb_79', 'emo_emb_80', 'emo_emb_81', 'emo_emb_82', 'emo_emb_83',
    'emo_emb_84', 'emo_emb_85', 'emo_emb_86', 'emo_emb_87', 'emo_emb_88', 'emo_emb_89',
    'emo_emb_90', 'emo_emb_91', 'emo_emb_92', 'emo_emb_93', 'emo_emb_94', 'emo_emb_95',
    'emo_emb_96', 'emo_emb_97', 'emo_emb_98', 'emo_emb_99', 'emo_emb_100', 'emo_emb_101',
    'emo_emb_102', 'emo_emb_103', 'emo_emb_104', 'emo_emb_105', 'emo_emb_106', 'emo_emb_107',
    'emo_emb_108', 'emo_emb_109', 'emo_emb_110', 'emo_emb_111', 'emo_emb_112', 'emo_emb_113',
    'emo_emb_114', 'emo_emb_115', 'emo_emb_116', 'emo_emb_117', 'emo_emb_118', 'emo_emb_119',
    'emo_emb_120', 'emo_emb_121', 'emo_emb_129', 'emo_emb_130', 'emo_emb_131', 'emo_emb_132',
    'emo_emb_133', 'emo_emb_134', 'emo_emb_135', 'emo_emb_136', 'emo_emb_137',
    'emo_val_code'
]

# =============================================================================
# VALIDATION
# =============================================================================

def validate_paths():
    """Check all required files exist before starting."""
    required = [FEATURES_FILE, SSL1_CSV, SSL2_CSV, SSL3_CSV, COTRAIN_CSV, LP_CSV]
    missing = [p for p in required if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Missing files: {missing}")
    logging.info("All input files validated.")

# =============================================================================
# UTILITIES
# =============================================================================

def softmax(z):
    z = np.asarray(z, dtype=np.float32)
    if z.ndim == 1:
        z = z.reshape(-1, 1)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    denom = np.clip(e.sum(axis=1, keepdims=True), 1e-9, None)
    return e / denom

def safe_catboost_load(path: str, use_gpu: bool = False):
    """Load CatBoost model; caller controls prediction_type."""
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    model = cb.CatBoost()
    model.load_model(path)
    return model

def get_latest_model(model_dir, stem, ext):
    """Return (path, iter) for latest available '*_iter_*.ext' under model_dir."""
    pattern = str(Path(model_dir) / f"{stem}_iter_*.{ext}")
    paths = glob.glob(pattern)
    if not paths:
        return None, None
    best_path, best_iter = None, -1
    for p in paths:
        m = re.search(r"iter_(\d+)", Path(p).name)
        if m:
            it = int(m.group(1))
            if it > best_iter:
                best_iter = it
                best_path = p
    return best_path, best_iter

def find_proba_columns(df, version='', n_classes=NUM_CLASSES):
    df_cols = list(df.columns)
    variants = []
    v_raw = version or ''
    v_strip = v_raw.replace('_', '')
    for i in range(n_classes):
        candidates = [
            f"f_proba_class_{i}{v_raw}",
            f"f_proba_class_{i}{v_strip}",
            f"f_proba_class_{i}",
            f"proba_class_{i}{v_raw}",
            f"fprobaclass{i}{v_strip}",
            f"proba_{i}{v_strip}",
        ]
        found = None
        for cand in candidates:
            for col in df_cols:
                if col.lower().replace('_', '') == cand.lower().replace('_', ''):
                    found = col; break
            if found: break
        if not found:
            variants = None; break
        variants.append(found)
    if variants: return variants

    candidates = [c for c in df_cols if 'proba' in c.lower()]
    class_map = {}
    for col in candidates:
        m = re.search(r'(\d+)', col)
        if m:
            idx = int(m.group(1))
            if 0 <= idx < n_classes and idx not in class_map:
                class_map[idx] = col
    if len(class_map) == n_classes:
        return [class_map[i] for i in range(n_classes)]

    proba_group = [c for c in df_cols if 'proba' in c.lower()]
    if len(proba_group) >= n_classes:
        return proba_group[:n_classes]
    return None

def load_ensemble_models(model_dir, num_folds=5, use_gpu=False):
    model_dir = Path(model_dir)
    lgb_models = []
    cat_models = []
    for i in range(1, num_folds + 1):
        lgb_path = model_dir / f"lgb_fold_{i}.txt"
        cat_path = model_dir / f"catboost_fold_{i}.cbm"
        if lgb_path.exists():
            try: lgb_models.append(lgb.Booster(model_file=str(lgb_path)))
            except Exception as e: logging.warning(f"Failed to load LGB {lgb_path}: {e}")
        if cat_path.exists():
            try: cat_models.append(safe_catboost_load(str(cat_path), use_gpu=use_gpu))
            except Exception as e: logging.warning(f"Failed to load CatBoost {cat_path}: {e}")
    logging.info(f"Loaded ensemble models from {model_dir}: lgb={len(lgb_models)}, cat={len(cat_models)}")
    return lgb_models, cat_models

def predict_ensemble_batched(X, lgb_models, cat_models, alpha=0.47, batch_size=BATCH_SIZE):
    X = np.asarray(X)
    if X.ndim == 1: X = X.reshape(-1, 1)
    n_samples = X.shape[0]
    final_proba = np.zeros((n_samples, NUM_CLASSES), dtype=np.float32)
    has_lgb, has_cat = bool(lgb_models), bool(cat_models)
    if not (has_lgb or has_cat):
        logging.warning("No ensemble models found. Returning NaN probabilities.")
        return np.full((n_samples, NUM_CLASSES), np.nan, dtype=np.float32)
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        Xb = X[start:end]
        lgb_avg = None
        if has_lgb:
            lgb_preds = [np.asarray(m.predict(Xb), dtype=np.float32) for m in lgb_models]
            lgb_avg = np.mean(lgb_preds, axis=0)
        cat_avg = None
        if has_cat:
            cat_preds = []
            for m in cat_models:
                try:
                    p = m.predict(Xb, prediction_type='Probability')
                except Exception:
                    raw = m.predict(Xb, prediction_type='RawFormulaVal')
                    p = softmax(raw)
                cat_preds.append(np.asarray(p, dtype=np.float32))
            cat_avg = np.mean(cat_preds, axis=0)
        if lgb_avg is None: combined = cat_avg
        elif cat_avg is None: combined = lgb_avg
        else:
            if lgb_avg.ndim == 1: lgb_avg = np.vstack([1 - lgb_avg, lgb_avg]).T
            if cat_avg.ndim == 1: cat_avg = np.vstack([1 - cat_avg, cat_avg]).T
            combined = alpha * lgb_avg + (1.0 - alpha) * cat_avg
        final_proba[start:end] = combined.astype(np.float32)
    return final_proba

def load_features():
    df = pd.read_csv(FEATURES_FILE)
    if 'feature' not in df.columns:
        raise KeyError("Feature CSV must contain 'feature' column")
    return df['feature'].tolist()

def load_alpha(alpha_csv):
    try:
        df = pd.read_csv(alpha_csv)
        if 'alpha' in df.columns:
            return float(df['alpha'].mean())
    except Exception:
        pass
    return 0.47

def find_pred_column(df, version=''):
    patterns = [
        f'predicted_class{version}',
        f'predictedclass{version}',
        f'predicted{version}',
        f'pred_class{version}'
    ]
    for col in df.columns:
        col_lower = col.lower().replace('_', '')
        for pattern in patterns:
            if col_lower == pattern.replace('_', ''):
                return col
    return None

def multiclass_brier(y_true, y_proba, n_classes=NUM_CLASSES):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_proba = np.asarray(y_proba, dtype=np.float32)
    n = len(y_true)
    Y = np.zeros((n, n_classes), dtype=np.float32)
    Y[np.arange(n), y_true] = 1.0
    per_sample = np.sum((y_proba - Y) ** 2, axis=1)
    overall = float(np.mean(per_sample))
    per_class = []
    for c in range(n_classes):
        y_c = (y_true == c).astype(np.float32)
        per_class.append(float(np.mean((y_proba[:, c] - y_c) ** 2)))
    return overall, per_class

def compute_metrics_for_stage(y_true, y_pred, y_proba=None, stage_name=""):
    results = []
    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    acc = float(accuracy_score(y_true, y_pred))
    precision = precision_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    recall = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    f1 = f1_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)

    macro_precision = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    macro_recall = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))

    weighted_precision = float(precision_score(y_true, y_pred, average='weighted', zero_division=0))
    weighted_recall = float(recall_score(y_true, y_pred, average='weighted', zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))

    if y_proba is not None:
        try:
            roc_auc_ovr_macro = float(roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro'))
            roc_auc_ovr_weighted = float(roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted'))
            roc_auc_per_class = []
            for cls in range(NUM_CLASSES):
                y_bin = (y_true == cls).astype(int)
                roc_auc_per_class.append(float(roc_auc_score(y_bin, y_proba[:, cls])) if len(np.unique(y_bin)) > 1 else np.nan)
        except Exception as e:
            logging.warning(f"ROC-AUC computation failed for {stage_name}: {e}")
            roc_auc_ovr_macro = np.nan
            roc_auc_ovr_weighted = np.nan
            roc_auc_per_class = [np.nan] * NUM_CLASSES
    else:
        roc_auc_ovr_macro = np.nan
        roc_auc_ovr_weighted = np.nan
        roc_auc_per_class = [np.nan] * NUM_CLASSES

    try: mcc = float(matthews_corrcoef(y_true, y_pred))
    except Exception: mcc = np.nan
    try:
        kappa = float(cohen_kappa_score(y_true, y_pred))
        kappa_weighted = float(cohen_kappa_score(y_true, y_pred, weights='quadratic')) if ORDINAL_CLASSES else np.nan
    except Exception:
        kappa = np.nan; kappa_weighted = np.nan

    if y_proba is not None:
        try:
            brier_overall, brier_per_class = multiclass_brier(y_true, y_proba, NUM_CLASSES)
        except Exception as e:
            logging.warning(f"Brier computation failed for {stage_name}: {e}")
            brier_overall, brier_per_class = np.nan, [np.nan] * NUM_CLASSES
    else:
        brier_overall, brier_per_class = np.nan, [np.nan] * NUM_CLASSES

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    for cls in range(NUM_CLASSES):
        results.append({
            'Stage': stage_name,
            'Class': int(cls),
            'Metric_Type': 'Per_Class',
            'Accuracy': acc,
            'Precision': float(precision[cls]) if len(precision) > cls else np.nan,
            'Recall': float(recall[cls]) if len(recall) > cls else np.nan,
            'F1_Score': float(f1[cls]) if len(f1) > cls else np.nan,
            'ROC_AUC': float(roc_auc_per_class[cls]) if roc_auc_per_class[cls] is not np.nan else np.nan,
            'Brier': float(brier_per_class[cls]) if brier_per_class[cls] is not np.nan else np.nan,
            'MCC': np.nan,
            'Kappa': np.nan,
            'Kappa_Weighted': np.nan,
            'Support': int(np.sum(y_true == cls)),
            'TP': int(cm[cls, cls]),
            'FP': int(np.sum(cm[:, cls]) - cm[cls, cls]),
            'FN': int(np.sum(cm[cls, :]) - cm[cls, cls]),
            'TN': int(np.sum(cm) - np.sum(cm[cls, :]) - np.sum(cm[:, cls]) + cm[cls, cls])
        })

    results.append({
        'Stage': stage_name, 'Class': 'Macro_Avg', 'Metric_Type': 'Macro',
        'Accuracy': acc, 'Precision': macro_precision, 'Recall': macro_recall, 'F1_Score': macro_f1,
        'ROC_AUC': roc_auc_ovr_macro, 'Brier': brier_overall, 'MCC': mcc, 'Kappa': kappa,
        'Kappa_Weighted': kappa_weighted, 'Support': int(len(y_true)),
        'TP': np.nan, 'FP': np.nan, 'FN': np.nan, 'TN': np.nan
    })

    results.append({
        'Stage': stage_name, 'Class': 'Weighted_Avg', 'Metric_Type': 'Weighted',
        'Accuracy': acc, 'Precision': weighted_precision, 'Recall': weighted_recall, 'F1_Score': weighted_f1,
        'ROC_AUC': roc_auc_ovr_weighted, 'Brier': brier_overall, 'MCC': mcc, 'Kappa': kappa,
        'Kappa_Weighted': kappa_weighted, 'Support': int(len(y_true)),
        'TP': np.nan, 'FP': np.nan, 'FN': np.nan, 'TN': np.nan
    })
    return results

# =============================================================================
# STAGE EVALUATORS
# =============================================================================

def evaluate_ssl1():
    print("\n" + "="*80); print("EVALUATING SSL-1"); print("="*80)
    df = pd.read_csv(SSL1_CSV); print(f"Loaded: {len(df):,} rows")
    features = load_features()
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0: print("No gold labels found. Skipping."); return []
    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values
    pred_col = find_pred_column(df_gold, version='')
    if pred_col:
        y_pred = df_gold[pred_col].astype(int).values; print(f"Using prediction column: {pred_col}")
    else:
        print("Predicted class column not found. Regenerating predictions...")
        X = df_gold[features].fillna(0).values
        lgb_models, cat_models = load_ensemble_models(MODEL_DIR_SSL1)
        alpha = load_alpha(ALPHA_CSV)
        y_proba_tmp = predict_ensemble_batched(X, lgb_models, cat_models, alpha)
        y_pred = np.argmax(y_proba_tmp, axis=1)
        del lgb_models, cat_models; gc.collect()
    proba_cols = find_proba_columns(df_gold, version='')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32); print(f"Using probability columns: {proba_cols}")
    else:
        print("Probability columns not found. Regenerating...")
        X = df_gold[features].fillna(0).values
        lgb_models, cat_models = load_ensemble_models(MODEL_DIR_SSL1)
        alpha = load_alpha(ALPHA_CSV)
        y_proba = predict_ensemble_batched(X, lgb_models, cat_models, alpha)
        del lgb_models, cat_models; gc.collect()
    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-1")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    return results

def evaluate_ssl2():
    print("\n" + "="*80); print("EVALUATING SSL-2"); print("="*80)
    df = pd.read_csv(SSL2_CSV); print(f"Loaded: {len(df):,} rows")
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0: print("No gold labels found. Skipping."); return []
    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values
    pred_col = find_pred_column(df_gold, version='_v2')
    if not pred_col: raise KeyError("Predicted class column for SSL-2 not found")
    y_pred = df_gold[pred_col].astype(int).values; print(f"Using prediction column: {pred_col}")
    proba_cols = find_proba_columns(df_gold, version='_v2')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32); print(f"Found {len(proba_cols)} probability columns")
    else:
        print("Warning: No probability columns found. ROC-AUC/Brier will be NaN."); y_proba = None
    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-2")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    return results

def evaluate_ssl3():
    print("\n" + "="*80); print("EVALUATING SSL-3"); print("="*80)
    df = pd.read_csv(SSL3_CSV); print(f"Loaded: {len(df):,} rows")
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0: print("No gold labels found. Skipping."); return []
    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values
    pred_col = find_pred_column(df_gold, version='_v3')
    if not pred_col: raise KeyError("Predicted class column for SSL-3 not found")
    y_pred = df_gold[pred_col].astype(int).values; print(f"Using prediction column: {pred_col}")
    proba_cols = find_proba_columns(df_gold, version='_v3')
    if proba_cols:
        y_proba = df_gold[proba_cols].values.astype(np.float32); print(f"Found {len(proba_cols)} probability columns")
    else:
        print("Warning: No probability columns found. ROC-AUC/Brier will be NaN."); y_proba = None
    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "SSL-3")
    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}, Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    return results

def evaluate_cotraining():
    """
    Evaluate Co-Training in a way that's actually informative for your pipeline.

    It computes two complementary views:
      1) CoTrain_FullGold: On all gold rows, score argmax of regenerated ViewA/B ensemble
         probabilities (not co_train_final_pred which is preserved from SSL-3 on gold).
      2) CoTrain_ChangedOnly: On rows co-training actually labeled (SSL-3 == -1 and
         co_train_final_pred != -1) that also have gold labels; if none, this slice is skipped.

    Both slices include ROC-AUC/Brier from regenerated probabilities.
    """
    print("\n" + "="*80)
    print("EVALUATING CO-TRAINING (ARGMAX ENSEMBLE + CHANGED-ONLY)")
    print("="*80)

    df = pd.read_csv(COTRAIN_CSV)
    print(f"Loaded: {len(df):,} rows")

    # Full gold set (used for CoTrain_FullGold)
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0:
        print("No gold labels found. Skipping Co-Training.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true_gold = df_gold[GOLD_LABEL_COL].astype(int).values

    # Regenerate probabilities using exact semantic views
    print("Regenerating probabilities from co-training models (semantic views)...")
    X_A_gold = df_gold[VIEW_A_FEATURES].fillna(0).values
    X_B_gold = df_gold[VIEW_B_FEATURES].fillna(0).values

    # Find latest models per view
    lgb_A_path, itA_lgb = get_latest_model(MODEL_DIR_COTRAIN, "lgb_model_A", "txt")
    cat_A_path, itA_cat = get_latest_model(MODEL_DIR_COTRAIN, "catboost_model_A", "cbm")
    lgb_B_path, itB_lgb = get_latest_model(MODEL_DIR_COTRAIN, "lgb_model_B", "txt")
    cat_B_path, itB_cat = get_latest_model(MODEL_DIR_COTRAIN, "catboost_model_B", "cbm")

    have_A = bool(lgb_A_path or cat_A_path)
    have_B = bool(lgb_B_path or cat_B_path)
    y_proba_gold = None

    try:
        proba_A = None
        if lgb_A_path:
            try:
                lgb_A = lgb.Booster(model_file=str(lgb_A_path))
                proba_A = lgb_A.predict(X_A_gold)
                logging.info(f"Used LGB A model @ iter={itA_lgb}")
            except Exception as e:
                logging.warning(f"Failed LGB A predict: {e}")
        if cat_A_path:
            try:
                cat_A = safe_catboost_load(str(cat_A_path), use_gpu=USE_GPU)
                try:
                    proba_catA = cat_A.predict(X_A_gold, prediction_type='Probability')
                except Exception:
                    rawA = cat_A.predict(X_A_gold, prediction_type='RawFormulaVal')
                    proba_catA = softmax(rawA)
                proba_A = proba_catA if proba_A is None else (np.asarray(proba_A) + np.asarray(proba_catA)) / 2.0
                logging.info(f"Used CatBoost A model @ iter={itA_cat}")
            except Exception as e:
                logging.warning(f"Failed CatBoost A predict: {e}")

        proba_B = None
        if lgb_B_path:
            try:
                lgb_B = lgb.Booster(model_file=str(lgb_B_path))
                proba_B = lgb_B.predict(X_B_gold)
                logging.info(f"Used LGB B model @ iter={itB_lgb}")
            except Exception as e:
                logging.warning(f"Failed LGB B predict: {e}")
        if cat_B_path:
            try:
                cat_B = safe_catboost_load(str(cat_B_path), use_gpu=USE_GPU)
                try:
                    proba_catB = cat_B.predict(X_B_gold, prediction_type='Probability')
                except Exception:
                    rawB = cat_B.predict(X_B_gold, prediction_type='RawFormulaVal')
                    proba_catB = softmax(rawB)
                proba_B = proba_catB if proba_B is None else (np.asarray(proba_B) + np.asarray(proba_catB)) / 2.0
                logging.info(f"Used CatBoost B model @ iter={itB_cat}")
            except Exception as e:
                logging.warning(f"Failed CatBoost B predict: {e}")

        if proba_A is not None and proba_B is not None:
            y_proba_gold = ((np.asarray(proba_A, dtype=np.float32) + np.asarray(proba_B, dtype=np.float32)) / 2.0).astype(np.float32)
        elif proba_A is not None:
            y_proba_gold = np.asarray(proba_A, dtype=np.float32)
        elif proba_B is not None:
            y_proba_gold = np.asarray(proba_B, dtype=np.float32)
        else:
            y_proba_gold = None

        if y_proba_gold is not None:
            # Normalize for safety
            row_sums = y_proba_gold.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1.0
            y_proba_gold = (y_proba_gold / row_sums).astype(np.float32)
            logging.info(f"Co-training probabilities generated on gold: shape {y_proba_gold.shape}")
        else:
            logging.warning("No co-training probabilities available for gold; metrics will lack ROC/Brier.")
    except Exception as e:
        logging.warning(f"Probability regeneration failed for Co-Training: {e}")
        y_proba_gold = None
    finally:
        gc.collect()

    results = []

    # 1) CoTrain_FullGold: score argmax of regenerated probs on all gold rows
    if y_proba_gold is not None:
        y_pred_fullgold = np.argmax(y_proba_gold, axis=1).astype(int)
        results.extend(compute_metrics_for_stage(y_true_gold, y_pred_fullgold, y_proba_gold, "CoTrain_FullGold"))
        print(f"✓ CoTrain_FullGold — Acc: {accuracy_score(y_true_gold, y_pred_fullgold):.4f}, "
              f"Macro F1: {f1_score(y_true_gold, y_pred_fullgold, average='macro'):.4f}")
    else:
        # Fall back to preserved labels (will likely be 1.0000 due to SSL-3 overwrite)
        y_pred_preserved = df_gold['co_train_final_pred'].astype(int).values
        results.extend(compute_metrics_for_stage(y_true_gold, y_pred_preserved, None, "CoTrain_FullGold"))
        print(f"✓ CoTrain_FullGold — Acc: {accuracy_score(y_true_gold, y_pred_preserved):.4f}, "
              f"Macro F1: {f1_score(y_true_gold, y_pred_preserved, average='macro'):.4f} (preserved labels)")

    # 2) CoTrain_ChangedOnly: rows actually labeled by co-training and with gold labels
    #    Condition in full DF is: (final_label_v3 == -1) & (co_train_final_pred != -1)
    if {'final_label_v3', 'co_train_final_pred'}.issubset(df.columns):
        changed_mask_full = (df['final_label_v3'].astype(int) == -1) & (df['co_train_final_pred'].astype(int) != -1)
        # Intersect with gold indices
        idx_gold = df_gold.index
        changed_mask_gold = changed_mask_full.loc[idx_gold] if isinstance(changed_mask_full, pd.Series) else changed_mask_full[idx_gold]
        if changed_mask_gold.any():
            sel = changed_mask_gold.values
            y_true_changed = y_true_gold[sel]
            if y_proba_gold is not None:
                y_pred_changed = np.argmax(y_proba_gold, axis=1).astype(int)[sel]
                y_proba_changed = y_proba_gold[sel]
            else:
                # fallback: use preserved labels (less informative)
                y_pred_changed = df_gold['co_train_final_pred'].astype(int).values[sel]
                y_proba_changed = None
            results.extend(compute_metrics_for_stage(y_true_changed, y_pred_changed, y_proba_changed, "CoTrain_ChangedOnly"))
            print(f"✓ CoTrain_ChangedOnly — n={len(y_true_changed)}: "
                  f"Acc: {accuracy_score(y_true_changed, y_pred_changed):.4f}, "
                  f"Macro F1: {f1_score(y_true_changed, y_pred_changed, average='macro'):.4f}")
        else:
            print("No intersection between co-training pseudo-labeled rows and gold; skipping CoTrain_ChangedOnly.")
    else:
        print("Columns final_label_v3 / co_train_final_pred missing; skipping CoTrain_ChangedOnly.")

    return results


def evaluate_labelprop():
    """
    Evaluate Label Propagation with full diagnostics and safe probability handling.

    Features:
    - Uses labelprop_final (or fallback labelprop_pred)
    - Loads LP model and extracts label_distributions_
    - Aligns using: original_index → lp_row_id → DataFrame index
    - Fixes zero-mass rows using tiny uniform prior
    - Normalizes to valid probability simplex
    - Prints detailed diagnostics pre/post normalization
    """

    print("\n" + "="*80)
    print("EVALUATING LABEL PROPAGATION — CLEAN REWRITE")
    print("="*80)

    # ---------------------------------------------------------
    # Load CSV
    # ---------------------------------------------------------
    df = pd.read_csv(LP_CSV)
    print(f"Loaded: {len(df):,} rows")

    # Select gold rows only
    df_gold = df[df[GOLD_LABEL_COL].notna()].copy()
    if len(df_gold) == 0:
        print("No gold labels found. Skipping Label Propagation.")
        return []

    print(f"Gold-labeled samples: {len(df_gold):,}")
    y_true = df_gold[GOLD_LABEL_COL].astype(int).values

    # ---------------------------------------------------------
    # Pick prediction column
    # ---------------------------------------------------------
    if "labelprop_final" in df_gold.columns:
        pred_col = "labelprop_final"
    elif "labelprop_pred" in df_gold.columns:
        pred_col = "labelprop_pred"
    else:
        print("No LP prediction column found. Skipping.")
        return []

    y_pred = df_gold[pred_col].astype(int).values
    print(f"Using predictions: {pred_col}")

    # ---------------------------------------------------------
    # Load LP model to extract probabilities
    # ---------------------------------------------------------
    y_proba = None
    try:
        logging.info("Loading LP model...")
        lp = joblib.load(LP_MODEL)

        if not hasattr(lp, "label_distributions_"):
            raise AttributeError("LP model does not contain label_distributions_")

        label_distributions = np.asarray(lp.label_distributions_, dtype=np.float32)
        print(f"Loaded LP distributions: shape = {label_distributions.shape}")

        # -----------------------------------------------------
        # Alignment logic
        # -----------------------------------------------------
        aligned = False

        # Priority 1: original_index
        if "original_index" in df_gold.columns:
            print("Attempting alignment via original_index…")
            try:
                idx = df_gold["original_index"].astype(int).values
                y_proba = label_distributions[idx]
                print("✓ Alignment via original_index successful.")
                aligned = True
            except Exception as e:
                logging.warning(f"original_index alignment failed: {e}")

        # Priority 2: lp_row_id
        if not aligned and "lp_row_id" in df_gold.columns:
            print("Attempting alignment via lp_row_id…")
            try:
                idx = df_gold["lp_row_id"].astype(int).values
                y_proba = label_distributions[idx]
                print("✓ Alignment via lp_row_id successful.")
                aligned = True
            except Exception as e:
                logging.warning(f"lp_row_id alignment failed: {e}")

        # Priority 3: DataFrame index
        if not aligned:
            print("Attempting alignment via DataFrame index (last resort)…")
            try:
                idx = df_gold.index.values
                y_proba = label_distributions[idx]
                print("✓ Alignment via DataFrame index successful.")
                aligned = True
            except Exception as e:
                logging.warning(f"DataFrame index alignment failed: {e}")
                y_proba = None

        # -----------------------------------------------------
        # If alignment failed → skip prob-based metrics
        # -----------------------------------------------------
        if y_proba is None:
            logging.warning("LP probability extraction failed — ROC/AUC and Brier will be NaN.")
            return compute_metrics_for_stage(y_true, y_pred, None, "Label_Propagation")

        # -----------------------------------------------------
        # Diagnostics BEFORE normalization
        # -----------------------------------------------------
        print("---- Probability Diagnostics (Before Normalization) ----")
        row_sums_before = y_proba.sum(axis=1)
        print(f"Min row sum: {row_sums_before.min():.5f}")
        print(f"Max row sum: {row_sums_before.max():.5f}")
        print(f"Mean row sum: {row_sums_before.mean():.5f}")

        # Identify zero-mass rows
        zero_rows = (row_sums_before <= 1e-12)
        if zero_rows.any():
            print(f"⚠ WARNING: {zero_rows.sum()} rows have zero probability mass — injecting uniform prior.")
            eps = 1e-6
            y_proba[zero_rows, :] = eps

        # -----------------------------------------------------
        # Normalize probabilities
        # -----------------------------------------------------
        row_sums = y_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        y_proba = (y_proba / row_sums).astype(np.float32)

        # -----------------------------------------------------
        # Diagnostics AFTER normalization
        # -----------------------------------------------------
        print("---- Probability Diagnostics (After Normalization) ----")
        row_sums_after = y_proba.sum(axis=1)
        print(f"Min row sum: {row_sums_after.min():.5f}")
        print(f"Max row sum: {row_sums_after.max():.5f}")
        print(f"Mean row sum: {row_sums_after.mean():.5f}")

        # Check for bad rows even after normalization
        if not np.allclose(row_sums_after, 1.0, atol=1e-3):
            print("⚠ WARNING: Some rows still do not sum to 1.00 — alignment might still be ambiguous.")
        else:
            print("✓ All rows are valid probability distributions.")

    except Exception as e:
        logging.warning(f"Failed LP probability extraction: {e}")
        y_proba = None

    # ---------------------------------------------------------
    # Compute all metrics
    # ---------------------------------------------------------
    print("Computing LP metrics…")
    results = compute_metrics_for_stage(y_true, y_pred, y_proba, "Label_Propagation")

    print(f"✓ Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"✓ Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    return results




# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*80)
    print("COMPREHENSIVE SSL METRICS EVALUATION - CORRECTED")
    print("="*80)
    print(f"GPU Acceleration: {'Enabled' if USE_GPU else 'Disabled'}")
    print(f"Ordinal Classes: {ORDINAL_CLASSES}")
    print(f"View A features: {len(VIEW_A_FEATURES)}")
    print(f"View B features: {len(VIEW_B_FEATURES)}")
    print("="*80)

    try: validate_paths()
    except FileNotFoundError as e:
        logging.error(e); logging.info("Continuing; some stages may regenerate predictions.")

    all_rows = []
    for fn in (evaluate_ssl1, evaluate_ssl2, evaluate_ssl3, evaluate_cotraining, evaluate_labelprop):
        try:
            rows = fn(); all_rows.extend(rows); gc.collect()
        except Exception as e:
            logging.error(f"Stage {fn.__name__} failed: {e}")
            import traceback; traceback.print_exc()

    if not all_rows:
        print("\n✗ No metrics generated. Check file paths and columns."); return

    df_out = pd.DataFrame(all_rows)
    cols = ['Stage','Class','Metric_Type','Accuracy','Precision','Recall','F1_Score',
            'ROC_AUC','Brier','MCC','Kappa','Kappa_Weighted','Support','TP','FP','FN','TN']
    for c in cols:
        if c not in df_out.columns: df_out[c] = np.nan
    df_out = df_out[cols]
    df_out.to_csv(OUTPUT_CSV, index=False, float_format='%.5f')

    print("\n" + "="*80)
    print("EVALUATION COMPLETE")
    print("="*80)
    print(f"Output: {OUTPUT_CSV}")
    print(f"Total rows: {len(df_out)}")
    print("\nPer-Stage Summary (Macro F1):")
    try:
        print(df_out[df_out['Metric_Type'] == 'Macro'][['Stage', 'F1_Score', 'ROC_AUC', 'Brier', 'MCC', 'Kappa']])
    except Exception:
        pass
    print("="*80)

if __name__ == '__main__':
    main()


COMPREHENSIVE SSL METRICS EVALUATION - CORRECTED
GPU Acceleration: Disabled
Ordinal Classes: True
View A features: 60
View B features: 140

EVALUATING SSL-1
Loaded: 83,289 rows
Gold-labeled samples: 3,287
Using prediction column: predicted_class
Using probability columns: ['proba_class_0', 'proba_class_1', 'proba_class_2', 'proba_class_3']
✓ Accuracy: 0.9811, Macro F1: 0.9716

EVALUATING SSL-2
Loaded: 83,289 rows
Gold-labeled samples: 3,287
Using prediction column: predicted_class_v2
Found 4 probability columns
✓ Accuracy: 0.9601, Macro F1: 0.9699

EVALUATING SSL-3
Loaded: 83,289 rows
Gold-labeled samples: 3,287
Using prediction column: predicted_class_v3
Found 4 probability columns
✓ Accuracy: 0.9416, Macro F1: 0.9604

EVALUATING CO-TRAINING (ARGMAX ENSEMBLE + CHANGED-ONLY)
Loaded: 83,289 rows
Gold-labeled samples: 3,287
Regenerating probabilities from co-training models (semantic views)...
✓ CoTrain_FullGold — Acc: 0.9851, Macro F1: 0.9834
No intersection between co-training pseudo-l

Graph

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/content/drive/MyDrive/ML_Project_Data/Manual_labelled_data"
METRICS_CSV = os.path.join(BASE_DIR, "ssl_comprehensive_metrics.csv")
OUT_DIR = os.path.join(BASE_DIR, "plots_per_class_metrics")
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# 1) Load CSV
# =========================================================
df = pd.read_csv(METRICS_CSV)

# Strip all column names
df.columns = [c.strip() for c in df.columns]

# =========================================================
# 2) UNIVERSAL STRING NORMALIZATION (fixes all mismatches)
# =========================================================
def normalize_token(x):
    x = str(x).strip().lower()
    x = x.replace("-", "").replace("_", "").replace(" ", "")
    return x

df["Stage_norm"] = df["Stage"].apply(normalize_token)
df["Metric_norm"] = df["Metric_Type"].apply(normalize_token)
df["Class_raw"] = df["Class"]

# =========================================================
# 3) CANONICAL TOKEN MAPS
# =========================================================
METRIC_PERCLASS = {"perclass", "per_class", "classper", "metricperclass"}

STAGE_MAP = {
    "ssl1": "SSL-1",
    "ssl2": "SSL-2",
    "ssl3": "SSL-3",
    "cotrainingfullgold": "CoTrain_FullGold",
    "cotrainingchangedonly": "CoTrain_ChangedOnly",
    "cotraining": "CoTraining",  # fallback
    "labelpropagation": "Label_Propagation",
    "labelprop": "Label_Propagation",
    "labelpropfinal": "Label_Propagation",
}

# =========================================================
# 4) FILTER TO PER-CLASS METRICS
# =========================================================
df = df[df["Metric_norm"].isin(METRIC_PERCLASS)].copy()

# Map stages
df["Stage"] = df["Stage_norm"].map(STAGE_MAP)

# Keep only rows with valid mapped stage
df = df[df["Stage"].notna()].copy()

# Convert Class values
def safe_int(x):
    try:
        return int(float(x))
    except:
        return np.nan

df["Class"] = df["Class_raw"].apply(safe_int)
df = df[df["Class"].isin([0, 1, 2, 3])].copy()

if df.empty:
    print("❌ AFTER NORMALIZATION: No per-class rows detected.")
    print("Unique Metric_Type:", df["Metric_Type"].unique())
    print("Unique Stage:", df["Stage_raw"].unique())
    print("Unique Class:", df["Class_raw"].unique())
    raise SystemExit

# =========================================================
# 5) FORCE NUMERIC METRICS
# =========================================================
METRICS = ["Accuracy","Precision","Recall","F1_Score","ROC_AUC","Brier"]

for c in METRICS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Keep only rows with at least one non-NaN metric
df = df.dropna(subset=METRICS, how="all")

if df.empty:
    print("❌ No numeric metric values found after coercion!")
    raise SystemExit

# =========================================================
# 6) STAGE ORDERING
# =========================================================
ordered_stages = [
    "SSL-1", "SSL-2", "SSL-3", "CoTrain_FullGold", "CoTrain_ChangedOnly", "Label_Propagation"
]

present_stages = [s for s in ordered_stages if s in df["Stage"].unique()]
if not present_stages:
    present_stages = sorted(df["Stage"].unique().tolist())

df["Stage"] = pd.Categorical(df["Stage"], categories=present_stages, ordered=True)

# =========================================================
# 7) PLOTTING FUNCTION
# =========================================================
sns.set(style="whitegrid")
palette = sns.color_palette("Set2", n_colors=len(present_stages))

def plot_metric(metric_name, ylim):
    if metric_name not in df.columns:
        print(f"⚠ Skipping {metric_name}: not found in CSV.")
        return

    d = df[["Stage","Class",metric_name]].copy()
    d[metric_name] = pd.to_numeric(d[metric_name], errors="coerce")

    # Wide pivot: Class rows, Stage columns
    pivot = d.pivot_table(index="Class", columns="Stage", values=metric_name, aggfunc="first")
    pivot = pivot.reindex(index=[0,1,2,3])  # stable class order

    if pivot.dropna(how="all").empty:
        print(f"⚠ Skipping {metric_name}: all values NaN after filtering.")
        return

    # Debug: Show mismatch between CSV and plot
    print(f"\n--- {metric_name} PIVOT TABLE (will be plotted) ---")
    print(pivot)

    ax = pivot.plot(kind="bar", figsize=(10, 6), color=palette, width=0.8)
    ax.set_title(f"{metric_name} by Class and Stage", pad = 25)
    ax.set_xlabel("Class")
    ax.set_ylabel(metric_name)
    ax.set_ylim(*ylim)
    ax.legend(title="Stage")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # Annotate bars with numeric values
    for p in ax.patches:
        height = p.get_height()
        if pd.notna(height):
            ax.annotate(f"{height:.3f}",
                        (p.get_x() + p.get_width()/2, height),
                        ha="center", va="bottom",
                        fontsize=8, xytext=(0, 3),
                        textcoords="offset points")

    plt.subplots_adjust(top=0.85)
    plt.tight_layout()

    path = os.path.join(OUT_DIR, f"per_class_{metric_name.lower()}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"✓ Saved: {path}")

# =========================================================
# 8) GENERATE ALL METRICS
# =========================================================
RANGES = {
    "Accuracy": (0,1),
    "Precision": (0,1),
    "Recall": (0,1),
    "F1_Score": (0,1),
    "ROC_AUC": (0,1),
    "Brier": (0,0.25),
}

for m, r in RANGES.items():
    plot_metric(m, r)

print("\n🎉 ALL PLOTS GENERATED SUCCESSFULLY!")



--- Accuracy PIVOT TABLE (will be plotted) ---
Stage    SSL-1    SSL-2    SSL-3  Label_Propagation
Class                                              
0      0.98114  0.96015  0.94159            0.96015
1      0.98114  0.96015  0.94159            0.96015
2      0.98114  0.96015  0.94159            0.96015
3      0.98114  0.96015  0.94159            0.96015
✓ Saved: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/plots_per_class_metrics/per_class_accuracy.png

--- Precision PIVOT TABLE (will be plotted) ---
Stage    SSL-1    SSL-2    SSL-3  Label_Propagation
Class                                              
0      0.99552  0.92928  0.98347            0.94813
1      0.97048  0.99744  0.88951            0.98049
2      0.96985  0.99000  0.95305            0.90868
3      0.95098  0.97561  0.99502            0.99444
✓ Saved: /content/drive/MyDrive/ML_Project_Data/Manual_labelled_data/plots_per_class_metrics/per_class_precision.png

--- Recall PIVOT TABLE (will be plotted) ---
